In [3]:
import pygame
import random
import math

pygame.init()

# -----------------------------
# SETTINGS
# -----------------------------
WIDTH = 1000
HEIGHT = 600
FPS = 60

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()

# Colors
WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 50, 50)
GREEN = (50, 200, 80)
BLUE = (50, 100, 220)
YELLOW = (255, 210, 50)
GRAY = (100, 100, 100)
GROUND = (70, 180, 90)

font = pygame.font.Font(None, 36)
big_font = pygame.font.Font(None, 72)


# -----------------------------
# PLAYER
# -----------------------------
class Player:
    def __init__(self):
        self.x = 200
        self.y = 450
        self.width = 35
        self.height = 90

        self.vel_y = 0
        self.speed = 5
        self.jump_power = -13
        self.gravity = 0.6

        self.health = 100
        self.max_health = 100

        self.attack_cooldown = 0
        self.attack_timer = 0

        self.facing = 1

    def rect(self):
        return pygame.Rect(
            self.x - self.width // 2,
            self.y - self.height,
            self.width,
            self.height
        )

    def update(self, keys):
        # Movement
        if keys[pygame.K_a] or keys[pygame.K_LEFT]:
            self.x -= self.speed
            self.facing = -1

        if keys[pygame.K_d] or keys[pygame.K_RIGHT]:
            self.x += self.speed
            self.facing = 1

        # Jump
        if (keys[pygame.K_w] or keys[pygame.K_UP]) and self.y >= 450:
            self.vel_y = self.jump_power

        # Gravity
        self.vel_y += self.gravity
        self.y += self.vel_y

        # Ground
        if self.y > 450:
            self.y = 450
            self.vel_y = 0

        # Screen boundaries
        self.x = max(30, min(WIDTH - 30, self.x))

        if self.attack_cooldown > 0:
            self.attack_cooldown -= 1

        if self.attack_timer > 0:
            self.attack_timer -= 1

    def attack(self):
        if self.attack_cooldown == 0:
            self.attack_cooldown = 25
            self.attack_timer = 10
            return True

        return False

    def attack_box(self):
        direction = self.facing

        return pygame.Rect(
            self.x + direction * 20 - (40 if direction < 0 else 0),
            self.y - 75,
            60,
            55
        )

    def draw(self):
        x = int(self.x)
        y = int(self.y)

        # Head
        pygame.draw.circle(screen, BLACK, (x, y - 78), 12, 3)

        # Body
        pygame.draw.line(screen, BLACK, (x, y - 66), (x, y - 30), 4)

        # Arms
        pygame.draw.line(screen, BLACK,
                         (x, y - 58),
                         (x - 22, y - 42), 4)

        pygame.draw.line(screen, BLACK,
                         (x, y - 58),
                         (x + 22, y - 42), 4)

        # Legs
        pygame.draw.line(screen, BLACK,
                         (x, y - 30),
                         (x - 18, y), 4)

        pygame.draw.line(screen, BLACK,
                         (x, y - 30),
                         (x + 18, y), 4)

        # Sword while attacking
        if self.attack_timer > 0:
            sword_start = (x + self.facing * 15, y - 50)
            sword_end = (x + self.facing * 65, y - 90)

            pygame.draw.line(
                screen,
                GRAY,
                sword_start,
                sword_end,
                7
            )

            pygame.draw.circle(
                screen,
                YELLOW,
                sword_end,
                5
            )

        # Health bar
        bar_width = 60
        health_width = int(
            bar_width * self.health / self.max_health
        )

        pygame.draw.rect(
            screen,
            RED,
            (x - 30, y - 110, bar_width, 8)
        )

        pygame.draw.rect(
            screen,
            GREEN,
            (x - 30, y - 110, health_width, 8)
        )


# -----------------------------
# ENEMY
# -----------------------------
class Enemy:
    def __init__(self, difficulty):
        side = random.choice([-1, 1])

        if side == -1:
            self.x = random.randint(30, 150)
        else:
            self.x = random.randint(WIDTH - 150, WIDTH - 30)

        self.y = 450

        self.speed = random.uniform(
            1.2 + difficulty * 0.1,
            2.0 + difficulty * 0.15
        )

        self.max_health = 50 + difficulty * 5
        self.health = self.max_health

        self.attack_cooldown = random.randint(30, 80)
        self.attack_damage = 5 + difficulty

        self.facing = -1

    def rect(self):
        return pygame.Rect(
            self.x - 18,
            self.y - 90,
            36,
            90
        )

    def update(self, player):
        distance = player.x - self.x

        if abs(distance) > 65:
            if distance > 0:
                self.x += self.speed
                self.facing = 1
            else:
                self.x -= self.speed
                self.facing = -1

        else:
            if self.attack_cooldown > 0:
                self.attack_cooldown -= 1
            else:
                if abs(distance) < 85:
                    player.health -= self.attack_damage

                self.attack_cooldown = 70

        self.x = max(20, min(WIDTH - 20, self.x))

    def draw(self):
        x = int(self.x)
        y = int(self.y)

        # Head
        pygame.draw.circle(
            screen,
            RED,
            (x, y - 78),
            12,
            3
        )

        # Body
        pygame.draw.line(
            screen,
            RED,
            (x, y - 66),
            (x, y - 30),
            4
        )

        # Arms
        pygame.draw.line(
            screen,
            RED,
            (x, y - 58),
            (x - 22, y - 42),
            4
        )

        pygame.draw.line(
            screen,
            RED,
            (x, y - 58),
            (x + 22, y - 42),
            4
        )

        # Legs
        pygame.draw.line(
            screen,
            RED,
            (x, y - 30),
            (x - 18, y),
            4
        )

        pygame.draw.line(
            screen,
            RED,
            (x, y - 30),
            (x + 18, y),
            4
        )

        # Health bar
        bar_width = 50
        health_width = int(
            bar_width * self.health / self.max_health
        )

        pygame.draw.rect(
            screen,
            BLACK,
            (x - 25, y - 108, bar_width, 7)
        )

        pygame.draw.rect(
            screen,
            GREEN,
            (x - 25, y - 108, health_width, 7)
        )


# -----------------------------
# DRAW BACKGROUND
# -----------------------------
def draw_background():
    screen.fill(WHITE)

    # Sky
    pygame.draw.rect(
        screen,
        (180, 220, 255),
        (0, 0, WIDTH, 450)
    )

    # Sun
    pygame.draw.circle(
        screen,
        YELLOW,
        (850, 90),
        45
    )

    # Ground
    pygame.draw.rect(
        screen,
        GROUND,
        (0, 450, WIDTH, 150)
    )

    # Ground line
    pygame.draw.line(
        screen,
        BLACK,
        (0, 450),
        (WIDTH, 450),
        5
    )


# -----------------------------
# MAIN GAME
# -----------------------------
player = Player()

enemies = []

score = 0
wave = 1
spawn_timer = 0

game_over = False

running = True

while running:

    clock.tick(FPS)

    # -------------------------
    # EVENTS
    # -------------------------
    for event in pygame.event.get():

        if event.type == pygame.QUIT:
            running = False

        if event.type == pygame.KEYDOWN:

            if event.key == pygame.K_ESCAPE:
                running = False

            if event.key == pygame.K_SPACE and not game_over:
                player.attack()

            if event.key == pygame.K_r and game_over:
                player = Player()
                enemies = []
                score = 0
                wave = 1
                spawn_timer = 0
                game_over = False

    # -------------------------
    # GAME LOGIC
    # -------------------------
    if not game_over:

        keys = pygame.key.get_pressed()
        player.update(keys)

        # Spawn enemies
        spawn_timer -= 1

        max_enemies = min(10, 2 + wave // 2)

        if spawn_timer <= 0 and len(enemies) < max_enemies:

            enemies.append(
                Enemy(wave)
            )

            spawn_timer = max(
                30,
                100 - wave * 5
            )

        # Update enemies
        for enemy in enemies:
            enemy.update(player)

        # Player attack
        if player.attack_timer > 0:

            hitbox = player.attack_box()

            for enemy in enemies[:]:

                if hitbox.colliderect(enemy.rect()):

                    enemy.health -= 20

                    # Knockback
                    enemy.x += player.facing * 25

                    if enemy.health <= 0:
                        enemies.remove(enemy)
                        score += 10

        # Wave progression
        if score >= wave * 100:
            wave += 1

        # Game over
        if player.health <= 0:
            player.health = 0
            game_over = True

    # -------------------------
    # DRAW
    # -------------------------
    draw_background()

    player.draw()

    for enemy in enemies:
        enemy.draw()

    # UI
    score_text = font.render(
        f"Score: {score}",
        True,
        BLACK
    )

    wave_text = font.render(
        f"Wave: {wave}",
        True,
        BLACK
    )

    controls_text = font.render(
        "A/D: Move   W: Jump   SPACE: Attack",
        True,
        BLACK
    )

    screen.blit(score_text, (20, 20))
    screen.blit(wave_text, (20, 55))
    screen.blit(controls_text, (WIDTH - 450, 20))

    # Game over
    if game_over:

        overlay = pygame.Surface(
            (WIDTH, HEIGHT),
            pygame.SRCALPHA
        )

        overlay.fill((0, 0, 0, 140))
        screen.blit(overlay, (0, 0))

        game_over_text = big_font.render(
            "GAME OVER",
            True,
            WHITE
        )

        restart_text = font.render(
            f"Score: {score}   |   Press R to restart",
            True,
            WHITE
        )

        screen.blit(
            game_over_text,
            (
                WIDTH // 2 - game_over_text.get_width() // 2,
                230
            )
        )

        screen.blit(
            restart_text,
            (
                WIDTH // 2 - restart_text.get_width() // 2,
                310
            )
        )

    pygame.display.flip()

pygame.quit()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


2026-09-10 14:12:22.436 Python[78439:16566090] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit


In [6]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame.

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch, L to kick

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import random

pygame.init()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
SKY_TOP = (135, 190, 230)
SKY_BOTTOM = (220, 235, 245)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing  # 1 = right, -1 = left
        self.controls = controls
        self.name = name

        self.health = MAX_HEALTH
        self.on_ground = True

        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0  # frames remaining showing an attack pose
        self.attack_type = None
        self.hit_stun = 0

        self.walk_cycle = 0
        self.moving = False

        self.wins = 0

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height,
                            self.width, self.height)

    def attack_hitbox(self):
        if self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def handle_input(self, keys, opponent):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            self.attack_type = "punch"
            self.attack_anim = 10
            self.punch_cd = PUNCH_COOLDOWN

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y
        if self.y >= GROUND_Y:
            self.y = GROUND_Y
            self.vel_y = 0
            self.on_ground = True

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        # legs (simple walk animation)
        import math
        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)

        # torso
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        # arms
        if self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        # head
        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)

        # simple face direction indicator (eye)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)


def draw_background(surf):
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = SKY_TOP[0] + (SKY_BOTTOM[0] - SKY_TOP[0]) * t
        g = SKY_TOP[1] + (SKY_BOTTOM[1] - SKY_TOP[1]) * t
        b = SKY_TOP[2] + (SKY_BOTTOM[2] - SKY_TOP[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))
    pygame.draw.rect(surf, (110, 90, 70), (0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y))
    pygame.draw.rect(surf, (80, 150, 80), (0, GROUND_Y, WIDTH, 12))


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)

    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)

    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type and attacker.attack_anim == (
            9 if attacker.attack_type == "punch" else 12
        ):
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                dmg = PUNCH_DAMAGE if attacker.attack_type == "punch" else KICK_DAMAGE
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))


def reset_fighters(p1_wins, p2_wins):
    p1 = Fighter(WIDTH * 0.25, RED, DARK_RED, 1,
                 (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g),
                 "Player 1")
    p2 = Fighter(WIDTH * 0.75, BLUE, DARK_BLUE, -1,
                 (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l),
                 "Player 2")
    p1.wins = p1_wins
    p2.wins = p2_wins
    return p1, p2


def main():
    p1, p2 = reset_fighters(0, 0)
    game_over = False
    winner = None

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                if event.key == pygame.K_r and game_over:
                    p1, p2 = reset_fighters(p1.wins, p2.wins)
                    game_over = False
                    winner = None

        keys = pygame.key.get_pressed()

        if not game_over:
            p1.handle_input(keys, p2)
            p2.handle_input(keys, p1)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)

            # keep fighters from overlapping too much
            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1

        # ---- draw ----
        draw_background(screen)
        p1.draw(screen)
        p2.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))

            if winner == "Draw":
                msg = "DRAW!"
            else:
                msg = f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))

            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F punch, G kick   |   P2: Arrows move/jump, K punch, L kick",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [7]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Weapons (sword, bat, spear) spawn randomly on the ground. Walk over one
while unarmed to pick it up automatically. Your "punch" button (F / K)
becomes a weapon swing with more range and damage while armed. Weapons
have limited durability and break after a number of hits.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random

pygame.init()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
SKY_TOP = (135, 190, 230)
SKY_BOTTOM = (220, 235, 245)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

# Unarmed attacks
PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

# Weapon definitions: reach, damage, cooldown (frames), durability (hits), color
WEAPON_TYPES = {
    "sword": {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5, "color": SILVER},
    "bat":   {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8, "color": BROWN},
    "spear": {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6, "color": (90, 60, 30)},
}

WEAPON_PICKUP_RADIUS = 45
MAX_WEAPONS_ON_FIELD = 2
WEAPON_SPAWN_INTERVAL = 300  # frames between spawn attempts

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class WeaponPickup:
    def __init__(self, x, wtype):
        self.x = x
        self.y = GROUND_Y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        # glowing circle behind it
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [
                (self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)
            ])
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing  # 1 = right, -1 = left
        self.controls = controls
        self.name = name

        self.health = MAX_HEALTH
        self.on_ground = True

        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0  # frames remaining showing an attack pose
        self.attack_type = None  # "punch", "kick", or "weapon"
        self.hit_stun = 0

        self.walk_cycle = 0
        self.moving = False

        self.wins = 0
        self.weapon = None  # dict: {"type": str, "durability": int}

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height,
                            self.width, self.height)

    def current_stats(self):
        """Return (reach, damage, cooldown) for the primary attack button."""
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def handle_input(self, keys, opponent, weapons):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup(weapons)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y
        if self.y >= GROUND_Y:
            self.y = GROUND_Y
            self.vel_y = 0
            self.on_ground = True

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        """Called once when a weapon attack connects. Reduces durability."""
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        # legs (simple walk animation)
        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)

        # torso
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        # arms
        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        # weapon drawing (in hand, extends from forward hand outward)
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                # idle carry pose: weapon rests forward-diagonally
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [
                    tipend,
                    (tip[0] + perp[0], tip[1] + perp[1]),
                    (tip[0] - perp[0], tip[1] - perp[1]),
                ])
            else:  # sword
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK,
                                  (guard_center[0] + perp[0], guard_center[1] + perp[1]),
                                  (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        # head
        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)

        # simple face direction indicator (eye)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        # weapon durability label above head
        if self.weapon:
            label = font_tiny.render(
                f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}",
                True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, int(head_y) - head_r - 20))


def draw_background(surf):
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = SKY_TOP[0] + (SKY_BOTTOM[0] - SKY_TOP[0]) * t
        g = SKY_TOP[1] + (SKY_BOTTOM[1] - SKY_TOP[1]) * t
        b = SKY_TOP[2] + (SKY_BOTTOM[2] - SKY_TOP[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))
    pygame.draw.rect(surf, (110, 90, 70), (0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y))
    pygame.draw.rect(surf, (80, 150, 80), (0, GROUND_Y, WIDTH, 12))


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)

    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)

    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_type and attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    x = random.randint(150, WIDTH - 150)
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, wtype))


def reset_fighters(p1_wins, p2_wins):
    p1 = Fighter(WIDTH * 0.25, RED, DARK_RED, 1,
                 (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g),
                 "Player 1")
    p2 = Fighter(WIDTH * 0.75, BLUE, DARK_BLUE, -1,
                 (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l),
                 "Player 2")
    p1.wins = p1_wins
    p2.wins = p2_wins
    weapons = []
    # start with one weapon on the field
    spawn_weapon(weapons)
    return p1, p2, weapons


def main():
    p1, p2, weapons = reset_fighters(0, 0)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                if event.key == pygame.K_r and game_over:
                    p1, p2, weapons = reset_fighters(p1.wins, p2.wins)
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL

        keys = pygame.key.get_pressed()

        if not game_over:
            p1.handle_input(keys, p2, weapons)
            p2.handle_input(keys, p1, weapons)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL

            # keep fighters from overlapping too much
            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1

        # ---- draw ----
        draw_background(screen)
        for wp in weapons:
            wp.draw(screen)
        p1.draw(screen)
        p2.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))

            if winner == "Draw":
                msg = "DRAW!"
            else:
                msg = f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))

            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Walk over a weapon to pick it up",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

/Users/feilu/Library/Python/3.12/lib/python/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [8]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

More weapons (sword, bat, spear) also spawn randomly on the ground during
the match. Walk over one while unarmed to pick it up automatically. Your
"punch" button (F / K) becomes a weapon swing with more range and damage
while armed. Weapons have limited durability and break after a number of
hits.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random

pygame.init()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
SKY_TOP = (135, 190, 230)
SKY_BOTTOM = (220, 235, 245)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

# Unarmed attacks
PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

# Weapon definitions: reach, damage, cooldown (frames), durability (hits), color
WEAPON_TYPES = {
    "sword": {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5, "color": SILVER},
    "bat":   {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8, "color": BROWN},
    "spear": {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6, "color": (90, 60, 30)},
}

WEAPON_PICKUP_RADIUS = 45
MAX_WEAPONS_ON_FIELD = 2
WEAPON_SPAWN_INTERVAL = 300  # frames between spawn attempts

# Selectable starting loadouts (None = fists)
WEAPON_CHOICES = [None, "sword", "bat", "spear"]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class WeaponPickup:
    def __init__(self, x, wtype):
        self.x = x
        self.y = GROUND_Y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        # glowing circle behind it
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [
                (self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)
            ])
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing  # 1 = right, -1 = left
        self.controls = controls
        self.name = name

        self.health = MAX_HEALTH
        self.on_ground = True

        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0  # frames remaining showing an attack pose
        self.attack_type = None  # "punch", "kick", or "weapon"
        self.hit_stun = 0

        self.walk_cycle = 0
        self.moving = False

        self.wins = 0
        self.weapon = None  # dict: {"type": str, "durability": int}

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height,
                            self.width, self.height)

    def current_stats(self):
        """Return (reach, damage, cooldown) for the primary attack button."""
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def handle_input(self, keys, opponent, weapons):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup(weapons)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y
        if self.y >= GROUND_Y:
            self.y = GROUND_Y
            self.vel_y = 0
            self.on_ground = True

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        """Called once when a weapon attack connects. Reduces durability."""
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        # legs (simple walk animation)
        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)

        # torso
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        # arms
        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        # weapon drawing (in hand, extends from forward hand outward)
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                # idle carry pose: weapon rests forward-diagonally
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [
                    tipend,
                    (tip[0] + perp[0], tip[1] + perp[1]),
                    (tip[0] - perp[0], tip[1] - perp[1]),
                ])
            else:  # sword
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK,
                                  (guard_center[0] + perp[0], guard_center[1] + perp[1]),
                                  (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        # head
        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)

        # simple face direction indicator (eye)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        # weapon durability label above head
        if self.weapon:
            label = font_tiny.render(
                f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}",
                True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, int(head_y) - head_r - 20))


def draw_background(surf):
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = SKY_TOP[0] + (SKY_BOTTOM[0] - SKY_TOP[0]) * t
        g = SKY_TOP[1] + (SKY_BOTTOM[1] - SKY_TOP[1]) * t
        b = SKY_TOP[2] + (SKY_BOTTOM[2] - SKY_TOP[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))
    pygame.draw.rect(surf, (110, 90, 70), (0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y))
    pygame.draw.rect(surf, (80, 150, 80), (0, GROUND_Y, WIDTH, 12))


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)

    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)

    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_type and attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    x = random.randint(150, WIDTH - 150)
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, wtype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    p1 = Fighter(WIDTH * 0.25, RED, DARK_RED, 1,
                 (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g),
                 "Player 1")
    p2 = Fighter(WIDTH * 0.75, BLUE, DARK_BLUE, -1,
                 (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l),
                 "Player 2")
    p1.wins = p1_wins
    p2.wins = p2_wins

    if p1_start_weapon:
        p1.weapon = {"type": p1_start_weapon,
                     "durability": WEAPON_TYPES[p1_start_weapon]["durability"]}
    if p2_start_weapon:
        p2.weapon = {"type": p2_start_weapon,
                     "durability": WEAPON_TYPES[p2_start_weapon]["durability"]}

    weapons = []
    # start with one extra weapon on the field for grabbing mid-fight
    spawn_weapon(weapons)
    return p1, p2, weapons


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    """Draw a small standalone icon of a weapon choice (or fists if None)."""
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [
            (cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)
        ])


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    col_w = 180
    start_x = WIDTH // 2 - (len(WEAPON_CHOICES) * col_w) // 2 + col_w // 2
    icon_y = 220

    # Player 1 row (above icons) and Player 2 row (below) each highlight their pick
    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - 70, icon_y - 70, 140, 140)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + 80))

        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))

    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE

    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))

    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def main():
    state = "select"  # "select" -> "playing" -> "over"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0

    p1, p2, weapons = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False

                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True

                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:  # brief pause on "Get ready..."
                    p1, p2, weapons = reset_fighters(
                        p1_wins, p2_wins,
                        WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    state = "playing"
                    select_confirm_timer = 0
            continue

        keys = pygame.key.get_pressed()

        if not game_over:
            p1.handle_input(keys, p2, weapons)
            p2.handle_input(keys, p1, weapons)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL

            # keep fighters from overlapping too much
            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen)
        for wp in weapons:
            wp.draw(screen)
        p1.draw(screen)
        p2.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))

            if winner == "Draw":
                msg = "DRAW!"
            else:
                msg = f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))

            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Walk over a weapon to pick it up",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [10]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

More weapons (sword, bat, spear) also spawn randomly on the ground during
the match. Walk over one while unarmed to pick it up automatically. Your
"punch" button (F / K) becomes a weapon swing with more range and damage
while armed. Weapons have limited durability and break after a number of
hits.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random

pygame.init()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
SKY_TOP = (135, 190, 230)
SKY_BOTTOM = (220, 235, 245)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

# Unarmed attacks
PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

# Weapon definitions: reach, damage, cooldown (frames), durability (hits), color
# "ranged": True marks weapons that fire projectiles instead of melee swings;
# for those, "durability" means number of shots (arrows) instead of hits.
WEAPON_TYPES = {
    "sword": {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5, "color": SILVER, "ranged": False},
    "bat":   {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8, "color": BROWN, "ranged": False},
    "spear": {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6, "color": (90, 60, 30), "ranged": False},
    "bow":   {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6, "color": (101, 67, 33), "ranged": True},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

WEAPON_PICKUP_RADIUS = 45
MAX_WEAPONS_ON_FIELD = 2
WEAPON_SPAWN_INTERVAL = 300  # frames between spawn attempts

# Selectable starting loadouts (None = fists)
WEAPON_CHOICES = [None, "sword", "bat", "spear", "bow"]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    def __init__(self, x, y, facing, damage, owner):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner  # Fighter who fired it, so it can't hit itself
        self.speed = ARROW_SPEED
        self.length = ARROW_LENGTH
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        return pygame.Rect(left, self.y - 4, self.length, 8)

    def draw(self, surf):
        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        # arrowhead
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        # fletching
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class WeaponPickup:
    def __init__(self, x, wtype):
        self.x = x
        self.y = GROUND_Y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        # glowing circle behind it
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [
                (self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)
            ])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing  # 1 = right, -1 = left
        self.controls = controls
        self.name = name

        self.health = MAX_HEALTH
        self.on_ground = True

        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0  # frames remaining showing an attack pose
        self.attack_type = None  # "punch", "kick", or "weapon"
        self.hit_stun = 0

        self.walk_cycle = 0
        self.moving = False

        self.wins = 0
        self.weapon = None  # dict: {"type": str, "durability": int}

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height,
                            self.width, self.height)

    def current_stats(self):
        """Return (reach, damage, cooldown) for the primary attack button."""
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self))

    def handle_input(self, keys, opponent, weapons, arrows):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.is_ranged():
                self.attack_type = "shoot"
                self.attack_anim = 10
                self.shoot(arrows)
                self.weapon["durability"] -= 1
                if self.weapon["durability"] <= 0:
                    self.weapon = None
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup(weapons)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y
        if self.y >= GROUND_Y:
            self.y = GROUND_Y
            self.vel_y = 0
            self.on_ground = True

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        """Called once when a weapon attack connects. Reduces durability."""
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        # legs (simple walk animation)
        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)

        # torso
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        # arms
        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        # weapon drawing (in hand, extends from forward hand outward)
        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200),
                              (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200),
                              (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                # idle carry pose: weapon rests forward-diagonally
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [
                    tipend,
                    (tip[0] + perp[0], tip[1] + perp[1]),
                    (tip[0] - perp[0], tip[1] - perp[1]),
                ])
            else:  # sword
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK,
                                  (guard_center[0] + perp[0], guard_center[1] + perp[1]),
                                  (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        # head
        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)

        # simple face direction indicator (eye)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        # weapon durability label above head
        if self.weapon:
            label = font_tiny.render(
                f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}",
                True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, int(head_y) - head_r - 20))


def draw_background(surf):
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = SKY_TOP[0] + (SKY_BOTTOM[0] - SKY_TOP[0]) * t
        g = SKY_TOP[1] + (SKY_BOTTOM[1] - SKY_TOP[1]) * t
        b = SKY_TOP[2] + (SKY_BOTTOM[2] - SKY_TOP[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))
    pygame.draw.rect(surf, (110, 90, 70), (0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y))
    pygame.draw.rect(surf, (80, 150, 80), (0, GROUND_Y, WIDTH, 12))


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)

    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)

    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    x = random.randint(150, WIDTH - 150)
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, wtype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    p1 = Fighter(WIDTH * 0.25, RED, DARK_RED, 1,
                 (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g),
                 "Player 1")
    p2 = Fighter(WIDTH * 0.75, BLUE, DARK_BLUE, -1,
                 (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l),
                 "Player 2")
    p1.wins = p1_wins
    p2.wins = p2_wins

    if p1_start_weapon:
        p1.weapon = {"type": p1_start_weapon,
                     "durability": WEAPON_TYPES[p1_start_weapon]["durability"]}
    if p2_start_weapon:
        p2.weapon = {"type": p2_start_weapon,
                     "durability": WEAPON_TYPES[p2_start_weapon]["durability"]}

    weapons = []
    # start with one extra weapon on the field for grabbing mid-fight
    spawn_weapon(weapons)
    arrows = []
    return p1, p2, weapons, arrows


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    """Draw a small standalone icon of a weapon choice (or fists if None)."""
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [
            (cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)
        ])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    col_w = 180
    start_x = WIDTH // 2 - (len(WEAPON_CHOICES) * col_w) // 2 + col_w // 2
    icon_y = 220

    # Player 1 row (above icons) and Player 2 row (below) each highlight their pick
    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - 70, icon_y - 70, 140, 140)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + 80))

        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))

    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE

    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))

    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def main():
    state = "select"  # "select" -> "playing" -> "over"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0

    p1, p2, weapons, arrows = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False

                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True

                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:  # brief pause on "Get ready..."
                    p1, p2, weapons, arrows = reset_fighters(
                        p1_wins, p2_wins,
                        WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    state = "playing"
                    select_confirm_timer = 0
            continue

        keys = pygame.key.get_pressed()

        if not game_over:
            p1.handle_input(keys, p2, weapons, arrows)
            p2.handle_input(keys, p1, weapons, arrows)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL

            # keep fighters from overlapping too much
            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen)
        for wp in weapons:
            wp.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))

            if winner == "Draw":
                msg = "DRAW!"
            else:
                msg = f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))

            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Bow fires arrows instead of a melee swing",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [12]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

More weapons (sword, bat, spear, bow) also spawn randomly on the ground
during the match. Walk over one while unarmed to pick it up automatically.
Your "punch" button (F / K) becomes a weapon attack while armed — a melee
swing for sword/bat/spear, or an arrow shot for the bow. Weapons have
limited durability (or ammo, for the bow) and are lost after a number of
uses.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random

pygame.init()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
SKY_TOP = (135, 190, 230)
SKY_BOTTOM = (220, 235, 245)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

# Unarmed attacks
PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

# Weapon definitions: reach, damage, cooldown (frames), durability (hits), color
# "ranged": True marks weapons that fire projectiles instead of melee swings;
# for those, "durability" means number of shots (arrows) instead of hits.
WEAPON_TYPES = {
    "sword": {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5, "color": SILVER, "ranged": False},
    "bat":   {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8, "color": BROWN, "ranged": False},
    "spear": {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6, "color": (90, 60, 30), "ranged": False},
    "bow":   {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6, "color": (101, 67, 33), "ranged": True},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

WEAPON_PICKUP_RADIUS = 45
MAX_WEAPONS_ON_FIELD = 2
WEAPON_SPAWN_INTERVAL = 300  # frames between spawn attempts

# Selectable starting loadouts (None = fists)
WEAPON_CHOICES = [None, "sword", "bat", "spear", "bow"]

# Selectable battle worlds/stages, each with its own palette and decorations
STAGES = [
    {
        "name": "Meadow",
        "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245),
        "ground": (110, 90, 70), "ground_edge": (80, 150, 80),
        "decor": "meadow",
    },
    {
        "name": "Desert",
        "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190),
        "ground": (200, 165, 100), "ground_edge": (225, 195, 130),
        "decor": "desert",
    },
    {
        "name": "Night City",
        "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90),
        "ground": (40, 40, 50), "ground_edge": (90, 90, 110),
        "decor": "city",
    },
    {
        "name": "Volcano",
        "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30),
        "ground": (50, 35, 30), "ground_edge": (200, 80, 30),
        "decor": "volcano",
    },
    {
        "name": "Snow Peak",
        "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245),
        "ground": (225, 235, 240), "ground_edge": (255, 255, 255),
        "decor": "snow",
    },
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    def __init__(self, x, y, facing, damage, owner):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner  # Fighter who fired it, so it can't hit itself
        self.speed = ARROW_SPEED
        self.length = ARROW_LENGTH
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        return pygame.Rect(left, self.y - 4, self.length, 8)

    def draw(self, surf):
        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        # arrowhead
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        # fletching
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class WeaponPickup:
    def __init__(self, x, wtype):
        self.x = x
        self.y = GROUND_Y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        # glowing circle behind it
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [
                (self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)
            ])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing  # 1 = right, -1 = left
        self.controls = controls
        self.name = name

        self.health = MAX_HEALTH
        self.on_ground = True

        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0  # frames remaining showing an attack pose
        self.attack_type = None  # "punch", "kick", or "weapon"
        self.hit_stun = 0

        self.walk_cycle = 0
        self.moving = False

        self.wins = 0
        self.weapon = None  # dict: {"type": str, "durability": int}

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height,
                            self.width, self.height)

    def current_stats(self):
        """Return (reach, damage, cooldown) for the primary attack button."""
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self))

    def handle_input(self, keys, opponent, weapons, arrows):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.is_ranged():
                self.attack_type = "shoot"
                self.attack_anim = 10
                self.shoot(arrows)
                self.weapon["durability"] -= 1
                if self.weapon["durability"] <= 0:
                    self.weapon = None
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup(weapons)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y
        if self.y >= GROUND_Y:
            self.y = GROUND_Y
            self.vel_y = 0
            self.on_ground = True

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        """Called once when a weapon attack connects. Reduces durability."""
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        # legs (simple walk animation)
        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)

        # torso
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        # arms
        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        # weapon drawing (in hand, extends from forward hand outward)
        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200),
                              (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200),
                              (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                # idle carry pose: weapon rests forward-diagonally
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [
                    tipend,
                    (tip[0] + perp[0], tip[1] + perp[1]),
                    (tip[0] - perp[0], tip[1] - perp[1]),
                ])
            else:  # sword
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK,
                                  (guard_center[0] + perp[0], guard_center[1] + perp[1]),
                                  (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        # head
        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)

        # simple face direction indicator (eye)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        # weapon durability label above head
        if self.weapon:
            label = font_tiny.render(
                f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}",
                True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, int(head_y) - head_r - 20))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]

    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    pygame.draw.rect(surf, stage["ground"], (0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y))
    pygame.draw.rect(surf, stage["ground_edge"], (0, GROUND_Y, WIDTH, 12))


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)

    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)

    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    x = random.randint(150, WIDTH - 150)
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, wtype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    p1 = Fighter(WIDTH * 0.25, RED, DARK_RED, 1,
                 (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g),
                 "Player 1")
    p2 = Fighter(WIDTH * 0.75, BLUE, DARK_BLUE, -1,
                 (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l),
                 "Player 2")
    p1.wins = p1_wins
    p2.wins = p2_wins

    if p1_start_weapon:
        p1.weapon = {"type": p1_start_weapon,
                     "durability": WEAPON_TYPES[p1_start_weapon]["durability"]}
    if p2_start_weapon:
        p2.weapon = {"type": p2_start_weapon,
                     "durability": WEAPON_TYPES[p2_start_weapon]["durability"]}

    weapons = []
    # start with one extra weapon on the field for grabbing mid-fight
    spawn_weapon(weapons)
    arrows = []
    return p1, p2, weapons, arrows


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    """Draw a small standalone icon of a weapon choice (or fists if None)."""
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [
            (cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)
        ])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    col_w = 180
    start_x = WIDTH // 2 - (len(WEAPON_CHOICES) * col_w) // 2 + col_w // 2
    icon_y = 220

    # Player 1 row (above icons) and Player 2 row (below) each highlight their pick
    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - 70, icon_y - 70, 140, 140)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + 80))

        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))

    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE

    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))

    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)

    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))

    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))

    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))

    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)

    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render(
            "Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    state = "select"  # "select" -> "world_select" -> "playing" -> "over"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0

    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, arrows = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False

                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True

                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True

                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:  # brief pause on "Get ready..."
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, arrows = reset_fighters(
                        p1_wins, p2_wins,
                        WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, arrows)
            p2.handle_input(keys, p1, weapons, arrows)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL

            # keep fighters from overlapping too much
            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))

            if winner == "Draw":
                msg = "DRAW!"
            else:
                msg = f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))

            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Bow fires arrows instead of a melee swing",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [13]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

More weapons (sword, bat, spear, bow, gatling) also spawn randomly on the
ground during the match. Walk over one while unarmed to pick it up
automatically. Your "punch" button (F / K) becomes a weapon attack while
armed — a melee swing for sword/bat/spear, an arrow shot for the bow, or
a rapid stream of bullets for the gatling gun. Weapons have limited
durability (or ammo, for the bow and gatling) and are lost after a number
of uses.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random

pygame.init()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
SKY_TOP = (135, 190, 230)
SKY_BOTTOM = (220, 235, 245)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

# Unarmed attacks
PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

# Weapon definitions: reach, damage, cooldown (frames), durability (hits), color
# "ranged": True marks weapons that fire projectiles instead of melee swings;
# for those, "durability" means number of shots (arrows) instead of hits.
WEAPON_TYPES = {
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow"},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

WEAPON_PICKUP_RADIUS = 45
MAX_WEAPONS_ON_FIELD = 2
WEAPON_SPAWN_INTERVAL = 300  # frames between spawn attempts

# Selectable starting loadouts (None = fists)
WEAPON_CHOICES = [None, "sword", "bat", "spear", "bow", "gatling"]

# Selectable battle worlds/stages, each with its own palette and decorations
STAGES = [
    {
        "name": "Meadow",
        "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245),
        "ground": (110, 90, 70), "ground_edge": (80, 150, 80),
        "decor": "meadow",
    },
    {
        "name": "Desert",
        "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190),
        "ground": (200, 165, 100), "ground_edge": (225, 195, 130),
        "decor": "desert",
    },
    {
        "name": "Night City",
        "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90),
        "ground": (40, 40, 50), "ground_edge": (90, 90, 110),
        "decor": "city",
    },
    {
        "name": "Volcano",
        "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30),
        "ground": (50, 35, 30), "ground_edge": (200, 80, 30),
        "decor": "volcano",
    },
    {
        "name": "Snow Peak",
        "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245),
        "ground": (225, 235, 240), "ground_edge": (255, 255, 255),
        "decor": "snow",
    },
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    """A ranged projectile fired by a Fighter — visually an arrow or a bullet
    depending on the weapon that fired it."""

    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner  # Fighter who fired it, so it can't hit itself
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return

        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        # arrowhead
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        # fletching
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class WeaponPickup:
    def __init__(self, x, wtype):
        self.x = x
        self.y = GROUND_Y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        # glowing circle behind it
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [
                (self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)
            ])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing  # 1 = right, -1 = left
        self.controls = controls
        self.name = name

        self.health = MAX_HEALTH
        self.on_ground = True

        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0  # frames remaining showing an attack pose
        self.attack_type = None  # "punch", "kick", or "weapon"
        self.hit_stun = 0

        self.walk_cycle = 0
        self.moving = False

        self.wins = 0
        self.weapon = None  # dict: {"type": str, "durability": int}

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height,
                            self.width, self.height)

    def current_stats(self):
        """Return (reach, damage, cooldown) for the primary attack button."""
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)  # slight spread for rapid fire
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))

    def handle_input(self, keys, opponent, weapons, arrows):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.is_ranged():
                self.attack_type = "shoot"
                self.attack_anim = 10
                self.shoot(arrows)
                self.weapon["durability"] -= 1
                if self.weapon["durability"] <= 0:
                    self.weapon = None
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup(weapons)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y
        if self.y >= GROUND_Y:
            self.y = GROUND_Y
            self.vel_y = 0
            self.on_ground = True

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        """Called once when a weapon attack connects. Reduces durability."""
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        # legs (simple walk animation)
        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)

        # torso
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        # arms
        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        # weapon drawing (in hand, extends from forward hand outward)
        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200),
                              (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200),
                              (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                # idle carry pose: weapon rests forward-diagonally
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [
                    tipend,
                    (tip[0] + perp[0], tip[1] + perp[1]),
                    (tip[0] - perp[0], tip[1] - perp[1]),
                ])
            else:  # sword
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK,
                                  (guard_center[0] + perp[0], guard_center[1] + perp[1]),
                                  (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        # head
        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)

        # simple face direction indicator (eye)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        # weapon durability label above head
        if self.weapon:
            label = font_tiny.render(
                f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}",
                True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, int(head_y) - head_r - 20))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]

    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    pygame.draw.rect(surf, stage["ground"], (0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y))
    pygame.draw.rect(surf, stage["ground_edge"], (0, GROUND_Y, WIDTH, 12))


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)

    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)

    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    x = random.randint(150, WIDTH - 150)
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, wtype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    p1 = Fighter(WIDTH * 0.25, RED, DARK_RED, 1,
                 (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g),
                 "Player 1")
    p2 = Fighter(WIDTH * 0.75, BLUE, DARK_BLUE, -1,
                 (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l),
                 "Player 2")
    p1.wins = p1_wins
    p2.wins = p2_wins

    if p1_start_weapon:
        p1.weapon = {"type": p1_start_weapon,
                     "durability": WEAPON_TYPES[p1_start_weapon]["durability"]}
    if p2_start_weapon:
        p2.weapon = {"type": p2_start_weapon,
                     "durability": WEAPON_TYPES[p2_start_weapon]["durability"]}

    weapons = []
    # start with one extra weapon on the field for grabbing mid-fight
    spawn_weapon(weapons)
    arrows = []
    return p1, p2, weapons, arrows


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    """Draw a small standalone icon of a weapon choice (or fists if None)."""
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [
            (cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)
        ])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220

    # Player 1 row (above icons) and Player 2 row (below) each highlight their pick
    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))

        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))

    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE

    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))

    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)

    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))

    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))

    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))

    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)

    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render(
            "Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    state = "select"  # "select" -> "world_select" -> "playing" -> "over"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0

    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, arrows = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False

                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True

                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True

                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:  # brief pause on "Get ready..."
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, arrows = reset_fighters(
                        p1_wins, p2_wins,
                        WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, arrows)
            p2.handle_input(keys, p1, weapons, arrows)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL

            # keep fighters from overlapping too much
            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))

            if winner == "Draw":
                msg = "DRAW!"
            else:
                msg = f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))

            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Bow/Gatling fire ranged shots",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [14]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

More weapons (sword, bat, spear, bow, gatling) also spawn randomly on the
ground during the match. Walk over one while unarmed to pick it up
automatically. Your "punch" button (F / K) becomes a weapon attack while
armed — a melee swing for sword/bat/spear, an arrow shot for the bow, or
a rapid stream of bullets for the gatling gun. Weapons have limited
durability (or ammo, for the bow and gatling) and are lost after a number
of uses.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound (procedurally generated, no external asset files needed)
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        """A short crack/snap noise burst for when a weapon breaks."""
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2  # sharp attack, quick decay
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
except pygame.error:
    break_sound = None  # no audio device available; game still runs silently


def play_break_sound():
    if break_sound:
        break_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
SKY_TOP = (135, 190, 230)
SKY_BOTTOM = (220, 235, 245)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

# Unarmed attacks
PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

# Weapon definitions: reach, damage, cooldown (frames), durability (hits), color
# "ranged": True marks weapons that fire projectiles instead of melee swings;
# for those, "durability" means number of shots (arrows) instead of hits.
WEAPON_TYPES = {
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow"},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

WEAPON_PICKUP_RADIUS = 45
MAX_WEAPONS_ON_FIELD = 2
WEAPON_SPAWN_INTERVAL = 300  # frames between spawn attempts

# Selectable starting loadouts (None = fists)
WEAPON_CHOICES = [None, "sword", "bat", "spear", "bow", "gatling"]

# Selectable battle worlds/stages, each with its own palette and decorations
STAGES = [
    {
        "name": "Meadow",
        "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245),
        "ground": (110, 90, 70), "ground_edge": (80, 150, 80),
        "decor": "meadow",
    },
    {
        "name": "Desert",
        "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190),
        "ground": (200, 165, 100), "ground_edge": (225, 195, 130),
        "decor": "desert",
    },
    {
        "name": "Night City",
        "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90),
        "ground": (40, 40, 50), "ground_edge": (90, 90, 110),
        "decor": "city",
    },
    {
        "name": "Volcano",
        "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30),
        "ground": (50, 35, 30), "ground_edge": (200, 80, 30),
        "decor": "volcano",
    },
    {
        "name": "Snow Peak",
        "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245),
        "ground": (225, 235, 240), "ground_edge": (255, 255, 255),
        "decor": "snow",
    },
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    """A ranged projectile fired by a Fighter — visually an arrow or a bullet
    depending on the weapon that fired it."""

    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner  # Fighter who fired it, so it can't hit itself
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return

        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        # arrowhead
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        # fletching
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class WeaponPickup:
    def __init__(self, x, wtype):
        self.x = x
        self.y = GROUND_Y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        # glowing circle behind it
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [
                (self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)
            ])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing  # 1 = right, -1 = left
        self.controls = controls
        self.name = name

        self.health = MAX_HEALTH
        self.on_ground = True

        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0  # frames remaining showing an attack pose
        self.attack_type = None  # "punch", "kick", or "weapon"
        self.hit_stun = 0

        self.walk_cycle = 0
        self.moving = False

        self.wins = 0
        self.weapon = None  # dict: {"type": str, "durability": int}

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height,
                            self.width, self.height)

    def current_stats(self):
        """Return (reach, damage, cooldown) for the primary attack button."""
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)  # slight spread for rapid fire
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))

    def handle_input(self, keys, opponent, weapons, arrows):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.is_ranged():
                self.attack_type = "shoot"
                self.attack_anim = 10
                self.shoot(arrows)
                self.weapon["durability"] -= 1
                if self.weapon["durability"] <= 0:
                    self.weapon = None
                    play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup(weapons)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y
        if self.y >= GROUND_Y:
            self.y = GROUND_Y
            self.vel_y = 0
            self.on_ground = True

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        """Called once when a weapon attack connects. Reduces durability."""
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        # legs (simple walk animation)
        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)

        # torso
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        # arms
        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        # weapon drawing (in hand, extends from forward hand outward)
        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200),
                              (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200),
                              (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                # idle carry pose: weapon rests forward-diagonally
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [
                    tipend,
                    (tip[0] + perp[0], tip[1] + perp[1]),
                    (tip[0] - perp[0], tip[1] - perp[1]),
                ])
            else:  # sword
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK,
                                  (guard_center[0] + perp[0], guard_center[1] + perp[1]),
                                  (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        # head
        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)

        # simple face direction indicator (eye)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        # weapon durability label above head
        if self.weapon:
            label = font_tiny.render(
                f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}",
                True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, int(head_y) - head_r - 20))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]

    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    pygame.draw.rect(surf, stage["ground"], (0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y))
    pygame.draw.rect(surf, stage["ground_edge"], (0, GROUND_Y, WIDTH, 12))


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)

    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)

    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    x = random.randint(150, WIDTH - 150)
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, wtype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    p1 = Fighter(WIDTH * 0.25, RED, DARK_RED, 1,
                 (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g),
                 "Player 1")
    p2 = Fighter(WIDTH * 0.75, BLUE, DARK_BLUE, -1,
                 (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l),
                 "Player 2")
    p1.wins = p1_wins
    p2.wins = p2_wins

    if p1_start_weapon:
        p1.weapon = {"type": p1_start_weapon,
                     "durability": WEAPON_TYPES[p1_start_weapon]["durability"]}
    if p2_start_weapon:
        p2.weapon = {"type": p2_start_weapon,
                     "durability": WEAPON_TYPES[p2_start_weapon]["durability"]}

    weapons = []
    # start with one extra weapon on the field for grabbing mid-fight
    spawn_weapon(weapons)
    arrows = []
    return p1, p2, weapons, arrows


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    """Draw a small standalone icon of a weapon choice (or fists if None)."""
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [
            (cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)
        ])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220

    # Player 1 row (above icons) and Player 2 row (below) each highlight their pick
    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))

        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))

    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE

    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))

    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)

    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))

    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))

    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))

    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)

    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render(
            "Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    state = "select"  # "select" -> "world_select" -> "playing" -> "over"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0

    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, arrows = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False

                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True

                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True

                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:  # brief pause on "Get ready..."
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, arrows = reset_fighters(
                        p1_wins, p2_wins,
                        WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, arrows)
            p2.handle_input(keys, p1, weapons, arrows)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL

            # keep fighters from overlapping too much
            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))

            if winner == "Draw":
                msg = "DRAW!"
            else:
                msg = f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))

            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Bow/Gatling fire ranged shots",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [15]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

More weapons (sword, bat, spear, bow, gatling) also spawn randomly on the
ground during the match. Walk over one while unarmed to pick it up
automatically. Your "punch" button (F / K) becomes a weapon attack while
armed — a melee swing for sword/bat/spear, an arrow shot for the bow, or
a rapid stream of bullets for the gatling gun. Weapons have limited
durability (or ammo, for the bow and gatling) and are lost after a number
of uses.

The battle floor is a floating platform with a lava/spike pit exposed on
both sides. Walk, get knocked, or fall off the edge and you'll take a
burn hit and bounce back onto the platform — watch your footing near the
edges, especially after taking a knockback hit.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound (procedurally generated, no external asset files needed)
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        """A short crack/snap noise burst for when a weapon breaks."""
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2  # sharp attack, quick decay
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        """A short sizzling hiss for falling into the lava/spike pit."""
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6  # swell then fade
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
except pygame.error:
    break_sound = None  # no audio device available; game still runs silently
    hazard_sound = None


def play_break_sound():
    if break_sound:
        break_sound.play()


def play_hazard_sound():
    if hazard_sound:
        hazard_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
SKY_TOP = (135, 190, 230)
SKY_BOTTOM = (220, 235, 245)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

# The battle floor is a floating platform with a lava/spike pit visible on
# either side. Walking or getting knocked past the platform's edge means
# a fall straight into the hazard below.
PLATFORM_MARGIN = 170
PLATFORM_LEFT = PLATFORM_MARGIN
PLATFORM_RIGHT = WIDTH - PLATFORM_MARGIN
HAZARD_Y = GROUND_Y + 90       # depth at which the fall becomes a hazard hit
HAZARD_DAMAGE = 22
HAZARD_STUN = 24

# Unarmed attacks
PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

# Weapon definitions: reach, damage, cooldown (frames), durability (hits), color
# "ranged": True marks weapons that fire projectiles instead of melee swings;
# for those, "durability" means number of shots (arrows) instead of hits.
WEAPON_TYPES = {
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow"},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

WEAPON_PICKUP_RADIUS = 45
MAX_WEAPONS_ON_FIELD = 2
WEAPON_SPAWN_INTERVAL = 300  # frames between spawn attempts

# Selectable starting loadouts (None = fists)
WEAPON_CHOICES = [None, "sword", "bat", "spear", "bow", "gatling"]

# Selectable battle worlds/stages, each with its own palette and decorations
STAGES = [
    {
        "name": "Meadow",
        "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245),
        "ground": (110, 90, 70), "ground_edge": (80, 150, 80),
        "decor": "meadow",
    },
    {
        "name": "Desert",
        "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190),
        "ground": (200, 165, 100), "ground_edge": (225, 195, 130),
        "decor": "desert",
    },
    {
        "name": "Night City",
        "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90),
        "ground": (40, 40, 50), "ground_edge": (90, 90, 110),
        "decor": "city",
    },
    {
        "name": "Volcano",
        "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30),
        "ground": (50, 35, 30), "ground_edge": (200, 80, 30),
        "decor": "volcano",
    },
    {
        "name": "Snow Peak",
        "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245),
        "ground": (225, 235, 240), "ground_edge": (255, 255, 255),
        "decor": "snow",
    },
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    """A ranged projectile fired by a Fighter — visually an arrow or a bullet
    depending on the weapon that fired it."""

    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner  # Fighter who fired it, so it can't hit itself
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return

        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        # arrowhead
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        # fletching
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class WeaponPickup:
    def __init__(self, x, wtype):
        self.x = x
        self.y = GROUND_Y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        # glowing circle behind it
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [
                (self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)
            ])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing  # 1 = right, -1 = left
        self.controls = controls
        self.name = name

        self.health = MAX_HEALTH
        self.on_ground = True

        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0  # frames remaining showing an attack pose
        self.attack_type = None  # "punch", "kick", or "weapon"
        self.hit_stun = 0

        self.walk_cycle = 0
        self.moving = False

        self.wins = 0
        self.weapon = None  # dict: {"type": str, "durability": int}

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height,
                            self.width, self.height)

    def current_stats(self):
        """Return (reach, damage, cooldown) for the primary attack button."""
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)  # slight spread for rapid fire
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))

    def handle_input(self, keys, opponent, weapons, arrows):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.is_ranged():
                self.attack_type = "shoot"
                self.attack_anim = 10
                self.shoot(arrows)
                self.weapon["durability"] -= 1
                if self.weapon["durability"] <= 0:
                    self.weapon = None
                    play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup(weapons)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y

        on_platform_x = PLATFORM_LEFT <= self.x <= PLATFORM_RIGHT
        if on_platform_x and self.y >= GROUND_Y:
            self.y = GROUND_Y
            self.vel_y = 0
            self.on_ground = True
        else:
            self.on_ground = False
            if self.y >= HAZARD_Y:
                self.hazard_hit()

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        """Called once when a weapon attack connects. Reduces durability."""
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        """Called when the fighter falls into the lava/spike pit off either
        side of the platform. Burns them, stuns them, and pops them back up
        onto the nearest platform edge."""
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        if self.x < WIDTH / 2:
            self.x = PLATFORM_LEFT + 30
        else:
            self.x = PLATFORM_RIGHT - 30
        self.y = GROUND_Y
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        # legs (simple walk animation)
        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)

        # torso
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        # arms
        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        # weapon drawing (in hand, extends from forward hand outward)
        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200),
                              (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200),
                              (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                # idle carry pose: weapon rests forward-diagonally
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [
                    tipend,
                    (tip[0] + perp[0], tip[1] + perp[1]),
                    (tip[0] - perp[0], tip[1] - perp[1]),
                ])
            else:  # sword
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK,
                                  (guard_center[0] + perp[0], guard_center[1] + perp[1]),
                                  (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        # head
        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)

        # simple face direction indicator (eye)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        # weapon durability label above head
        if self.weapon:
            label = font_tiny.render(
                f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}",
                True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, int(head_y) - head_r - 20))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]

    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    # Lava/spike pit fills the full width at floor level; the platform is
    # then drawn on top of it in the middle, leaving hazard visible on
    # either side where a fighter can fall through.
    draw_hazard_pit(surf)

    pygame.draw.rect(surf, stage["ground"],
                      (PLATFORM_LEFT, GROUND_Y, PLATFORM_RIGHT - PLATFORM_LEFT, HEIGHT - GROUND_Y))
    pygame.draw.rect(surf, stage["ground_edge"],
                      (PLATFORM_LEFT, GROUND_Y, PLATFORM_RIGHT - PLATFORM_LEFT, 12))
    # small rock underside hanging from each edge, so the platform reads as
    # floating above the pit rather than just stopping mid-air
    pygame.draw.polygon(surf, stage["ground"], [
        (PLATFORM_LEFT, GROUND_Y + 12), (PLATFORM_LEFT - 18, GROUND_Y + 55), (PLATFORM_LEFT + 22, GROUND_Y + 55)
    ])
    pygame.draw.polygon(surf, stage["ground"], [
        (PLATFORM_RIGHT, GROUND_Y + 12), (PLATFORM_RIGHT + 18, GROUND_Y + 55), (PLATFORM_RIGHT - 22, GROUND_Y + 55)
    ])


def draw_hazard_pit(surf):
    """Draw the lava/spike hazard that fills the floor on either side of
    the central platform."""
    pit_rect = pygame.Rect(0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)

    # bubbling lava glow, layered bands getting brighter toward the bottom
    for i, (band_y, color) in enumerate([
        (GROUND_Y + 15, (150, 40, 15)),
        (GROUND_Y + 35, (200, 70, 20)),
        (GROUND_Y + 55, (240, 110, 30)),
    ]):
        for gx in range(0, WIDTH, 26):
            wobble = math.sin((gx + i * 40) * 0.15) * 4
            pygame.draw.circle(surf, color, (gx + 13, int(band_y + wobble)), 9)

    # bright glowing highlights
    for gx in range(0, WIDTH, 52):
        pygame.draw.circle(surf, (255, 200, 90), (gx + 26, GROUND_Y + 60), 4)

    # spikes jutting up from the very bottom of the pit
    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [
            (gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)
        ])
        pygame.draw.polygon(surf, (90, 90, 95), [
            (gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)
        ])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)

    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)

    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    x = random.randint(PLATFORM_LEFT + 25, PLATFORM_RIGHT - 25)
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, wtype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    p1 = Fighter(WIDTH * 0.25, RED, DARK_RED, 1,
                 (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g),
                 "Player 1")
    p2 = Fighter(WIDTH * 0.75, BLUE, DARK_BLUE, -1,
                 (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l),
                 "Player 2")
    p1.wins = p1_wins
    p2.wins = p2_wins

    if p1_start_weapon:
        p1.weapon = {"type": p1_start_weapon,
                     "durability": WEAPON_TYPES[p1_start_weapon]["durability"]}
    if p2_start_weapon:
        p2.weapon = {"type": p2_start_weapon,
                     "durability": WEAPON_TYPES[p2_start_weapon]["durability"]}

    weapons = []
    # start with one extra weapon on the field for grabbing mid-fight
    spawn_weapon(weapons)
    arrows = []
    return p1, p2, weapons, arrows


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    """Draw a small standalone icon of a weapon choice (or fists if None)."""
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [
            (cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)
        ])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220

    # Player 1 row (above icons) and Player 2 row (below) each highlight their pick
    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))

        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))

    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE

    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))

    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)

    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))

    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))

    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))

    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)

    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render(
            "Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    state = "select"  # "select" -> "world_select" -> "playing" -> "over"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0

    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, arrows = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False

                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True

                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True

                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:  # brief pause on "Get ready..."
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, arrows = reset_fighters(
                        p1_wins, p2_wins,
                        WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, arrows)
            p2.handle_input(keys, p1, weapons, arrows)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL

            # keep fighters from overlapping too much
            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))

            if winner == "Draw":
                msg = "DRAW!"
            else:
                msg = f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))

            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Watch the ledge — lava/spikes below!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [2]:
pip install pygame

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 18.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons and multiple platforms!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

More weapons (uzi, sword, bat, spear, bow, gatling) also spawn randomly on the
ground during the match. Walk over one while unarmed to pick it up
automatically. Your "punch" button (F / K) becomes a weapon attack while
armed — a melee swing for sword/bat/spear, an arrow shot for the bow, or
a rapid stream of bullets for the gatling gun. Weapons have limited
durability (or ammo, for the bow and gatling) and are lost after a number
of uses.

The battle floor features a main floating platform with additional mid-air 
platforms. A lava/spike pit is exposed on both sides. Walk, get knocked, 
or fall off the edge and you'll take instant death — watch your footing!

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound (procedurally generated, no external asset files needed)
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        """A short crack/snap noise burst for when a weapon breaks."""
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2  # sharp attack, quick decay
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        """A short sizzling hiss for falling into the lava/spike pit."""
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6  # swell then fade
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
except pygame.error:
    break_sound = None  # no audio device available; game still runs silently
    hazard_sound = None


def play_break_sound():
    if break_sound:
        break_sound.play()


def play_hazard_sound():
    if hazard_sound:
        hazard_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
SKY_TOP = (135, 190, 230)
SKY_BOTTOM = (220, 235, 245)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

# The battle floor is a floating platform with a lava/spike pit visible on
# either side. Walking or getting knocked past the platform's edge means
# a fall straight into the hazard below.
PLATFORM_MARGIN = 170
PLATFORM_LEFT = PLATFORM_MARGIN
PLATFORM_RIGHT = WIDTH - PLATFORM_MARGIN
HAZARD_Y = GROUND_Y + 90       # depth at which the fall becomes a hazard hit
HAZARD_DAMAGE = 100
HAZARD_STUN = 24

# Multiple platforms for vertical gameplay (Main ground + 3 floating platforms)
# Jump strength (-15) and gravity (0.8) allow ~140px of jump height, making 
# these platforms comfortably reachable from one another.
PLATFORMS = [
    # Main ground (extends to bottom of screen)
    pygame.Rect(PLATFORM_LEFT, GROUND_Y, PLATFORM_RIGHT - PLATFORM_LEFT, HEIGHT - GROUND_Y),
    # Left mid-air platform
    pygame.Rect(250, 380, 150, 20),
    # Right mid-air platform
    pygame.Rect(600, 380, 150, 20),
    # Center high platform
    pygame.Rect(425, 260, 150, 20),
]

# Unarmed attacks
PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

# Weapon definitions: reach, damage, cooldown (frames), durability (hits), color
# "ranged": True marks weapons that fire projectiles instead of melee swings;
# for those, "durability" means number of shots (arrows) instead of hits.
WEAPON_TYPES = {
    "uzi":     {"reach": 0,   "damage": 5, "cooldown": 5, "durability": 50, "color": (45, 45, 50),
                "ranged": True, "proj_speed": 28, "proj_kind": "bullet"},
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow"},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

WEAPON_PICKUP_RADIUS = 45
MAX_WEAPONS_ON_FIELD = 2
WEAPON_SPAWN_INTERVAL = 300  # frames between spawn attempts

# Selectable starting loadouts (None = fists)
WEAPON_CHOICES = [None, "uzi", "sword", "bat", "spear", "bow", "gatling"]

# Selectable battle worlds/stages, each with its own palette and decorations
STAGES = [
    {
        "name": "Meadow",
        "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245),
        "ground": (110, 90, 70), "ground_edge": (80, 150, 80),
        "decor": "meadow",
    },
    {
        "name": "Desert",
        "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190),
        "ground": (200, 165, 100), "ground_edge": (225, 195, 130),
        "decor": "desert",
    },
    {
        "name": "Night City",
        "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90),
        "ground": (40, 40, 50), "ground_edge": (90, 90, 110),
        "decor": "city",
    },
    {
        "name": "Volcano",
        "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30),
        "ground": (50, 35, 30), "ground_edge": (200, 80, 30),
        "decor": "volcano",
    },
    {
        "name": "Snow Peak",
        "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245),
        "ground": (225, 235, 240), "ground_edge": (255, 255, 255),
        "decor": "snow",
    },
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    """A ranged projectile fired by a Fighter — visually an arrow or a bullet
    depending on the weapon that fired it."""

    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner  # Fighter who fired it, so it can't hit itself
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return

        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        # arrowhead
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        # fletching
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class WeaponPickup:
    def __init__(self, x, y, wtype):
        self.x = x
        self.y = y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        # glowing circle behind it
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [
                (self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)
            ])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing  # 1 = right, -1 = left
        self.controls = controls
        self.name = name

        self.health = MAX_HEALTH
        self.on_ground = True

        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0  # frames remaining showing an attack pose
        self.attack_type = None  # "punch", "kick", or "weapon"
        self.hit_stun = 0

        self.walk_cycle = 0
        self.moving = False

        self.wins = 0
        self.weapon = None  # dict: {"type": str, "durability": int}

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height,
                            self.width, self.height)

    def current_stats(self):
        """Return (reach, damage, cooldown) for the primary attack button."""
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)  # slight spread for rapid fire
        elif self.weapon["type"] == "uzi":
            spawn_y += random.randint(-3, 3)  # tiny SMG spread
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))

    def handle_input(self, keys, opponent, weapons, arrows):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.is_ranged():
                self.attack_type = "shoot"
                self.attack_anim = 10
                self.shoot(arrows)
                self.weapon["durability"] -= 1
                if self.weapon["durability"] <= 0:
                    self.weapon = None
                    play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup(weapons)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y

        self.on_ground = False
        # Check collision with all platforms (allows jumping up through them)
        for plat in PLATFORMS:
            prev_y = self.y - self.vel_y
            # Check if falling down onto the platform
            if prev_y <= plat.top and self.y >= plat.top and plat.left <= self.x <= plat.right and self.vel_y >= 0:
                self.y = plat.top
                self.vel_y = 0
                self.on_ground = True
                break

        if not self.on_ground:
            if self.y >= HAZARD_Y:
                self.hazard_hit()

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        """Called once when a weapon attack connects. Reduces durability."""
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        """Called when the fighter falls into the lava/spike pit off either
        side of the platform. Burns them, stuns them, and pops them back up
        onto the nearest platform edge."""
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        if self.x < WIDTH / 2:
            self.x = PLATFORM_LEFT + 30
        else:
            self.x = PLATFORM_RIGHT - 30
        self.y = GROUND_Y
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        # legs (simple walk animation)
        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)

        # torso
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        # arms
        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        # weapon drawing (in hand, extends from forward hand outward)
        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200),
                              (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200),
                              (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "uzi":
            # Compact Uzi-style SMG.
            wcolor = WEAPON_TYPES["uzi"]["color"]
            ux, uy = fwd_hand[0], fwd_hand[1] - 3
            barrel_tip = (ux + self.facing * 27, uy)
            grip_bottom = (ux - self.facing * 5, uy + 18)
            pygame.draw.rect(
                surf, wcolor,
                pygame.Rect(min(ux, barrel_tip[0]), uy - 7,
                            abs(barrel_tip[0] - ux) + 8, 14),
                border_radius=3
            )
            pygame.draw.line(surf, (25, 25, 28), (ux, uy + 4), grip_bottom, 6)
            pygame.draw.line(surf, (25, 25, 28), barrel_tip,
                             (barrel_tip[0] + self.facing * 10, barrel_tip[1]), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 8, barrel_tip[1])
                pygame.draw.circle(surf, (255, 220, 90),
                                   (int(flash[0]), int(flash[1])), 6)

        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                # idle carry pose: weapon rests forward-diagonally
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [
                    tipend,
                    (tip[0] + perp[0], tip[1] + perp[1]),
                    (tip[0] - perp[0], tip[1] - perp[1]),
                ])
            else:  # sword
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK,
                                  (guard_center[0] + perp[0], guard_center[1] + perp[1]),
                                  (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        # head
        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)

        # simple face direction indicator (eye)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        # weapon durability label above head
        if self.weapon:
            label = font_tiny.render(
                f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}",
                True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, int(head_y) - head_r - 20))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]

    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    # Lava/spike pit fills the full width at floor level; the platform is
    # then drawn on top of it in the middle, leaving hazard visible on
    # either side where a fighter can fall through.
    draw_hazard_pit(surf)

    # Draw main ground (first platform) extending to bottom
    main_plat = PLATFORMS[0]
    pygame.draw.rect(surf, stage["ground"], main_plat)
    pygame.draw.rect(surf, stage["ground_edge"], (main_plat.left, main_plat.top, main_plat.width, 12))
    # small rock underside hanging from each edge, so the platform reads as
    # floating above the pit rather than just stopping mid-air
    pygame.draw.polygon(surf, stage["ground"], [
        (main_plat.left, main_plat.top + 12), (main_plat.left - 18, main_plat.top + 55), (main_plat.left + 22, main_plat.top + 55)
    ])
    pygame.draw.polygon(surf, stage["ground"], [
        (main_plat.right, main_plat.top + 12), (main_plat.right + 18, main_plat.top + 55), (main_plat.right - 22, main_plat.top + 55)
    ])

    # Draw additional floating platforms
    for plat in PLATFORMS[1:]:
        pygame.draw.rect(surf, stage["ground"], plat)
        pygame.draw.rect(surf, stage["ground_edge"], (plat.left, plat.top, plat.width, 10))
        # subtle underside detail for floating platforms
        pygame.draw.polygon(surf, stage["ground"], [
            (plat.left, plat.bottom), (plat.left + 12, plat.bottom + 18), (plat.left + 24, plat.bottom)
        ])
        pygame.draw.polygon(surf, stage["ground"], [
            (plat.right, plat.bottom), (plat.right - 12, plat.bottom + 18), (plat.right - 24, plat.bottom)
        ])


def draw_hazard_pit(surf):
    """Draw the lava/spike hazard that fills the floor on either side of
    the central platform."""
    pit_rect = pygame.Rect(0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)

    # bubbling lava glow, layered bands getting brighter toward the bottom
    for i, (band_y, color) in enumerate([
        (GROUND_Y + 15, (150, 40, 15)),
        (GROUND_Y + 35, (200, 70, 20)),
        (GROUND_Y + 55, (240, 110, 30)),
    ]):
        for gx in range(0, WIDTH, 26):
            wobble = math.sin((gx + i * 40) * 0.15) * 4
            pygame.draw.circle(surf, color, (gx + 13, int(band_y + wobble)), 9)

    # bright glowing highlights
    for gx in range(0, WIDTH, 52):
        pygame.draw.circle(surf, (255, 200, 90), (gx + 26, GROUND_Y + 60), 4)

    # spikes jutting up from the very bottom of the pit
    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [
            (gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)
        ])
        pygame.draw.polygon(surf, (90, 90, 95), [
            (gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)
        ])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)

    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)

    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    # Pick a random platform to spawn the weapon on
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, y, wtype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    p1 = Fighter(WIDTH * 0.25, RED, DARK_RED, 1,
                 (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g),
                 "Player 1")
    p2 = Fighter(WIDTH * 0.75, BLUE, DARK_BLUE, -1,
                 (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l),
                 "Player 2")
    p1.wins = p1_wins
    p2.wins = p2_wins

    if p1_start_weapon:
        p1.weapon = {"type": p1_start_weapon,
                     "durability": WEAPON_TYPES[p1_start_weapon]["durability"]}
    if p2_start_weapon:
        p2.weapon = {"type": p2_start_weapon,
                     "durability": WEAPON_TYPES[p2_start_weapon]["durability"]}

    weapons = []
    # start with one extra weapon on the field for grabbing mid-fight
    spawn_weapon(weapons)
    arrows = []
    return p1, p2, weapons, arrows


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    """Draw a small standalone icon of a weapon choice (or fists if None)."""
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [
            (cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)
        ])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "uzi":
        pygame.draw.rect(surf, (45, 45, 50), (cx - 22, cy - 8, 44, 16), border_radius=3)
        pygame.draw.line(surf, (25, 25, 28), (cx - 4, cy + 5), (cx - 10, cy + 25), 7)
        pygame.draw.line(surf, (25, 25, 28), (cx + 20, cy), (cx + 34, cy), 5)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220

    # Player 1 row (above icons) and Player 2 row (below) each highlight their pick
    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))

        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))

    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE

    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))

    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)

    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))

    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))

    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))

    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)

    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render(
            "Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    state = "select"  # "select" -> "world_select" -> "playing" -> "over"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0

    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, arrows = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False

                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True

                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True

                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:  # brief pause on "Get ready..."
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, arrows = reset_fighters(
                        p1_wins, p2_wins,
                        WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, arrows)
            p2.handle_input(keys, p1, weapons, arrows)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL

            # keep fighters from overlapping too much
            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))

            if winner == "Draw":
                msg = "DRAW!"
            else:
                msg = f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))

            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Watch the ledges & platforms!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [17]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons and floating platforms!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

More weapons (uzi, sword, bat, spear, bow, gatling) also spawn randomly on the
platforms during the match. Walk over one while unarmed to pick it up
automatically. Your "punch" button (F / K) becomes a weapon attack while
armed — a melee swing for sword/bat/spear, an arrow shot for the bow, or
a rapid stream of bullets for the gatling gun. Weapons have limited
durability (or ammo, for the bow and gatling) and are lost after a number
of uses.

The arena consists of multiple floating platforms above a deadly lava/spike 
pit. Walk, get knocked, or fall off the edge and you'll take instant death — 
watch your footing!

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound (procedurally generated, no external asset files needed)
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        """A short crack/snap noise burst for when a weapon breaks."""
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2  # sharp attack, quick decay
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        """A short sizzling hiss for falling into the lava/spike pit."""
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6  # swell then fade
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
except pygame.error:
    break_sound = None  # no audio device available; game still runs silently
    hazard_sound = None


def play_break_sound():
    if break_sound:
        break_sound.play()


def play_hazard_sound():
    if hazard_sound:
        hazard_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100  # Used as visual baseline for the pit and background decor
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
SKY_TOP = (135, 190, 230)
SKY_BOTTOM = (220, 235, 245)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

HAZARD_Y = GROUND_Y + 90       # depth at which the fall becomes a hazard hit
HAZARD_DAMAGE = 100
HAZARD_STUN = 24

# Multiple floating platforms for vertical gameplay. 
# Jump strength (-15) and gravity (0.8) allow ~140px of jump height, making 
# these platforms comfortably reachable from one another.
PLATFORMS = [
    # Left mid-air platform
    pygame.Rect(250, 380, 150, 20),
    # Right mid-air platform
    pygame.Rect(600, 380, 150, 20),
    # Center high platform
    pygame.Rect(425, 260, 150, 20),
]

# Unarmed attacks
PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

# Weapon definitions: reach, damage, cooldown (frames), durability (hits), color
# "ranged": True marks weapons that fire projectiles instead of melee swings;
# for those, "durability" means number of shots (arrows) instead of hits.
WEAPON_TYPES = {
    "uzi":     {"reach": 0,   "damage": 5, "cooldown": 5, "durability": 50, "color": (45, 45, 50),
                "ranged": True, "proj_speed": 28, "proj_kind": "bullet"},
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow"},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

WEAPON_PICKUP_RADIUS = 45
MAX_WEAPONS_ON_FIELD = 2
WEAPON_SPAWN_INTERVAL = 300  # frames between spawn attempts

# Selectable starting loadouts (None = fists)
WEAPON_CHOICES = [None, "uzi", "sword", "bat", "spear", "bow", "gatling"]

# Selectable battle worlds/stages, each with its own palette and decorations
STAGES = [
    {
        "name": "Meadow",
        "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245),
        "ground": (110, 90, 70), "ground_edge": (80, 150, 80),
        "decor": "meadow",
    },
    {
        "name": "Desert",
        "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190),
        "ground": (200, 165, 100), "ground_edge": (225, 195, 130),
        "decor": "desert",
    },
    {
        "name": "Night City",
        "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90),
        "ground": (40, 40, 50), "ground_edge": (90, 90, 110),
        "decor": "city",
    },
    {
        "name": "Volcano",
        "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30),
        "ground": (50, 35, 30), "ground_edge": (200, 80, 30),
        "decor": "volcano",
    },
    {
        "name": "Snow Peak",
        "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245),
        "ground": (225, 235, 240), "ground_edge": (255, 255, 255),
        "decor": "snow",
    },
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    """A ranged projectile fired by a Fighter — visually an arrow or a bullet
    depending on the weapon that fired it."""

    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner  # Fighter who fired it, so it can't hit itself
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return

        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        # arrowhead
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        # fletching
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class WeaponPickup:
    def __init__(self, x, y, wtype):
        self.x = x
        self.y = y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        # glowing circle behind it
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [
                (self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)
            ])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing  # 1 = right, -1 = left
        self.controls = controls
        self.name = name

        self.health = MAX_HEALTH
        self.on_ground = True

        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0  # frames remaining showing an attack pose
        self.attack_type = None  # "punch", "kick", or "weapon"
        self.hit_stun = 0

        self.walk_cycle = 0
        self.moving = False

        self.wins = 0
        self.weapon = None  # dict: {"type": str, "durability": int}

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height,
                            self.width, self.height)

    def current_stats(self):
        """Return (reach, damage, cooldown) for the primary attack button."""
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)  # slight spread for rapid fire
        elif self.weapon["type"] == "uzi":
            spawn_y += random.randint(-3, 3)  # tiny SMG spread
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))

    def handle_input(self, keys, opponent, weapons, arrows):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.is_ranged():
                self.attack_type = "shoot"
                self.attack_anim = 10
                self.shoot(arrows)
                self.weapon["durability"] -= 1
                if self.weapon["durability"] <= 0:
                    self.weapon = None
                    play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup(weapons)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y

        self.on_ground = False
        # Check collision with all platforms (allows jumping up through them)
        for plat in PLATFORMS:
            prev_y = self.y - self.vel_y
            # Check if falling down onto the platform
            if prev_y <= plat.top and self.y >= plat.top and plat.left <= self.x <= plat.right and self.vel_y >= 0:
                self.y = plat.top
                self.vel_y = 0
                self.on_ground = True
                break

        if not self.on_ground:
            if self.y >= HAZARD_Y:
                self.hazard_hit()

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        """Called once when a weapon attack connects. Reduces durability."""
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        """Called when the fighter falls into the lava/spike pit. 
        Burns them, stuns them, and pops them back up onto a safe platform."""
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        if self.x < WIDTH / 2:
            self.x = 325  # Center of left platform
        else:
            self.x = 675  # Center of right platform
        self.y = 380      # Top of the mid-air platforms
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        # legs (simple walk animation)
        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)

        # torso
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        # arms
        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        # weapon drawing (in hand, extends from forward hand outward)
        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200),
                              (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200),
                              (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "uzi":
            # Compact Uzi-style SMG.
            wcolor = WEAPON_TYPES["uzi"]["color"]
            ux, uy = fwd_hand[0], fwd_hand[1] - 3
            barrel_tip = (ux + self.facing * 27, uy)
            grip_bottom = (ux - self.facing * 5, uy + 18)
            pygame.draw.rect(
                surf, wcolor,
                pygame.Rect(min(ux, barrel_tip[0]), uy - 7,
                            abs(barrel_tip[0] - ux) + 8, 14),
                border_radius=3
            )
            pygame.draw.line(surf, (25, 25, 28), (ux, uy + 4), grip_bottom, 6)
            pygame.draw.line(surf, (25, 25, 28), barrel_tip,
                             (barrel_tip[0] + self.facing * 10, barrel_tip[1]), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 8, barrel_tip[1])
                pygame.draw.circle(surf, (255, 220, 90),
                                   (int(flash[0]), int(flash[1])), 6)

        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                # idle carry pose: weapon rests forward-diagonally
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [
                    tipend,
                    (tip[0] + perp[0], tip[1] + perp[1]),
                    (tip[0] - perp[0], tip[1] - perp[1]),
                ])
            else:  # sword
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx = tip[0] - grip[0]
                dy = tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK,
                                  (guard_center[0] + perp[0], guard_center[1] + perp[1]),
                                  (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        # head
        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)

        # simple face direction indicator (eye)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        # weapon durability label above head
        if self.weapon:
            label = font_tiny.render(
                f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}",
                True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, int(head_y) - head_r - 20))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]

    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    # Lava/spike pit fills the full width at the bottom of the screen
    draw_hazard_pit(surf)

    # Draw all floating platforms
    for plat in PLATFORMS:
        pygame.draw.rect(surf, stage["ground"], plat)
        pygame.draw.rect(surf, stage["ground_edge"], (plat.left, plat.top, plat.width, 10))
        # subtle underside detail for floating platforms
        pygame.draw.polygon(surf, stage["ground"], [
            (plat.left, plat.bottom), (plat.left + 12, plat.bottom + 18), (plat.left + 24, plat.bottom)
        ])
        pygame.draw.polygon(surf, stage["ground"], [
            (plat.right, plat.bottom), (plat.right - 12, plat.bottom + 18), (plat.right - 24, plat.bottom)
        ])


def draw_hazard_pit(surf):
    """Draw the lava/spike hazard that fills the floor at the bottom."""
    pit_rect = pygame.Rect(0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)

    # bubbling lava glow, layered bands getting brighter toward the bottom
    for i, (band_y, color) in enumerate([
        (GROUND_Y + 15, (150, 40, 15)),
        (GROUND_Y + 35, (200, 70, 20)),
        (GROUND_Y + 55, (240, 110, 30)),
    ]):
        for gx in range(0, WIDTH, 26):
            wobble = math.sin((gx + i * 40) * 0.15) * 4
            pygame.draw.circle(surf, color, (gx + 13, int(band_y + wobble)), 9)

    # bright glowing highlights
    for gx in range(0, WIDTH, 52):
        pygame.draw.circle(surf, (255, 200, 90), (gx + 26, GROUND_Y + 60), 4)

    # spikes jutting up from the very bottom of the pit
    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [
            (gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)
        ])
        pygame.draw.polygon(surf, (90, 90, 95), [
            (gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)
        ])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)

    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)

    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    # Pick a random floating platform to spawn the weapon on
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, y, wtype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    p1 = Fighter(WIDTH * 0.25, RED, DARK_RED, 1,
                 (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g),
                 "Player 1")
    p1.x = 325  # Center of left platform
    p1.y = 380  # Top of left platform
    
    p2 = Fighter(WIDTH * 0.75, BLUE, DARK_BLUE, -1,
                 (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l),
                 "Player 2")
    p2.x = 675  # Center of right platform
    p2.y = 380  # Top of right platform
    
    p1.wins = p1_wins
    p2.wins = p2_wins

    if p1_start_weapon:
        p1.weapon = {"type": p1_start_weapon,
                     "durability": WEAPON_TYPES[p1_start_weapon]["durability"]}
    if p2_start_weapon:
        p2.weapon = {"type": p2_start_weapon,
                     "durability": WEAPON_TYPES[p2_start_weapon]["durability"]}

    weapons = []
    # start with one extra weapon on the field for grabbing mid-fight
    spawn_weapon(weapons)
    arrows = []
    return p1, p2, weapons, arrows


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    """Draw a small standalone icon of a weapon choice (or fists if None)."""
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [
            (cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)
        ])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "uzi":
        pygame.draw.rect(surf, (45, 45, 50), (cx - 22, cy - 8, 44, 16), border_radius=3)
        pygame.draw.line(surf, (25, 25, 28), (cx - 4, cy + 5), (cx - 10, cy + 25), 7)
        pygame.draw.line(surf, (25, 25, 28), (cx + 20, cy), (cx + 34, cy), 5)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220

    # Player 1 row (above icons) and Player 2 row (below) each highlight their pick
    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))

        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))

    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE

    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))

    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)

    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))

    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))

    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))

    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)

    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render(
            "Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    state = "select"  # "select" -> "world_select" -> "playing" -> "over"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0

    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, arrows = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False

                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True

                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True

                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:  # brief pause on "Get ready..."
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, arrows = reset_fighters(
                        p1_wins, p2_wins,
                        WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, arrows)
            p2.handle_input(keys, p1, weapons, arrows)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL

            # keep fighters from overlapping too much
            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))

            if winner == "Draw":
                msg = "DRAW!"
            else:
                msg = f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))

            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Watch the edges — platforms float over the pit!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

2026-09-10 15:34:00.461 Python[78439:16566090] error messaging the mach port for IMKCFRunLoopWakeUpReliable


SystemExit: 

In [18]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons, floating platforms, and bombs!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

More weapons (uzi, sword, bat, spear, bow, gatling, bomb) also spawn randomly 
on the platforms during the match. Walk over one while unarmed to pick it up
automatically. Your "punch" button (F / K) becomes a weapon attack while
armed — a melee swing for sword/bat/spear, an arrow shot for the bow, a rapid 
stream of bullets for the gatling, or a thrown explosive for the bomb! 
Weapons have limited durability and are lost after a number of uses.

The arena consists of multiple floating platforms above a deadly lava/spike 
pit. Walk, get knocked, or fall off the edge and you'll take instant death — 
watch your footing!

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound (procedurally generated, no external asset files needed)
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None
explosion_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_explosion_sound():
        duration = 0.45
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 120 * (1 - progress * 0.8)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.7 * noise + 0.3 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
    explosion_sound = _generate_explosion_sound()
except pygame.error:
    break_sound = None
    hazard_sound = None
    explosion_sound = None


def play_break_sound():
    if break_sound:
        break_sound.play()

def play_hazard_sound():
    if hazard_sound:
        hazard_sound.play()

def play_explosion_sound():
    if explosion_sound:
        explosion_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100  # Used as visual baseline for the pit and background decor
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

HAZARD_Y = GROUND_Y + 90
HAZARD_DAMAGE = 100
HAZARD_STUN = 24

# Multiple floating platforms for vertical gameplay.
PLATFORMS = [
    pygame.Rect(250, 380, 150, 20),   # Left mid-air platform
    pygame.Rect(600, 380, 150, 20),   # Right mid-air platform
    pygame.Rect(425, 260, 150, 20),   # Center high platform
]

# Unarmed attacks
PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

# Weapon definitions
WEAPON_TYPES = {
    "uzi":     {"reach": 0,   "damage": 5, "cooldown": 5, "durability": 50, "color": (45, 45, 50),
                "ranged": True, "proj_speed": 28, "proj_kind": "bullet"},
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow"},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet"},
    "bomb":    {"reach": 0,   "damage": 35, "cooldown": 50, "durability": 3,  "color": (40, 40, 40),
                "ranged": True, "proj_speed": 9, "proj_kind": "bomb"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

MAX_WEAPONS_ON_FIELD = 2
WEAPON_SPAWN_INTERVAL = 300

WEAPON_CHOICES = [None, "uzi", "sword", "bat", "spear", "bow", "gatling", "bomb"]

STAGES = [
    {"name": "Meadow", "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245), "ground": (110, 90, 70), "ground_edge": (80, 150, 80), "decor": "meadow"},
    {"name": "Desert", "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190), "ground": (200, 165, 100), "ground_edge": (225, 195, 130), "decor": "desert"},
    {"name": "Night City", "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90), "ground": (40, 40, 50), "ground_edge": (90, 90, 110), "decor": "city"},
    {"name": "Volcano", "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30), "ground": (50, 35, 30), "ground_edge": (200, 80, 30), "decor": "volcano"},
    {"name": "Snow Peak", "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245), "ground": (225, 235, 240), "ground_edge": (255, 255, 255), "decor": "snow"},
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return
        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class Bomb:
    def __init__(self, x, y, facing, damage, owner):
        self.x = x
        self.y = y
        self.vel_x = facing * 9
        self.vel_y = -11
        self.damage = damage
        self.owner = owner
        self.timer = 70  # frames until explosion
        self.dead = False
        self.exploded = False

    def update(self, platforms):
        if self.exploded:
            return
        self.vel_y += GRAVITY
        self.x += self.vel_x
        self.y += self.vel_y
        self.timer -= 1
        
        # Check platform collision
        for plat in platforms:
            if plat.left <= self.x <= plat.right and self.y >= plat.top and self.vel_y >= 0:
                if self.y - self.vel_y <= plat.top + 10:
                    self.y = plat.top
                    self.vel_y = 0
                    self.vel_x *= 0.7  # friction
                    self.timer -= 3    # explode slightly faster on impact
        
        # Explode if it hits the hazard level or timer runs out
        if self.y >= GROUND_Y or self.timer <= 0:
            self.exploded = True
            self.dead = True

    def explode(self):
        play_explosion_sound()
        return Explosion(self.x, self.y, self.damage, self.owner)

    def draw(self, surf):
        pygame.draw.circle(surf, (30, 30, 30), (int(self.x), int(self.y)), 8)
        # Fuse spark
        if self.timer % 8 < 4:
            pygame.draw.circle(surf, (255, 200, 50), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 3)
        pygame.draw.line(surf, (100, 100, 100), (int(self.x), int(self.y)), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 2)


class Explosion:
    def __init__(self, x, y, damage, owner):
        self.x = x
        self.y = y
        self.damage = damage
        self.owner = owner
        self.radius = 10
        self.max_radius = 110
        self.life = 20
        self.has_damaged = False

    def update(self, p1, p2):
        self.life -= 1
        if self.life > 10:
            self.radius += (self.max_radius - self.radius) * 0.4
        else:
            self.radius += (self.max_radius - self.radius) * 0.1
            
        if not self.has_damaged and self.life == 19:
            for target in (p1, p2):
                target_cx = target.x
                target_cy = target.y - target.height / 2
                dist = math.hypot(target_cx - self.x, target_cy - self.y)
                
                if dist < self.max_radius:
                    from_left = (self.x < target.x)
                    target.take_hit(self.damage, from_left)
                    # Extra explosion knockback
                    push = 20 if from_left else -20
                    target.x = max(target.width, min(WIDTH - target.width, target.x + push))
                    target.vel_y = -12  # pop up
            self.has_damaged = True

    def draw(self, surf):
        if self.life > 0:
            alpha = int(255 * (self.life / 20))
            surf_exp = pygame.Surface((self.max_radius * 2, self.max_radius * 2), pygame.SRCALPHA)
            pygame.draw.circle(surf_exp, (255, 80, 20, alpha), (self.max_radius, self.max_radius), int(self.radius))
            pygame.draw.circle(surf_exp, (255, 200, 50, int(alpha * 0.8)), (self.max_radius, self.max_radius), int(self.radius * 0.6))
            pygame.draw.circle(surf_exp, (255, 255, 220, int(alpha * 0.9)), (self.max_radius, self.max_radius), int(self.radius * 0.25))
            surf.blit(surf_exp, (int(self.x - self.max_radius), int(self.y - self.max_radius)))


class WeaponPickup:
    def __init__(self, x, y, wtype):
        self.x = x
        self.y = y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [(self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        elif self.wtype == "bomb":
            pygame.draw.circle(surf, (30, 30, 30), (self.x, self.y - 20), 10)
            pygame.draw.line(surf, (100, 100, 100), (self.x, self.y - 20), (self.x + 6, self.y - 30), 2)
            pygame.draw.circle(surf, (255, 200, 50), (self.x + 6, self.y - 30), 3)
            
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing
        self.controls = controls
        self.name = name
        self.health = MAX_HEALTH
        self.on_ground = True
        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0
        self.attack_type = None
        self.hit_stun = 0
        self.walk_cycle = 0
        self.moving = False
        self.wins = 0
        self.weapon = None

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height, self.width, self.height)

    def current_stats(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)
        elif self.weapon["type"] == "uzi":
            spawn_y += random.randint(-3, 3)
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))

    def throw_bomb(self, bombs):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 20
        spawn_y = self.y - self.height * 0.7
        bombs.append(Bomb(spawn_x, spawn_y, self.facing, info["damage"], self))
        self.weapon["durability"] -= 1
        if self.weapon["durability"] <= 0:
            self.weapon = None
            play_break_sound()

    def handle_input(self, keys, opponent, weapons, arrows, bombs):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.weapon and self.weapon["type"] == "bomb":
                self.attack_type = "throw"
                self.attack_anim = 15
                self.throw_bomb(bombs)
            elif self.is_ranged():
                self.attack_type = "shoot"
                self.attack_anim = 10
                self.shoot(arrows)
                self.weapon["durability"] -= 1
                if self.weapon["durability"] <= 0:
                    self.weapon = None
                    play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup(weapons)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y

        self.on_ground = False
        for plat in PLATFORMS:
            prev_y = self.y - self.vel_y
            if prev_y <= plat.top and self.y >= plat.top and plat.left <= self.x <= plat.right and self.vel_y >= 0:
                self.y = plat.top
                self.vel_y = 0
                self.on_ground = True
                break

        if not self.on_ground:
            if self.y >= HAZARD_Y:
                self.hazard_hit()

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        if self.x < WIDTH / 2:
            self.x = 325
        else:
            self.x = 675
        self.y = 380
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        elif self.attack_type == "throw":
            fwd_hand = (cx + self.facing * 20, shoulder_y - 10)
            back_hand = (cx - self.facing * 10, shoulder_y + 10)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        # Weapon drawing
        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "uzi":
            wcolor = WEAPON_TYPES["uzi"]["color"]
            ux, uy = fwd_hand[0], fwd_hand[1] - 3
            barrel_tip = (ux + self.facing * 27, uy)
            grip_bottom = (ux - self.facing * 5, uy + 18)
            pygame.draw.rect(surf, wcolor, pygame.Rect(min(ux, barrel_tip[0]), uy - 7, abs(barrel_tip[0] - ux) + 8, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (ux, uy + 4), grip_bottom, 6)
            pygame.draw.line(surf, (25, 25, 28), barrel_tip, (barrel_tip[0] + self.facing * 10, barrel_tip[1]), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 8, barrel_tip[1])
                pygame.draw.circle(surf, (255, 220, 90), (int(flash[0]), int(flash[1])), 6)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [tipend, (tip[0] + perp[0], tip[1] + perp[1]), (tip[0] - perp[0], tip[1] - perp[1])])
            else:  # sword
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK, (guard_center[0] + perp[0], guard_center[1] + perp[1]), (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        if self.weapon:
            label = font_tiny.render(f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}", True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, int(head_y) - head_r - 20))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]
    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    draw_hazard_pit(surf)

    for plat in PLATFORMS:
        pygame.draw.rect(surf, stage["ground"], plat)
        pygame.draw.rect(surf, stage["ground_edge"], (plat.left, plat.top, plat.width, 10))
        pygame.draw.polygon(surf, stage["ground"], [(plat.left, plat.bottom), (plat.left + 12, plat.bottom + 18), (plat.left + 24, plat.bottom)])
        pygame.draw.polygon(surf, stage["ground"], [(plat.right, plat.bottom), (plat.right - 12, plat.bottom + 18), (plat.right - 24, plat.bottom)])


def draw_hazard_pit(surf):
    pit_rect = pygame.Rect(0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)
    for i, (band_y, color) in enumerate([(GROUND_Y + 15, (150, 40, 15)), (GROUND_Y + 35, (200, 70, 20)), (GROUND_Y + 55, (240, 110, 30))]):
        for gx in range(0, WIDTH, 26):
            wobble = math.sin((gx + i * 40) * 0.15) * 4
            pygame.draw.circle(surf, color, (gx + 13, int(band_y + wobble)), 9)
    for gx in range(0, WIDTH, 52):
        pygame.draw.circle(surf, (255, 200, 90), (gx + 26, GROUND_Y + 60), 4)
    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [(gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)])
        pygame.draw.polygon(surf, (90, 90, 95), [(gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)
    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)
    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, y, wtype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    p1 = Fighter(325, RED, DARK_RED, 1, (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g), "Player 1")
    p1.y = 380
    p2 = Fighter(675, BLUE, DARK_BLUE, -1, (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l), "Player 2")
    p2.y = 380
    p1.wins = p1_wins
    p2.wins = p2_wins

    if p1_start_weapon:
        p1.weapon = {"type": p1_start_weapon, "durability": WEAPON_TYPES[p1_start_weapon]["durability"]}
    if p2_start_weapon:
        p2.weapon = {"type": p2_start_weapon, "durability": WEAPON_TYPES[p2_start_weapon]["durability"]}

    weapons, bombs, explosions, arrows = [], [], [], []
    spawn_weapon(weapons)
    return p1, p2, weapons, arrows, bombs, explosions


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [(cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "uzi":
        pygame.draw.rect(surf, (45, 45, 50), (cx - 22, cy - 8, 44, 16), border_radius=3)
        pygame.draw.line(surf, (25, 25, 28), (cx - 4, cy + 5), (cx - 10, cy + 25), 7)
        pygame.draw.line(surf, (25, 25, 28), (cx + 20, cy), (cx + 34, cy), 5)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)
    elif wtype == "bomb":
        pygame.draw.circle(surf, (40, 40, 40), (cx, cy), 14)
        pygame.draw.line(surf, (100, 100, 100), (cx, cy), (cx + 8, cy - 12), 3)
        pygame.draw.circle(surf, (255, 200, 50), (cx + 8, cy - 12), 4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220

    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))

        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))

    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE

    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))

    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)
    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))

    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))

    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))

    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)

    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render("Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    state = "select"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0

    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, arrows, bombs, explosions = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False

                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True

                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True

                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, arrows, bombs, explosions = reset_fighters(
                        p1_wins, p2_wins, WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, arrows, bombs)
            p2.handle_input(keys, p1, weapons, arrows, bombs)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            # Update bombs
            for bomb in bombs[:]:
                bomb.update(PLATFORMS)
                if bomb.exploded:
                    explosions.append(bomb.explode())
                    bombs.remove(bomb)

            # Update explosions
            for exp in explosions[:]:
                exp.update(p1, p2)
                if exp.life <= 0:
                    explosions.remove(exp)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL

            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)
        for bomb in bombs:
            bomb.draw(screen)
        for exp in explosions:
            exp.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))
            msg = "DRAW!" if winner == "Draw" else f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))
            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Watch the edges & bomb blast radius!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [19]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons, floating platforms, and bombs with devastating knockback!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

More weapons (uzi, sword, bat, spear, bow, gatling, bomb) also spawn randomly 
on the platforms during the match. Walk over one while unarmed to pick it up
automatically. Your "punch" button (F / K) becomes a weapon attack while
armed — a melee swing for sword/bat/spear, an arrow shot for the bow, a rapid 
stream of bullets for the gatling, or a thrown explosive for the bomb! 
Weapons have limited durability and are lost after a number of uses.

The arena consists of multiple floating platforms above a deadly lava/spike 
pit. Walk, get knocked, or fall off the edge and you'll take instant death — 
watch your footing! Bombs now launch players with devastating knockback!

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound (procedurally generated, no external asset files needed)
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None
explosion_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_explosion_sound():
        duration = 0.45
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 120 * (1 - progress * 0.8)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.7 * noise + 0.3 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
    explosion_sound = _generate_explosion_sound()
except pygame.error:
    break_sound = None
    hazard_sound = None
    explosion_sound = None


def play_break_sound():
    if break_sound:
        break_sound.play()

def play_hazard_sound():
    if hazard_sound:
        hazard_sound.play()

def play_explosion_sound():
    if explosion_sound:
        explosion_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

HAZARD_Y = GROUND_Y + 90
HAZARD_DAMAGE = 100
HAZARD_STUN = 24

PLATFORMS = [
    pygame.Rect(250, 380, 150, 20),
    pygame.Rect(600, 380, 150, 20),
    pygame.Rect(425, 260, 150, 20),
]

PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

WEAPON_TYPES = {
    "uzi":     {"reach": 0,   "damage": 5, "cooldown": 5, "durability": 50, "color": (45, 45, 50),
                "ranged": True, "proj_speed": 28, "proj_kind": "bullet"},
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow"},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet"},
    "bomb":    {"reach": 0,   "damage": 35, "cooldown": 50, "durability": 3,  "color": (40, 40, 40),
                "ranged": True, "proj_speed": 9, "proj_kind": "bomb"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

MAX_WEAPONS_ON_FIELD = 2
WEAPON_SPAWN_INTERVAL = 300

WEAPON_CHOICES = [None, "uzi", "sword", "bat", "spear", "bow", "gatling", "bomb"]

STAGES = [
    {"name": "Meadow", "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245), "ground": (110, 90, 70), "ground_edge": (80, 150, 80), "decor": "meadow"},
    {"name": "Desert", "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190), "ground": (200, 165, 100), "ground_edge": (225, 195, 130), "decor": "desert"},
    {"name": "Night City", "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90), "ground": (40, 40, 50), "ground_edge": (90, 90, 110), "decor": "city"},
    {"name": "Volcano", "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30), "ground": (50, 35, 30), "ground_edge": (200, 80, 30), "decor": "volcano"},
    {"name": "Snow Peak", "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245), "ground": (225, 235, 240), "ground_edge": (255, 255, 255), "decor": "snow"},
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return
        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class Bomb:
    def __init__(self, x, y, facing, damage, owner):
        self.x = x
        self.y = y
        self.vel_x = facing * 9
        self.vel_y = -11
        self.damage = damage
        self.owner = owner
        self.timer = 70
        self.dead = False
        self.exploded = False

    def update(self, platforms):
        if self.exploded:
            return
        self.vel_y += GRAVITY
        self.x += self.vel_x
        self.y += self.vel_y
        self.timer -= 1
        
        for plat in platforms:
            if plat.left <= self.x <= plat.right and self.y >= plat.top and self.vel_y >= 0:
                if self.y - self.vel_y <= plat.top + 10:
                    self.y = plat.top
                    self.vel_y = 0
                    self.vel_x *= 0.7
                    self.timer -= 3
        
        if self.y >= GROUND_Y or self.timer <= 0:
            self.exploded = True
            self.dead = True

    def explode(self):
        play_explosion_sound()
        return Explosion(self.x, self.y, self.damage, self.owner)

    def draw(self, surf):
        pygame.draw.circle(surf, (30, 30, 30), (int(self.x), int(self.y)), 8)
        if self.timer % 8 < 4:
            pygame.draw.circle(surf, (255, 200, 50), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 3)
        pygame.draw.line(surf, (100, 100, 100), (int(self.x), int(self.y)), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 2)


class Explosion:
    def __init__(self, x, y, damage, owner):
        self.x = x
        self.y = y
        self.damage = damage
        self.owner = owner
        self.radius = 10
        self.max_radius = 110
        self.life = 20
        self.has_damaged = False

    def update(self, p1, p2):
        self.life -= 1
        if self.life > 10:
            self.radius += (self.max_radius - self.radius) * 0.4
        else:
            self.radius += (self.max_radius - self.radius) * 0.1
            
        if not self.has_damaged and self.life == 19:
            for target in (p1, p2):
                target_cx = target.x
                target_cy = target.y - target.height / 2
                dist = math.hypot(target_cx - self.x, target_cy - self.y)
                
                if dist < self.max_radius:
                    from_left = (self.x < target.x)
                    target.take_hit(self.damage, from_left)
                    
                    # MASSIVE knockback - distance-based for extra punch
                    distance_factor = 1.0 - (dist / self.max_radius)  # closer = stronger
                    base_push = 120  # horizontal knockback
                    base_pop = -22   # vertical launch
                    
                    push = (base_push * distance_factor) if from_left else -(base_push * distance_factor)
                    target.x = max(target.width, min(WIDTH - target.width, target.x + push))
                    target.vel_y = base_pop * distance_factor
                    
            self.has_damaged = True

    def draw(self, surf):
        if self.life > 0:
            alpha = int(255 * (self.life / 20))
            surf_exp = pygame.Surface((self.max_radius * 2, self.max_radius * 2), pygame.SRCALPHA)
            pygame.draw.circle(surf_exp, (255, 80, 20, alpha), (self.max_radius, self.max_radius), int(self.radius))
            pygame.draw.circle(surf_exp, (255, 200, 50, int(alpha * 0.8)), (self.max_radius, self.max_radius), int(self.radius * 0.6))
            pygame.draw.circle(surf_exp, (255, 255, 220, int(alpha * 0.9)), (self.max_radius, self.max_radius), int(self.radius * 0.25))
            surf.blit(surf_exp, (int(self.x - self.max_radius), int(self.y - self.max_radius)))


class WeaponPickup:
    def __init__(self, x, y, wtype):
        self.x = x
        self.y = y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [(self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        elif self.wtype == "bomb":
            pygame.draw.circle(surf, (30, 30, 30), (self.x, self.y - 20), 10)
            pygame.draw.line(surf, (100, 100, 100), (self.x, self.y - 20), (self.x + 6, self.y - 30), 2)
            pygame.draw.circle(surf, (255, 200, 50), (self.x + 6, self.y - 30), 3)
            
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing
        self.controls = controls
        self.name = name
        self.health = MAX_HEALTH
        self.on_ground = True
        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0
        self.attack_type = None
        self.hit_stun = 0
        self.walk_cycle = 0
        self.moving = False
        self.wins = 0
        self.weapon = None

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height, self.width, self.height)

    def current_stats(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)
        elif self.weapon["type"] == "uzi":
            spawn_y += random.randint(-3, 3)
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))

    def throw_bomb(self, bombs):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 20
        spawn_y = self.y - self.height * 0.7
        bombs.append(Bomb(spawn_x, spawn_y, self.facing, info["damage"], self))
        self.weapon["durability"] -= 1
        if self.weapon["durability"] <= 0:
            self.weapon = None
            play_break_sound()

    def handle_input(self, keys, opponent, weapons, arrows, bombs):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.weapon and self.weapon["type"] == "bomb":
                self.attack_type = "throw"
                self.attack_anim = 15
                self.throw_bomb(bombs)
            elif self.is_ranged():
                self.attack_type = "shoot"
                self.attack_anim = 10
                self.shoot(arrows)
                self.weapon["durability"] -= 1
                if self.weapon["durability"] <= 0:
                    self.weapon = None
                    play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup(weapons)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y

        self.on_ground = False
        for plat in PLATFORMS:
            prev_y = self.y - self.vel_y
            if prev_y <= plat.top and self.y >= plat.top and plat.left <= self.x <= plat.right and self.vel_y >= 0:
                self.y = plat.top
                self.vel_y = 0
                self.on_ground = True
                break

        if not self.on_ground:
            if self.y >= HAZARD_Y:
                self.hazard_hit()

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        if self.x < WIDTH / 2:
            self.x = 325
        else:
            self.x = 675
        self.y = 380
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        elif self.attack_type == "throw":
            fwd_hand = (cx + self.facing * 20, shoulder_y - 10)
            back_hand = (cx - self.facing * 10, shoulder_y + 10)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "uzi":
            wcolor = WEAPON_TYPES["uzi"]["color"]
            ux, uy = fwd_hand[0], fwd_hand[1] - 3
            barrel_tip = (ux + self.facing * 27, uy)
            grip_bottom = (ux - self.facing * 5, uy + 18)
            pygame.draw.rect(surf, wcolor, pygame.Rect(min(ux, barrel_tip[0]), uy - 7, abs(barrel_tip[0] - ux) + 8, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (ux, uy + 4), grip_bottom, 6)
            pygame.draw.line(surf, (25, 25, 28), barrel_tip, (barrel_tip[0] + self.facing * 10, barrel_tip[1]), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 8, barrel_tip[1])
                pygame.draw.circle(surf, (255, 220, 90), (int(flash[0]), int(flash[1])), 6)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [tipend, (tip[0] + perp[0], tip[1] + perp[1]), (tip[0] - perp[0], tip[1] - perp[1])])
            else:
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK, (guard_center[0] + perp[0], guard_center[1] + perp[1]), (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        if self.weapon:
            label = font_tiny.render(f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}", True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, int(head_y) - head_r - 20))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]
    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    draw_hazard_pit(surf)

    for plat in PLATFORMS:
        pygame.draw.rect(surf, stage["ground"], plat)
        pygame.draw.rect(surf, stage["ground_edge"], (plat.left, plat.top, plat.width, 10))
        pygame.draw.polygon(surf, stage["ground"], [(plat.left, plat.bottom), (plat.left + 12, plat.bottom + 18), (plat.left + 24, plat.bottom)])
        pygame.draw.polygon(surf, stage["ground"], [(plat.right, plat.bottom), (plat.right - 12, plat.bottom + 18), (plat.right - 24, plat.bottom)])


def draw_hazard_pit(surf):
    pit_rect = pygame.Rect(0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)
    for i, (band_y, color) in enumerate([(GROUND_Y + 15, (150, 40, 15)), (GROUND_Y + 35, (200, 70, 20)), (GROUND_Y + 55, (240, 110, 30))]):
        for gx in range(0, WIDTH, 26):
            wobble = math.sin((gx + i * 40) * 0.15) * 4
            pygame.draw.circle(surf, color, (gx + 13, int(band_y + wobble)), 9)
    for gx in range(0, WIDTH, 52):
        pygame.draw.circle(surf, (255, 200, 90), (gx + 26, GROUND_Y + 60), 4)
    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [(gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)])
        pygame.draw.polygon(surf, (90, 90, 95), [(gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)
    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)
    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, y, wtype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    p1 = Fighter(325, RED, DARK_RED, 1, (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g), "Player 1")
    p1.y = 380
    p2 = Fighter(675, BLUE, DARK_BLUE, -1, (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l), "Player 2")
    p2.y = 380
    p1.wins = p1_wins
    p2.wins = p2_wins

    if p1_start_weapon:
        p1.weapon = {"type": p1_start_weapon, "durability": WEAPON_TYPES[p1_start_weapon]["durability"]}
    if p2_start_weapon:
        p2.weapon = {"type": p2_start_weapon, "durability": WEAPON_TYPES[p2_start_weapon]["durability"]}

    weapons, bombs, explosions, arrows = [], [], [], []
    spawn_weapon(weapons)
    return p1, p2, weapons, arrows, bombs, explosions


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [(cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "uzi":
        pygame.draw.rect(surf, (45, 45, 50), (cx - 22, cy - 8, 44, 16), border_radius=3)
        pygame.draw.line(surf, (25, 25, 28), (cx - 4, cy + 5), (cx - 10, cy + 25), 7)
        pygame.draw.line(surf, (25, 25, 28), (cx + 20, cy), (cx + 34, cy), 5)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)
    elif wtype == "bomb":
        pygame.draw.circle(surf, (40, 40, 40), (cx, cy), 14)
        pygame.draw.line(surf, (100, 100, 100), (cx, cy), (cx + 8, cy - 12), 3)
        pygame.draw.circle(surf, (255, 200, 50), (cx + 8, cy - 12), 4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220

    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))

        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))

    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE

    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))

    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)
    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))

    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))

    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))

    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)

    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render("Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    state = "select"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0

    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, arrows, bombs, explosions = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False

                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True

                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True

                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, arrows, bombs, explosions = reset_fighters(
                        p1_wins, p2_wins, WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, arrows, bombs)
            p2.handle_input(keys, p1, weapons, arrows, bombs)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            for bomb in bombs[:]:
                bomb.update(PLATFORMS)
                if bomb.exploded:
                    explosions.append(bomb.explode())
                    bombs.remove(bomb)

            for exp in explosions[:]:
                exp.update(p1, p2)
                if exp.life <= 0:
                    explosions.remove(exp)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL

            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)
        for bomb in bombs:
            bomb.draw(screen)
        for exp in explosions:
            exp.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))
            msg = "DRAW!" if winner == "Draw" else f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))
            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Bombs launch you FAR — watch the edges!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

KeyboardInterrupt: 

In [20]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons, 
floating platforms, and bombs that DESTROY platforms!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

More weapons (uzi, sword, bat, spear, bow, gatling, bomb) also spawn randomly 
on the platforms during the match. Walk over one while unarmed to pick it up
automatically. Your "punch" button (F / K) becomes a weapon attack while
armed — a melee swing for sword/bat/spear, an arrow shot for the bow, a rapid 
stream of bullets for the gatling, or a thrown explosive for the bomb! 
Weapons have limited durability and are lost after a number of uses.

The arena consists of multiple floating platforms above a deadly lava/spike 
pit. BOMB EXPLOSIONS NOW DESTROY PLATFORMS — shatter your opponent's footing 
and send them plummeting into the hazard! The arena gets more dangerous 
the more bombs go off.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound (procedurally generated, no external asset files needed)
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None
explosion_sound = None
crumble_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_explosion_sound():
        duration = 0.45
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 120 * (1 - progress * 0.8)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.7 * noise + 0.3 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_crumble_sound():
        duration = 0.6
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.2
            freq = 180 * (1 - progress * 0.5) + 40 * math.sin(2 * math.pi * 8 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.85 * noise + 0.15 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
    explosion_sound = _generate_explosion_sound()
    crumble_sound = _generate_crumble_sound()
except pygame.error:
    break_sound = None
    hazard_sound = None
    explosion_sound = None
    crumble_sound = None


def play_break_sound():
    if break_sound:
        break_sound.play()

def play_hazard_sound():
    if hazard_sound:
        hazard_sound.play()

def play_explosion_sound():
    if explosion_sound:
        explosion_sound.play()

def play_crumble_sound():
    if crumble_sound:
        crumble_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

HAZARD_Y = GROUND_Y + 90
HAZARD_DAMAGE = 100
HAZARD_STUN = 24

# Multiple floating platforms for vertical gameplay.
# This list is MUTABLE — bombs can remove platforms from it during gameplay.
def _initial_platforms():
    return [
        pygame.Rect(250, 380, 150, 20),   # Left mid-air platform
        pygame.Rect(600, 380, 150, 20),   # Right mid-air platform
        pygame.Rect(425, 260, 150, 20),   # Center high platform
    ]

PLATFORMS = _initial_platforms()

PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

WEAPON_TYPES = {
    "uzi":     {"reach": 0,   "damage": 5, "cooldown": 5, "durability": 50, "color": (45, 45, 50),
                "ranged": True, "proj_speed": 28, "proj_kind": "bullet"},
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow"},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet"},
    "bomb":    {"reach": 0,   "damage": 35, "cooldown": 50, "durability": 3,  "color": (40, 40, 40),
                "ranged": True, "proj_speed": 9, "proj_kind": "bomb"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

MAX_WEAPONS_ON_FIELD = 2
WEAPON_SPAWN_INTERVAL = 300

WEAPON_CHOICES = [None, "uzi", "sword", "bat", "spear", "bow", "gatling", "bomb"]

STAGES = [
    {"name": "Meadow", "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245), "ground": (110, 90, 70), "ground_edge": (80, 150, 80), "decor": "meadow"},
    {"name": "Desert", "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190), "ground": (200, 165, 100), "ground_edge": (225, 195, 130), "decor": "desert"},
    {"name": "Night City", "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90), "ground": (40, 40, 50), "ground_edge": (90, 90, 110), "decor": "city"},
    {"name": "Volcano", "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30), "ground": (50, 35, 30), "ground_edge": (200, 80, 30), "decor": "volcano"},
    {"name": "Snow Peak", "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245), "ground": (225, 235, 240), "ground_edge": (255, 255, 255), "decor": "snow"},
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return
        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class Bomb:
    def __init__(self, x, y, facing, damage, owner):
        self.x = x
        self.y = y
        self.vel_x = facing * 9
        self.vel_y = -11
        self.damage = damage
        self.owner = owner
        self.timer = 70
        self.dead = False
        self.exploded = False

    def update(self, platforms):
        if self.exploded:
            return
        self.vel_y += GRAVITY
        self.x += self.vel_x
        self.y += self.vel_y
        self.timer -= 1
        
        for plat in platforms:
            if plat.left <= self.x <= plat.right and self.y >= plat.top and self.vel_y >= 0:
                if self.y - self.vel_y <= plat.top + 10:
                    self.y = plat.top
                    self.vel_y = 0
                    self.vel_x *= 0.7
                    self.timer -= 3
        
        if self.y >= GROUND_Y or self.timer <= 0:
            self.exploded = True
            self.dead = True

    def explode(self):
        play_explosion_sound()
        return Explosion(self.x, self.y, self.damage, self.owner)

    def draw(self, surf):
        pygame.draw.circle(surf, (30, 30, 30), (int(self.x), int(self.y)), 8)
        if self.timer % 8 < 4:
            pygame.draw.circle(surf, (255, 200, 50), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 3)
        pygame.draw.line(surf, (100, 100, 100), (int(self.x), int(self.y)), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 2)


class Debris:
    """A chunk of a destroyed platform flying through the air."""
    def __init__(self, x, y, color):
        self.x = x
        self.y = y
        self.vel_x = random.uniform(-8, 8)
        self.vel_y = random.uniform(-10, -2)
        self.size = random.randint(4, 10)
        self.color = color
        self.rotation = random.uniform(0, 360)
        self.rot_speed = random.uniform(-15, 15)
        self.life = random.randint(40, 70)

    def update(self):
        self.vel_y += GRAVITY * 0.8
        self.x += self.vel_x
        self.y += self.vel_y
        self.rotation += self.rot_speed
        self.life -= 1
        self.vel_x *= 0.98

    def draw(self, surf):
        if self.life <= 0:
            return
        alpha = min(255, self.life * 6)
        surf_deb = pygame.Surface((self.size * 2, self.size * 2), pygame.SRCALPHA)
        # Draw a rotated square
        rect = pygame.Rect(0, 0, self.size, self.size)
        pygame.draw.rect(surf_deb, (*self.color, alpha), rect)
        pygame.draw.rect(surf_deb, (0, 0, 0, alpha), rect, 1)
        rotated = pygame.transform.rotate(surf_deb, self.rotation)
        new_rect = rotated.get_rect(center=(int(self.x), int(self.y)))
        surf.blit(rotated, new_rect)


class Explosion:
    def __init__(self, x, y, damage, owner):
        self.x = x
        self.y = y
        self.damage = damage
        self.owner = owner
        self.radius = 10
        self.max_radius = 110
        self.life = 20
        self.has_damaged = False
        self.has_destroyed_platforms = False
        self.destroyed_color = None  # color of the platform being destroyed

    def update(self, p1, p2, debris_list):
        self.life -= 1
        if self.life > 10:
            self.radius += (self.max_radius - self.radius) * 0.4
        else:
            self.radius += (self.max_radius - self.radius) * 0.1
            
        if not self.has_damaged and self.life == 19:
            # Damage players
            for target in (p1, p2):
                target_cx = target.x
                target_cy = target.y - target.height / 2
                dist = math.hypot(target_cx - self.x, target_cy - self.y)
                
                if dist < self.max_radius:
                    from_left = (self.x < target.x)
                    target.take_hit(self.damage, from_left)
                    distance_factor = 1.0 - (dist / self.max_radius)
                    base_push = 120
                    base_pop = -22
                    push = (base_push * distance_factor) if from_left else -(base_push * distance_factor)
                    target.x = max(target.width, min(WIDTH - target.width, target.x + push))
                    target.vel_y = base_pop * distance_factor
            self.has_damaged = True

        # Destroy platforms within blast radius (only once, on frame 17)
        if not self.has_destroyed_platforms and self.life == 17:
            global PLATFORMS
            platforms_to_destroy = []
            for plat in PLATFORMS:
                # Find closest point on platform rect to explosion center
                closest_x = max(plat.left, min(self.x, plat.right))
                closest_y = max(plat.top, min(self.y, plat.bottom))
                dist = math.hypot(closest_x - self.x, closest_y - self.y)
                if dist < self.max_radius:
                    platforms_to_destroy.append(plat)
            
            # Don't destroy the last platform — keep the game playable
            if len(PLATFORMS) - len(platforms_to_destroy) >= 1:
                for plat in platforms_to_destroy:
                    # Spawn debris chunks from the destroyed platform
                    plat_color = (110, 90, 70)  # default dirt color
                    # Try to match the current stage's ground color if possible
                    for chunk_i in range(18):
                        chunk_x = random.uniform(plat.left, plat.right)
                        chunk_y = random.uniform(plat.top, plat.bottom)
                        debris_list.append(Debris(chunk_x, chunk_y, plat_color))
                    PLATFORMS.remove(plat)
                    play_crumble_sound()
            self.has_destroyed_platforms = True

    def draw(self, surf):
        if self.life > 0:
            alpha = int(255 * (self.life / 20))
            surf_exp = pygame.Surface((self.max_radius * 2, self.max_radius * 2), pygame.SRCALPHA)
            pygame.draw.circle(surf_exp, (255, 80, 20, alpha), (self.max_radius, self.max_radius), int(self.radius))
            pygame.draw.circle(surf_exp, (255, 200, 50, int(alpha * 0.8)), (self.max_radius, self.max_radius), int(self.radius * 0.6))
            pygame.draw.circle(surf_exp, (255, 255, 220, int(alpha * 0.9)), (self.max_radius, self.max_radius), int(self.radius * 0.25))
            surf.blit(surf_exp, (int(self.x - self.max_radius), int(self.y - self.max_radius)))


class WeaponPickup:
    def __init__(self, x, y, wtype):
        self.x = x
        self.y = y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [(self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        elif self.wtype == "bomb":
            pygame.draw.circle(surf, (30, 30, 30), (self.x, self.y - 20), 10)
            pygame.draw.line(surf, (100, 100, 100), (self.x, self.y - 20), (self.x + 6, self.y - 30), 2)
            pygame.draw.circle(surf, (255, 200, 50), (self.x + 6, self.y - 30), 3)
            
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing
        self.controls = controls
        self.name = name
        self.health = MAX_HEALTH
        self.on_ground = True
        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0
        self.attack_type = None
        self.hit_stun = 0
        self.walk_cycle = 0
        self.moving = False
        self.wins = 0
        self.weapon = None

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height, self.width, self.height)

    def current_stats(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)
        elif self.weapon["type"] == "uzi":
            spawn_y += random.randint(-3, 3)
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))

    def throw_bomb(self, bombs):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 20
        spawn_y = self.y - self.height * 0.7
        bombs.append(Bomb(spawn_x, spawn_y, self.facing, info["damage"], self))
        self.weapon["durability"] -= 1
        if self.weapon["durability"] <= 0:
            self.weapon = None
            play_break_sound()

    def handle_input(self, keys, opponent, weapons, arrows, bombs):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.weapon and self.weapon["type"] == "bomb":
                self.attack_type = "throw"
                self.attack_anim = 15
                self.throw_bomb(bombs)
            elif self.is_ranged():
                self.attack_type = "shoot"
                self.attack_anim = 10
                self.shoot(arrows)
                self.weapon["durability"] -= 1
                if self.weapon["durability"] <= 0:
                    self.weapon = None
                    play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup(weapons)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y

        self.on_ground = False
        for plat in PLATFORMS:
            prev_y = self.y - self.vel_y
            if prev_y <= plat.top and self.y >= plat.top and plat.left <= self.x <= plat.right and self.vel_y >= 0:
                self.y = plat.top
                self.vel_y = 0
                self.on_ground = True
                break

        if not self.on_ground:
            if self.y >= HAZARD_Y:
                self.hazard_hit()

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        # Respawn on a surviving platform, or the hazard edge if none left
        if PLATFORMS:
            # Pick the platform closest to the player's x
            best = min(PLATFORMS, key=lambda p: abs(p.centerx - self.x))
            self.x = best.centerx
            self.y = best.top
        else:
            if self.x < WIDTH / 2:
                self.x = 325
            else:
                self.x = 675
            self.y = 380
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        elif self.attack_type == "throw":
            fwd_hand = (cx + self.facing * 20, shoulder_y - 10)
            back_hand = (cx - self.facing * 10, shoulder_y + 10)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "uzi":
            wcolor = WEAPON_TYPES["uzi"]["color"]
            ux, uy = fwd_hand[0], fwd_hand[1] - 3
            barrel_tip = (ux + self.facing * 27, uy)
            grip_bottom = (ux - self.facing * 5, uy + 18)
            pygame.draw.rect(surf, wcolor, pygame.Rect(min(ux, barrel_tip[0]), uy - 7, abs(barrel_tip[0] - ux) + 8, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (ux, uy + 4), grip_bottom, 6)
            pygame.draw.line(surf, (25, 25, 28), barrel_tip, (barrel_tip[0] + self.facing * 10, barrel_tip[1]), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 8, barrel_tip[1])
                pygame.draw.circle(surf, (255, 220, 90), (int(flash[0]), int(flash[1])), 6)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [tipend, (tip[0] + perp[0], tip[1] + perp[1]), (tip[0] - perp[0], tip[1] - perp[1])])
            else:
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK, (guard_center[0] + perp[0], guard_center[1] + perp[1]), (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        if self.weapon:
            label = font_tiny.render(f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}", True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, int(head_y) - head_r - 20))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]
    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    draw_hazard_pit(surf)

    # Draw current (possibly destroyed) platforms
    for plat in PLATFORMS:
        pygame.draw.rect(surf, stage["ground"], plat)
        pygame.draw.rect(surf, stage["ground_edge"], (plat.left, plat.top, plat.width, 10))
        pygame.draw.polygon(surf, stage["ground"], [(plat.left, plat.bottom), (plat.left + 12, plat.bottom + 18), (plat.left + 24, plat.bottom)])
        pygame.draw.polygon(surf, stage["ground"], [(plat.right, plat.bottom), (plat.right - 12, plat.bottom + 18), (plat.right - 24, plat.bottom)])


def draw_hazard_pit(surf):
    pit_rect = pygame.Rect(0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)
    for i, (band_y, color) in enumerate([(GROUND_Y + 15, (150, 40, 15)), (GROUND_Y + 35, (200, 70, 20)), (GROUND_Y + 55, (240, 110, 30))]):
        for gx in range(0, WIDTH, 26):
            wobble = math.sin((gx + i * 40) * 0.15) * 4
            pygame.draw.circle(surf, color, (gx + 13, int(band_y + wobble)), 9)
    for gx in range(0, WIDTH, 52):
        pygame.draw.circle(surf, (255, 200, 90), (gx + 26, GROUND_Y + 60), 4)
    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [(gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)])
        pygame.draw.polygon(surf, (90, 90, 95), [(gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)
    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)
    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, y, wtype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    global PLATFORMS
    PLATFORMS = _initial_platforms()  # reset platforms for new match
    
    p1 = Fighter(325, RED, DARK_RED, 1, (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g), "Player 1")
    p1.y = 380
    p2 = Fighter(675, BLUE, DARK_BLUE, -1, (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l), "Player 2")
    p2.y = 380
    p1.wins = p1_wins
    p2.wins = p2_wins

    if p1_start_weapon:
        p1.weapon = {"type": p1_start_weapon, "durability": WEAPON_TYPES[p1_start_weapon]["durability"]}
    if p2_start_weapon:
        p2.weapon = {"type": p2_start_weapon, "durability": WEAPON_TYPES[p2_start_weapon]["durability"]}

    weapons, bombs, explosions, arrows, debris = [], [], [], [], []
    spawn_weapon(weapons)
    return p1, p2, weapons, arrows, bombs, explosions, debris


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [(cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "uzi":
        pygame.draw.rect(surf, (45, 45, 50), (cx - 22, cy - 8, 44, 16), border_radius=3)
        pygame.draw.line(surf, (25, 25, 28), (cx - 4, cy + 5), (cx - 10, cy + 25), 7)
        pygame.draw.line(surf, (25, 25, 28), (cx + 20, cy), (cx + 34, cy), 5)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)
    elif wtype == "bomb":
        pygame.draw.circle(surf, (40, 40, 40), (cx, cy), 14)
        pygame.draw.line(surf, (100, 100, 100), (cx, cy), (cx + 8, cy - 12), 3)
        pygame.draw.circle(surf, (255, 200, 50), (cx + 8, cy - 12), 4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220

    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))

        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))

    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE

    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))

    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)
    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))

    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))

    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))

    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)

    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render("Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    state = "select"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0

    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, arrows, bombs, explosions, debris = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False

                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True

                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True

                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, arrows, bombs, explosions, debris = reset_fighters(
                        p1_wins, p2_wins, WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, arrows, bombs)
            p2.handle_input(keys, p1, weapons, arrows, bombs)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            for bomb in bombs[:]:
                bomb.update(PLATFORMS)
                if bomb.exploded:
                    explosions.append(bomb.explode())
                    bombs.remove(bomb)

            for exp in explosions[:]:
                exp.update(p1, p2, debris)
                if exp.life <= 0:
                    explosions.remove(exp)

            # Update debris particles
            for d in debris[:]:
                d.update()
                if d.life <= 0:
                    debris.remove(d)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL

            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)
        for bomb in bombs:
            bomb.draw(screen)
        for exp in explosions:
            exp.draw(screen)
        for d in debris:
            d.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        # Platform count indicator (shows how many platforms remain)
        plat_count = font_tiny.render(f"Platforms: {len(PLATFORMS)}", True, WHITE)
        screen.blit(plat_count, (WIDTH // 2 - plat_count.get_width() // 2, 50))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))
            msg = "DRAW!" if winner == "Draw" else f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))
            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Bombs DESTROY platforms!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [21]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons, 
floating platforms, and bombs that DESTROY platforms!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

More weapons (uzi, sword, bat, spear, bow, gatling, bomb) also spawn randomly 
on the platforms during the match. Walk over one while unarmed to pick it up
automatically. Your "punch" button (F / K) becomes a weapon attack while
armed — a melee swing for sword/bat/spear, an arrow shot for the bow, a rapid 
stream of bullets for the gatling, or a thrown explosive for the bomb! 
Weapons have limited durability and are lost after a number of uses.

The arena consists of MULTIPLE floating platforms at various heights above a 
deadly lava/spike pit. BOMB EXPLOSIONS DESTROY PLATFORMS — shatter your 
opponent's footing and send them plummeting into the hazard! The arena gets 
more dangerous the more bombs go off.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound (procedurally generated, no external asset files needed)
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None
explosion_sound = None
crumble_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_explosion_sound():
        duration = 0.45
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 120 * (1 - progress * 0.8)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.7 * noise + 0.3 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_crumble_sound():
        duration = 0.6
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.2
            freq = 180 * (1 - progress * 0.5) + 40 * math.sin(2 * math.pi * 8 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.85 * noise + 0.15 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
    explosion_sound = _generate_explosion_sound()
    crumble_sound = _generate_crumble_sound()
except pygame.error:
    break_sound = None
    hazard_sound = None
    explosion_sound = None
    crumble_sound = None


def play_break_sound():
    if break_sound:
        break_sound.play()

def play_hazard_sound():
    if hazard_sound:
        hazard_sound.play()

def play_explosion_sound():
    if explosion_sound:
        explosion_sound.play()

def play_crumble_sound():
    if crumble_sound:
        crumble_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

HAZARD_Y = GROUND_Y + 90
HAZARD_DAMAGE = 100
HAZARD_STUN = 24

# Multiple floating platforms for vertical gameplay.
# This list is MUTABLE — bombs can remove platforms from it during gameplay.
def _initial_platforms():
    return [
        # Low platforms (y=420-450)
        pygame.Rect(150, 450, 120, 18),   # Far left low
        pygame.Rect(730, 450, 120, 18),   # Far right low
        
        # Mid platforms (y=340-400)
        pygame.Rect(280, 400, 140, 20),   # Left mid
        pygame.Rect(580, 400, 140, 20),   # Right mid
        pygame.Rect(430, 360, 140, 20),   # Center mid (slightly higher)
        
        # High platforms (y=260-320)
        pygame.Rect(200, 320, 130, 18),   # Left high
        pygame.Rect(670, 320, 130, 18),   # Right high
        pygame.Rect(440, 280, 120, 18),   # Center high
        
        # Very high platforms (y=180-220)
        pygame.Rect(320, 220, 110, 16),   # Left very high
        pygame.Rect(570, 220, 110, 16),   # Right very high
        pygame.Rect(450, 180, 100, 16),   # Top center
    ]

PLATFORMS = _initial_platforms()

PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

WEAPON_TYPES = {
    "uzi":     {"reach": 0,   "damage": 5, "cooldown": 5, "durability": 50, "color": (45, 45, 50),
                "ranged": True, "proj_speed": 28, "proj_kind": "bullet"},
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow"},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet"},
    "bomb":    {"reach": 0,   "damage": 35, "cooldown": 50, "durability": 3,  "color": (40, 40, 40),
                "ranged": True, "proj_speed": 9, "proj_kind": "bomb"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

MAX_WEAPONS_ON_FIELD = 2
WEAPON_SPAWN_INTERVAL = 300

WEAPON_CHOICES = [None, "uzi", "sword", "bat", "spear", "bow", "gatling", "bomb"]

STAGES = [
    {"name": "Meadow", "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245), "ground": (110, 90, 70), "ground_edge": (80, 150, 80), "decor": "meadow"},
    {"name": "Desert", "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190), "ground": (200, 165, 100), "ground_edge": (225, 195, 130), "decor": "desert"},
    {"name": "Night City", "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90), "ground": (40, 40, 50), "ground_edge": (90, 90, 110), "decor": "city"},
    {"name": "Volcano", "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30), "ground": (50, 35, 30), "ground_edge": (200, 80, 30), "decor": "volcano"},
    {"name": "Snow Peak", "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245), "ground": (225, 235, 240), "ground_edge": (255, 255, 255), "decor": "snow"},
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return
        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class Bomb:
    def __init__(self, x, y, facing, damage, owner):
        self.x = x
        self.y = y
        self.vel_x = facing * 9
        self.vel_y = -11
        self.damage = damage
        self.owner = owner
        self.timer = 70
        self.dead = False
        self.exploded = False

    def update(self, platforms):
        if self.exploded:
            return
        self.vel_y += GRAVITY
        self.x += self.vel_x
        self.y += self.vel_y
        self.timer -= 1
        
        for plat in platforms:
            if plat.left <= self.x <= plat.right and self.y >= plat.top and self.vel_y >= 0:
                if self.y - self.vel_y <= plat.top + 10:
                    self.y = plat.top
                    self.vel_y = 0
                    self.vel_x *= 0.7
                    self.timer -= 3
        
        if self.y >= GROUND_Y or self.timer <= 0:
            self.exploded = True
            self.dead = True

    def explode(self):
        play_explosion_sound()
        return Explosion(self.x, self.y, self.damage, self.owner)

    def draw(self, surf):
        pygame.draw.circle(surf, (30, 30, 30), (int(self.x), int(self.y)), 8)
        if self.timer % 8 < 4:
            pygame.draw.circle(surf, (255, 200, 50), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 3)
        pygame.draw.line(surf, (100, 100, 100), (int(self.x), int(self.y)), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 2)


class Debris:
    """A chunk of a destroyed platform flying through the air."""
    def __init__(self, x, y, color):
        self.x = x
        self.y = y
        self.vel_x = random.uniform(-8, 8)
        self.vel_y = random.uniform(-10, -2)
        self.size = random.randint(4, 10)
        self.color = color
        self.rotation = random.uniform(0, 360)
        self.rot_speed = random.uniform(-15, 15)
        self.life = random.randint(40, 70)

    def update(self):
        self.vel_y += GRAVITY * 0.8
        self.x += self.vel_x
        self.y += self.vel_y
        self.rotation += self.rot_speed
        self.life -= 1
        self.vel_x *= 0.98

    def draw(self, surf):
        if self.life <= 0:
            return
        alpha = min(255, self.life * 6)
        surf_deb = pygame.Surface((self.size * 2, self.size * 2), pygame.SRCALPHA)
        # Draw a rotated square
        rect = pygame.Rect(0, 0, self.size, self.size)
        pygame.draw.rect(surf_deb, (*self.color, alpha), rect)
        pygame.draw.rect(surf_deb, (0, 0, 0, alpha), rect, 1)
        rotated = pygame.transform.rotate(surf_deb, self.rotation)
        new_rect = rotated.get_rect(center=(int(self.x), int(self.y)))
        surf.blit(rotated, new_rect)


class Explosion:
    def __init__(self, x, y, damage, owner):
        self.x = x
        self.y = y
        self.damage = damage
        self.owner = owner
        self.radius = 10
        self.max_radius = 110
        self.life = 20
        self.has_damaged = False
        self.has_destroyed_platforms = False
        self.destroyed_color = None

    def update(self, p1, p2, debris_list):
        self.life -= 1
        if self.life > 10:
            self.radius += (self.max_radius - self.radius) * 0.4
        else:
            self.radius += (self.max_radius - self.radius) * 0.1
            
        if not self.has_damaged and self.life == 19:
            # Damage players
            for target in (p1, p2):
                target_cx = target.x
                target_cy = target.y - target.height / 2
                dist = math.hypot(target_cx - self.x, target_cy - self.y)
                
                if dist < self.max_radius:
                    from_left = (self.x < target.x)
                    target.take_hit(self.damage, from_left)
                    distance_factor = 1.0 - (dist / self.max_radius)
                    base_push = 120
                    base_pop = -22
                    push = (base_push * distance_factor) if from_left else -(base_push * distance_factor)
                    target.x = max(target.width, min(WIDTH - target.width, target.x + push))
                    target.vel_y = base_pop * distance_factor
            self.has_damaged = True

        # Destroy platforms within blast radius (only once, on frame 17)
        if not self.has_destroyed_platforms and self.life == 17:
            global PLATFORMS
            platforms_to_destroy = []
            for plat in PLATFORMS:
                # Find closest point on platform rect to explosion center
                closest_x = max(plat.left, min(self.x, plat.right))
                closest_y = max(plat.top, min(self.y, plat.bottom))
                dist = math.hypot(closest_x - self.x, closest_y - self.y)
                if dist < self.max_radius:
                    platforms_to_destroy.append(plat)
            
            # Don't destroy the last platform — keep the game playable
            if len(PLATFORMS) - len(platforms_to_destroy) >= 1:
                for plat in platforms_to_destroy:
                    # Spawn debris chunks from the destroyed platform
                    plat_color = (110, 90, 70)  # default dirt color
                    for chunk_i in range(18):
                        chunk_x = random.uniform(plat.left, plat.right)
                        chunk_y = random.uniform(plat.top, plat.bottom)
                        debris_list.append(Debris(chunk_x, chunk_y, plat_color))
                    PLATFORMS.remove(plat)
                    play_crumble_sound()
            self.has_destroyed_platforms = True

    def draw(self, surf):
        if self.life > 0:
            alpha = int(255 * (self.life / 20))
            surf_exp = pygame.Surface((self.max_radius * 2, self.max_radius * 2), pygame.SRCALPHA)
            pygame.draw.circle(surf_exp, (255, 80, 20, alpha), (self.max_radius, self.max_radius), int(self.radius))
            pygame.draw.circle(surf_exp, (255, 200, 50, int(alpha * 0.8)), (self.max_radius, self.max_radius), int(self.radius * 0.6))
            pygame.draw.circle(surf_exp, (255, 255, 220, int(alpha * 0.9)), (self.max_radius, self.max_radius), int(self.radius * 0.25))
            surf.blit(surf_exp, (int(self.x - self.max_radius), int(self.y - self.max_radius)))


class WeaponPickup:
    def __init__(self, x, y, wtype):
        self.x = x
        self.y = y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [(self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        elif self.wtype == "bomb":
            pygame.draw.circle(surf, (30, 30, 30), (self.x, self.y - 20), 10)
            pygame.draw.line(surf, (100, 100, 100), (self.x, self.y - 20), (self.x + 6, self.y - 30), 2)
            pygame.draw.circle(surf, (255, 200, 50), (self.x + 6, self.y - 30), 3)
            
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing
        self.controls = controls
        self.name = name
        self.health = MAX_HEALTH
        self.on_ground = True
        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0
        self.attack_type = None
        self.hit_stun = 0
        self.walk_cycle = 0
        self.moving = False
        self.wins = 0
        self.weapon = None

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height, self.width, self.height)

    def current_stats(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)
        elif self.weapon["type"] == "uzi":
            spawn_y += random.randint(-3, 3)
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))

    def throw_bomb(self, bombs):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 20
        spawn_y = self.y - self.height * 0.7
        bombs.append(Bomb(spawn_x, spawn_y, self.facing, info["damage"], self))
        self.weapon["durability"] -= 1
        if self.weapon["durability"] <= 0:
            self.weapon = None
            play_break_sound()

    def handle_input(self, keys, opponent, weapons, arrows, bombs):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.weapon and self.weapon["type"] == "bomb":
                self.attack_type = "throw"
                self.attack_anim = 15
                self.throw_bomb(bombs)
            elif self.is_ranged():
                self.attack_type = "shoot"
                self.attack_anim = 10
                self.shoot(arrows)
                self.weapon["durability"] -= 1
                if self.weapon["durability"] <= 0:
                    self.weapon = None
                    play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup(weapons)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y

        self.on_ground = False
        for plat in PLATFORMS:
            prev_y = self.y - self.vel_y
            if prev_y <= plat.top and self.y >= plat.top and plat.left <= self.x <= plat.right and self.vel_y >= 0:
                self.y = plat.top
                self.vel_y = 0
                self.on_ground = True
                break

        if not self.on_ground:
            if self.y >= HAZARD_Y:
                self.hazard_hit()

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        # Respawn on a surviving platform, or the hazard edge if none left
        if PLATFORMS:
            # Pick the platform closest to the player's x
            best = min(PLATFORMS, key=lambda p: abs(p.centerx - self.x))
            self.x = best.centerx
            self.y = best.top
        else:
            if self.x < WIDTH / 2:
                self.x = 325
            else:
                self.x = 675
            self.y = 380
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        elif self.attack_type == "throw":
            fwd_hand = (cx + self.facing * 20, shoulder_y - 10)
            back_hand = (cx - self.facing * 10, shoulder_y + 10)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "uzi":
            wcolor = WEAPON_TYPES["uzi"]["color"]
            ux, uy = fwd_hand[0], fwd_hand[1] - 3
            barrel_tip = (ux + self.facing * 27, uy)
            grip_bottom = (ux - self.facing * 5, uy + 18)
            pygame.draw.rect(surf, wcolor, pygame.Rect(min(ux, barrel_tip[0]), uy - 7, abs(barrel_tip[0] - ux) + 8, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (ux, uy + 4), grip_bottom, 6)
            pygame.draw.line(surf, (25, 25, 28), barrel_tip, (barrel_tip[0] + self.facing * 10, barrel_tip[1]), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 8, barrel_tip[1])
                pygame.draw.circle(surf, (255, 220, 90), (int(flash[0]), int(flash[1])), 6)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [tipend, (tip[0] + perp[0], tip[1] + perp[1]), (tip[0] - perp[0], tip[1] - perp[1])])
            else:
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK, (guard_center[0] + perp[0], guard_center[1] + perp[1]), (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        if self.weapon:
            label = font_tiny.render(f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}", True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, int(head_y) - head_r - 20))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]
    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    draw_hazard_pit(surf)

    # Draw current (possibly destroyed) platforms
    for plat in PLATFORMS:
        pygame.draw.rect(surf, stage["ground"], plat)
        pygame.draw.rect(surf, stage["ground_edge"], (plat.left, plat.top, plat.width, 10))
        pygame.draw.polygon(surf, stage["ground"], [(plat.left, plat.bottom), (plat.left + 12, plat.bottom + 18), (plat.left + 24, plat.bottom)])
        pygame.draw.polygon(surf, stage["ground"], [(plat.right, plat.bottom), (plat.right - 12, plat.bottom + 18), (plat.right - 24, plat.bottom)])


def draw_hazard_pit(surf):
    pit_rect = pygame.Rect(0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)
    for i, (band_y, color) in enumerate([(GROUND_Y + 15, (150, 40, 15)), (GROUND_Y + 35, (200, 70, 20)), (GROUND_Y + 55, (240, 110, 30))]):
        for gx in range(0, WIDTH, 26):
            wobble = math.sin((gx + i * 40) * 0.15) * 4
            pygame.draw.circle(surf, color, (gx + 13, int(band_y + wobble)), 9)
    for gx in range(0, WIDTH, 52):
        pygame.draw.circle(surf, (255, 200, 90), (gx + 26, GROUND_Y + 60), 4)
    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [(gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)])
        pygame.draw.polygon(surf, (90, 90, 95), [(gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)
    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)
    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, y, wtype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    global PLATFORMS
    PLATFORMS = _initial_platforms()  # reset platforms for new match
    
    p1 = Fighter(210, RED, DARK_RED, 1, (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g), "Player 1")
    p1.y = 450  # Spawn on left low platform
    p2 = Fighter(790, BLUE, DARK_BLUE, -1, (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l), "Player 2")
    p2.y = 450  # Spawn on right low platform
    p1.wins = p1_wins
    p2.wins = p2_wins

    if p1_start_weapon:
        p1.weapon = {"type": p1_start_weapon, "durability": WEAPON_TYPES[p1_start_weapon]["durability"]}
    if p2_start_weapon:
        p2.weapon = {"type": p2_start_weapon, "durability": WEAPON_TYPES[p2_start_weapon]["durability"]}

    weapons, bombs, explosions, arrows, debris = [], [], [], [], []
    spawn_weapon(weapons)
    return p1, p2, weapons, arrows, bombs, explosions, debris


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [(cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "uzi":
        pygame.draw.rect(surf, (45, 45, 50), (cx - 22, cy - 8, 44, 16), border_radius=3)
        pygame.draw.line(surf, (25, 25, 28), (cx - 4, cy + 5), (cx - 10, cy + 25), 7)
        pygame.draw.line(surf, (25, 25, 28), (cx + 20, cy), (cx + 34, cy), 5)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)
    elif wtype == "bomb":
        pygame.draw.circle(surf, (40, 40, 40), (cx, cy), 14)
        pygame.draw.line(surf, (100, 100, 100), (cx, cy), (cx + 8, cy - 12), 3)
        pygame.draw.circle(surf, (255, 200, 50), (cx + 8, cy - 12), 4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220

    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))

        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))

    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE

    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))

    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)
    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))

    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))

    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))

    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)

    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render("Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    state = "select"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0

    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, arrows, bombs, explosions, debris = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False

                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True

                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True

                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, arrows, bombs, explosions, debris = reset_fighters(
                        p1_wins, p2_wins, WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, arrows, bombs)
            p2.handle_input(keys, p1, weapons, arrows, bombs)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            for bomb in bombs[:]:
                bomb.update(PLATFORMS)
                if bomb.exploded:
                    explosions.append(bomb.explode())
                    bombs.remove(bomb)

            for exp in explosions[:]:
                exp.update(p1, p2, debris)
                if exp.life <= 0:
                    explosions.remove(exp)

            # Update debris particles
            for d in debris[:]:
                d.update()
                if d.life <= 0:
                    debris.remove(d)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL

            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)
        for bomb in bombs:
            bomb.draw(screen)
        for exp in explosions:
            exp.draw(screen)
        for d in debris:
            d.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        # Platform count indicator (shows how many platforms remain)
        plat_count = font_tiny.render(f"Platforms: {len(PLATFORMS)}", True, WHITE)
        screen.blit(plat_count, (WIDTH // 2 - plat_count.get_width() // 2, 50))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))
            msg = "DRAW!" if winner == "Draw" else f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))
            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Bombs DESTROY platforms!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [ ]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons, 
floating platforms, and bombs that DESTROY platforms!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

More weapons (uzi, sword, bat, spear, bow, gatling, bomb) also spawn randomly 
on the platforms during the match. Walk over one while unarmed to pick it up
automatically. Your "punch" button (F / K) becomes a weapon attack while
armed — a melee swing for sword/bat/spear, an arrow shot for the bow, a rapid 
stream of bullets for the gatling, or a thrown explosive for the bomb! 
Weapons have limited durability and are lost after a number of uses.

The arena consists of MULTIPLE floating platforms at various heights above a 
deadly lava/spike pit. BOMB EXPLOSIONS DESTROY PLATFORMS — shatter your 
opponent's footing and send them plummeting into the hazard! Falling into 
the pit deals heavy damage (40 HP) but isn't instant death — you can 
survive a fall and keep fighting, but three hits in the pit will finish you.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound (procedurally generated, no external asset files needed)
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None
explosion_sound = None
crumble_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_explosion_sound():
        duration = 0.45
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 120 * (1 - progress * 0.8)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.7 * noise + 0.3 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_crumble_sound():
        duration = 0.6
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.2
            freq = 180 * (1 - progress * 0.5) + 40 * math.sin(2 * math.pi * 8 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.85 * noise + 0.15 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
    explosion_sound = _generate_explosion_sound()
    crumble_sound = _generate_crumble_sound()
except pygame.error:
    break_sound = None
    hazard_sound = None
    explosion_sound = None
    crumble_sound = None


def play_break_sound():
    if break_sound:
        break_sound.play()

def play_hazard_sound():
    if hazard_sound:
        hazard_sound.play()

def play_explosion_sound():
    if explosion_sound:
        explosion_sound.play()

def play_crumble_sound():
    if crumble_sound:
        crumble_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

HAZARD_Y = GROUND_Y + 90
# Reduced from 100 (instant death) to 40 — still very punishing, but 
# survivable. Three hits in the pit will finish you off.
HAZARD_DAMAGE = 40
HAZARD_STUN = 24

# Multiple floating platforms for vertical gameplay.
# This list is MUTABLE — bombs can remove platforms from it during gameplay.
def _initial_platforms():
    return [
        # Low platforms (y=420-450)
        pygame.Rect(150, 450, 120, 18),   # Far left low
        pygame.Rect(730, 450, 120, 18),   # Far right low
        
        # Mid platforms (y=340-400)
        pygame.Rect(280, 400, 140, 20),   # Left mid
        pygame.Rect(580, 400, 140, 20),   # Right mid
        pygame.Rect(430, 360, 140, 20),   # Center mid (slightly higher)
        
        # High platforms (y=260-320)
        pygame.Rect(200, 320, 130, 18),   # Left high
        pygame.Rect(670, 320, 130, 18),   # Right high
        pygame.Rect(440, 280, 120, 18),   # Center high
        
        # Very high platforms (y=180-220)
        pygame.Rect(320, 220, 110, 16),   # Left very high
        pygame.Rect(570, 220, 110, 16),   # Right very high
        pygame.Rect(450, 180, 100, 16),   # Top center
    ]

PLATFORMS = _initial_platforms()

PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

WEAPON_TYPES = {
    "uzi":     {"reach": 0,   "damage": 5, "cooldown": 5, "durability": 50, "color": (45, 45, 50),
                "ranged": True, "proj_speed": 28, "proj_kind": "bullet"},
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow"},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet"},
    "bomb":    {"reach": 0,   "damage": 35, "cooldown": 50, "durability": 3,  "color": (40, 40, 40),
                "ranged": True, "proj_speed": 9, "proj_kind": "bomb"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

MAX_WEAPONS_ON_FIELD = 2
WEAPON_SPAWN_INTERVAL = 300

WEAPON_CHOICES = [None, "uzi", "sword", "bat", "spear", "bow", "gatling", "bomb"]

STAGES = [
    {"name": "Meadow", "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245), "ground": (110, 90, 70), "ground_edge": (80, 150, 80), "decor": "meadow"},
    {"name": "Desert", "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190), "ground": (200, 165, 100), "ground_edge": (225, 195, 130), "decor": "desert"},
    {"name": "Night City", "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90), "ground": (40, 40, 50), "ground_edge": (90, 90, 110), "decor": "city"},
    {"name": "Volcano", "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30), "ground": (50, 35, 30), "ground_edge": (200, 80, 30), "decor": "volcano"},
    {"name": "Snow Peak", "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245), "ground": (225, 235, 240), "ground_edge": (255, 255, 255), "decor": "snow"},
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return
        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class Bomb:
    def __init__(self, x, y, facing, damage, owner):
        self.x = x
        self.y = y
        self.vel_x = facing * 9
        self.vel_y = -11
        self.damage = damage
        self.owner = owner
        self.timer = 70
        self.dead = False
        self.exploded = False

    def update(self, platforms):
        if self.exploded:
            return
        self.vel_y += GRAVITY
        self.x += self.vel_x
        self.y += self.vel_y
        self.timer -= 1
        
        for plat in platforms:
            if plat.left <= self.x <= plat.right and self.y >= plat.top and self.vel_y >= 0:
                if self.y - self.vel_y <= plat.top + 10:
                    self.y = plat.top
                    self.vel_y = 0
                    self.vel_x *= 0.7
                    self.timer -= 3
        
        if self.y >= GROUND_Y or self.timer <= 0:
            self.exploded = True
            self.dead = True

    def explode(self):
        play_explosion_sound()
        return Explosion(self.x, self.y, self.damage, self.owner)

    def draw(self, surf):
        pygame.draw.circle(surf, (30, 30, 30), (int(self.x), int(self.y)), 8)
        if self.timer % 8 < 4:
            pygame.draw.circle(surf, (255, 200, 50), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 3)
        pygame.draw.line(surf, (100, 100, 100), (int(self.x), int(self.y)), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 2)


class Debris:
    """A chunk of a destroyed platform flying through the air."""
    def __init__(self, x, y, color):
        self.x = x
        self.y = y
        self.vel_x = random.uniform(-8, 8)
        self.vel_y = random.uniform(-10, -2)
        self.size = random.randint(4, 10)
        self.color = color
        self.rotation = random.uniform(0, 360)
        self.rot_speed = random.uniform(-15, 15)
        self.life = random.randint(40, 70)

    def update(self):
        self.vel_y += GRAVITY * 0.8
        self.x += self.vel_x
        self.y += self.vel_y
        self.rotation += self.rot_speed
        self.life -= 1
        self.vel_x *= 0.98

    def draw(self, surf):
        if self.life <= 0:
            return
        alpha = min(255, self.life * 6)
        surf_deb = pygame.Surface((self.size * 2, self.size * 2), pygame.SRCALPHA)
        rect = pygame.Rect(0, 0, self.size, self.size)
        pygame.draw.rect(surf_deb, (*self.color, alpha), rect)
        pygame.draw.rect(surf_deb, (0, 0, 0, alpha), rect, 1)
        rotated = pygame.transform.rotate(surf_deb, self.rotation)
        new_rect = rotated.get_rect(center=(int(self.x), int(self.y)))
        surf.blit(rotated, new_rect)


class Explosion:
    def __init__(self, x, y, damage, owner):
        self.x = x
        self.y = y
        self.damage = damage
        self.owner = owner
        self.radius = 10
        self.max_radius = 110
        self.life = 20
        self.has_damaged = False
        self.has_destroyed_platforms = False
        self.destroyed_color = None

    def update(self, p1, p2, debris_list):
        self.life -= 1
        if self.life > 10:
            self.radius += (self.max_radius - self.radius) * 0.4
        else:
            self.radius += (self.max_radius - self.radius) * 0.1
            
        if not self.has_damaged and self.life == 19:
            for target in (p1, p2):
                target_cx = target.x
                target_cy = target.y - target.height / 2
                dist = math.hypot(target_cx - self.x, target_cy - self.y)
                
                if dist < self.max_radius:
                    from_left = (self.x < target.x)
                    target.take_hit(self.damage, from_left)
                    distance_factor = 1.0 - (dist / self.max_radius)
                    base_push = 120
                    base_pop = -22
                    push = (base_push * distance_factor) if from_left else -(base_push * distance_factor)
                    target.x = max(target.width, min(WIDTH - target.width, target.x + push))
                    target.vel_y = base_pop * distance_factor
            self.has_damaged = True

        if not self.has_destroyed_platforms and self.life == 17:
            global PLATFORMS
            platforms_to_destroy = []
            for plat in PLATFORMS:
                closest_x = max(plat.left, min(self.x, plat.right))
                closest_y = max(plat.top, min(self.y, plat.bottom))
                dist = math.hypot(closest_x - self.x, closest_y - self.y)
                if dist < self.max_radius:
                    platforms_to_destroy.append(plat)
            
            if len(PLATFORMS) - len(platforms_to_destroy) >= 1:
                for plat in platforms_to_destroy:
                    plat_color = (110, 90, 70)
                    for chunk_i in range(18):
                        chunk_x = random.uniform(plat.left, plat.right)
                        chunk_y = random.uniform(plat.top, plat.bottom)
                        debris_list.append(Debris(chunk_x, chunk_y, plat_color))
                    PLATFORMS.remove(plat)
                    play_crumble_sound()
            self.has_destroyed_platforms = True

    def draw(self, surf):
        if self.life > 0:
            alpha = int(255 * (self.life / 20))
            surf_exp = pygame.Surface((self.max_radius * 2, self.max_radius * 2), pygame.SRCALPHA)
            pygame.draw.circle(surf_exp, (255, 80, 20, alpha), (self.max_radius, self.max_radius), int(self.radius))
            pygame.draw.circle(surf_exp, (255, 200, 50, int(alpha * 0.8)), (self.max_radius, self.max_radius), int(self.radius * 0.6))
            pygame.draw.circle(surf_exp, (255, 255, 220, int(alpha * 0.9)), (self.max_radius, self.max_radius), int(self.radius * 0.25))
            surf.blit(surf_exp, (int(self.x - self.max_radius), int(self.y - self.max_radius)))


class WeaponPickup:
    def __init__(self, x, y, wtype):
        self.x = x
        self.y = y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [(self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        elif self.wtype == "bomb":
            pygame.draw.circle(surf, (30, 30, 30), (self.x, self.y - 20), 10)
            pygame.draw.line(surf, (100, 100, 100), (self.x, self.y - 20), (self.x + 6, self.y - 30), 2)
            pygame.draw.circle(surf, (255, 200, 50), (self.x + 6, self.y - 30), 3)
            
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing
        self.controls = controls
        self.name = name
        self.health = MAX_HEALTH
        self.on_ground = True
        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0
        self.attack_type = None
        self.hit_stun = 0
        self.walk_cycle = 0
        self.moving = False
        self.wins = 0
        self.weapon = None

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height, self.width, self.height)

    def current_stats(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)
        elif self.weapon["type"] == "uzi":
            spawn_y += random.randint(-3, 3)
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))

    def throw_bomb(self, bombs):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 20
        spawn_y = self.y - self.height * 0.7
        bombs.append(Bomb(spawn_x, spawn_y, self.facing, info["damage"], self))
        self.weapon["durability"] -= 1
        if self.weapon["durability"] <= 0:
            self.weapon = None
            play_break_sound()

    def handle_input(self, keys, opponent, weapons, arrows, bombs):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.weapon and self.weapon["type"] == "bomb":
                self.attack_type = "throw"
                self.attack_anim = 15
                self.throw_bomb(bombs)
            elif self.is_ranged():
                self.attack_type = "shoot"
                self.attack_anim = 10
                self.shoot(arrows)
                self.weapon["durability"] -= 1
                if self.weapon["durability"] <= 0:
                    self.weapon = None
                    play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup(weapons)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y

        self.on_ground = False
        for plat in PLATFORMS:
            prev_y = self.y - self.vel_y
            if prev_y <= plat.top and self.y >= plat.top and plat.left <= self.x <= plat.right and self.vel_y >= 0:
                self.y = plat.top
                self.vel_y = 0
                self.on_ground = True
                break

        if not self.on_ground:
            if self.y >= HAZARD_Y:
                self.hazard_hit()

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        # Reduced damage: 40 HP instead of 100 (instant death).
        # Still very punishing, but survivable — three pit falls = KO.
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        if PLATFORMS:
            best = min(PLATFORMS, key=lambda p: abs(p.centerx - self.x))
            self.x = best.centerx
            self.y = best.top
        else:
            if self.x < WIDTH / 2:
                self.x = 325
            else:
                self.x = 675
            self.y = 380
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        elif self.attack_type == "throw":
            fwd_hand = (cx + self.facing * 20, shoulder_y - 10)
            back_hand = (cx - self.facing * 10, shoulder_y + 10)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "uzi":
            wcolor = WEAPON_TYPES["uzi"]["color"]
            ux, uy = fwd_hand[0], fwd_hand[1] - 3
            barrel_tip = (ux + self.facing * 27, uy)
            grip_bottom = (ux - self.facing * 5, uy + 18)
            pygame.draw.rect(surf, wcolor, pygame.Rect(min(ux, barrel_tip[0]), uy - 7, abs(barrel_tip[0] - ux) + 8, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (ux, uy + 4), grip_bottom, 6)
            pygame.draw.line(surf, (25, 25, 28), barrel_tip, (barrel_tip[0] + self.facing * 10, barrel_tip[1]), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 8, barrel_tip[1])
                pygame.draw.circle(surf, (255, 220, 90), (int(flash[0]), int(flash[1])), 6)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [tipend, (tip[0] + perp[0], tip[1] + perp[1]), (tip[0] - perp[0], tip[1] - perp[1])])
            else:
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK, (guard_center[0] + perp[0], guard_center[1] + perp[1]), (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        if self.weapon:
            label = font_tiny.render(f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}", True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, int(head_y) - head_r - 20))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]
    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    draw_hazard_pit(surf)

    for plat in PLATFORMS:
        pygame.draw.rect(surf, stage["ground"], plat)
        pygame.draw.rect(surf, stage["ground_edge"], (plat.left, plat.top, plat.width, 10))
        pygame.draw.polygon(surf, stage["ground"], [(plat.left, plat.bottom), (plat.left + 12, plat.bottom + 18), (plat.left + 24, plat.bottom)])
        pygame.draw.polygon(surf, stage["ground"], [(plat.right, plat.bottom), (plat.right - 12, plat.bottom + 18), (plat.right - 24, plat.bottom)])


def draw_hazard_pit(surf):
    pit_rect = pygame.Rect(0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)
    for i, (band_y, color) in enumerate([(GROUND_Y + 15, (150, 40, 15)), (GROUND_Y + 35, (200, 70, 20)), (GROUND_Y + 55, (240, 110, 30))]):
        for gx in range(0, WIDTH, 26):
            wobble = math.sin((gx + i * 40) * 0.15) * 4
            pygame.draw.circle(surf, color, (gx + 13, int(band_y + wobble)), 9)
    for gx in range(0, WIDTH, 52):
        pygame.draw.circle(surf, (255, 200, 90), (gx + 26, GROUND_Y + 60), 4)
    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [(gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)])
        pygame.draw.polygon(surf, (90, 90, 95), [(gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)
    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)
    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, y, wtype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    global PLATFORMS
    PLATFORMS = _initial_platforms()
    
    p1 = Fighter(210, RED, DARK_RED, 1, (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g), "Player 1")
    p1.y = 450
    p2 = Fighter(790, BLUE, DARK_BLUE, -1, (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l), "Player 2")
    p2.y = 450
    p1.wins = p1_wins
    p2.wins = p2_wins

    if p1_start_weapon:
        p1.weapon = {"type": p1_start_weapon, "durability": WEAPON_TYPES[p1_start_weapon]["durability"]}
    if p2_start_weapon:
        p2.weapon = {"type": p2_start_weapon, "durability": WEAPON_TYPES[p2_start_weapon]["durability"]}

    weapons, bombs, explosions, arrows, debris = [], [], [], [], []
    spawn_weapon(weapons)
    return p1, p2, weapons, arrows, bombs, explosions, debris


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [(cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "uzi":
        pygame.draw.rect(surf, (45, 45, 50), (cx - 22, cy - 8, 44, 16), border_radius=3)
        pygame.draw.line(surf, (25, 25, 28), (cx - 4, cy + 5), (cx - 10, cy + 25), 7)
        pygame.draw.line(surf, (25, 25, 28), (cx + 20, cy), (cx + 34, cy), 5)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)
    elif wtype == "bomb":
        pygame.draw.circle(surf, (40, 40, 40), (cx, cy), 14)
        pygame.draw.line(surf, (100, 100, 100), (cx, cy), (cx + 8, cy - 12), 3)
        pygame.draw.circle(surf, (255, 200, 50), (cx + 8, cy - 12), 4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220

    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))

        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))

    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE

    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))

    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)
    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))

    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))

    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))

    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)

    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render("Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    state = "select"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0

    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, arrows, bombs, explosions, debris = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False

                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True

                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True

                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, arrows, bombs, explosions, debris = reset_fighters(
                        p1_wins, p2_wins, WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, arrows, bombs)
            p2.handle_input(keys, p1, weapons, arrows, bombs)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            for bomb in bombs[:]:
                bomb.update(PLATFORMS)
                if bomb.exploded:
                    explosions.append(bomb.explode())
                    bombs.remove(bomb)

            for exp in explosions[:]:
                exp.update(p1, p2, debris)
                if exp.life <= 0:
                    explosions.remove(exp)

            for d in debris[:]:
                d.update()
                if d.life <= 0:
                    debris.remove(d)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL

            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)
        for bomb in bombs:
            bomb.draw(screen)
        for exp in explosions:
            exp.draw(screen)
        for d in debris:
            d.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        plat_count = font_tiny.render(f"Platforms: {len(PLATFORMS)}", True, WHITE)
        screen.blit(plat_count, (WIDTH // 2 - plat_count.get_width() // 2, 50))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))
            msg = "DRAW!" if winner == "Draw" else f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))
            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Pit deals 40 dmg — survivable!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

In [22]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons, 
floating platforms, and bombs that DESTROY platforms!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

More weapons (uzi, sword, bat, spear, bow, gatling, bomb) also spawn randomly 
on the platforms during the match. Walk over one while unarmed to pick it up
automatically. Your "punch" button (F / K) becomes a weapon attack while
armed — a melee swing for sword/bat/spear, an arrow shot for the bow, a rapid 
stream of bullets for the gatling, or a thrown explosive for the bomb! 
Weapons have limited durability and are lost after a number of uses.

The arena consists of MULTIPLE floating platforms at various heights above a 
deadly lava/spike pit. BOMB EXPLOSIONS DESTROY PLATFORMS — shatter your 
opponent's footing and send them plummeting into the hazard! Falling into 
the pit deals heavy damage (40 HP) but isn't instant death — you can 
survive a fall and keep fighting, but three hits in the pit will finish you.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound (procedurally generated, no external asset files needed)
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None
explosion_sound = None
crumble_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_explosion_sound():
        duration = 0.45
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 120 * (1 - progress * 0.8)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.7 * noise + 0.3 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_crumble_sound():
        duration = 0.6
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.2
            freq = 180 * (1 - progress * 0.5) + 40 * math.sin(2 * math.pi * 8 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.85 * noise + 0.15 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
    explosion_sound = _generate_explosion_sound()
    crumble_sound = _generate_crumble_sound()
except pygame.error:
    break_sound = None
    hazard_sound = None
    explosion_sound = None
    crumble_sound = None


def play_break_sound():
    if break_sound:
        break_sound.play()

def play_hazard_sound():
    if hazard_sound:
        hazard_sound.play()

def play_explosion_sound():
    if explosion_sound:
        explosion_sound.play()

def play_crumble_sound():
    if crumble_sound:
        crumble_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

HAZARD_Y = GROUND_Y + 90
# Reduced from 100 (instant death) to 40 — still very punishing, but 
# survivable. Three hits in the pit will finish you off.
HAZARD_DAMAGE = 40
HAZARD_STUN = 24

# Multiple floating platforms for vertical gameplay.
# This list is MUTABLE — bombs can remove platforms from it during gameplay.
def _initial_platforms():
    return [
        # Low platforms (y=420-450)
        pygame.Rect(150, 450, 120, 18),   # Far left low
        pygame.Rect(730, 450, 120, 18),   # Far right low
        
        # Mid platforms (y=340-400)
        pygame.Rect(280, 400, 140, 20),   # Left mid
        pygame.Rect(580, 400, 140, 20),   # Right mid
        pygame.Rect(430, 360, 140, 20),   # Center mid (slightly higher)
        
        # High platforms (y=260-320)
        pygame.Rect(200, 320, 130, 18),   # Left high
        pygame.Rect(670, 320, 130, 18),   # Right high
        pygame.Rect(440, 280, 120, 18),   # Center high
        
        # Very high platforms (y=180-220)
        pygame.Rect(320, 220, 110, 16),   # Left very high
        pygame.Rect(570, 220, 110, 16),   # Right very high
        pygame.Rect(450, 180, 100, 16),   # Top center
    ]

PLATFORMS = _initial_platforms()

PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

WEAPON_TYPES = {
    "uzi":     {"reach": 0,   "damage": 5, "cooldown": 5, "durability": 50, "color": (45, 45, 50),
                "ranged": True, "proj_speed": 28, "proj_kind": "bullet"},
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow"},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet"},
    "bomb":    {"reach": 0,   "damage": 35, "cooldown": 50, "durability": 3,  "color": (40, 40, 40),
                "ranged": True, "proj_speed": 9, "proj_kind": "bomb"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

MAX_WEAPONS_ON_FIELD = 2
WEAPON_SPAWN_INTERVAL = 300

WEAPON_CHOICES = [None, "uzi", "sword", "bat", "spear", "bow", "gatling", "bomb"]

STAGES = [
    {"name": "Meadow", "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245), "ground": (110, 90, 70), "ground_edge": (80, 150, 80), "decor": "meadow"},
    {"name": "Desert", "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190), "ground": (200, 165, 100), "ground_edge": (225, 195, 130), "decor": "desert"},
    {"name": "Night City", "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90), "ground": (40, 40, 50), "ground_edge": (90, 90, 110), "decor": "city"},
    {"name": "Volcano", "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30), "ground": (50, 35, 30), "ground_edge": (200, 80, 30), "decor": "volcano"},
    {"name": "Snow Peak", "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245), "ground": (225, 235, 240), "ground_edge": (255, 255, 255), "decor": "snow"},
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return
        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class Bomb:
    def __init__(self, x, y, facing, damage, owner):
        self.x = x
        self.y = y
        self.vel_x = facing * 9
        self.vel_y = -11
        self.damage = damage
        self.owner = owner
        self.timer = 70
        self.dead = False
        self.exploded = False

    def update(self, platforms):
        if self.exploded:
            return
        self.vel_y += GRAVITY
        self.x += self.vel_x
        self.y += self.vel_y
        self.timer -= 1
        
        for plat in platforms:
            if plat.left <= self.x <= plat.right and self.y >= plat.top and self.vel_y >= 0:
                if self.y - self.vel_y <= plat.top + 10:
                    self.y = plat.top
                    self.vel_y = 0
                    self.vel_x *= 0.7
                    self.timer -= 3
        
        if self.y >= GROUND_Y or self.timer <= 0:
            self.exploded = True
            self.dead = True

    def explode(self):
        play_explosion_sound()
        return Explosion(self.x, self.y, self.damage, self.owner)

    def draw(self, surf):
        pygame.draw.circle(surf, (30, 30, 30), (int(self.x), int(self.y)), 8)
        if self.timer % 8 < 4:
            pygame.draw.circle(surf, (255, 200, 50), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 3)
        pygame.draw.line(surf, (100, 100, 100), (int(self.x), int(self.y)), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 2)


class Debris:
    """A chunk of a destroyed platform flying through the air."""
    def __init__(self, x, y, color):
        self.x = x
        self.y = y
        self.vel_x = random.uniform(-8, 8)
        self.vel_y = random.uniform(-10, -2)
        self.size = random.randint(4, 10)
        self.color = color
        self.rotation = random.uniform(0, 360)
        self.rot_speed = random.uniform(-15, 15)
        self.life = random.randint(40, 70)

    def update(self):
        self.vel_y += GRAVITY * 0.8
        self.x += self.vel_x
        self.y += self.vel_y
        self.rotation += self.rot_speed
        self.life -= 1
        self.vel_x *= 0.98

    def draw(self, surf):
        if self.life <= 0:
            return
        alpha = min(255, self.life * 6)
        surf_deb = pygame.Surface((self.size * 2, self.size * 2), pygame.SRCALPHA)
        rect = pygame.Rect(0, 0, self.size, self.size)
        pygame.draw.rect(surf_deb, (*self.color, alpha), rect)
        pygame.draw.rect(surf_deb, (0, 0, 0, alpha), rect, 1)
        rotated = pygame.transform.rotate(surf_deb, self.rotation)
        new_rect = rotated.get_rect(center=(int(self.x), int(self.y)))
        surf.blit(rotated, new_rect)


class Explosion:
    def __init__(self, x, y, damage, owner):
        self.x = x
        self.y = y
        self.damage = damage
        self.owner = owner
        self.radius = 10
        self.max_radius = 110
        self.life = 20
        self.has_damaged = False
        self.has_destroyed_platforms = False
        self.destroyed_color = None

    def update(self, p1, p2, debris_list):
        self.life -= 1
        if self.life > 10:
            self.radius += (self.max_radius - self.radius) * 0.4
        else:
            self.radius += (self.max_radius - self.radius) * 0.1
            
        if not self.has_damaged and self.life == 19:
            for target in (p1, p2):
                target_cx = target.x
                target_cy = target.y - target.height / 2
                dist = math.hypot(target_cx - self.x, target_cy - self.y)
                
                if dist < self.max_radius:
                    from_left = (self.x < target.x)
                    target.take_hit(self.damage, from_left)
                    distance_factor = 1.0 - (dist / self.max_radius)
                    base_push = 120
                    base_pop = -22
                    push = (base_push * distance_factor) if from_left else -(base_push * distance_factor)
                    target.x = max(target.width, min(WIDTH - target.width, target.x + push))
                    target.vel_y = base_pop * distance_factor
            self.has_damaged = True

        if not self.has_destroyed_platforms and self.life == 17:
            global PLATFORMS
            platforms_to_destroy = []
            for plat in PLATFORMS:
                closest_x = max(plat.left, min(self.x, plat.right))
                closest_y = max(plat.top, min(self.y, plat.bottom))
                dist = math.hypot(closest_x - self.x, closest_y - self.y)
                if dist < self.max_radius:
                    platforms_to_destroy.append(plat)
            
            if len(PLATFORMS) - len(platforms_to_destroy) >= 1:
                for plat in platforms_to_destroy:
                    plat_color = (110, 90, 70)
                    for chunk_i in range(18):
                        chunk_x = random.uniform(plat.left, plat.right)
                        chunk_y = random.uniform(plat.top, plat.bottom)
                        debris_list.append(Debris(chunk_x, chunk_y, plat_color))
                    PLATFORMS.remove(plat)
                    play_crumble_sound()
            self.has_destroyed_platforms = True

    def draw(self, surf):
        if self.life > 0:
            alpha = int(255 * (self.life / 20))
            surf_exp = pygame.Surface((self.max_radius * 2, self.max_radius * 2), pygame.SRCALPHA)
            pygame.draw.circle(surf_exp, (255, 80, 20, alpha), (self.max_radius, self.max_radius), int(self.radius))
            pygame.draw.circle(surf_exp, (255, 200, 50, int(alpha * 0.8)), (self.max_radius, self.max_radius), int(self.radius * 0.6))
            pygame.draw.circle(surf_exp, (255, 255, 220, int(alpha * 0.9)), (self.max_radius, self.max_radius), int(self.radius * 0.25))
            surf.blit(surf_exp, (int(self.x - self.max_radius), int(self.y - self.max_radius)))


class WeaponPickup:
    def __init__(self, x, y, wtype):
        self.x = x
        self.y = y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [(self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        elif self.wtype == "bomb":
            pygame.draw.circle(surf, (30, 30, 30), (self.x, self.y - 20), 10)
            pygame.draw.line(surf, (100, 100, 100), (self.x, self.y - 20), (self.x + 6, self.y - 30), 2)
            pygame.draw.circle(surf, (255, 200, 50), (self.x + 6, self.y - 30), 3)
            
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing
        self.controls = controls
        self.name = name
        self.health = MAX_HEALTH
        self.on_ground = True
        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0
        self.attack_type = None
        self.hit_stun = 0
        self.walk_cycle = 0
        self.moving = False
        self.wins = 0
        self.weapon = None

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height, self.width, self.height)

    def current_stats(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)
        elif self.weapon["type"] == "uzi":
            spawn_y += random.randint(-3, 3)
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))

    def throw_bomb(self, bombs):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 20
        spawn_y = self.y - self.height * 0.7
        bombs.append(Bomb(spawn_x, spawn_y, self.facing, info["damage"], self))
        self.weapon["durability"] -= 1
        if self.weapon["durability"] <= 0:
            self.weapon = None
            play_break_sound()

    def handle_input(self, keys, opponent, weapons, arrows, bombs):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.weapon and self.weapon["type"] == "bomb":
                self.attack_type = "throw"
                self.attack_anim = 15
                self.throw_bomb(bombs)
            elif self.is_ranged():
                self.attack_type = "shoot"
                self.attack_anim = 10
                self.shoot(arrows)
                self.weapon["durability"] -= 1
                if self.weapon["durability"] <= 0:
                    self.weapon = None
                    play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup(weapons)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y

        self.on_ground = False
        for plat in PLATFORMS:
            prev_y = self.y - self.vel_y
            if prev_y <= plat.top and self.y >= plat.top and plat.left <= self.x <= plat.right and self.vel_y >= 0:
                self.y = plat.top
                self.vel_y = 0
                self.on_ground = True
                break

        if not self.on_ground:
            if self.y >= HAZARD_Y:
                self.hazard_hit()

        if self.punch_cd > 0:
            self.punch_cd -= 1
        if self.kick_cd > 0:
            self.kick_cd -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        # Reduced damage: 40 HP instead of 100 (instant death).
        # Still very punishing, but survivable — three pit falls = KO.
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        if PLATFORMS:
            best = min(PLATFORMS, key=lambda p: abs(p.centerx - self.x))
            self.x = best.centerx
            self.y = best.top
        else:
            if self.x < WIDTH / 2:
                self.x = 325
            else:
                self.x = 675
            self.y = 380
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        elif self.attack_type == "throw":
            fwd_hand = (cx + self.facing * 20, shoulder_y - 10)
            back_hand = (cx - self.facing * 10, shoulder_y + 10)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "uzi":
            wcolor = WEAPON_TYPES["uzi"]["color"]
            ux, uy = fwd_hand[0], fwd_hand[1] - 3
            barrel_tip = (ux + self.facing * 27, uy)
            grip_bottom = (ux - self.facing * 5, uy + 18)
            pygame.draw.rect(surf, wcolor, pygame.Rect(min(ux, barrel_tip[0]), uy - 7, abs(barrel_tip[0] - ux) + 8, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (ux, uy + 4), grip_bottom, 6)
            pygame.draw.line(surf, (25, 25, 28), barrel_tip, (barrel_tip[0] + self.facing * 10, barrel_tip[1]), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 8, barrel_tip[1])
                pygame.draw.circle(surf, (255, 220, 90), (int(flash[0]), int(flash[1])), 6)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [tipend, (tip[0] + perp[0], tip[1] + perp[1]), (tip[0] - perp[0], tip[1] - perp[1])])
            else:
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK, (guard_center[0] + perp[0], guard_center[1] + perp[1]), (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        if self.weapon:
            label = font_tiny.render(f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}", True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, int(head_y) - head_r - 20))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]
    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    draw_hazard_pit(surf)

    for plat in PLATFORMS:
        pygame.draw.rect(surf, stage["ground"], plat)
        pygame.draw.rect(surf, stage["ground_edge"], (plat.left, plat.top, plat.width, 10))
        pygame.draw.polygon(surf, stage["ground"], [(plat.left, plat.bottom), (plat.left + 12, plat.bottom + 18), (plat.left + 24, plat.bottom)])
        pygame.draw.polygon(surf, stage["ground"], [(plat.right, plat.bottom), (plat.right - 12, plat.bottom + 18), (plat.right - 24, plat.bottom)])


def draw_hazard_pit(surf):
    pit_rect = pygame.Rect(0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)
    for i, (band_y, color) in enumerate([(GROUND_Y + 15, (150, 40, 15)), (GROUND_Y + 35, (200, 70, 20)), (GROUND_Y + 55, (240, 110, 30))]):
        for gx in range(0, WIDTH, 26):
            wobble = math.sin((gx + i * 40) * 0.15) * 4
            pygame.draw.circle(surf, color, (gx + 13, int(band_y + wobble)), 9)
    for gx in range(0, WIDTH, 52):
        pygame.draw.circle(surf, (255, 200, 90), (gx + 26, GROUND_Y + 60), 4)
    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [(gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)])
        pygame.draw.polygon(surf, (90, 90, 95), [(gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)
    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)
    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, y, wtype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    global PLATFORMS
    PLATFORMS = _initial_platforms()
    
    p1 = Fighter(210, RED, DARK_RED, 1, (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g), "Player 1")
    p1.y = 450
    p2 = Fighter(790, BLUE, DARK_BLUE, -1, (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l), "Player 2")
    p2.y = 450
    p1.wins = p1_wins
    p2.wins = p2_wins

    if p1_start_weapon:
        p1.weapon = {"type": p1_start_weapon, "durability": WEAPON_TYPES[p1_start_weapon]["durability"]}
    if p2_start_weapon:
        p2.weapon = {"type": p2_start_weapon, "durability": WEAPON_TYPES[p2_start_weapon]["durability"]}

    weapons, bombs, explosions, arrows, debris = [], [], [], [], []
    spawn_weapon(weapons)
    return p1, p2, weapons, arrows, bombs, explosions, debris


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [(cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "uzi":
        pygame.draw.rect(surf, (45, 45, 50), (cx - 22, cy - 8, 44, 16), border_radius=3)
        pygame.draw.line(surf, (25, 25, 28), (cx - 4, cy + 5), (cx - 10, cy + 25), 7)
        pygame.draw.line(surf, (25, 25, 28), (cx + 20, cy), (cx + 34, cy), 5)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)
    elif wtype == "bomb":
        pygame.draw.circle(surf, (40, 40, 40), (cx, cy), 14)
        pygame.draw.line(surf, (100, 100, 100), (cx, cy), (cx + 8, cy - 12), 3)
        pygame.draw.circle(surf, (255, 200, 50), (cx + 8, cy - 12), 4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))

    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220

    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))

        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))

    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE

    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))

    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)
    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))

    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))

    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))

    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)

    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render("Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    state = "select"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0

    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, arrows, bombs, explosions, debris = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False

                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True

                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True

                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, arrows, bombs, explosions, debris = reset_fighters(
                        p1_wins, p2_wins, WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, arrows, bombs)
            p2.handle_input(keys, p1, weapons, arrows, bombs)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            for bomb in bombs[:]:
                bomb.update(PLATFORMS)
                if bomb.exploded:
                    explosions.append(bomb.explode())
                    bombs.remove(bomb)

            for exp in explosions[:]:
                exp.update(p1, p2, debris)
                if exp.life <= 0:
                    explosions.remove(exp)

            for d in debris[:]:
                d.update()
                if d.life <= 0:
                    debris.remove(d)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL

            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)
        for bomb in bombs:
            bomb.draw(screen)
        for exp in explosions:
            exp.draw(screen)
        for d in debris:
            d.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        plat_count = font_tiny.render(f"Platforms: {len(PLATFORMS)}", True, WHITE)
        screen.blit(plat_count, (WIDTH // 2 - plat_count.get_width() // 2, 50))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))
            msg = "DRAW!" if winner == "Draw" else f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))
            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Pit deals 40 dmg — survivable!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [23]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons, 
floating platforms, bombs that DESTROY platforms, and ARMOR PICKUPS!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

Weapons (uzi, sword, bat, spear, bow, gatling, bomb) AND armor (light/medium/
heavy) spawn randomly on the platforms during the match. Walk over a pickup 
to grab it. Your "punch" button (F / K) becomes a weapon attack while armed.
Armor absorbs damage from hits — each tier reduces incoming damage and has 
limited durability before it shatters.

The arena consists of MULTIPLE floating platforms above a deadly lava/spike 
pit. BOMB EXPLOSIONS DESTROY PLATFORMS. Falling into the pit deals 40 HP 
damage but isn't instant death.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None
explosion_sound = None
crumble_sound = None
armor_pickup_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_explosion_sound():
        duration = 0.45
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 120 * (1 - progress * 0.8)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.7 * noise + 0.3 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_crumble_sound():
        duration = 0.6
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.2
            freq = 180 * (1 - progress * 0.5) + 40 * math.sin(2 * math.pi * 8 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.85 * noise + 0.15 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_armor_pickup_sound():
        duration = 0.22
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            # Metallic "clink" - two quick tones
            freq = 1200 - 400 * progress
            tone = math.sin(2 * math.pi * freq * t)
            tone2 = math.sin(2 * math.pi * (freq * 1.5) * t) * 0.5
            value = int(amplitude * envelope * (tone + tone2) * 0.6)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
    explosion_sound = _generate_explosion_sound()
    crumble_sound = _generate_crumble_sound()
    armor_pickup_sound = _generate_armor_pickup_sound()
except pygame.error:
    break_sound = None
    hazard_sound = None
    explosion_sound = None
    crumble_sound = None
    armor_pickup_sound = None


def play_break_sound():
    if break_sound: break_sound.play()
def play_hazard_sound():
    if hazard_sound: hazard_sound.play()
def play_explosion_sound():
    if explosion_sound: explosion_sound.play()
def play_crumble_sound():
    if crumble_sound: crumble_sound.play()
def play_armor_pickup_sound():
    if armor_pickup_sound: armor_pickup_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
GOLD = (230, 190, 60)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

HAZARD_Y = GROUND_Y + 90
HAZARD_DAMAGE = 40
HAZARD_STUN = 24

def _initial_platforms():
    return [
        pygame.Rect(150, 450, 120, 18),
        pygame.Rect(730, 450, 120, 18),
        pygame.Rect(280, 400, 140, 20),
        pygame.Rect(580, 400, 140, 20),
        pygame.Rect(430, 360, 140, 20),
        pygame.Rect(200, 320, 130, 18),
        pygame.Rect(670, 320, 130, 18),
        pygame.Rect(440, 280, 120, 18),
        pygame.Rect(320, 220, 110, 16),
        pygame.Rect(570, 220, 110, 16),
        pygame.Rect(450, 180, 100, 16),
    ]

PLATFORMS = _initial_platforms()

PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

WEAPON_TYPES = {
    "uzi":     {"reach": 0,   "damage": 5, "cooldown": 5, "durability": 50, "color": (45, 45, 50),
                "ranged": True, "proj_speed": 28, "proj_kind": "bullet"},
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow"},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet"},
    "bomb":    {"reach": 0,   "damage": 35, "cooldown": 50, "durability": 3,  "color": (40, 40, 40),
                "ranged": True, "proj_speed": 9, "proj_kind": "bomb"},
}

# Armor tiers: absorb = damage reduction per hit, durability = number of hits
# before it shatters. Color is used for the aura/shield visual.
ARMOR_TYPES = {
    "light":  {"absorb": 15, "durability": 3, "color": (120, 220, 120), "name": "Light"},
    "medium": {"absorb": 30, "durability": 4, "color": (120, 170, 240), "name": "Medium"},
    "heavy":  {"absorb": 50, "durability": 5, "color": (240, 200, 80),  "name": "Heavy"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

MAX_WEAPONS_ON_FIELD = 2
MAX_ARMOR_ON_FIELD = 1
WEAPON_SPAWN_INTERVAL = 300
ARMOR_SPAWN_INTERVAL = 480

WEAPON_CHOICES = [None, "uzi", "sword", "bat", "spear", "bow", "gatling", "bomb"]

STAGES = [
    {"name": "Meadow", "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245), "ground": (110, 90, 70), "ground_edge": (80, 150, 80), "decor": "meadow"},
    {"name": "Desert", "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190), "ground": (200, 165, 100), "ground_edge": (225, 195, 130), "decor": "desert"},
    {"name": "Night City", "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90), "ground": (40, 40, 50), "ground_edge": (90, 90, 110), "decor": "city"},
    {"name": "Volcano", "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30), "ground": (50, 35, 30), "ground_edge": (200, 80, 30), "decor": "volcano"},
    {"name": "Snow Peak", "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245), "ground": (225, 235, 240), "ground_edge": (255, 255, 255), "decor": "snow"},
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return
        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class Bomb:
    def __init__(self, x, y, facing, damage, owner):
        self.x = x
        self.y = y
        self.vel_x = facing * 9
        self.vel_y = -11
        self.damage = damage
        self.owner = owner
        self.timer = 70
        self.dead = False
        self.exploded = False

    def update(self, platforms):
        if self.exploded:
            return
        self.vel_y += GRAVITY
        self.x += self.vel_x
        self.y += self.vel_y
        self.timer -= 1
        for plat in platforms:
            if plat.left <= self.x <= plat.right and self.y >= plat.top and self.vel_y >= 0:
                if self.y - self.vel_y <= plat.top + 10:
                    self.y = plat.top
                    self.vel_y = 0
                    self.vel_x *= 0.7
                    self.timer -= 3
        if self.y >= GROUND_Y or self.timer <= 0:
            self.exploded = True
            self.dead = True

    def explode(self):
        play_explosion_sound()
        return Explosion(self.x, self.y, self.damage, self.owner)

    def draw(self, surf):
        pygame.draw.circle(surf, (30, 30, 30), (int(self.x), int(self.y)), 8)
        if self.timer % 8 < 4:
            pygame.draw.circle(surf, (255, 200, 50), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 3)
        pygame.draw.line(surf, (100, 100, 100), (int(self.x), int(self.y)), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 2)


class Debris:
    def __init__(self, x, y, color):
        self.x = x
        self.y = y
        self.vel_x = random.uniform(-8, 8)
        self.vel_y = random.uniform(-10, -2)
        self.size = random.randint(4, 10)
        self.color = color
        self.rotation = random.uniform(0, 360)
        self.rot_speed = random.uniform(-15, 15)
        self.life = random.randint(40, 70)

    def update(self):
        self.vel_y += GRAVITY * 0.8
        self.x += self.vel_x
        self.y += self.vel_y
        self.rotation += self.rot_speed
        self.life -= 1
        self.vel_x *= 0.98

    def draw(self, surf):
        if self.life <= 0:
            return
        alpha = min(255, self.life * 6)
        surf_deb = pygame.Surface((self.size * 2, self.size * 2), pygame.SRCALPHA)
        rect = pygame.Rect(0, 0, self.size, self.size)
        pygame.draw.rect(surf_deb, (*self.color, alpha), rect)
        pygame.draw.rect(surf_deb, (0, 0, 0, alpha), rect, 1)
        rotated = pygame.transform.rotate(surf_deb, self.rotation)
        new_rect = rotated.get_rect(center=(int(self.x), int(self.y)))
        surf.blit(rotated, new_rect)


class Explosion:
    def __init__(self, x, y, damage, owner):
        self.x = x
        self.y = y
        self.damage = damage
        self.owner = owner
        self.radius = 10
        self.max_radius = 110
        self.life = 20
        self.has_damaged = False
        self.has_destroyed_platforms = False

    def update(self, p1, p2, debris_list):
        self.life -= 1
        if self.life > 10:
            self.radius += (self.max_radius - self.radius) * 0.4
        else:
            self.radius += (self.max_radius - self.radius) * 0.1
        if not self.has_damaged and self.life == 19:
            for target in (p1, p2):
                target_cx = target.x
                target_cy = target.y - target.height / 2
                dist = math.hypot(target_cx - self.x, target_cy - self.y)
                if dist < self.max_radius:
                    from_left = (self.x < target.x)
                    target.take_hit(self.damage, from_left)
                    distance_factor = 1.0 - (dist / self.max_radius)
                    base_push = 120
                    base_pop = -22
                    push = (base_push * distance_factor) if from_left else -(base_push * distance_factor)
                    target.x = max(target.width, min(WIDTH - target.width, target.x + push))
                    target.vel_y = base_pop * distance_factor
            self.has_damaged = True
        if not self.has_destroyed_platforms and self.life == 17:
            global PLATFORMS
            platforms_to_destroy = []
            for plat in PLATFORMS:
                closest_x = max(plat.left, min(self.x, plat.right))
                closest_y = max(plat.top, min(self.y, plat.bottom))
                dist = math.hypot(closest_x - self.x, closest_y - self.y)
                if dist < self.max_radius:
                    platforms_to_destroy.append(plat)
            if len(PLATFORMS) - len(platforms_to_destroy) >= 1:
                for plat in platforms_to_destroy:
                    plat_color = (110, 90, 70)
                    for _ in range(18):
                        chunk_x = random.uniform(plat.left, plat.right)
                        chunk_y = random.uniform(plat.top, plat.bottom)
                        debris_list.append(Debris(chunk_x, chunk_y, plat_color))
                    PLATFORMS.remove(plat)
                    play_crumble_sound()
            self.has_destroyed_platforms = True

    def draw(self, surf):
        if self.life > 0:
            alpha = int(255 * (self.life / 20))
            surf_exp = pygame.Surface((self.max_radius * 2, self.max_radius * 2), pygame.SRCALPHA)
            pygame.draw.circle(surf_exp, (255, 80, 20, alpha), (self.max_radius, self.max_radius), int(self.radius))
            pygame.draw.circle(surf_exp, (255, 200, 50, int(alpha * 0.8)), (self.max_radius, self.max_radius), int(self.radius * 0.6))
            pygame.draw.circle(surf_exp, (255, 255, 220, int(alpha * 0.9)), (self.max_radius, self.max_radius), int(self.radius * 0.25))
            surf.blit(surf_exp, (int(self.x - self.max_radius), int(self.y - self.max_radius)))


class WeaponPickup:
    def __init__(self, x, y, wtype):
        self.x = x
        self.y = y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [(self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        elif self.wtype == "bomb":
            pygame.draw.circle(surf, (30, 30, 30), (self.x, self.y - 20), 10)
            pygame.draw.line(surf, (100, 100, 100), (self.x, self.y - 20), (self.x + 6, self.y - 30), 2)
            pygame.draw.circle(surf, (255, 200, 50), (self.x + 6, self.y - 30), 3)
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class ArmorPickup:
    """A glowing armor chestpiece that spawns on platforms. Walk over it to equip."""
    def __init__(self, x, y, atype):
        self.x = x
        self.y = y
        self.atype = atype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = ARMOR_TYPES[self.atype]
        color = info["color"]
        # Colored glow matching armor tier
        glow = pygame.Surface((80, 80), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*color, 80), (40, 40), 34)
        surf.blit(glow, (self.x - 40, self.y - 60))

        # Chestplate shape
        chest_points = [
            (self.x - 14, self.y - 38),
            (self.x + 14, self.y - 38),
            (self.x + 16, self.y - 18),
            (self.x + 10, self.y - 8),
            (self.x - 10, self.y - 8),
            (self.x - 16, self.y - 18),
        ]
        pygame.draw.polygon(surf, color, chest_points)
        pygame.draw.polygon(surf, BLACK, chest_points, 2)
        # Shoulder straps
        pygame.draw.line(surf, color, (self.x - 12, self.y - 38), (self.x - 14, self.y - 44), 4)
        pygame.draw.line(surf, color, (self.x + 12, self.y - 38), (self.x + 14, self.y - 44), 4)
        # Center emblem
        pygame.draw.circle(surf, BLACK, (int(self.x), int(self.y - 24)), 3)

        label = font_tiny.render(info["name"] + " Armor", True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 64))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing
        self.controls = controls
        self.name = name
        self.health = MAX_HEALTH
        self.on_ground = True
        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0
        self.attack_type = None
        self.hit_stun = 0
        self.walk_cycle = 0
        self.moving = False
        self.wins = 0
        self.weapon = None
        self.armor = None  # dict: {"type": str, "durability": int}
        self.armor_flash = 0  # frames of flash when armor absorbs a hit

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height, self.width, self.height)

    def current_stats(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup_weapon(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {"type": wp.wtype, "durability": info["durability"]}
                weapons.remove(wp)
                return

    def try_pickup_armor(self, armors):
        for ap in armors:
            if ap.rect().colliderect(self.rect()):
                info = ARMOR_TYPES[ap.atype]
                # New armor replaces old armor
                self.armor = {"type": ap.atype, "durability": info["durability"]}
                self.armor_flash = 12
                armors.remove(ap)
                play_armor_pickup_sound()
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)
        elif self.weapon["type"] == "uzi":
            spawn_y += random.randint(-3, 3)
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))

    def throw_bomb(self, bombs):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 20
        spawn_y = self.y - self.height * 0.7
        bombs.append(Bomb(spawn_x, spawn_y, self.facing, info["damage"], self))
        self.weapon["durability"] -= 1
        if self.weapon["durability"] <= 0:
            self.weapon = None
            play_break_sound()

    def handle_input(self, keys, opponent, weapons, armors, arrows, bombs):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.weapon and self.weapon["type"] == "bomb":
                self.attack_type = "throw"
                self.attack_anim = 15
                self.throw_bomb(bombs)
            elif self.is_ranged():
                self.attack_type = "shoot"
                self.attack_anim = 10
                self.shoot(arrows)
                self.weapon["durability"] -= 1
                if self.weapon["durability"] <= 0:
                    self.weapon = None
                    play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup_weapon(weapons)
        self.try_pickup_armor(armors)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y

        self.on_ground = False
        for plat in PLATFORMS:
            prev_y = self.y - self.vel_y
            if prev_y <= plat.top and self.y >= plat.top and plat.left <= self.x <= plat.right and self.vel_y >= 0:
                self.y = plat.top
                self.vel_y = 0
                self.on_ground = True
                break

        if not self.on_ground:
            if self.y >= HAZARD_Y:
                self.hazard_hit()

        if self.punch_cd > 0: self.punch_cd -= 1
        if self.kick_cd > 0: self.kick_cd -= 1
        if self.armor_flash > 0: self.armor_flash -= 1

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        # Armor absorbs damage first
        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            absorbed = min(damage, info["absorb"])
            damage = damage - absorbed
            self.armor["durability"] -= 1
            self.armor_flash = 10
            if self.armor["durability"] <= 0:
                self.armor = None
                play_break_sound()
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        if PLATFORMS:
            best = min(PLATFORMS, key=lambda p: abs(p.centerx - self.x))
            self.x = best.centerx
            self.y = best.top
        else:
            if self.x < WIDTH / 2:
                self.x = 325
            else:
                self.x = 675
            self.y = 380
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        # Draw armor aura BEHIND the stickman (glowing shield)
        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            aura_color = info["color"]
            flash = self.armor_flash > 0
            # Outer glow
            aura_surf = pygame.Surface((90, 130), pygame.SRCALPHA)
            alpha = 90 if not flash else 180
            pygame.draw.ellipse(aura_surf, (*aura_color, alpha), (0, 0, 90, 130))
            if flash:
                pygame.draw.ellipse(aura_surf, (255, 255, 255, 120), (10, 10, 70, 110))
            surf.blit(aura_surf, (cx - 45, self.y - self.height - 10))

        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        elif self.attack_type == "throw":
            fwd_hand = (cx + self.facing * 20, shoulder_y - 10)
            back_hand = (cx - self.facing * 10, shoulder_y + 10)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        # Draw armor chestplate ON TOP of torso
        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            ac = info["color"]
            chest_points = [
                (cx - 12, shoulder_y - 2),
                (cx + 12, shoulder_y - 2),
                (cx + 14, shoulder_y + 18),
                (cx + 8, shoulder_y + 26),
                (cx - 8, shoulder_y + 26),
                (cx - 14, shoulder_y + 18),
            ]
            pygame.draw.polygon(surf, ac, chest_points)
            pygame.draw.polygon(surf, BLACK, chest_points, 2)
            pygame.draw.circle(surf, BLACK, (cx, shoulder_y + 12), 2)

        # Weapon drawing
        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "uzi":
            wcolor = WEAPON_TYPES["uzi"]["color"]
            ux, uy = fwd_hand[0], fwd_hand[1] - 3
            barrel_tip = (ux + self.facing * 27, uy)
            grip_bottom = (ux - self.facing * 5, uy + 18)
            pygame.draw.rect(surf, wcolor, pygame.Rect(min(ux, barrel_tip[0]), uy - 7, abs(barrel_tip[0] - ux) + 8, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (ux, uy + 4), grip_bottom, 6)
            pygame.draw.line(surf, (25, 25, 28), barrel_tip, (barrel_tip[0] + self.facing * 10, barrel_tip[1]), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 8, barrel_tip[1])
                pygame.draw.circle(surf, (255, 220, 90), (int(flash[0]), int(flash[1])), 6)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [tipend, (tip[0] + perp[0], tip[1] + perp[1]), (tip[0] - perp[0], tip[1] - perp[1])])
            else:
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK, (guard_center[0] + perp[0], guard_center[1] + perp[1]), (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        # Labels above head
        label_y = int(head_y) - head_r - 20
        if self.weapon:
            label = font_tiny.render(f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}", True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, label_y))
            label_y -= 16
        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            label = font_tiny.render(f"{info['name']} Armor x{self.armor['durability']}", True, info["color"])
            pygame.draw.rect(surf, BLACK, (cx - label.get_width() // 2 - 2, label_y - 1, label.get_width() + 4, label.get_height() + 2))
            surf.blit(label, (cx - label.get_width() // 2, label_y))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]
    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    draw_hazard_pit(surf)

    for plat in PLATFORMS:
        pygame.draw.rect(surf, stage["ground"], plat)
        pygame.draw.rect(surf, stage["ground_edge"], (plat.left, plat.top, plat.width, 10))
        pygame.draw.polygon(surf, stage["ground"], [(plat.left, plat.bottom), (plat.left + 12, plat.bottom + 18), (plat.left + 24, plat.bottom)])
        pygame.draw.polygon(surf, stage["ground"], [(plat.right, plat.bottom), (plat.right - 12, plat.bottom + 18), (plat.right - 24, plat.bottom)])


def draw_hazard_pit(surf):
    pit_rect = pygame.Rect(0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)
    for i, (band_y, color) in enumerate([(GROUND_Y + 15, (150, 40, 15)), (GROUND_Y + 35, (200, 70, 20)), (GROUND_Y + 55, (240, 110, 30))]):
        for gx in range(0, WIDTH, 26):
            wobble = math.sin((gx + i * 40) * 0.15) * 4
            pygame.draw.circle(surf, color, (gx + 13, int(band_y + wobble)), 9)
    for gx in range(0, WIDTH, 52):
        pygame.draw.circle(surf, (255, 200, 90), (gx + 26, GROUND_Y + 60), 4)
    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [(gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)])
        pygame.draw.polygon(surf, (90, 90, 95), [(gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)
    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)
    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def draw_armor_bar(surf, x, y, fighter, align_left=True):
    """Draw a small armor durability bar under the health bar."""
    if not fighter.armor:
        return
    info = ARMOR_TYPES[fighter.armor["type"]]
    max_dur = info["durability"]
    cur_dur = fighter.armor["durability"]
    bar_w, bar_h = 320, 10
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=3)
    fill_w = int((bar_w - 4) * (cur_dur / max_dur))
    if align_left:
        fill_rect = pygame.Rect(x + 2, y + 2, fill_w, bar_h - 4)
    else:
        fill_rect = pygame.Rect(x + bar_w - 2 - fill_w, y + 2, fill_w, bar_h - 4)
    pygame.draw.rect(surf, info["color"], fill_rect, border_radius=2)
    armor_label = font_tiny.render(f"{info['name']} Armor ({cur_dur})", True, info["color"])
    if align_left:
        surf.blit(armor_label, (x, y - 14))
    else:
        surf.blit(armor_label, (x + bar_w - armor_label.get_width(), y - 14))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, y, wtype))


def spawn_armor(armors):
    if len(armors) >= MAX_ARMOR_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    # Weighted: light more common, heavy rarer
    atype = random.choices(list(ARMOR_TYPES.keys()), weights=[5, 3, 1], k=1)[0]
    armors.append(ArmorPickup(x, y, atype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    global PLATFORMS
    PLATFORMS = _initial_platforms()
    p1 = Fighter(210, RED, DARK_RED, 1, (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g), "Player 1")
    p1.y = 450
    p2 = Fighter(790, BLUE, DARK_BLUE, -1, (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l), "Player 2")
    p2.y = 450
    p1.wins = p1_wins
    p2.wins = p2_wins
    if p1_start_weapon:
        p1.weapon = {"type": p1_start_weapon, "durability": WEAPON_TYPES[p1_start_weapon]["durability"]}
    if p2_start_weapon:
        p2.weapon = {"type": p2_start_weapon, "durability": WEAPON_TYPES[p2_start_weapon]["durability"]}
    weapons, armors, bombs, explosions, arrows, debris = [], [], [], [], [], []
    spawn_weapon(weapons)
    spawn_armor(armors)
    return p1, p2, weapons, armors, arrows, bombs, explosions, debris


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [(cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "uzi":
        pygame.draw.rect(surf, (45, 45, 50), (cx - 22, cy - 8, 44, 16), border_radius=3)
        pygame.draw.line(surf, (25, 25, 28), (cx - 4, cy + 5), (cx - 10, cy + 25), 7)
        pygame.draw.line(surf, (25, 25, 28), (cx + 20, cy), (cx + 34, cy), 5)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)
    elif wtype == "bomb":
        pygame.draw.circle(surf, (40, 40, 40), (cx, cy), 14)
        pygame.draw.line(surf, (100, 100, 100), (cx, cy), (cx + 8, cy - 12), 3)
        pygame.draw.circle(surf, (255, 200, 50), (cx + 8, cy - 12), 4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220
    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))
        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))
    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE
    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))
    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)
    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))
    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))
    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))
    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)
    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render("Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    state = "select"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0
    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, armors, arrows, bombs, explosions, debris = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL
    armor_spawn_timer = ARMOR_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True
                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True
                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, armors, arrows, bombs, explosions, debris = reset_fighters(
                        p1_wins, p2_wins, WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    armor_spawn_timer = ARMOR_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, armors, arrows, bombs)
            p2.handle_input(keys, p1, weapons, armors, arrows, bombs)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            for bomb in bombs[:]:
                bomb.update(PLATFORMS)
                if bomb.exploded:
                    explosions.append(bomb.explode())
                    bombs.remove(bomb)
            for exp in explosions[:]:
                exp.update(p1, p2, debris)
                if exp.life <= 0:
                    explosions.remove(exp)
            for d in debris[:]:
                d.update()
                if d.life <= 0:
                    debris.remove(d)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL
            armor_spawn_timer -= 1
            if armor_spawn_timer <= 0:
                spawn_armor(armors)
                armor_spawn_timer = ARMOR_SPAWN_INTERVAL

            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        for ap in armors:
            ap.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)
        for bomb in bombs:
            bomb.draw(screen)
        for exp in explosions:
            exp.draw(screen)
        for d in debris:
            d.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)
        draw_armor_bar(screen, 30, 62, p1, align_left=True)
        draw_armor_bar(screen, WIDTH - 30 - 320, 62, p2, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        plat_count = font_tiny.render(f"Platforms: {len(PLATFORMS)}", True, WHITE)
        screen.blit(plat_count, (WIDTH // 2 - plat_count.get_width() // 2, 50))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))
            msg = "DRAW!" if winner == "Draw" else f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))
            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Grab armor for protection!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [24]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons, 
floating platforms, bombs that DESTROY platforms, ARMOR PICKUPS, and GUN RELOADING!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

Weapons (uzi, sword, bat, spear, bow, gatling, bomb) AND armor (light/medium/
heavy) spawn randomly on the platforms during the match. Walk over a pickup 
to grab it.

GUN RELOADING: The uzi, gatling, and bow now have 4-round magazines. After 
firing 4 shots, the gun auto-reloads for ~1 second — you can move and jump 
during reload, but can't fire. Watch your ammo count above your head!

Armor absorbs damage from hits — each tier reduces incoming damage and has 
limited durability before it shatters.

The arena consists of MULTIPLE floating platforms above a deadly lava/spike 
pit. BOMB EXPLOSIONS DESTROY PLATFORMS. Falling into the pit deals 40 HP 
damage but isn't instant death.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None
explosion_sound = None
crumble_sound = None
armor_pickup_sound = None
reload_sound = None
empty_click_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_explosion_sound():
        duration = 0.45
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 120 * (1 - progress * 0.8)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.7 * noise + 0.3 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_crumble_sound():
        duration = 0.6
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.2
            freq = 180 * (1 - progress * 0.5) + 40 * math.sin(2 * math.pi * 8 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.85 * noise + 0.15 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_armor_pickup_sound():
        duration = 0.22
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 1200 - 400 * progress
            tone = math.sin(2 * math.pi * freq * t)
            tone2 = math.sin(2 * math.pi * (freq * 1.5) * t) * 0.5
            value = int(amplitude * envelope * (tone + tone2) * 0.6)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_reload_sound():
        """A mechanical click-clack for reloading a gun."""
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.3
            # Two distinct clicks: one at start, one at end
            click1 = math.sin(2 * math.pi * 1800 * t) * (1 if progress < 0.1 else 0)
            click2 = math.sin(2 * math.pi * 2200 * t) * (1 if 0.45 < progress < 0.55 else 0)
            metallic = math.sin(2 * math.pi * 900 * t) * 0.3
            mix = click1 + click2 + metallic
            value = int(amplitude * envelope * mix * 0.5)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_empty_click_sound():
        """A short dry click when trying to fire an empty gun."""
        duration = 0.08
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 3
            freq = 3500 * (1 - progress * 0.5)
            tone = math.sin(2 * math.pi * freq * t)
            value = int(amplitude * envelope * tone * 0.4)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
    explosion_sound = _generate_explosion_sound()
    crumble_sound = _generate_crumble_sound()
    armor_pickup_sound = _generate_armor_pickup_sound()
    reload_sound = _generate_reload_sound()
    empty_click_sound = _generate_empty_click_sound()
except pygame.error:
    break_sound = None
    hazard_sound = None
    explosion_sound = None
    crumble_sound = None
    armor_pickup_sound = None
    reload_sound = None
    empty_click_sound = None


def play_break_sound():
    if break_sound: break_sound.play()
def play_hazard_sound():
    if hazard_sound: hazard_sound.play()
def play_explosion_sound():
    if explosion_sound: explosion_sound.play()
def play_crumble_sound():
    if crumble_sound: crumble_sound.play()
def play_armor_pickup_sound():
    if armor_pickup_sound: armor_pickup_sound.play()
def play_reload_sound():
    if reload_sound: reload_sound.play()
def play_empty_click_sound():
    if empty_click_sound: empty_click_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
GOLD = (230, 190, 60)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

HAZARD_Y = GROUND_Y + 90
HAZARD_DAMAGE = 40
HAZARD_STUN = 24

# Reload time in frames (60 frames = 1 second at 60 FPS)
RELOAD_TIME = 60

def _initial_platforms():
    return [
        pygame.Rect(150, 450, 120, 18),
        pygame.Rect(730, 450, 120, 18),
        pygame.Rect(280, 400, 140, 20),
        pygame.Rect(580, 400, 140, 20),
        pygame.Rect(430, 360, 140, 20),
        pygame.Rect(200, 320, 130, 18),
        pygame.Rect(670, 320, 130, 18),
        pygame.Rect(440, 280, 120, 18),
        pygame.Rect(320, 220, 110, 16),
        pygame.Rect(570, 220, 110, 16),
        pygame.Rect(450, 180, 100, 16),
    ]

PLATFORMS = _initial_platforms()

PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

# Weapons now have mag_size for ranged guns. Melee weapons ignore this field.
WEAPON_TYPES = {
    "uzi":     {"reach": 0,   "damage": 5, "cooldown": 5, "durability": 50, "color": (45, 45, 50),
                "ranged": True, "proj_speed": 28, "proj_kind": "bullet", "mag_size": 4},
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow", "mag_size": 4},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet", "mag_size": 4},
    "bomb":    {"reach": 0,   "damage": 35, "cooldown": 50, "durability": 3,  "color": (40, 40, 40),
                "ranged": True, "proj_speed": 9, "proj_kind": "bomb"},
}

ARMOR_TYPES = {
    "light":  {"absorb": 15, "durability": 3, "color": (120, 220, 120), "name": "Light"},
    "medium": {"absorb": 30, "durability": 4, "color": (120, 170, 240), "name": "Medium"},
    "heavy":  {"absorb": 50, "durability": 5, "color": (240, 200, 80),  "name": "Heavy"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

MAX_WEAPONS_ON_FIELD = 2
MAX_ARMOR_ON_FIELD = 1
WEAPON_SPAWN_INTERVAL = 300
ARMOR_SPAWN_INTERVAL = 480

WEAPON_CHOICES = [None, "uzi", "sword", "bat", "spear", "bow", "gatling", "bomb"]

STAGES = [
    {"name": "Meadow", "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245), "ground": (110, 90, 70), "ground_edge": (80, 150, 80), "decor": "meadow"},
    {"name": "Desert", "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190), "ground": (200, 165, 100), "ground_edge": (225, 195, 130), "decor": "desert"},
    {"name": "Night City", "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90), "ground": (40, 40, 50), "ground_edge": (90, 90, 110), "decor": "city"},
    {"name": "Volcano", "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30), "ground": (50, 35, 30), "ground_edge": (200, 80, 30), "decor": "volcano"},
    {"name": "Snow Peak", "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245), "ground": (225, 235, 240), "ground_edge": (255, 255, 255), "decor": "snow"},
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return
        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class Bomb:
    def __init__(self, x, y, facing, damage, owner):
        self.x = x
        self.y = y
        self.vel_x = facing * 9
        self.vel_y = -11
        self.damage = damage
        self.owner = owner
        self.timer = 70
        self.dead = False
        self.exploded = False

    def update(self, platforms):
        if self.exploded:
            return
        self.vel_y += GRAVITY
        self.x += self.vel_x
        self.y += self.vel_y
        self.timer -= 1
        for plat in platforms:
            if plat.left <= self.x <= plat.right and self.y >= plat.top and self.vel_y >= 0:
                if self.y - self.vel_y <= plat.top + 10:
                    self.y = plat.top
                    self.vel_y = 0
                    self.vel_x *= 0.7
                    self.timer -= 3
        if self.y >= GROUND_Y or self.timer <= 0:
            self.exploded = True
            self.dead = True

    def explode(self):
        play_explosion_sound()
        return Explosion(self.x, self.y, self.damage, self.owner)

    def draw(self, surf):
        pygame.draw.circle(surf, (30, 30, 30), (int(self.x), int(self.y)), 8)
        if self.timer % 8 < 4:
            pygame.draw.circle(surf, (255, 200, 50), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 3)
        pygame.draw.line(surf, (100, 100, 100), (int(self.x), int(self.y)), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 2)


class Debris:
    def __init__(self, x, y, color):
        self.x = x
        self.y = y
        self.vel_x = random.uniform(-8, 8)
        self.vel_y = random.uniform(-10, -2)
        self.size = random.randint(4, 10)
        self.color = color
        self.rotation = random.uniform(0, 360)
        self.rot_speed = random.uniform(-15, 15)
        self.life = random.randint(40, 70)

    def update(self):
        self.vel_y += GRAVITY * 0.8
        self.x += self.vel_x
        self.y += self.vel_y
        self.rotation += self.rot_speed
        self.life -= 1
        self.vel_x *= 0.98

    def draw(self, surf):
        if self.life <= 0:
            return
        alpha = min(255, self.life * 6)
        surf_deb = pygame.Surface((self.size * 2, self.size * 2), pygame.SRCALPHA)
        rect = pygame.Rect(0, 0, self.size, self.size)
        pygame.draw.rect(surf_deb, (*self.color, alpha), rect)
        pygame.draw.rect(surf_deb, (0, 0, 0, alpha), rect, 1)
        rotated = pygame.transform.rotate(surf_deb, self.rotation)
        new_rect = rotated.get_rect(center=(int(self.x), int(self.y)))
        surf.blit(rotated, new_rect)


class Explosion:
    def __init__(self, x, y, damage, owner):
        self.x = x
        self.y = y
        self.damage = damage
        self.owner = owner
        self.radius = 10
        self.max_radius = 110
        self.life = 20
        self.has_damaged = False
        self.has_destroyed_platforms = False

    def update(self, p1, p2, debris_list):
        self.life -= 1
        if self.life > 10:
            self.radius += (self.max_radius - self.radius) * 0.4
        else:
            self.radius += (self.max_radius - self.radius) * 0.1
        if not self.has_damaged and self.life == 19:
            for target in (p1, p2):
                target_cx = target.x
                target_cy = target.y - target.height / 2
                dist = math.hypot(target_cx - self.x, target_cy - self.y)
                if dist < self.max_radius:
                    from_left = (self.x < target.x)
                    target.take_hit(self.damage, from_left)
                    distance_factor = 1.0 - (dist / self.max_radius)
                    base_push = 120
                    base_pop = -22
                    push = (base_push * distance_factor) if from_left else -(base_push * distance_factor)
                    target.x = max(target.width, min(WIDTH - target.width, target.x + push))
                    target.vel_y = base_pop * distance_factor
            self.has_damaged = True
        if not self.has_destroyed_platforms and self.life == 17:
            global PLATFORMS
            platforms_to_destroy = []
            for plat in PLATFORMS:
                closest_x = max(plat.left, min(self.x, plat.right))
                closest_y = max(plat.top, min(self.y, plat.bottom))
                dist = math.hypot(closest_x - self.x, closest_y - self.y)
                if dist < self.max_radius:
                    platforms_to_destroy.append(plat)
            if len(PLATFORMS) - len(platforms_to_destroy) >= 1:
                for plat in platforms_to_destroy:
                    plat_color = (110, 90, 70)
                    for _ in range(18):
                        chunk_x = random.uniform(plat.left, plat.right)
                        chunk_y = random.uniform(plat.top, plat.bottom)
                        debris_list.append(Debris(chunk_x, chunk_y, plat_color))
                    PLATFORMS.remove(plat)
                    play_crumble_sound()
            self.has_destroyed_platforms = True

    def draw(self, surf):
        if self.life > 0:
            alpha = int(255 * (self.life / 20))
            surf_exp = pygame.Surface((self.max_radius * 2, self.max_radius * 2), pygame.SRCALPHA)
            pygame.draw.circle(surf_exp, (255, 80, 20, alpha), (self.max_radius, self.max_radius), int(self.radius))
            pygame.draw.circle(surf_exp, (255, 200, 50, int(alpha * 0.8)), (self.max_radius, self.max_radius), int(self.radius * 0.6))
            pygame.draw.circle(surf_exp, (255, 255, 220, int(alpha * 0.9)), (self.max_radius, self.max_radius), int(self.radius * 0.25))
            surf.blit(surf_exp, (int(self.x - self.max_radius), int(self.y - self.max_radius)))


class WeaponPickup:
    def __init__(self, x, y, wtype):
        self.x = x
        self.y = y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [(self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        elif self.wtype == "uzi":
            pygame.draw.rect(surf, color, (self.x - 12, self.y - 26, 24, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (self.x - 2, self.y - 15), (self.x - 6, self.y - 4), 5)
        elif self.wtype == "bomb":
            pygame.draw.circle(surf, (30, 30, 30), (self.x, self.y - 20), 10)
            pygame.draw.line(surf, (100, 100, 100), (self.x, self.y - 20), (self.x + 6, self.y - 30), 2)
            pygame.draw.circle(surf, (255, 200, 50), (self.x + 6, self.y - 30), 3)
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class ArmorPickup:
    def __init__(self, x, y, atype):
        self.x = x
        self.y = y
        self.atype = atype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = ARMOR_TYPES[self.atype]
        color = info["color"]
        glow = pygame.Surface((80, 80), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*color, 80), (40, 40), 34)
        surf.blit(glow, (self.x - 40, self.y - 60))

        chest_points = [
            (self.x - 14, self.y - 38),
            (self.x + 14, self.y - 38),
            (self.x + 16, self.y - 18),
            (self.x + 10, self.y - 8),
            (self.x - 10, self.y - 8),
            (self.x - 16, self.y - 18),
        ]
        pygame.draw.polygon(surf, color, chest_points)
        pygame.draw.polygon(surf, BLACK, chest_points, 2)
        pygame.draw.line(surf, color, (self.x - 12, self.y - 38), (self.x - 14, self.y - 44), 4)
        pygame.draw.line(surf, color, (self.x + 12, self.y - 38), (self.x + 14, self.y - 44), 4)
        pygame.draw.circle(surf, BLACK, (int(self.x), int(self.y - 24)), 3)

        label = font_tiny.render(info["name"] + " Armor", True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 64))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing
        self.controls = controls
        self.name = name
        self.health = MAX_HEALTH
        self.on_ground = True
        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0
        self.attack_type = None
        self.hit_stun = 0
        self.walk_cycle = 0
        self.moving = False
        self.wins = 0
        self.weapon = None
        self.armor = None
        self.armor_flash = 0
        # Reload system: reload_timer > 0 means currently reloading
        self.reload_timer = 0

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height, self.width, self.height)

    def current_stats(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup_weapon(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {
                    "type": wp.wtype,
                    "durability": info["durability"],
                    # Initialize magazine for ranged weapons
                    "current_mag": info.get("mag_size", 0),
                }
                weapons.remove(wp)
                return

    def try_pickup_armor(self, armors):
        for ap in armors:
            if ap.rect().colliderect(self.rect()):
                info = ARMOR_TYPES[ap.atype]
                self.armor = {"type": ap.atype, "durability": info["durability"]}
                self.armor_flash = 12
                armors.remove(ap)
                play_armor_pickup_sound()
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def is_reloading(self):
        return self.reload_timer > 0

    def start_reload(self):
        """Begin reloading the current gun. Called when mag empties."""
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            if info.get("mag_size", 0) > 0:
                self.reload_timer = RELOAD_TIME
                play_reload_sound()

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)
        elif self.weapon["type"] == "uzi":
            spawn_y += random.randint(-3, 3)
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))
        # Decrement magazine
        if "current_mag" in self.weapon:
            self.weapon["current_mag"] -= 1
            # If magazine empty, start reload
            if self.weapon["current_mag"] <= 0:
                self.start_reload()

    def throw_bomb(self, bombs):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 20
        spawn_y = self.y - self.height * 0.7
        bombs.append(Bomb(spawn_x, spawn_y, self.facing, info["damage"], self))
        self.weapon["durability"] -= 1
        if self.weapon["durability"] <= 0:
            self.weapon = None
            play_break_sound()

    def handle_input(self, keys, opponent, weapons, armors, arrows, bombs):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            # Can't fire while reloading!
            if self.is_reloading():
                # Play a click to indicate reload is in progress
                pass
            elif self.weapon and self.weapon["type"] == "bomb":
                self.attack_type = "throw"
                self.attack_anim = 15
                self.throw_bomb(bombs)
            elif self.is_ranged():
                # Check if magazine is empty (shouldn't happen since reload auto-starts, but safety check)
                if "current_mag" in self.weapon and self.weapon["current_mag"] <= 0:
                    self.start_reload()
                    play_empty_click_sound()
                else:
                    self.attack_type = "shoot"
                    self.attack_anim = 10
                    self.shoot(arrows)
                    self.weapon["durability"] -= 1
                    if self.weapon["durability"] <= 0:
                        self.weapon = None
                        play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup_weapon(weapons)
        self.try_pickup_armor(armors)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y

        self.on_ground = False
        for plat in PLATFORMS:
            prev_y = self.y - self.vel_y
            if prev_y <= plat.top and self.y >= plat.top and plat.left <= self.x <= plat.right and self.vel_y >= 0:
                self.y = plat.top
                self.vel_y = 0
                self.on_ground = True
                break

        if not self.on_ground:
            if self.y >= HAZARD_Y:
                self.hazard_hit()

        if self.punch_cd > 0: self.punch_cd -= 1
        if self.kick_cd > 0: self.kick_cd -= 1
        if self.armor_flash > 0: self.armor_flash -= 1

        # Reload timer countdown
        if self.reload_timer > 0:
            self.reload_timer -= 1
            if self.reload_timer == 0 and self.weapon:
                # Reload complete — refill magazine
                info = WEAPON_TYPES[self.weapon["type"]]
                if "current_mag" in self.weapon:
                    self.weapon["current_mag"] = info.get("mag_size", 0)

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            absorbed = min(damage, info["absorb"])
            damage = damage - absorbed
            self.armor["durability"] -= 1
            self.armor_flash = 10
            if self.armor["durability"] <= 0:
                self.armor = None
                play_break_sound()
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        if PLATFORMS:
            best = min(PLATFORMS, key=lambda p: abs(p.centerx - self.x))
            self.x = best.centerx
            self.y = best.top
        else:
            if self.x < WIDTH / 2:
                self.x = 325
            else:
                self.x = 675
            self.y = 380
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        # Armor aura
        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            aura_color = info["color"]
            flash = self.armor_flash > 0
            aura_surf = pygame.Surface((90, 130), pygame.SRCALPHA)
            alpha = 90 if not flash else 180
            pygame.draw.ellipse(aura_surf, (*aura_color, alpha), (0, 0, 90, 130))
            if flash:
                pygame.draw.ellipse(aura_surf, (255, 255, 255, 120), (10, 10, 70, 110))
            surf.blit(aura_surf, (cx - 45, self.y - self.height - 10))

        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        elif self.attack_type == "throw":
            fwd_hand = (cx + self.facing * 20, shoulder_y - 10)
            back_hand = (cx - self.facing * 10, shoulder_y + 10)
        elif self.is_reloading():
            # Reload pose: both hands down at the hip, tilting the gun
            fwd_hand = (cx + self.facing * 8, shoulder_y + 30)
            back_hand = (cx - self.facing * 6, shoulder_y + 28)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        # Armor chestplate
        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            ac = info["color"]
            chest_points = [
                (cx - 12, shoulder_y - 2),
                (cx + 12, shoulder_y - 2),
                (cx + 14, shoulder_y + 18),
                (cx + 8, shoulder_y + 26),
                (cx - 8, shoulder_y + 26),
                (cx - 14, shoulder_y + 18),
            ]
            pygame.draw.polygon(surf, ac, chest_points)
            pygame.draw.polygon(surf, BLACK, chest_points, 2)
            pygame.draw.circle(surf, BLACK, (cx, shoulder_y + 12), 2)

        # Weapon drawing
        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "uzi":
            wcolor = WEAPON_TYPES["uzi"]["color"]
            ux, uy = fwd_hand[0], fwd_hand[1] - 3
            barrel_tip = (ux + self.facing * 27, uy)
            grip_bottom = (ux - self.facing * 5, uy + 18)
            pygame.draw.rect(surf, wcolor, pygame.Rect(min(ux, barrel_tip[0]), uy - 7, abs(barrel_tip[0] - ux) + 8, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (ux, uy + 4), grip_bottom, 6)
            pygame.draw.line(surf, (25, 25, 28), barrel_tip, (barrel_tip[0] + self.facing * 10, barrel_tip[1]), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 8, barrel_tip[1])
                pygame.draw.circle(surf, (255, 220, 90), (int(flash[0]), int(flash[1])), 6)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [tipend, (tip[0] + perp[0], tip[1] + perp[1]), (tip[0] - perp[0], tip[1] - perp[1])])
            else:
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK, (guard_center[0] + perp[0], guard_center[1] + perp[1]), (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        # Labels above head
        label_y = int(head_y) - head_r - 20
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            mag_size = info.get("mag_size", 0)
            if mag_size > 0 and "current_mag" in self.weapon:
                # Gun with magazine — show mag count
                cur = self.weapon["current_mag"]
                mag_color = YELLOW if cur > 0 else RED
                mag_label = font_tiny.render(f"[{cur}/{mag_size}]", True, mag_color)
                surf.blit(mag_label, (cx - mag_label.get_width() // 2, label_y))
                label_y -= 14
            label = font_tiny.render(f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}", True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, label_y))
            label_y -= 16

        # Show RELOADING indicator
        if self.is_reloading():
            reload_label = font_tiny.render("RELOADING...", True, YELLOW)
            pygame.draw.rect(surf, BLACK, (cx - reload_label.get_width() // 2 - 2, label_y - 1,
                                            reload_label.get_width() + 4, reload_label.get_height() + 2))
            surf.blit(reload_label, (cx - reload_label.get_width() // 2, label_y))
            label_y -= 16

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            label = font_tiny.render(f"{info['name']} Armor x{self.armor['durability']}", True, info["color"])
            pygame.draw.rect(surf, BLACK, (cx - label.get_width() // 2 - 2, label_y - 1, label.get_width() + 4, label.get_height() + 2))
            surf.blit(label, (cx - label.get_width() // 2, label_y))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]
    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    draw_hazard_pit(surf)

    for plat in PLATFORMS:
        pygame.draw.rect(surf, stage["ground"], plat)
        pygame.draw.rect(surf, stage["ground_edge"], (plat.left, plat.top, plat.width, 10))
        pygame.draw.polygon(surf, stage["ground"], [(plat.left, plat.bottom), (plat.left + 12, plat.bottom + 18), (plat.left + 24, plat.bottom)])
        pygame.draw.polygon(surf, stage["ground"], [(plat.right, plat.bottom), (plat.right - 12, plat.bottom + 18), (plat.right - 24, plat.bottom)])


def draw_hazard_pit(surf):
    pit_rect = pygame.Rect(0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)
    for i, (band_y, color) in enumerate([(GROUND_Y + 15, (150, 40, 15)), (GROUND_Y + 35, (200, 70, 20)), (GROUND_Y + 55, (240, 110, 30))]):
        for gx in range(0, WIDTH, 26):
            wobble = math.sin((gx + i * 40) * 0.15) * 4
            pygame.draw.circle(surf, color, (gx + 13, int(band_y + wobble)), 9)
    for gx in range(0, WIDTH, 52):
        pygame.draw.circle(surf, (255, 200, 90), (gx + 26, GROUND_Y + 60), 4)
    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [(gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)])
        pygame.draw.polygon(surf, (90, 90, 95), [(gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)
    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)
    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def draw_armor_bar(surf, x, y, fighter, align_left=True):
    if not fighter.armor:
        return
    info = ARMOR_TYPES[fighter.armor["type"]]
    max_dur = info["durability"]
    cur_dur = fighter.armor["durability"]
    bar_w, bar_h = 320, 10
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=3)
    fill_w = int((bar_w - 4) * (cur_dur / max_dur))
    if align_left:
        fill_rect = pygame.Rect(x + 2, y + 2, fill_w, bar_h - 4)
    else:
        fill_rect = pygame.Rect(x + bar_w - 2 - fill_w, y + 2, fill_w, bar_h - 4)
    pygame.draw.rect(surf, info["color"], fill_rect, border_radius=2)
    armor_label = font_tiny.render(f"{info['name']} Armor ({cur_dur})", True, info["color"])
    if align_left:
        surf.blit(armor_label, (x, y - 14))
    else:
        surf.blit(armor_label, (x + bar_w - armor_label.get_width(), y - 14))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, y, wtype))


def spawn_armor(armors):
    if len(armors) >= MAX_ARMOR_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    atype = random.choices(list(ARMOR_TYPES.keys()), weights=[5, 3, 1], k=1)[0]
    armors.append(ArmorPickup(x, y, atype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    global PLATFORMS
    PLATFORMS = _initial_platforms()
    p1 = Fighter(210, RED, DARK_RED, 1, (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g), "Player 1")
    p1.y = 450
    p2 = Fighter(790, BLUE, DARK_BLUE, -1, (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l), "Player 2")
    p2.y = 450
    p1.wins = p1_wins
    p2.wins = p2_wins
    if p1_start_weapon:
        info = WEAPON_TYPES[p1_start_weapon]
        p1.weapon = {
            "type": p1_start_weapon,
            "durability": info["durability"],
            "current_mag": info.get("mag_size", 0),
        }
    if p2_start_weapon:
        info = WEAPON_TYPES[p2_start_weapon]
        p2.weapon = {
            "type": p2_start_weapon,
            "durability": info["durability"],
            "current_mag": info.get("mag_size", 0),
        }
    weapons, armors, bombs, explosions, arrows, debris = [], [], [], [], [], []
    spawn_weapon(weapons)
    spawn_armor(armors)
    return p1, p2, weapons, armors, arrows, bombs, explosions, debris


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [(cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "uzi":
        pygame.draw.rect(surf, (45, 45, 50), (cx - 22, cy - 8, 44, 16), border_radius=3)
        pygame.draw.line(surf, (25, 25, 28), (cx - 4, cy + 5), (cx - 10, cy + 25), 7)
        pygame.draw.line(surf, (25, 25, 28), (cx + 20, cy), (cx + 34, cy), 5)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)
    elif wtype == "bomb":
        pygame.draw.circle(surf, (40, 40, 40), (cx, cy), 14)
        pygame.draw.line(surf, (100, 100, 100), (cx, cy), (cx + 8, cy - 12), 3)
        pygame.draw.circle(surf, (255, 200, 50), (cx + 8, cy - 12), 4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220
    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))
        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))
    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE
    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))
    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)
    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))
    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))
    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))
    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)
    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render("Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    state = "select"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0
    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, armors, arrows, bombs, explosions, debris = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL
    armor_spawn_timer = ARMOR_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True
                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True
                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, armors, arrows, bombs, explosions, debris = reset_fighters(
                        p1_wins, p2_wins, WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    armor_spawn_timer = ARMOR_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, armors, arrows, bombs)
            p2.handle_input(keys, p1, weapons, armors, arrows, bombs)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            for bomb in bombs[:]:
                bomb.update(PLATFORMS)
                if bomb.exploded:
                    explosions.append(bomb.explode())
                    bombs.remove(bomb)
            for exp in explosions[:]:
                exp.update(p1, p2, debris)
                if exp.life <= 0:
                    explosions.remove(exp)
            for d in debris[:]:
                d.update()
                if d.life <= 0:
                    debris.remove(d)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL
            armor_spawn_timer -= 1
            if armor_spawn_timer <= 0:
                spawn_armor(armors)
                armor_spawn_timer = ARMOR_SPAWN_INTERVAL

            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        for ap in armors:
            ap.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)
        for bomb in bombs:
            bomb.draw(screen)
        for exp in explosions:
            exp.draw(screen)
        for d in debris:
            d.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)
        draw_armor_bar(screen, 30, 62, p1, align_left=True)
        draw_armor_bar(screen, WIDTH - 30 - 320, 62, p2, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        plat_count = font_tiny.render(f"Platforms: {len(PLATFORMS)}", True, WHITE)
        screen.blit(plat_count, (WIDTH // 2 - plat_count.get_width() // 2, 50))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))
            msg = "DRAW!" if winner == "Draw" else f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))
            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Guns reload after 4 shots!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [25]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons, 
floating platforms, bombs that DESTROY platforms, ARMOR PICKUPS, and GUN RELOADING!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

Weapons (uzi, sword, bat, spear, bow, gatling, bomb) AND armor (light/medium/
heavy) spawn randomly on the platforms during the match. Walk over a pickup 
to grab it.

GUN RELOADING: The uzi, gatling, and bow now have 50-round magazines! After 
firing 50 shots, the gun auto-reloads for ~1 second — you can move and jump 
during reload, but can't fire. Watch your ammo count above your head!

Armor absorbs damage from hits — each tier reduces incoming damage and has 
limited durability before it shatters.

The arena consists of MULTIPLE floating platforms above a deadly lava/spike 
pit. BOMB EXPLOSIONS DESTROY PLATFORMS. Falling into the pit deals 40 HP 
damage but isn't instant death.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None
explosion_sound = None
crumble_sound = None
armor_pickup_sound = None
reload_sound = None
empty_click_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_explosion_sound():
        duration = 0.45
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 120 * (1 - progress * 0.8)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.7 * noise + 0.3 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_crumble_sound():
        duration = 0.6
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.2
            freq = 180 * (1 - progress * 0.5) + 40 * math.sin(2 * math.pi * 8 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.85 * noise + 0.15 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_armor_pickup_sound():
        duration = 0.22
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 1200 - 400 * progress
            tone = math.sin(2 * math.pi * freq * t)
            tone2 = math.sin(2 * math.pi * (freq * 1.5) * t) * 0.5
            value = int(amplitude * envelope * (tone + tone2) * 0.6)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_reload_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.3
            click1 = math.sin(2 * math.pi * 1800 * t) * (1 if progress < 0.1 else 0)
            click2 = math.sin(2 * math.pi * 2200 * t) * (1 if 0.45 < progress < 0.55 else 0)
            metallic = math.sin(2 * math.pi * 900 * t) * 0.3
            mix = click1 + click2 + metallic
            value = int(amplitude * envelope * mix * 0.5)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_empty_click_sound():
        duration = 0.08
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 3
            freq = 3500 * (1 - progress * 0.5)
            tone = math.sin(2 * math.pi * freq * t)
            value = int(amplitude * envelope * tone * 0.4)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
    explosion_sound = _generate_explosion_sound()
    crumble_sound = _generate_crumble_sound()
    armor_pickup_sound = _generate_armor_pickup_sound()
    reload_sound = _generate_reload_sound()
    empty_click_sound = _generate_empty_click_sound()
except pygame.error:
    break_sound = None
    hazard_sound = None
    explosion_sound = None
    crumble_sound = None
    armor_pickup_sound = None
    reload_sound = None
    empty_click_sound = None


def play_break_sound():
    if break_sound: break_sound.play()
def play_hazard_sound():
    if hazard_sound: hazard_sound.play()
def play_explosion_sound():
    if explosion_sound: explosion_sound.play()
def play_crumble_sound():
    if crumble_sound: crumble_sound.play()
def play_armor_pickup_sound():
    if armor_pickup_sound: armor_pickup_sound.play()
def play_reload_sound():
    if reload_sound: reload_sound.play()
def play_empty_click_sound():
    if empty_click_sound: empty_click_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
GOLD = (230, 190, 60)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

HAZARD_Y = GROUND_Y + 90
HAZARD_DAMAGE = 40
HAZARD_STUN = 24

RELOAD_TIME = 60

def _initial_platforms():
    return [
        pygame.Rect(150, 450, 120, 18),
        pygame.Rect(730, 450, 120, 18),
        pygame.Rect(280, 400, 140, 20),
        pygame.Rect(580, 400, 140, 20),
        pygame.Rect(430, 360, 140, 20),
        pygame.Rect(200, 320, 130, 18),
        pygame.Rect(670, 320, 130, 18),
        pygame.Rect(440, 280, 120, 18),
        pygame.Rect(320, 220, 110, 16),
        pygame.Rect(570, 220, 110, 16),
        pygame.Rect(450, 180, 100, 16),
    ]

PLATFORMS = _initial_platforms()

PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

# All guns now have 50-round magazines!
WEAPON_TYPES = {
    "uzi":     {"reach": 0,   "damage": 5, "cooldown": 5, "durability": 50, "color": (45, 45, 50),
                "ranged": True, "proj_speed": 28, "proj_kind": "bullet", "mag_size": 50},
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow", "mag_size": 50},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet", "mag_size": 50},
    "bomb":    {"reach": 0,   "damage": 35, "cooldown": 50, "durability": 3,  "color": (40, 40, 40),
                "ranged": True, "proj_speed": 9, "proj_kind": "bomb"},
}

ARMOR_TYPES = {
    "light":  {"absorb": 15, "durability": 3, "color": (120, 220, 120), "name": "Light"},
    "medium": {"absorb": 30, "durability": 4, "color": (120, 170, 240), "name": "Medium"},
    "heavy":  {"absorb": 50, "durability": 5, "color": (240, 200, 80),  "name": "Heavy"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

MAX_WEAPONS_ON_FIELD = 2
MAX_ARMOR_ON_FIELD = 1
WEAPON_SPAWN_INTERVAL = 300
ARMOR_SPAWN_INTERVAL = 480

WEAPON_CHOICES = [None, "uzi", "sword", "bat", "spear", "bow", "gatling", "bomb"]

STAGES = [
    {"name": "Meadow", "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245), "ground": (110, 90, 70), "ground_edge": (80, 150, 80), "decor": "meadow"},
    {"name": "Desert", "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190), "ground": (200, 165, 100), "ground_edge": (225, 195, 130), "decor": "desert"},
    {"name": "Night City", "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90), "ground": (40, 40, 50), "ground_edge": (90, 90, 110), "decor": "city"},
    {"name": "Volcano", "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30), "ground": (50, 35, 30), "ground_edge": (200, 80, 30), "decor": "volcano"},
    {"name": "Snow Peak", "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245), "ground": (225, 235, 240), "ground_edge": (255, 255, 255), "decor": "snow"},
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return
        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class Bomb:
    def __init__(self, x, y, facing, damage, owner):
        self.x = x
        self.y = y
        self.vel_x = facing * 9
        self.vel_y = -11
        self.damage = damage
        self.owner = owner
        self.timer = 70
        self.dead = False
        self.exploded = False

    def update(self, platforms):
        if self.exploded:
            return
        self.vel_y += GRAVITY
        self.x += self.vel_x
        self.y += self.vel_y
        self.timer -= 1
        for plat in platforms:
            if plat.left <= self.x <= plat.right and self.y >= plat.top and self.vel_y >= 0:
                if self.y - self.vel_y <= plat.top + 10:
                    self.y = plat.top
                    self.vel_y = 0
                    self.vel_x *= 0.7
                    self.timer -= 3
        if self.y >= GROUND_Y or self.timer <= 0:
            self.exploded = True
            self.dead = True

    def explode(self):
        play_explosion_sound()
        return Explosion(self.x, self.y, self.damage, self.owner)

    def draw(self, surf):
        pygame.draw.circle(surf, (30, 30, 30), (int(self.x), int(self.y)), 8)
        if self.timer % 8 < 4:
            pygame.draw.circle(surf, (255, 200, 50), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 3)
        pygame.draw.line(surf, (100, 100, 100), (int(self.x), int(self.y)), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 2)


class Debris:
    def __init__(self, x, y, color):
        self.x = x
        self.y = y
        self.vel_x = random.uniform(-8, 8)
        self.vel_y = random.uniform(-10, -2)
        self.size = random.randint(4, 10)
        self.color = color
        self.rotation = random.uniform(0, 360)
        self.rot_speed = random.uniform(-15, 15)
        self.life = random.randint(40, 70)

    def update(self):
        self.vel_y += GRAVITY * 0.8
        self.x += self.vel_x
        self.y += self.vel_y
        self.rotation += self.rot_speed
        self.life -= 1
        self.vel_x *= 0.98

    def draw(self, surf):
        if self.life <= 0:
            return
        alpha = min(255, self.life * 6)
        surf_deb = pygame.Surface((self.size * 2, self.size * 2), pygame.SRCALPHA)
        rect = pygame.Rect(0, 0, self.size, self.size)
        pygame.draw.rect(surf_deb, (*self.color, alpha), rect)
        pygame.draw.rect(surf_deb, (0, 0, 0, alpha), rect, 1)
        rotated = pygame.transform.rotate(surf_deb, self.rotation)
        new_rect = rotated.get_rect(center=(int(self.x), int(self.y)))
        surf.blit(rotated, new_rect)


class Explosion:
    def __init__(self, x, y, damage, owner):
        self.x = x
        self.y = y
        self.damage = damage
        self.owner = owner
        self.radius = 10
        self.max_radius = 110
        self.life = 20
        self.has_damaged = False
        self.has_destroyed_platforms = False

    def update(self, p1, p2, debris_list):
        self.life -= 1
        if self.life > 10:
            self.radius += (self.max_radius - self.radius) * 0.4
        else:
            self.radius += (self.max_radius - self.radius) * 0.1
        if not self.has_damaged and self.life == 19:
            for target in (p1, p2):
                target_cx = target.x
                target_cy = target.y - target.height / 2
                dist = math.hypot(target_cx - self.x, target_cy - self.y)
                if dist < self.max_radius:
                    from_left = (self.x < target.x)
                    target.take_hit(self.damage, from_left)
                    distance_factor = 1.0 - (dist / self.max_radius)
                    base_push = 120
                    base_pop = -22
                    push = (base_push * distance_factor) if from_left else -(base_push * distance_factor)
                    target.x = max(target.width, min(WIDTH - target.width, target.x + push))
                    target.vel_y = base_pop * distance_factor
            self.has_damaged = True
        if not self.has_destroyed_platforms and self.life == 17:
            global PLATFORMS
            platforms_to_destroy = []
            for plat in PLATFORMS:
                closest_x = max(plat.left, min(self.x, plat.right))
                closest_y = max(plat.top, min(self.y, plat.bottom))
                dist = math.hypot(closest_x - self.x, closest_y - self.y)
                if dist < self.max_radius:
                    platforms_to_destroy.append(plat)
            if len(PLATFORMS) - len(platforms_to_destroy) >= 1:
                for plat in platforms_to_destroy:
                    plat_color = (110, 90, 70)
                    for _ in range(18):
                        chunk_x = random.uniform(plat.left, plat.right)
                        chunk_y = random.uniform(plat.top, plat.bottom)
                        debris_list.append(Debris(chunk_x, chunk_y, plat_color))
                    PLATFORMS.remove(plat)
                    play_crumble_sound()
            self.has_destroyed_platforms = True

    def draw(self, surf):
        if self.life > 0:
            alpha = int(255 * (self.life / 20))
            surf_exp = pygame.Surface((self.max_radius * 2, self.max_radius * 2), pygame.SRCALPHA)
            pygame.draw.circle(surf_exp, (255, 80, 20, alpha), (self.max_radius, self.max_radius), int(self.radius))
            pygame.draw.circle(surf_exp, (255, 200, 50, int(alpha * 0.8)), (self.max_radius, self.max_radius), int(self.radius * 0.6))
            pygame.draw.circle(surf_exp, (255, 255, 220, int(alpha * 0.9)), (self.max_radius, self.max_radius), int(self.radius * 0.25))
            surf.blit(surf_exp, (int(self.x - self.max_radius), int(self.y - self.max_radius)))


class WeaponPickup:
    def __init__(self, x, y, wtype):
        self.x = x
        self.y = y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [(self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        elif self.wtype == "uzi":
            pygame.draw.rect(surf, color, (self.x - 12, self.y - 26, 24, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (self.x - 2, self.y - 15), (self.x - 6, self.y - 4), 5)
        elif self.wtype == "bomb":
            pygame.draw.circle(surf, (30, 30, 30), (self.x, self.y - 20), 10)
            pygame.draw.line(surf, (100, 100, 100), (self.x, self.y - 20), (self.x + 6, self.y - 30), 2)
            pygame.draw.circle(surf, (255, 200, 50), (self.x + 6, self.y - 30), 3)
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class ArmorPickup:
    def __init__(self, x, y, atype):
        self.x = x
        self.y = y
        self.atype = atype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = ARMOR_TYPES[self.atype]
        color = info["color"]
        glow = pygame.Surface((80, 80), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*color, 80), (40, 40), 34)
        surf.blit(glow, (self.x - 40, self.y - 60))

        chest_points = [
            (self.x - 14, self.y - 38),
            (self.x + 14, self.y - 38),
            (self.x + 16, self.y - 18),
            (self.x + 10, self.y - 8),
            (self.x - 10, self.y - 8),
            (self.x - 16, self.y - 18),
        ]
        pygame.draw.polygon(surf, color, chest_points)
        pygame.draw.polygon(surf, BLACK, chest_points, 2)
        pygame.draw.line(surf, color, (self.x - 12, self.y - 38), (self.x - 14, self.y - 44), 4)
        pygame.draw.line(surf, color, (self.x + 12, self.y - 38), (self.x + 14, self.y - 44), 4)
        pygame.draw.circle(surf, BLACK, (int(self.x), int(self.y - 24)), 3)

        label = font_tiny.render(info["name"] + " Armor", True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 64))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing
        self.controls = controls
        self.name = name
        self.health = MAX_HEALTH
        self.on_ground = True
        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0
        self.attack_type = None
        self.hit_stun = 0
        self.walk_cycle = 0
        self.moving = False
        self.wins = 0
        self.weapon = None
        self.armor = None
        self.armor_flash = 0
        self.reload_timer = 0

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height, self.width, self.height)

    def current_stats(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup_weapon(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {
                    "type": wp.wtype,
                    "durability": info["durability"],
                    "current_mag": info.get("mag_size", 0),
                }
                weapons.remove(wp)
                return

    def try_pickup_armor(self, armors):
        for ap in armors:
            if ap.rect().colliderect(self.rect()):
                info = ARMOR_TYPES[ap.atype]
                self.armor = {"type": ap.atype, "durability": info["durability"]}
                self.armor_flash = 12
                armors.remove(ap)
                play_armor_pickup_sound()
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def is_reloading(self):
        return self.reload_timer > 0

    def start_reload(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            if info.get("mag_size", 0) > 0:
                self.reload_timer = RELOAD_TIME
                play_reload_sound()

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)
        elif self.weapon["type"] == "uzi":
            spawn_y += random.randint(-3, 3)
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))
        if "current_mag" in self.weapon:
            self.weapon["current_mag"] -= 1
            if self.weapon["current_mag"] <= 0:
                self.start_reload()

    def throw_bomb(self, bombs):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 20
        spawn_y = self.y - self.height * 0.7
        bombs.append(Bomb(spawn_x, spawn_y, self.facing, info["damage"], self))
        self.weapon["durability"] -= 1
        if self.weapon["durability"] <= 0:
            self.weapon = None
            play_break_sound()

    def handle_input(self, keys, opponent, weapons, armors, arrows, bombs):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.is_reloading():
                pass
            elif self.weapon and self.weapon["type"] == "bomb":
                self.attack_type = "throw"
                self.attack_anim = 15
                self.throw_bomb(bombs)
            elif self.is_ranged():
                if "current_mag" in self.weapon and self.weapon["current_mag"] <= 0:
                    self.start_reload()
                    play_empty_click_sound()
                else:
                    self.attack_type = "shoot"
                    self.attack_anim = 10
                    self.shoot(arrows)
                    self.weapon["durability"] -= 1
                    if self.weapon["durability"] <= 0:
                        self.weapon = None
                        play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup_weapon(weapons)
        self.try_pickup_armor(armors)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y

        self.on_ground = False
        for plat in PLATFORMS:
            prev_y = self.y - self.vel_y
            if prev_y <= plat.top and self.y >= plat.top and plat.left <= self.x <= plat.right and self.vel_y >= 0:
                self.y = plat.top
                self.vel_y = 0
                self.on_ground = True
                break

        if not self.on_ground:
            if self.y >= HAZARD_Y:
                self.hazard_hit()

        if self.punch_cd > 0: self.punch_cd -= 1
        if self.kick_cd > 0: self.kick_cd -= 1
        if self.armor_flash > 0: self.armor_flash -= 1

        if self.reload_timer > 0:
            self.reload_timer -= 1
            if self.reload_timer == 0 and self.weapon:
                info = WEAPON_TYPES[self.weapon["type"]]
                if "current_mag" in self.weapon:
                    self.weapon["current_mag"] = info.get("mag_size", 0)

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            absorbed = min(damage, info["absorb"])
            damage = damage - absorbed
            self.armor["durability"] -= 1
            self.armor_flash = 10
            if self.armor["durability"] <= 0:
                self.armor = None
                play_break_sound()
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        if PLATFORMS:
            best = min(PLATFORMS, key=lambda p: abs(p.centerx - self.x))
            self.x = best.centerx
            self.y = best.top
        else:
            if self.x < WIDTH / 2:
                self.x = 325
            else:
                self.x = 675
            self.y = 380
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            aura_color = info["color"]
            flash = self.armor_flash > 0
            aura_surf = pygame.Surface((90, 130), pygame.SRCALPHA)
            alpha = 90 if not flash else 180
            pygame.draw.ellipse(aura_surf, (*aura_color, alpha), (0, 0, 90, 130))
            if flash:
                pygame.draw.ellipse(aura_surf, (255, 255, 255, 120), (10, 10, 70, 110))
            surf.blit(aura_surf, (cx - 45, self.y - self.height - 10))

        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        elif self.attack_type == "throw":
            fwd_hand = (cx + self.facing * 20, shoulder_y - 10)
            back_hand = (cx - self.facing * 10, shoulder_y + 10)
        elif self.is_reloading():
            fwd_hand = (cx + self.facing * 8, shoulder_y + 30)
            back_hand = (cx - self.facing * 6, shoulder_y + 28)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            ac = info["color"]
            chest_points = [
                (cx - 12, shoulder_y - 2),
                (cx + 12, shoulder_y - 2),
                (cx + 14, shoulder_y + 18),
                (cx + 8, shoulder_y + 26),
                (cx - 8, shoulder_y + 26),
                (cx - 14, shoulder_y + 18),
            ]
            pygame.draw.polygon(surf, ac, chest_points)
            pygame.draw.polygon(surf, BLACK, chest_points, 2)
            pygame.draw.circle(surf, BLACK, (cx, shoulder_y + 12), 2)

        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "uzi":
            wcolor = WEAPON_TYPES["uzi"]["color"]
            ux, uy = fwd_hand[0], fwd_hand[1] - 3
            barrel_tip = (ux + self.facing * 27, uy)
            grip_bottom = (ux - self.facing * 5, uy + 18)
            pygame.draw.rect(surf, wcolor, pygame.Rect(min(ux, barrel_tip[0]), uy - 7, abs(barrel_tip[0] - ux) + 8, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (ux, uy + 4), grip_bottom, 6)
            pygame.draw.line(surf, (25, 25, 28), barrel_tip, (barrel_tip[0] + self.facing * 10, barrel_tip[1]), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 8, barrel_tip[1])
                pygame.draw.circle(surf, (255, 220, 90), (int(flash[0]), int(flash[1])), 6)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [tipend, (tip[0] + perp[0], tip[1] + perp[1]), (tip[0] - perp[0], tip[1] - perp[1])])
            else:
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK, (guard_center[0] + perp[0], guard_center[1] + perp[1]), (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        label_y = int(head_y) - head_r - 20
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            mag_size = info.get("mag_size", 0)
            if mag_size > 0 and "current_mag" in self.weapon:
                cur = self.weapon["current_mag"]
                mag_color = YELLOW if cur > 0 else RED
                mag_label = font_tiny.render(f"[{cur}/{mag_size}]", True, mag_color)
                surf.blit(mag_label, (cx - mag_label.get_width() // 2, label_y))
                label_y -= 14
            label = font_tiny.render(f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}", True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, label_y))
            label_y -= 16

        if self.is_reloading():
            reload_label = font_tiny.render("RELOADING...", True, YELLOW)
            pygame.draw.rect(surf, BLACK, (cx - reload_label.get_width() // 2 - 2, label_y - 1,
                                            reload_label.get_width() + 4, reload_label.get_height() + 2))
            surf.blit(reload_label, (cx - reload_label.get_width() // 2, label_y))
            label_y -= 16

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            label = font_tiny.render(f"{info['name']} Armor x{self.armor['durability']}", True, info["color"])
            pygame.draw.rect(surf, BLACK, (cx - label.get_width() // 2 - 2, label_y - 1, label.get_width() + 4, label.get_height() + 2))
            surf.blit(label, (cx - label.get_width() // 2, label_y))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]
    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    draw_hazard_pit(surf)

    for plat in PLATFORMS:
        pygame.draw.rect(surf, stage["ground"], plat)
        pygame.draw.rect(surf, stage["ground_edge"], (plat.left, plat.top, plat.width, 10))
        pygame.draw.polygon(surf, stage["ground"], [(plat.left, plat.bottom), (plat.left + 12, plat.bottom + 18), (plat.left + 24, plat.bottom)])
        pygame.draw.polygon(surf, stage["ground"], [(plat.right, plat.bottom), (plat.right - 12, plat.bottom + 18), (plat.right - 24, plat.bottom)])


def draw_hazard_pit(surf):
    pit_rect = pygame.Rect(0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)
    for i, (band_y, color) in enumerate([(GROUND_Y + 15, (150, 40, 15)), (GROUND_Y + 35, (200, 70, 20)), (GROUND_Y + 55, (240, 110, 30))]):
        for gx in range(0, WIDTH, 26):
            wobble = math.sin((gx + i * 40) * 0.15) * 4
            pygame.draw.circle(surf, color, (gx + 13, int(band_y + wobble)), 9)
    for gx in range(0, WIDTH, 52):
        pygame.draw.circle(surf, (255, 200, 90), (gx + 26, GROUND_Y + 60), 4)
    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [(gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)])
        pygame.draw.polygon(surf, (90, 90, 95), [(gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)
    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)
    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def draw_armor_bar(surf, x, y, fighter, align_left=True):
    if not fighter.armor:
        return
    info = ARMOR_TYPES[fighter.armor["type"]]
    max_dur = info["durability"]
    cur_dur = fighter.armor["durability"]
    bar_w, bar_h = 320, 10
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=3)
    fill_w = int((bar_w - 4) * (cur_dur / max_dur))
    if align_left:
        fill_rect = pygame.Rect(x + 2, y + 2, fill_w, bar_h - 4)
    else:
        fill_rect = pygame.Rect(x + bar_w - 2 - fill_w, y + 2, fill_w, bar_h - 4)
    pygame.draw.rect(surf, info["color"], fill_rect, border_radius=2)
    armor_label = font_tiny.render(f"{info['name']} Armor ({cur_dur})", True, info["color"])
    if align_left:
        surf.blit(armor_label, (x, y - 14))
    else:
        surf.blit(armor_label, (x + bar_w - armor_label.get_width(), y - 14))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, y, wtype))


def spawn_armor(armors):
    if len(armors) >= MAX_ARMOR_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    atype = random.choices(list(ARMOR_TYPES.keys()), weights=[5, 3, 1], k=1)[0]
    armors.append(ArmorPickup(x, y, atype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    global PLATFORMS
    PLATFORMS = _initial_platforms()
    p1 = Fighter(210, RED, DARK_RED, 1, (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g), "Player 1")
    p1.y = 450
    p2 = Fighter(790, BLUE, DARK_BLUE, -1, (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l), "Player 2")
    p2.y = 450
    p1.wins = p1_wins
    p2.wins = p2_wins
    if p1_start_weapon:
        info = WEAPON_TYPES[p1_start_weapon]
        p1.weapon = {
            "type": p1_start_weapon,
            "durability": info["durability"],
            "current_mag": info.get("mag_size", 0),
        }
    if p2_start_weapon:
        info = WEAPON_TYPES[p2_start_weapon]
        p2.weapon = {
            "type": p2_start_weapon,
            "durability": info["durability"],
            "current_mag": info.get("mag_size", 0),
        }
    weapons, armors, bombs, explosions, arrows, debris = [], [], [], [], [], []
    spawn_weapon(weapons)
    spawn_armor(armors)
    return p1, p2, weapons, armors, arrows, bombs, explosions, debris


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [(cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "uzi":
        pygame.draw.rect(surf, (45, 45, 50), (cx - 22, cy - 8, 44, 16), border_radius=3)
        pygame.draw.line(surf, (25, 25, 28), (cx - 4, cy + 5), (cx - 10, cy + 25), 7)
        pygame.draw.line(surf, (25, 25, 28), (cx + 20, cy), (cx + 34, cy), 5)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)
    elif wtype == "bomb":
        pygame.draw.circle(surf, (40, 40, 40), (cx, cy), 14)
        pygame.draw.line(surf, (100, 100, 100), (cx, cy), (cx + 8, cy - 12), 3)
        pygame.draw.circle(surf, (255, 200, 50), (cx + 8, cy - 12), 4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220
    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))
        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))
    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE
    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))
    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)
    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))
    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))
    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))
    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)
    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render("Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    state = "select"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0
    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, armors, arrows, bombs, explosions, debris = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL
    armor_spawn_timer = ARMOR_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True
                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True
                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, armors, arrows, bombs, explosions, debris = reset_fighters(
                        p1_wins, p2_wins, WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    armor_spawn_timer = ARMOR_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, armors, arrows, bombs)
            p2.handle_input(keys, p1, weapons, armors, arrows, bombs)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            for bomb in bombs[:]:
                bomb.update(PLATFORMS)
                if bomb.exploded:
                    explosions.append(bomb.explode())
                    bombs.remove(bomb)
            for exp in explosions[:]:
                exp.update(p1, p2, debris)
                if exp.life <= 0:
                    explosions.remove(exp)
            for d in debris[:]:
                d.update()
                if d.life <= 0:
                    debris.remove(d)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL
            armor_spawn_timer -= 1
            if armor_spawn_timer <= 0:
                spawn_armor(armors)
                armor_spawn_timer = ARMOR_SPAWN_INTERVAL

            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        for ap in armors:
            ap.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)
        for bomb in bombs:
            bomb.draw(screen)
        for exp in explosions:
            exp.draw(screen)
        for d in debris:
            d.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)
        draw_armor_bar(screen, 30, 62, p1, align_left=True)
        draw_armor_bar(screen, WIDTH - 30 - 320, 62, p2, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        plat_count = font_tiny.render(f"Platforms: {len(PLATFORMS)}", True, WHITE)
        screen.blit(plat_count, (WIDTH // 2 - plat_count.get_width() // 2, 50))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))
            msg = "DRAW!" if winner == "Draw" else f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))
            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Guns have 50-round mags!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [26]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons, 
floating platforms, bombs that DESTROY platforms, ARMOR PICKUPS, GUN RELOADING, 
and RISING LAVA!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

Weapons (uzi, sword, bat, spear, bow, gatling, bomb) AND armor (light/medium/
heavy) spawn randomly on the platforms during the match. Walk over a pickup 
to grab it.

GUN RELOADING: The uzi, gatling, and bow now have 50-round magazines! After 
firing 50 shots, the gun auto-reloads for ~1 second.

RISING LAVA: Every 10 seconds, the lava pit rises by 25 pixels, shrinking the 
safe arena and forcing players higher. A warning flashes 2 seconds before each 
rise! Falling into the lava deals 40 HP damage.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None
explosion_sound = None
crumble_sound = None
armor_pickup_sound = None
reload_sound = None
empty_click_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_explosion_sound():
        duration = 0.45
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 120 * (1 - progress * 0.8)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.7 * noise + 0.3 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_crumble_sound():
        duration = 0.6
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.2
            freq = 180 * (1 - progress * 0.5) + 40 * math.sin(2 * math.pi * 8 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.85 * noise + 0.15 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_armor_pickup_sound():
        duration = 0.22
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 1200 - 400 * progress
            tone = math.sin(2 * math.pi * freq * t)
            tone2 = math.sin(2 * math.pi * (freq * 1.5) * t) * 0.5
            value = int(amplitude * envelope * (tone + tone2) * 0.6)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_reload_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.3
            click1 = math.sin(2 * math.pi * 1800 * t) * (1 if progress < 0.1 else 0)
            click2 = math.sin(2 * math.pi * 2200 * t) * (1 if 0.45 < progress < 0.55 else 0)
            metallic = math.sin(2 * math.pi * 900 * t) * 0.3
            mix = click1 + click2 + metallic
            value = int(amplitude * envelope * mix * 0.5)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_empty_click_sound():
        duration = 0.08
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 3
            freq = 3500 * (1 - progress * 0.5)
            tone = math.sin(2 * math.pi * freq * t)
            value = int(amplitude * envelope * tone * 0.4)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
    explosion_sound = _generate_explosion_sound()
    crumble_sound = _generate_crumble_sound()
    armor_pickup_sound = _generate_armor_pickup_sound()
    reload_sound = _generate_reload_sound()
    empty_click_sound = _generate_empty_click_sound()
except pygame.error:
    break_sound = None
    hazard_sound = None
    explosion_sound = None
    crumble_sound = None
    armor_pickup_sound = None
    reload_sound = None
    empty_click_sound = None


def play_break_sound():
    if break_sound: break_sound.play()
def play_hazard_sound():
    if hazard_sound: hazard_sound.play()
def play_explosion_sound():
    if explosion_sound: explosion_sound.play()
def play_crumble_sound():
    if crumble_sound: crumble_sound.play()
def play_armor_pickup_sound():
    if armor_pickup_sound: armor_pickup_sound.play()
def play_reload_sound():
    if reload_sound: reload_sound.play()
def play_empty_click_sound():
    if empty_click_sound: empty_click_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
GOLD = (230, 190, 60)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

# Lava mechanics
LAVA_RISE_INTERVAL = 600  # 10 seconds at 60 FPS
LAVA_RISE_AMOUNT = 25     # Pixels the lava rises each interval
current_lava_y = GROUND_Y
lava_rise_timer = LAVA_RISE_INTERVAL
lava_rise_warning = 0     # Frames to show warning before rise

HAZARD_DAMAGE = 40
HAZARD_STUN = 24

RELOAD_TIME = 60

def _initial_platforms():
    return [
        pygame.Rect(150, 450, 120, 18),
        pygame.Rect(730, 450, 120, 18),
        pygame.Rect(280, 400, 140, 20),
        pygame.Rect(580, 400, 140, 20),
        pygame.Rect(430, 360, 140, 20),
        pygame.Rect(200, 320, 130, 18),
        pygame.Rect(670, 320, 130, 18),
        pygame.Rect(440, 280, 120, 18),
        pygame.Rect(320, 220, 110, 16),
        pygame.Rect(570, 220, 110, 16),
        pygame.Rect(450, 180, 100, 16),
    ]

PLATFORMS = _initial_platforms()

PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

WEAPON_TYPES = {
    "uzi":     {"reach": 0,   "damage": 5, "cooldown": 5, "durability": 50, "color": (45, 45, 50),
                "ranged": True, "proj_speed": 28, "proj_kind": "bullet", "mag_size": 50},
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow", "mag_size": 50},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet", "mag_size": 50},
    "bomb":    {"reach": 0,   "damage": 35, "cooldown": 50, "durability": 3,  "color": (40, 40, 40),
                "ranged": True, "proj_speed": 9, "proj_kind": "bomb"},
}

ARMOR_TYPES = {
    "light":  {"absorb": 15, "durability": 3, "color": (120, 220, 120), "name": "Light"},
    "medium": {"absorb": 30, "durability": 4, "color": (120, 170, 240), "name": "Medium"},
    "heavy":  {"absorb": 50, "durability": 5, "color": (240, 200, 80),  "name": "Heavy"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

MAX_WEAPONS_ON_FIELD = 2
MAX_ARMOR_ON_FIELD = 1
WEAPON_SPAWN_INTERVAL = 300
ARMOR_SPAWN_INTERVAL = 480

WEAPON_CHOICES = [None, "uzi", "sword", "bat", "spear", "bow", "gatling", "bomb"]

STAGES = [
    {"name": "Meadow", "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245), "ground": (110, 90, 70), "ground_edge": (80, 150, 80), "decor": "meadow"},
    {"name": "Desert", "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190), "ground": (200, 165, 100), "ground_edge": (225, 195, 130), "decor": "desert"},
    {"name": "Night City", "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90), "ground": (40, 40, 50), "ground_edge": (90, 90, 110), "decor": "city"},
    {"name": "Volcano", "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30), "ground": (50, 35, 30), "ground_edge": (200, 80, 30), "decor": "volcano"},
    {"name": "Snow Peak", "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245), "ground": (225, 235, 240), "ground_edge": (255, 255, 255), "decor": "snow"},
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


class Arrow:
    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return
        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class Bomb:
    def __init__(self, x, y, facing, damage, owner):
        self.x = x
        self.y = y
        self.vel_x = facing * 9
        self.vel_y = -11
        self.damage = damage
        self.owner = owner
        self.timer = 70
        self.dead = False
        self.exploded = False

    def update(self, platforms):
        if self.exploded:
            return
        self.vel_y += GRAVITY
        self.x += self.vel_x
        self.y += self.vel_y
        self.timer -= 1
        for plat in platforms:
            if plat.left <= self.x <= plat.right and self.y >= plat.top and self.vel_y >= 0:
                if self.y - self.vel_y <= plat.top + 10:
                    self.y = plat.top
                    self.vel_y = 0
                    self.vel_x *= 0.7
                    self.timer -= 3
        if self.y >= current_lava_y or self.timer <= 0:
            self.exploded = True
            self.dead = True

    def explode(self):
        play_explosion_sound()
        return Explosion(self.x, self.y, self.damage, self.owner)

    def draw(self, surf):
        pygame.draw.circle(surf, (30, 30, 30), (int(self.x), int(self.y)), 8)
        if self.timer % 8 < 4:
            pygame.draw.circle(surf, (255, 200, 50), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 3)
        pygame.draw.line(surf, (100, 100, 100), (int(self.x), int(self.y)), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 2)


class Debris:
    def __init__(self, x, y, color):
        self.x = x
        self.y = y
        self.vel_x = random.uniform(-8, 8)
        self.vel_y = random.uniform(-10, -2)
        self.size = random.randint(4, 10)
        self.color = color
        self.rotation = random.uniform(0, 360)
        self.rot_speed = random.uniform(-15, 15)
        self.life = random.randint(40, 70)

    def update(self):
        self.vel_y += GRAVITY * 0.8
        self.x += self.vel_x
        self.y += self.vel_y
        self.rotation += self.rot_speed
        self.life -= 1
        self.vel_x *= 0.98

    def draw(self, surf):
        if self.life <= 0:
            return
        alpha = min(255, self.life * 6)
        surf_deb = pygame.Surface((self.size * 2, self.size * 2), pygame.SRCALPHA)
        rect = pygame.Rect(0, 0, self.size, self.size)
        pygame.draw.rect(surf_deb, (*self.color, alpha), rect)
        pygame.draw.rect(surf_deb, (0, 0, 0, alpha), rect, 1)
        rotated = pygame.transform.rotate(surf_deb, self.rotation)
        new_rect = rotated.get_rect(center=(int(self.x), int(self.y)))
        surf.blit(rotated, new_rect)


class Explosion:
    def __init__(self, x, y, damage, owner):
        self.x = x
        self.y = y
        self.damage = damage
        self.owner = owner
        self.radius = 10
        self.max_radius = 110
        self.life = 20
        self.has_damaged = False
        self.has_destroyed_platforms = False

    def update(self, p1, p2, debris_list):
        self.life -= 1
        if self.life > 10:
            self.radius += (self.max_radius - self.radius) * 0.4
        else:
            self.radius += (self.max_radius - self.radius) * 0.1
        if not self.has_damaged and self.life == 19:
            for target in (p1, p2):
                target_cx = target.x
                target_cy = target.y - target.height / 2
                dist = math.hypot(target_cx - self.x, target_cy - self.y)
                if dist < self.max_radius:
                    from_left = (self.x < target.x)
                    target.take_hit(self.damage, from_left)
                    distance_factor = 1.0 - (dist / self.max_radius)
                    base_push = 120
                    base_pop = -22
                    push = (base_push * distance_factor) if from_left else -(base_push * distance_factor)
                    target.x = max(target.width, min(WIDTH - target.width, target.x + push))
                    target.vel_y = base_pop * distance_factor
            self.has_damaged = True
        if not self.has_destroyed_platforms and self.life == 17:
            global PLATFORMS
            platforms_to_destroy = []
            for plat in PLATFORMS:
                closest_x = max(plat.left, min(self.x, plat.right))
                closest_y = max(plat.top, min(self.y, plat.bottom))
                dist = math.hypot(closest_x - self.x, closest_y - self.y)
                if dist < self.max_radius:
                    platforms_to_destroy.append(plat)
            if len(PLATFORMS) - len(platforms_to_destroy) >= 1:
                for plat in platforms_to_destroy:
                    plat_color = (110, 90, 70)
                    for _ in range(18):
                        chunk_x = random.uniform(plat.left, plat.right)
                        chunk_y = random.uniform(plat.top, plat.bottom)
                        debris_list.append(Debris(chunk_x, chunk_y, plat_color))
                    PLATFORMS.remove(plat)
                    play_crumble_sound()
            self.has_destroyed_platforms = True

    def draw(self, surf):
        if self.life > 0:
            alpha = int(255 * (self.life / 20))
            surf_exp = pygame.Surface((self.max_radius * 2, self.max_radius * 2), pygame.SRCALPHA)
            pygame.draw.circle(surf_exp, (255, 80, 20, alpha), (self.max_radius, self.max_radius), int(self.radius))
            pygame.draw.circle(surf_exp, (255, 200, 50, int(alpha * 0.8)), (self.max_radius, self.max_radius), int(self.radius * 0.6))
            pygame.draw.circle(surf_exp, (255, 255, 220, int(alpha * 0.9)), (self.max_radius, self.max_radius), int(self.radius * 0.25))
            surf.blit(surf_exp, (int(self.x - self.max_radius), int(self.y - self.max_radius)))


class WeaponPickup:
    def __init__(self, x, y, wtype):
        self.x = x
        self.y = y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [(self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        elif self.wtype == "uzi":
            pygame.draw.rect(surf, color, (self.x - 12, self.y - 26, 24, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (self.x - 2, self.y - 15), (self.x - 6, self.y - 4), 5)
        elif self.wtype == "bomb":
            pygame.draw.circle(surf, (30, 30, 30), (self.x, self.y - 20), 10)
            pygame.draw.line(surf, (100, 100, 100), (self.x, self.y - 20), (self.x + 6, self.y - 30), 2)
            pygame.draw.circle(surf, (255, 200, 50), (self.x + 6, self.y - 30), 3)
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class ArmorPickup:
    def __init__(self, x, y, atype):
        self.x = x
        self.y = y
        self.atype = atype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = ARMOR_TYPES[self.atype]
        color = info["color"]
        glow = pygame.Surface((80, 80), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*color, 80), (40, 40), 34)
        surf.blit(glow, (self.x - 40, self.y - 60))

        chest_points = [
            (self.x - 14, self.y - 38),
            (self.x + 14, self.y - 38),
            (self.x + 16, self.y - 18),
            (self.x + 10, self.y - 8),
            (self.x - 10, self.y - 8),
            (self.x - 16, self.y - 18),
        ]
        pygame.draw.polygon(surf, color, chest_points)
        pygame.draw.polygon(surf, BLACK, chest_points, 2)
        pygame.draw.line(surf, color, (self.x - 12, self.y - 38), (self.x - 14, self.y - 44), 4)
        pygame.draw.line(surf, color, (self.x + 12, self.y - 38), (self.x + 14, self.y - 44), 4)
        pygame.draw.circle(surf, BLACK, (int(self.x), int(self.y - 24)), 3)

        label = font_tiny.render(info["name"] + " Armor", True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 64))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing
        self.controls = controls
        self.name = name
        self.health = MAX_HEALTH
        self.on_ground = True
        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0
        self.attack_type = None
        self.hit_stun = 0
        self.walk_cycle = 0
        self.moving = False
        self.wins = 0
        self.weapon = None
        self.armor = None
        self.armor_flash = 0
        self.reload_timer = 0

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height, self.width, self.height)

    def current_stats(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup_weapon(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {
                    "type": wp.wtype,
                    "durability": info["durability"],
                    "current_mag": info.get("mag_size", 0),
                }
                weapons.remove(wp)
                return

    def try_pickup_armor(self, armors):
        for ap in armors:
            if ap.rect().colliderect(self.rect()):
                info = ARMOR_TYPES[ap.atype]
                self.armor = {"type": ap.atype, "durability": info["durability"]}
                self.armor_flash = 12
                armors.remove(ap)
                play_armor_pickup_sound()
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def is_reloading(self):
        return self.reload_timer > 0

    def start_reload(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            if info.get("mag_size", 0) > 0:
                self.reload_timer = RELOAD_TIME
                play_reload_sound()

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)
        elif self.weapon["type"] == "uzi":
            spawn_y += random.randint(-3, 3)
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))
        if "current_mag" in self.weapon:
            self.weapon["current_mag"] -= 1
            if self.weapon["current_mag"] <= 0:
                self.start_reload()

    def throw_bomb(self, bombs):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 20
        spawn_y = self.y - self.height * 0.7
        bombs.append(Bomb(spawn_x, spawn_y, self.facing, info["damage"], self))
        self.weapon["durability"] -= 1
        if self.weapon["durability"] <= 0:
            self.weapon = None
            play_break_sound()

    def handle_input(self, keys, opponent, weapons, armors, arrows, bombs):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.is_reloading():
                pass
            elif self.weapon and self.weapon["type"] == "bomb":
                self.attack_type = "throw"
                self.attack_anim = 15
                self.throw_bomb(bombs)
            elif self.is_ranged():
                if "current_mag" in self.weapon and self.weapon["current_mag"] <= 0:
                    self.start_reload()
                    play_empty_click_sound()
                else:
                    self.attack_type = "shoot"
                    self.attack_anim = 10
                    self.shoot(arrows)
                    self.weapon["durability"] -= 1
                    if self.weapon["durability"] <= 0:
                        self.weapon = None
                        play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup_weapon(weapons)
        self.try_pickup_armor(armors)

    def physics(self):
        global current_lava_y
        self.vel_y += GRAVITY
        self.y += self.vel_y

        self.on_ground = False
        for plat in PLATFORMS:
            prev_y = self.y - self.vel_y
            if prev_y <= plat.top and self.y >= plat.top and plat.left <= self.x <= plat.right and self.vel_y >= 0:
                self.y = plat.top
                self.vel_y = 0
                self.on_ground = True
                break

        if not self.on_ground:
            # Check against the rising lava level
            if self.y >= current_lava_y:
                self.hazard_hit()

        if self.punch_cd > 0: self.punch_cd -= 1
        if self.kick_cd > 0: self.kick_cd -= 1
        if self.armor_flash > 0: self.armor_flash -= 1

        if self.reload_timer > 0:
            self.reload_timer -= 1
            if self.reload_timer == 0 and self.weapon:
                info = WEAPON_TYPES[self.weapon["type"]]
                if "current_mag" in self.weapon:
                    self.weapon["current_mag"] = info.get("mag_size", 0)

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            absorbed = min(damage, info["absorb"])
            damage = damage - absorbed
            self.armor["durability"] -= 1
            self.armor_flash = 10
            if self.armor["durability"] <= 0:
                self.armor = None
                play_break_sound()
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        global current_lava_y
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        if PLATFORMS:
            # Find the highest safe platform to respawn on
            best = min(PLATFORMS, key=lambda p: abs(p.centerx - self.x))
            # Ensure we don't respawn directly into the lava if it's risen too high
            if best.top < current_lava_y - 20:
                self.x = best.centerx
                self.y = best.top
            else:
                # Fallback to a safe default if lava is too high
                self.x = 500
                self.y = 150
        else:
            self.x = 500
            self.y = 150
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            aura_color = info["color"]
            flash = self.armor_flash > 0
            aura_surf = pygame.Surface((90, 130), pygame.SRCALPHA)
            alpha = 90 if not flash else 180
            pygame.draw.ellipse(aura_surf, (*aura_color, alpha), (0, 0, 90, 130))
            if flash:
                pygame.draw.ellipse(aura_surf, (255, 255, 255, 120), (10, 10, 70, 110))
            surf.blit(aura_surf, (cx - 45, self.y - self.height - 10))

        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        elif self.attack_type == "throw":
            fwd_hand = (cx + self.facing * 20, shoulder_y - 10)
            back_hand = (cx - self.facing * 10, shoulder_y + 10)
        elif self.is_reloading():
            fwd_hand = (cx + self.facing * 8, shoulder_y + 30)
            back_hand = (cx - self.facing * 6, shoulder_y + 28)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            ac = info["color"]
            chest_points = [
                (cx - 12, shoulder_y - 2),
                (cx + 12, shoulder_y - 2),
                (cx + 14, shoulder_y + 18),
                (cx + 8, shoulder_y + 26),
                (cx - 8, shoulder_y + 26),
                (cx - 14, shoulder_y + 18),
            ]
            pygame.draw.polygon(surf, ac, chest_points)
            pygame.draw.polygon(surf, BLACK, chest_points, 2)
            pygame.draw.circle(surf, BLACK, (cx, shoulder_y + 12), 2)

        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "uzi":
            wcolor = WEAPON_TYPES["uzi"]["color"]
            ux, uy = fwd_hand[0], fwd_hand[1] - 3
            barrel_tip = (ux + self.facing * 27, uy)
            grip_bottom = (ux - self.facing * 5, uy + 18)
            pygame.draw.rect(surf, wcolor, pygame.Rect(min(ux, barrel_tip[0]), uy - 7, abs(barrel_tip[0] - ux) + 8, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (ux, uy + 4), grip_bottom, 6)
            pygame.draw.line(surf, (25, 25, 28), barrel_tip, (barrel_tip[0] + self.facing * 10, barrel_tip[1]), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 8, barrel_tip[1])
                pygame.draw.circle(surf, (255, 220, 90), (int(flash[0]), int(flash[1])), 6)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [tipend, (tip[0] + perp[0], tip[1] + perp[1]), (tip[0] - perp[0], tip[1] - perp[1])])
            else:
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK, (guard_center[0] + perp[0], guard_center[1] + perp[1]), (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        label_y = int(head_y) - head_r - 20
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            mag_size = info.get("mag_size", 0)
            if mag_size > 0 and "current_mag" in self.weapon:
                cur = self.weapon["current_mag"]
                mag_color = YELLOW if cur > 0 else RED
                mag_label = font_tiny.render(f"[{cur}/{mag_size}]", True, mag_color)
                surf.blit(mag_label, (cx - mag_label.get_width() // 2, label_y))
                label_y -= 14
            label = font_tiny.render(f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}", True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, label_y))
            label_y -= 16

        if self.is_reloading():
            reload_label = font_tiny.render("RELOADING...", True, YELLOW)
            pygame.draw.rect(surf, BLACK, (cx - reload_label.get_width() // 2 - 2, label_y - 1,
                                            reload_label.get_width() + 4, reload_label.get_height() + 2))
            surf.blit(reload_label, (cx - reload_label.get_width() // 2, label_y))
            label_y -= 16

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            label = font_tiny.render(f"{info['name']} Armor x{self.armor['durability']}", True, info["color"])
            pygame.draw.rect(surf, BLACK, (cx - label.get_width() // 2 - 2, label_y - 1, label.get_width() + 4, label.get_height() + 2))
            surf.blit(label, (cx - label.get_width() // 2, label_y))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]
    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    draw_hazard_pit(surf, current_lava_y)

    for plat in PLATFORMS:
        pygame.draw.rect(surf, stage["ground"], plat)
        pygame.draw.rect(surf, stage["ground_edge"], (plat.left, plat.top, plat.width, 10))
        pygame.draw.polygon(surf, stage["ground"], [(plat.left, plat.bottom), (plat.left + 12, plat.bottom + 18), (plat.left + 24, plat.bottom)])
        pygame.draw.polygon(surf, stage["ground"], [(plat.right, plat.bottom), (plat.right - 12, plat.bottom + 18), (plat.right - 24, plat.bottom)])


def draw_hazard_pit(surf, lava_y):
    pit_rect = pygame.Rect(0, lava_y, WIDTH, HEIGHT - lava_y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)
    
    # Draw bands relative to the current lava level
    for i, (band_y_offset, color) in enumerate([
        (15, (150, 40, 15)),
        (35, (200, 70, 20)),
        (55, (240, 110, 30)),
    ]):
        actual_band_y = lava_y + band_y_offset
        if actual_band_y < HEIGHT:
            for gx in range(0, WIDTH, 26):
                wobble = math.sin((gx + i * 40) * 0.15) * 4
                pygame.draw.circle(surf, color, (gx + 13, int(actual_band_y + wobble)), 9)
                
    # Spikes at the very bottom of the screen
    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [
            (gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)
        ])
        pygame.draw.polygon(surf, (90, 90, 95), [
            (gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)
        ])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)
    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)
    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def draw_armor_bar(surf, x, y, fighter, align_left=True):
    if not fighter.armor:
        return
    info = ARMOR_TYPES[fighter.armor["type"]]
    max_dur = info["durability"]
    cur_dur = fighter.armor["durability"]
    bar_w, bar_h = 320, 10
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=3)
    fill_w = int((bar_w - 4) * (cur_dur / max_dur))
    if align_left:
        fill_rect = pygame.Rect(x + 2, y + 2, fill_w, bar_h - 4)
    else:
        fill_rect = pygame.Rect(x + bar_w - 2 - fill_w, y + 2, fill_w, bar_h - 4)
    pygame.draw.rect(surf, info["color"], fill_rect, border_radius=2)
    armor_label = font_tiny.render(f"{info['name']} Armor ({cur_dur})", True, info["color"])
    if align_left:
        surf.blit(armor_label, (x, y - 14))
    else:
        surf.blit(armor_label, (x + bar_w - armor_label.get_width(), y - 14))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, y, wtype))


def spawn_armor(armors):
    if len(armors) >= MAX_ARMOR_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    atype = random.choices(list(ARMOR_TYPES.keys()), weights=[5, 3, 1], k=1)[0]
    armors.append(ArmorPickup(x, y, atype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    global PLATFORMS, current_lava_y, lava_rise_timer, lava_rise_warning
    PLATFORMS = _initial_platforms()
    current_lava_y = GROUND_Y
    lava_rise_timer = LAVA_RISE_INTERVAL
    lava_rise_warning = 0
    
    p1 = Fighter(210, RED, DARK_RED, 1, (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g), "Player 1")
    p1.y = 450
    p2 = Fighter(790, BLUE, DARK_BLUE, -1, (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l), "Player 2")
    p2.y = 450
    p1.wins = p1_wins
    p2.wins = p2_wins
    if p1_start_weapon:
        info = WEAPON_TYPES[p1_start_weapon]
        p1.weapon = {
            "type": p1_start_weapon,
            "durability": info["durability"],
            "current_mag": info.get("mag_size", 0),
        }
    if p2_start_weapon:
        info = WEAPON_TYPES[p2_start_weapon]
        p2.weapon = {
            "type": p2_start_weapon,
            "durability": info["durability"],
            "current_mag": info.get("mag_size", 0),
        }
    weapons, armors, bombs, explosions, arrows, debris = [], [], [], [], [], []
    spawn_weapon(weapons)
    spawn_armor(armors)
    return p1, p2, weapons, armors, arrows, bombs, explosions, debris


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [(cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "uzi":
        pygame.draw.rect(surf, (45, 45, 50), (cx - 22, cy - 8, 44, 16), border_radius=3)
        pygame.draw.line(surf, (25, 25, 28), (cx - 4, cy + 5), (cx - 10, cy + 25), 7)
        pygame.draw.line(surf, (25, 25, 28), (cx + 20, cy), (cx + 34, cy), 5)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)
    elif wtype == "bomb":
        pygame.draw.circle(surf, (40, 40, 40), (cx, cy), 14)
        pygame.draw.line(surf, (100, 100, 100), (cx, cy), (cx + 8, cy - 12), 3)
        pygame.draw.circle(surf, (255, 200, 50), (cx + 8, cy - 12), 4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220
    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))
        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))
    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE
    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))
    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)
    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))
    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))
    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))
    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)
    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render("Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    global current_lava_y, lava_rise_timer, lava_rise_warning
    state = "select"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0
    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, armors, arrows, bombs, explosions, debris = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL
    armor_spawn_timer = ARMOR_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True
                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True
                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, armors, arrows, bombs, explosions, debris = reset_fighters(
                        p1_wins, p2_wins, WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    armor_spawn_timer = ARMOR_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, armors, arrows, bombs)
            p2.handle_input(keys, p1, weapons, armors, arrows, bombs)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            for bomb in bombs[:]:
                bomb.update(PLATFORMS)
                if bomb.exploded:
                    explosions.append(bomb.explode())
                    bombs.remove(bomb)
            for exp in explosions[:]:
                exp.update(p1, p2, debris)
                if exp.life <= 0:
                    explosions.remove(exp)
            for d in debris[:]:
                d.update()
                if d.life <= 0:
                    debris.remove(d)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL
            armor_spawn_timer -= 1
            if armor_spawn_timer <= 0:
                spawn_armor(armors)
                armor_spawn_timer = ARMOR_SPAWN_INTERVAL

            # Lava rising logic
            lava_rise_timer -= 1
            if lava_rise_timer <= 120:  # 2 seconds warning
                lava_rise_warning = 120
            
            if lava_rise_timer <= 0:
                current_lava_y -= LAVA_RISE_AMOUNT
                # Cap the lava so it doesn't cover the absolute top of the screen
                if current_lava_y < 120:
                    current_lava_y = 120
                lava_rise_timer = LAVA_RISE_INTERVAL
                play_hazard_sound()  # Warning rumble

            if lava_rise_warning > 0:
                lava_rise_warning -= 1

            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        for ap in armors:
            ap.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)
        for bomb in bombs:
            bomb.draw(screen)
        for exp in explosions:
            exp.draw(screen)
        for d in debris:
            d.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)
        draw_armor_bar(screen, 30, 62, p1, align_left=True)
        draw_armor_bar(screen, WIDTH - 30 - 320, 62, p2, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        plat_count = font_tiny.render(f"Platforms: {len(PLATFORMS)}", True, WHITE)
        screen.blit(plat_count, (WIDTH // 2 - plat_count.get_width() // 2, 50))

        # Lava rising warning overlay
        if lava_rise_warning > 0 and (lava_rise_warning // 5) % 2 == 0:
            warning_text = font_med.render("WARNING: LAVA RISING!", True, RED)
            screen.blit(warning_text, (WIDTH // 2 - warning_text.get_width() // 2, HEIGHT // 2 - 50))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))
            msg = "DRAW!" if winner == "Draw" else f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))
            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Watch the rising lava!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

KeyboardInterrupt: 

In [27]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons, 
floating platforms, bombs that DESTROY platforms, ARMOR PICKUPS, GUN RELOADING, 
and RISING LAVA with DYNAMIC PLATFORMS!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

Weapons (uzi, sword, bat, spear, bow, gatling, bomb) AND armor (light/medium/
heavy) spawn randomly on the platforms during the match. Walk over a pickup 
to grab it.

GUN RELOADING: The uzi, gatling, and bow now have 50-round magazines! After 
firing 50 shots, the gun auto-reloads for ~1 second.

RISING LAVA + DYNAMIC PLATFORMS: Every 10 seconds, the lava pit rises by 25 
pixels. Platforms submerged by lava are DESTROYED, and NEW platforms spawn 
above the lava line to keep the arena playable. The battle constantly shifts 
upward as the arena reshapes itself!

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None
explosion_sound = None
crumble_sound = None
armor_pickup_sound = None
reload_sound = None
empty_click_sound = None
platform_spawn_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_explosion_sound():
        duration = 0.45
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 120 * (1 - progress * 0.8)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.7 * noise + 0.3 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_crumble_sound():
        duration = 0.6
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.2
            freq = 180 * (1 - progress * 0.5) + 40 * math.sin(2 * math.pi * 8 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.85 * noise + 0.15 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_armor_pickup_sound():
        duration = 0.22
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 1200 - 400 * progress
            tone = math.sin(2 * math.pi * freq * t)
            tone2 = math.sin(2 * math.pi * (freq * 1.5) * t) * 0.5
            value = int(amplitude * envelope * (tone + tone2) * 0.6)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_reload_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.3
            click1 = math.sin(2 * math.pi * 1800 * t) * (1 if progress < 0.1 else 0)
            click2 = math.sin(2 * math.pi * 2200 * t) * (1 if 0.45 < progress < 0.55 else 0)
            metallic = math.sin(2 * math.pi * 900 * t) * 0.3
            mix = click1 + click2 + metallic
            value = int(amplitude * envelope * mix * 0.5)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_empty_click_sound():
        duration = 0.08
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 3
            freq = 3500 * (1 - progress * 0.5)
            tone = math.sin(2 * math.pi * freq * t)
            value = int(amplitude * envelope * tone * 0.4)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_platform_spawn_sound():
        """A rising whoosh/crystalline tone for new platforms appearing."""
        duration = 0.4
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) * (1 - progress * 0.5)
            freq = 400 + 800 * progress  # rising pitch
            tone = math.sin(2 * math.pi * freq * t)
            tone2 = math.sin(2 * math.pi * (freq * 2) * t) * 0.3
            value = int(amplitude * envelope * (tone + tone2) * 0.5)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
    explosion_sound = _generate_explosion_sound()
    crumble_sound = _generate_crumble_sound()
    armor_pickup_sound = _generate_armor_pickup_sound()
    reload_sound = _generate_reload_sound()
    empty_click_sound = _generate_empty_click_sound()
    platform_spawn_sound = _generate_platform_spawn_sound()
except pygame.error:
    break_sound = None
    hazard_sound = None
    explosion_sound = None
    crumble_sound = None
    armor_pickup_sound = None
    reload_sound = None
    empty_click_sound = None
    platform_spawn_sound = None


def play_break_sound():
    if break_sound: break_sound.play()
def play_hazard_sound():
    if hazard_sound: hazard_sound.play()
def play_explosion_sound():
    if explosion_sound: explosion_sound.play()
def play_crumble_sound():
    if crumble_sound: crumble_sound.play()
def play_armor_pickup_sound():
    if armor_pickup_sound: armor_pickup_sound.play()
def play_reload_sound():
    if reload_sound: reload_sound.play()
def play_empty_click_sound():
    if empty_click_sound: empty_click_sound.play()
def play_platform_spawn_sound():
    if platform_spawn_sound: platform_spawn_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
GOLD = (230, 190, 60)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

# Lava mechanics
LAVA_RISE_INTERVAL = 600  # 10 seconds at 60 FPS
LAVA_RISE_AMOUNT = 25
current_lava_y = GROUND_Y
lava_rise_timer = LAVA_RISE_INTERVAL
lava_rise_warning = 0

# Platform generation
MIN_PLATFORM_Y = 100  # Don't spawn above this (leaves room for HUD)
TARGET_PLATFORM_COUNT = 9  # Aim to keep this many platforms active
MAX_PLATFORMS = 12

HAZARD_DAMAGE = 40
HAZARD_STUN = 24

RELOAD_TIME = 60

def _initial_platforms():
    return [
        pygame.Rect(150, 450, 120, 18),
        pygame.Rect(730, 450, 120, 18),
        pygame.Rect(280, 400, 140, 20),
        pygame.Rect(580, 400, 140, 20),
        pygame.Rect(430, 360, 140, 20),
        pygame.Rect(200, 320, 130, 18),
        pygame.Rect(670, 320, 130, 18),
        pygame.Rect(440, 280, 120, 18),
        pygame.Rect(320, 220, 110, 16),
        pygame.Rect(570, 220, 110, 16),
        pygame.Rect(450, 180, 100, 16),
    ]

PLATFORMS = _initial_platforms()

# Track spawn animations for new platforms (list of (rect, frames_remaining))
PLATFORM_SPAWN_ANIMS = []

PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

WEAPON_TYPES = {
    "uzi":     {"reach": 0,   "damage": 5, "cooldown": 5, "durability": 50, "color": (45, 45, 50),
                "ranged": True, "proj_speed": 28, "proj_kind": "bullet", "mag_size": 50},
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow", "mag_size": 50},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet", "mag_size": 50},
    "bomb":    {"reach": 0,   "damage": 35, "cooldown": 50, "durability": 3,  "color": (40, 40, 40),
                "ranged": True, "proj_speed": 9, "proj_kind": "bomb"},
}

ARMOR_TYPES = {
    "light":  {"absorb": 15, "durability": 3, "color": (120, 220, 120), "name": "Light"},
    "medium": {"absorb": 30, "durability": 4, "color": (120, 170, 240), "name": "Medium"},
    "heavy":  {"absorb": 50, "durability": 5, "color": (240, 200, 80),  "name": "Heavy"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

MAX_WEAPONS_ON_FIELD = 2
MAX_ARMOR_ON_FIELD = 1
WEAPON_SPAWN_INTERVAL = 300
ARMOR_SPAWN_INTERVAL = 480

WEAPON_CHOICES = [None, "uzi", "sword", "bat", "spear", "bow", "gatling", "bomb"]

STAGES = [
    {"name": "Meadow", "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245), "ground": (110, 90, 70), "ground_edge": (80, 150, 80), "decor": "meadow"},
    {"name": "Desert", "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190), "ground": (200, 165, 100), "ground_edge": (225, 195, 130), "decor": "desert"},
    {"name": "Night City", "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90), "ground": (40, 40, 50), "ground_edge": (90, 90, 110), "decor": "city"},
    {"name": "Volcano", "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30), "ground": (50, 35, 30), "ground_edge": (200, 80, 30), "decor": "volcano"},
    {"name": "Snow Peak", "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245), "ground": (225, 235, 240), "ground_edge": (255, 255, 255), "decor": "snow"},
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


def generate_new_platforms():
    """Generate new platforms above the current lava level to replace 
    submerged ones. Called whenever lava rises."""
    global PLATFORMS, PLATFORM_SPAWN_ANIMS
    
    # First, remove platforms that are now submerged in lava
    submerged = [p for p in PLATFORMS if p.top >= current_lava_y]
    for p in submerged:
        PLATFORMS.remove(p)
    
    # Calculate how many new platforms we need
    needed = TARGET_PLATFORM_COUNT - len(PLATFORMS)
    if needed <= 0 or len(PLATFORMS) >= MAX_PLATFORMS:
        return
    
    # Determine the safe spawn zone: above lava but below MIN_PLATFORM_Y
    spawn_zone_top = max(MIN_PLATFORM_Y, current_lava_y - 350)
    spawn_zone_bottom = current_lava_y - 40  # at least 40px above lava
    
    if spawn_zone_bottom <= spawn_zone_top:
        # Not enough vertical space — spawn just above lava
        spawn_zone_top = max(MIN_PLATFORM_Y, current_lava_y - 100)
        spawn_zone_bottom = current_lava_y - 30
    
    new_platforms = []
    attempts = 0
    max_attempts = 50
    
    while len(new_platforms) < needed and attempts < max_attempts:
        attempts += 1
        
        # Random dimensions
        width = random.randint(80, 160)
        height = random.choice([16, 18, 20])
        
        # Random position in spawn zone
        x = random.randint(80, WIDTH - 80 - width)
        y = random.randint(spawn_zone_top, spawn_zone_bottom)
        
        new_plat = pygame.Rect(x, y, width, height)
        
        # Check for overlap with existing platforms (need at least 30px gap)
        overlap = False
        for existing in PLATFORMS + new_platforms:
            expanded = existing.inflate(30, 40)
            if new_plat.colliderect(expanded):
                overlap = True
                break
        
        if not overlap:
            new_platforms.append(new_plat)
            PLATFORM_SPAWN_ANIMS.append((new_plat, 20))  # 20 frames of spawn animation
    
    PLATFORMS.extend(new_platforms)
    
    if new_platforms:
        play_platform_spawn_sound()


class Arrow:
    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return
        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class Bomb:
    def __init__(self, x, y, facing, damage, owner):
        self.x = x
        self.y = y
        self.vel_x = facing * 9
        self.vel_y = -11
        self.damage = damage
        self.owner = owner
        self.timer = 70
        self.dead = False
        self.exploded = False

    def update(self, platforms):
        if self.exploded:
            return
        self.vel_y += GRAVITY
        self.x += self.vel_x
        self.y += self.vel_y
        self.timer -= 1
        for plat in platforms:
            if plat.left <= self.x <= plat.right and self.y >= plat.top and self.vel_y >= 0:
                if self.y - self.vel_y <= plat.top + 10:
                    self.y = plat.top
                    self.vel_y = 0
                    self.vel_x *= 0.7
                    self.timer -= 3
        if self.y >= current_lava_y or self.timer <= 0:
            self.exploded = True
            self.dead = True

    def explode(self):
        play_explosion_sound()
        return Explosion(self.x, self.y, self.damage, self.owner)

    def draw(self, surf):
        pygame.draw.circle(surf, (30, 30, 30), (int(self.x), int(self.y)), 8)
        if self.timer % 8 < 4:
            pygame.draw.circle(surf, (255, 200, 50), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 3)
        pygame.draw.line(surf, (100, 100, 100), (int(self.x), int(self.y)), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 2)


class Debris:
    def __init__(self, x, y, color):
        self.x = x
        self.y = y
        self.vel_x = random.uniform(-8, 8)
        self.vel_y = random.uniform(-10, -2)
        self.size = random.randint(4, 10)
        self.color = color
        self.rotation = random.uniform(0, 360)
        self.rot_speed = random.uniform(-15, 15)
        self.life = random.randint(40, 70)

    def update(self):
        self.vel_y += GRAVITY * 0.8
        self.x += self.vel_x
        self.y += self.vel_y
        self.rotation += self.rot_speed
        self.life -= 1
        self.vel_x *= 0.98

    def draw(self, surf):
        if self.life <= 0:
            return
        alpha = min(255, self.life * 6)
        surf_deb = pygame.Surface((self.size * 2, self.size * 2), pygame.SRCALPHA)
        rect = pygame.Rect(0, 0, self.size, self.size)
        pygame.draw.rect(surf_deb, (*self.color, alpha), rect)
        pygame.draw.rect(surf_deb, (0, 0, 0, alpha), rect, 1)
        rotated = pygame.transform.rotate(surf_deb, self.rotation)
        new_rect = rotated.get_rect(center=(int(self.x), int(self.y)))
        surf.blit(rotated, new_rect)


class Explosion:
    def __init__(self, x, y, damage, owner):
        self.x = x
        self.y = y
        self.damage = damage
        self.owner = owner
        self.radius = 10
        self.max_radius = 110
        self.life = 20
        self.has_damaged = False
        self.has_destroyed_platforms = False

    def update(self, p1, p2, debris_list):
        self.life -= 1
        if self.life > 10:
            self.radius += (self.max_radius - self.radius) * 0.4
        else:
            self.radius += (self.max_radius - self.radius) * 0.1
        if not self.has_damaged and self.life == 19:
            for target in (p1, p2):
                target_cx = target.x
                target_cy = target.y - target.height / 2
                dist = math.hypot(target_cx - self.x, target_cy - self.y)
                if dist < self.max_radius:
                    from_left = (self.x < target.x)
                    target.take_hit(self.damage, from_left)
                    distance_factor = 1.0 - (dist / self.max_radius)
                    base_push = 120
                    base_pop = -22
                    push = (base_push * distance_factor) if from_left else -(base_push * distance_factor)
                    target.x = max(target.width, min(WIDTH - target.width, target.x + push))
                    target.vel_y = base_pop * distance_factor
            self.has_damaged = True
        if not self.has_destroyed_platforms and self.life == 17:
            global PLATFORMS
            platforms_to_destroy = []
            for plat in PLATFORMS:
                closest_x = max(plat.left, min(self.x, plat.right))
                closest_y = max(plat.top, min(self.y, plat.bottom))
                dist = math.hypot(closest_x - self.x, closest_y - self.y)
                if dist < self.max_radius:
                    platforms_to_destroy.append(plat)
            if len(PLATFORMS) - len(platforms_to_destroy) >= 1:
                for plat in platforms_to_destroy:
                    plat_color = (110, 90, 70)
                    for _ in range(18):
                        chunk_x = random.uniform(plat.left, plat.right)
                        chunk_y = random.uniform(plat.top, plat.bottom)
                        debris_list.append(Debris(chunk_x, chunk_y, plat_color))
                    PLATFORMS.remove(plat)
                    play_crumble_sound()
            self.has_destroyed_platforms = True

    def draw(self, surf):
        if self.life > 0:
            alpha = int(255 * (self.life / 20))
            surf_exp = pygame.Surface((self.max_radius * 2, self.max_radius * 2), pygame.SRCALPHA)
            pygame.draw.circle(surf_exp, (255, 80, 20, alpha), (self.max_radius, self.max_radius), int(self.radius))
            pygame.draw.circle(surf_exp, (255, 200, 50, int(alpha * 0.8)), (self.max_radius, self.max_radius), int(self.radius * 0.6))
            pygame.draw.circle(surf_exp, (255, 255, 220, int(alpha * 0.9)), (self.max_radius, self.max_radius), int(self.radius * 0.25))
            surf.blit(surf_exp, (int(self.x - self.max_radius), int(self.y - self.max_radius)))


class WeaponPickup:
    def __init__(self, x, y, wtype):
        self.x = x
        self.y = y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [(self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        elif self.wtype == "uzi":
            pygame.draw.rect(surf, color, (self.x - 12, self.y - 26, 24, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (self.x - 2, self.y - 15), (self.x - 6, self.y - 4), 5)
        elif self.wtype == "bomb":
            pygame.draw.circle(surf, (30, 30, 30), (self.x, self.y - 20), 10)
            pygame.draw.line(surf, (100, 100, 100), (self.x, self.y - 20), (self.x + 6, self.y - 30), 2)
            pygame.draw.circle(surf, (255, 200, 50), (self.x + 6, self.y - 30), 3)
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class ArmorPickup:
    def __init__(self, x, y, atype):
        self.x = x
        self.y = y
        self.atype = atype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = ARMOR_TYPES[self.atype]
        color = info["color"]
        glow = pygame.Surface((80, 80), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*color, 80), (40, 40), 34)
        surf.blit(glow, (self.x - 40, self.y - 60))

        chest_points = [
            (self.x - 14, self.y - 38),
            (self.x + 14, self.y - 38),
            (self.x + 16, self.y - 18),
            (self.x + 10, self.y - 8),
            (self.x - 10, self.y - 8),
            (self.x - 16, self.y - 18),
        ]
        pygame.draw.polygon(surf, color, chest_points)
        pygame.draw.polygon(surf, BLACK, chest_points, 2)
        pygame.draw.line(surf, color, (self.x - 12, self.y - 38), (self.x - 14, self.y - 44), 4)
        pygame.draw.line(surf, color, (self.x + 12, self.y - 38), (self.x + 14, self.y - 44), 4)
        pygame.draw.circle(surf, BLACK, (int(self.x), int(self.y - 24)), 3)

        label = font_tiny.render(info["name"] + " Armor", True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 64))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing
        self.controls = controls
        self.name = name
        self.health = MAX_HEALTH
        self.on_ground = True
        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0
        self.attack_type = None
        self.hit_stun = 0
        self.walk_cycle = 0
        self.moving = False
        self.wins = 0
        self.weapon = None
        self.armor = None
        self.armor_flash = 0
        self.reload_timer = 0

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height, self.width, self.height)

    def current_stats(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup_weapon(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {
                    "type": wp.wtype,
                    "durability": info["durability"],
                    "current_mag": info.get("mag_size", 0),
                }
                weapons.remove(wp)
                return

    def try_pickup_armor(self, armors):
        for ap in armors:
            if ap.rect().colliderect(self.rect()):
                info = ARMOR_TYPES[ap.atype]
                self.armor = {"type": ap.atype, "durability": info["durability"]}
                self.armor_flash = 12
                armors.remove(ap)
                play_armor_pickup_sound()
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def is_reloading(self):
        return self.reload_timer > 0

    def start_reload(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            if info.get("mag_size", 0) > 0:
                self.reload_timer = RELOAD_TIME
                play_reload_sound()

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)
        elif self.weapon["type"] == "uzi":
            spawn_y += random.randint(-3, 3)
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))
        if "current_mag" in self.weapon:
            self.weapon["current_mag"] -= 1
            if self.weapon["current_mag"] <= 0:
                self.start_reload()

    def throw_bomb(self, bombs):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 20
        spawn_y = self.y - self.height * 0.7
        bombs.append(Bomb(spawn_x, spawn_y, self.facing, info["damage"], self))
        self.weapon["durability"] -= 1
        if self.weapon["durability"] <= 0:
            self.weapon = None
            play_break_sound()

    def handle_input(self, keys, opponent, weapons, armors, arrows, bombs):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.is_reloading():
                pass
            elif self.weapon and self.weapon["type"] == "bomb":
                self.attack_type = "throw"
                self.attack_anim = 15
                self.throw_bomb(bombs)
            elif self.is_ranged():
                if "current_mag" in self.weapon and self.weapon["current_mag"] <= 0:
                    self.start_reload()
                    play_empty_click_sound()
                else:
                    self.attack_type = "shoot"
                    self.attack_anim = 10
                    self.shoot(arrows)
                    self.weapon["durability"] -= 1
                    if self.weapon["durability"] <= 0:
                        self.weapon = None
                        play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup_weapon(weapons)
        self.try_pickup_armor(armors)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y

        self.on_ground = False
        for plat in PLATFORMS:
            prev_y = self.y - self.vel_y
            if prev_y <= plat.top and self.y >= plat.top and plat.left <= self.x <= plat.right and self.vel_y >= 0:
                self.y = plat.top
                self.vel_y = 0
                self.on_ground = True
                break

        if not self.on_ground:
            if self.y >= current_lava_y:
                self.hazard_hit()

        if self.punch_cd > 0: self.punch_cd -= 1
        if self.kick_cd > 0: self.kick_cd -= 1
        if self.armor_flash > 0: self.armor_flash -= 1

        if self.reload_timer > 0:
            self.reload_timer -= 1
            if self.reload_timer == 0 and self.weapon:
                info = WEAPON_TYPES[self.weapon["type"]]
                if "current_mag" in self.weapon:
                    self.weapon["current_mag"] = info.get("mag_size", 0)

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            absorbed = min(damage, info["absorb"])
            damage = damage - absorbed
            self.armor["durability"] -= 1
            self.armor_flash = 10
            if self.armor["durability"] <= 0:
                self.armor = None
                play_break_sound()
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        if PLATFORMS:
            # Find the highest safe platform to respawn on
            safe_platforms = [p for p in PLATFORMS if p.top < current_lava_y - 20]
            if safe_platforms:
                best = min(safe_platforms, key=lambda p: abs(p.centerx - self.x))
                self.x = best.centerx
                self.y = best.top
            else:
                self.x = 500
                self.y = max(MIN_PLATFORM_Y + 20, current_lava_y - 50)
        else:
            self.x = 500
            self.y = max(MIN_PLATFORM_Y + 20, current_lava_y - 50)
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            aura_color = info["color"]
            flash = self.armor_flash > 0
            aura_surf = pygame.Surface((90, 130), pygame.SRCALPHA)
            alpha = 90 if not flash else 180
            pygame.draw.ellipse(aura_surf, (*aura_color, alpha), (0, 0, 90, 130))
            if flash:
                pygame.draw.ellipse(aura_surf, (255, 255, 255, 120), (10, 10, 70, 110))
            surf.blit(aura_surf, (cx - 45, self.y - self.height - 10))

        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        elif self.attack_type == "throw":
            fwd_hand = (cx + self.facing * 20, shoulder_y - 10)
            back_hand = (cx - self.facing * 10, shoulder_y + 10)
        elif self.is_reloading():
            fwd_hand = (cx + self.facing * 8, shoulder_y + 30)
            back_hand = (cx - self.facing * 6, shoulder_y + 28)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            ac = info["color"]
            chest_points = [
                (cx - 12, shoulder_y - 2),
                (cx + 12, shoulder_y - 2),
                (cx + 14, shoulder_y + 18),
                (cx + 8, shoulder_y + 26),
                (cx - 8, shoulder_y + 26),
                (cx - 14, shoulder_y + 18),
            ]
            pygame.draw.polygon(surf, ac, chest_points)
            pygame.draw.polygon(surf, BLACK, chest_points, 2)
            pygame.draw.circle(surf, BLACK, (cx, shoulder_y + 12), 2)

        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "uzi":
            wcolor = WEAPON_TYPES["uzi"]["color"]
            ux, uy = fwd_hand[0], fwd_hand[1] - 3
            barrel_tip = (ux + self.facing * 27, uy)
            grip_bottom = (ux - self.facing * 5, uy + 18)
            pygame.draw.rect(surf, wcolor, pygame.Rect(min(ux, barrel_tip[0]), uy - 7, abs(barrel_tip[0] - ux) + 8, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (ux, uy + 4), grip_bottom, 6)
            pygame.draw.line(surf, (25, 25, 28), barrel_tip, (barrel_tip[0] + self.facing * 10, barrel_tip[1]), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 8, barrel_tip[1])
                pygame.draw.circle(surf, (255, 220, 90), (int(flash[0]), int(flash[1])), 6)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [tipend, (tip[0] + perp[0], tip[1] + perp[1]), (tip[0] - perp[0], tip[1] - perp[1])])
            else:
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK, (guard_center[0] + perp[0], guard_center[1] + perp[1]), (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        label_y = int(head_y) - head_r - 20
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            mag_size = info.get("mag_size", 0)
            if mag_size > 0 and "current_mag" in self.weapon:
                cur = self.weapon["current_mag"]
                mag_color = YELLOW if cur > 0 else RED
                mag_label = font_tiny.render(f"[{cur}/{mag_size}]", True, mag_color)
                surf.blit(mag_label, (cx - mag_label.get_width() // 2, label_y))
                label_y -= 14
            label = font_tiny.render(f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}", True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, label_y))
            label_y -= 16

        if self.is_reloading():
            reload_label = font_tiny.render("RELOADING...", True, YELLOW)
            pygame.draw.rect(surf, BLACK, (cx - reload_label.get_width() // 2 - 2, label_y - 1,
                                            reload_label.get_width() + 4, reload_label.get_height() + 2))
            surf.blit(reload_label, (cx - reload_label.get_width() // 2, label_y))
            label_y -= 16

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            label = font_tiny.render(f"{info['name']} Armor x{self.armor['durability']}", True, info["color"])
            pygame.draw.rect(surf, BLACK, (cx - label.get_width() // 2 - 2, label_y - 1, label.get_width() + 4, label.get_height() + 2))
            surf.blit(label, (cx - label.get_width() // 2, label_y))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]
    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    draw_hazard_pit(surf, current_lava_y)

    # Draw all current platforms
    for plat in PLATFORMS:
        # Check if this platform is currently in its spawn animation
        anim_frames = 0
        for anim_plat, frames in PLATFORM_SPAWN_ANIMS:
            if anim_plat is plat:
                anim_frames = frames
                break
        
        if anim_frames > 0:
            # Spawn animation: scale up from center with glow
            progress = 1.0 - (anim_frames / 20.0)  # 0 to 1
            scale = 0.3 + 0.7 * progress
            alpha = int(255 * progress)
            
            # Glowing outline
            glow_surf = pygame.Surface((plat.width + 20, plat.height + 20), pygame.SRCALPHA)
            glow_color = (255, 255, 150, int(alpha * 0.6))
            pygame.draw.rect(glow_surf, glow_color, (0, 0, plat.width + 20, plat.height + 20), border_radius=6)
            glow_surf_scaled = pygame.transform.scale(glow_surf, 
                (int((plat.width + 20) * scale), int((plat.height + 20) * scale)))
            surf.blit(glow_surf_scaled, 
                (int(plat.centerx - glow_surf_scaled.get_width() / 2),
                 int(plat.centery - glow_surf_scaled.get_height() / 2)))
        
        # Draw the platform itself
        pygame.draw.rect(surf, stage["ground"], plat)
        pygame.draw.rect(surf, stage["ground_edge"], (plat.left, plat.top, plat.width, 10))
        pygame.draw.polygon(surf, stage["ground"], [(plat.left, plat.bottom), (plat.left + 12, plat.bottom + 18), (plat.left + 24, plat.bottom)])
        pygame.draw.polygon(surf, stage["ground"], [(plat.right, plat.bottom), (plat.right - 12, plat.bottom + 18), (plat.right - 24, plat.bottom)])


def draw_hazard_pit(surf, lava_y):
    pit_rect = pygame.Rect(0, lava_y, WIDTH, HEIGHT - lava_y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)
    
    for i, (band_y_offset, color) in enumerate([
        (15, (150, 40, 15)),
        (35, (200, 70, 20)),
        (55, (240, 110, 30)),
    ]):
        actual_band_y = lava_y + band_y_offset
        if actual_band_y < HEIGHT:
            for gx in range(0, WIDTH, 26):
                wobble = math.sin((gx + i * 40) * 0.15) * 4
                pygame.draw.circle(surf, color, (gx + 13, int(actual_band_y + wobble)), 9)
                
    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [
            (gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)
        ])
        pygame.draw.polygon(surf, (90, 90, 95), [
            (gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)
        ])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)
    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)
    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def draw_armor_bar(surf, x, y, fighter, align_left=True):
    if not fighter.armor:
        return
    info = ARMOR_TYPES[fighter.armor["type"]]
    max_dur = info["durability"]
    cur_dur = fighter.armor["durability"]
    bar_w, bar_h = 320, 10
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=3)
    fill_w = int((bar_w - 4) * (cur_dur / max_dur))
    if align_left:
        fill_rect = pygame.Rect(x + 2, y + 2, fill_w, bar_h - 4)
    else:
        fill_rect = pygame.Rect(x + bar_w - 2 - fill_w, y + 2, fill_w, bar_h - 4)
    pygame.draw.rect(surf, info["color"], fill_rect, border_radius=2)
    armor_label = font_tiny.render(f"{info['name']} Armor ({cur_dur})", True, info["color"])
    if align_left:
        surf.blit(armor_label, (x, y - 14))
    else:
        surf.blit(armor_label, (x + bar_w - armor_label.get_width(), y - 14))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, y, wtype))


def spawn_armor(armors):
    if len(armors) >= MAX_ARMOR_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    atype = random.choices(list(ARMOR_TYPES.keys()), weights=[5, 3, 1], k=1)[0]
    armors.append(ArmorPickup(x, y, atype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    global PLATFORMS, current_lava_y, lava_rise_timer, lava_rise_warning, PLATFORM_SPAWN_ANIMS
    PLATFORMS = _initial_platforms()
    PLATFORM_SPAWN_ANIMS = []
    current_lava_y = GROUND_Y
    lava_rise_timer = LAVA_RISE_INTERVAL
    lava_rise_warning = 0
    
    p1 = Fighter(210, RED, DARK_RED, 1, (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g), "Player 1")
    p1.y = 450
    p2 = Fighter(790, BLUE, DARK_BLUE, -1, (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l), "Player 2")
    p2.y = 450
    p1.wins = p1_wins
    p2.wins = p2_wins
    if p1_start_weapon:
        info = WEAPON_TYPES[p1_start_weapon]
        p1.weapon = {
            "type": p1_start_weapon,
            "durability": info["durability"],
            "current_mag": info.get("mag_size", 0),
        }
    if p2_start_weapon:
        info = WEAPON_TYPES[p2_start_weapon]
        p2.weapon = {
            "type": p2_start_weapon,
            "durability": info["durability"],
            "current_mag": info.get("mag_size", 0),
        }
    weapons, armors, bombs, explosions, arrows, debris = [], [], [], [], [], []
    spawn_weapon(weapons)
    spawn_armor(armors)
    return p1, p2, weapons, armors, arrows, bombs, explosions, debris


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [(cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "uzi":
        pygame.draw.rect(surf, (45, 45, 50), (cx - 22, cy - 8, 44, 16), border_radius=3)
        pygame.draw.line(surf, (25, 25, 28), (cx - 4, cy + 5), (cx - 10, cy + 25), 7)
        pygame.draw.line(surf, (25, 25, 28), (cx + 20, cy), (cx + 34, cy), 5)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)
    elif wtype == "bomb":
        pygame.draw.circle(surf, (40, 40, 40), (cx, cy), 14)
        pygame.draw.line(surf, (100, 100, 100), (cx, cy), (cx + 8, cy - 12), 3)
        pygame.draw.circle(surf, (255, 200, 50), (cx + 8, cy - 12), 4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220
    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))
        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))
    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE
    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))
    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)
    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))
    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))
    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))
    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)
    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render("Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    global current_lava_y, lava_rise_timer, lava_rise_warning, PLATFORM_SPAWN_ANIMS
    state = "select"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0
    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, armors, arrows, bombs, explosions, debris = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL
    armor_spawn_timer = ARMOR_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a:
                            p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d:
                            p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f:
                            p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT:
                            p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT:
                            p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k:
                            p2_ready = True
                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT):
                        stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT):
                        stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k):
                        world_locked = True
                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, armors, arrows, bombs, explosions, debris = reset_fighters(
                        p1_wins, p2_wins, WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    armor_spawn_timer = ARMOR_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, armors, arrows, bombs)
            p2.handle_input(keys, p1, weapons, armors, arrows, bombs)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            for bomb in bombs[:]:
                bomb.update(PLATFORMS)
                if bomb.exploded:
                    explosions.append(bomb.explode())
                    bombs.remove(bomb)
            for exp in explosions[:]:
                exp.update(p1, p2, debris)
                if exp.life <= 0:
                    explosions.remove(exp)
            for d in debris[:]:
                d.update()
                if d.life <= 0:
                    debris.remove(d)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL
            armor_spawn_timer -= 1
            if armor_spawn_timer <= 0:
                spawn_armor(armors)
                armor_spawn_timer = ARMOR_SPAWN_INTERVAL

            # Update platform spawn animations
            for anim in PLATFORM_SPAWN_ANIMS[:]:
                plat, frames = anim
                frames -= 1
                if frames <= 0:
                    PLATFORM_SPAWN_ANIMS.remove(anim)
                else:
                    # Update the frames in the list
                    idx = PLATFORM_SPAWN_ANIMS.index(anim)
                    PLATFORM_SPAWN_ANIMS[idx] = (plat, frames)

            # Lava rising logic
            lava_rise_timer -= 1
            if lava_rise_timer <= 120:
                lava_rise_warning = 120
            
            if lava_rise_timer <= 0:
                current_lava_y -= LAVA_RISE_AMOUNT
                if current_lava_y < MIN_PLATFORM_Y - 50:
                    current_lava_y = MIN_PLATFORM_Y - 50
                lava_rise_timer = LAVA_RISE_INTERVAL
                play_hazard_sound()
                
                # Generate new platforms above the lava!
                generate_new_platforms()

            if lava_rise_warning > 0:
                lava_rise_warning -= 1

            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        for ap in armors:
            ap.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)
        for bomb in bombs:
            bomb.draw(screen)
        for exp in explosions:
            exp.draw(screen)
        for d in debris:
            d.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)
        draw_armor_bar(screen, 30, 62, p1, align_left=True)
        draw_armor_bar(screen, WIDTH - 30 - 320, 62, p2, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        plat_count = font_tiny.render(f"Platforms: {len(PLATFORMS)}", True, WHITE)
        screen.blit(plat_count, (WIDTH // 2 - plat_count.get_width() // 2, 50))

        if lava_rise_warning > 0 and (lava_rise_warning // 5) % 2 == 0:
            warning_text = font_med.render("WARNING: LAVA RISING!", True, RED)
            screen.blit(warning_text, (WIDTH // 2 - warning_text.get_width() // 2, HEIGHT // 2 - 50))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))
            msg = "DRAW!" if winner == "Draw" else f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))
            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   New platforms spawn as lava rises!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 

In [28]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons, 
floating platforms, bombs that DESTROY platforms, ARMOR PICKUPS, GUN RELOADING, 
and RISING LAVA with HIGH PLATFORM GENERATION!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

Weapons (uzi, sword, bat, spear, bow, gatling, bomb) AND armor (light/medium/
heavy) spawn randomly on the platforms during the match. Walk over a pickup 
to grab it.

GUN RELOADING: The uzi, gatling, and bow now have 50-round magazines! After 
firing 50 shots, the gun auto-reloads for ~1 second.

RISING LAVA + HIGH PLATFORMS: Every 10 seconds, the lava pit rises by 25 
pixels. Submerged platforms are DESTROYED, and NEW platforms spawn HIGH UP 
near the top of the screen — well above the lava. The battle constantly 
shifts upward as the arena reshapes itself!

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None
explosion_sound = None
crumble_sound = None
armor_pickup_sound = None
reload_sound = None
empty_click_sound = None
platform_spawn_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_explosion_sound():
        duration = 0.45
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 120 * (1 - progress * 0.8)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.7 * noise + 0.3 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_crumble_sound():
        duration = 0.6
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.2
            freq = 180 * (1 - progress * 0.5) + 40 * math.sin(2 * math.pi * 8 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.85 * noise + 0.15 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_armor_pickup_sound():
        duration = 0.22
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 1200 - 400 * progress
            tone = math.sin(2 * math.pi * freq * t)
            tone2 = math.sin(2 * math.pi * (freq * 1.5) * t) * 0.5
            value = int(amplitude * envelope * (tone + tone2) * 0.6)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_reload_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.3
            click1 = math.sin(2 * math.pi * 1800 * t) * (1 if progress < 0.1 else 0)
            click2 = math.sin(2 * math.pi * 2200 * t) * (1 if 0.45 < progress < 0.55 else 0)
            metallic = math.sin(2 * math.pi * 900 * t) * 0.3
            mix = click1 + click2 + metallic
            value = int(amplitude * envelope * mix * 0.5)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_empty_click_sound():
        duration = 0.08
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 3
            freq = 3500 * (1 - progress * 0.5)
            tone = math.sin(2 * math.pi * freq * t)
            value = int(amplitude * envelope * tone * 0.4)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_platform_spawn_sound():
        duration = 0.4
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) * (1 - progress * 0.5)
            freq = 400 + 800 * progress
            tone = math.sin(2 * math.pi * freq * t)
            tone2 = math.sin(2 * math.pi * (freq * 2) * t) * 0.3
            value = int(amplitude * envelope * (tone + tone2) * 0.5)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
    explosion_sound = _generate_explosion_sound()
    crumble_sound = _generate_crumble_sound()
    armor_pickup_sound = _generate_armor_pickup_sound()
    reload_sound = _generate_reload_sound()
    empty_click_sound = _generate_empty_click_sound()
    platform_spawn_sound = _generate_platform_spawn_sound()
except pygame.error:
    break_sound = None
    hazard_sound = None
    explosion_sound = None
    crumble_sound = None
    armor_pickup_sound = None
    reload_sound = None
    empty_click_sound = None
    platform_spawn_sound = None


def play_break_sound():
    if break_sound: break_sound.play()
def play_hazard_sound():
    if hazard_sound: hazard_sound.play()
def play_explosion_sound():
    if explosion_sound: explosion_sound.play()
def play_crumble_sound():
    if crumble_sound: crumble_sound.play()
def play_armor_pickup_sound():
    if armor_pickup_sound: armor_pickup_sound.play()
def play_reload_sound():
    if reload_sound: reload_sound.play()
def play_empty_click_sound():
    if empty_click_sound: empty_click_sound.play()
def play_platform_spawn_sound():
    if platform_spawn_sound: platform_spawn_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
GOLD = (230, 190, 60)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

LAVA_RISE_INTERVAL = 600
LAVA_RISE_AMOUNT = 25
current_lava_y = GROUND_Y
lava_rise_timer = LAVA_RISE_INTERVAL
lava_rise_warning = 0

# Platform generation — platforms spawn HIGH UP near the top of the screen
MIN_PLATFORM_Y = 80       # Absolute ceiling for platform spawns (below HUD)
TARGET_PLATFORM_COUNT = 9
MAX_PLATFORMS = 14

HAZARD_DAMAGE = 40
HAZARD_STUN = 24

RELOAD_TIME = 60

def _initial_platforms():
    return [
        pygame.Rect(150, 450, 120, 18),
        pygame.Rect(730, 450, 120, 18),
        pygame.Rect(280, 400, 140, 20),
        pygame.Rect(580, 400, 140, 20),
        pygame.Rect(430, 360, 140, 20),
        pygame.Rect(200, 320, 130, 18),
        pygame.Rect(670, 320, 130, 18),
        pygame.Rect(440, 280, 120, 18),
        pygame.Rect(320, 220, 110, 16),
        pygame.Rect(570, 220, 110, 16),
        pygame.Rect(450, 180, 100, 16),
    ]

PLATFORMS = _initial_platforms()
PLATFORM_SPAWN_ANIMS = []

PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

WEAPON_TYPES = {
    "uzi":     {"reach": 0,   "damage": 5, "cooldown": 5, "durability": 50, "color": (45, 45, 50),
                "ranged": True, "proj_speed": 28, "proj_kind": "bullet", "mag_size": 50},
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow", "mag_size": 50},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet", "mag_size": 50},
    "bomb":    {"reach": 0,   "damage": 35, "cooldown": 50, "durability": 3,  "color": (40, 40, 40),
                "ranged": True, "proj_speed": 9, "proj_kind": "bomb"},
}

ARMOR_TYPES = {
    "light":  {"absorb": 15, "durability": 3, "color": (120, 220, 120), "name": "Light"},
    "medium": {"absorb": 30, "durability": 4, "color": (120, 170, 240), "name": "Medium"},
    "heavy":  {"absorb": 50, "durability": 5, "color": (240, 200, 80),  "name": "Heavy"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

MAX_WEAPONS_ON_FIELD = 2
MAX_ARMOR_ON_FIELD = 1
WEAPON_SPAWN_INTERVAL = 300
ARMOR_SPAWN_INTERVAL = 480

WEAPON_CHOICES = [None, "uzi", "sword", "bat", "spear", "bow", "gatling", "bomb"]

STAGES = [
    {"name": "Meadow", "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245), "ground": (110, 90, 70), "ground_edge": (80, 150, 80), "decor": "meadow"},
    {"name": "Desert", "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190), "ground": (200, 165, 100), "ground_edge": (225, 195, 130), "decor": "desert"},
    {"name": "Night City", "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90), "ground": (40, 40, 50), "ground_edge": (90, 90, 110), "decor": "city"},
    {"name": "Volcano", "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30), "ground": (50, 35, 30), "ground_edge": (200, 80, 30), "decor": "volcano"},
    {"name": "Snow Peak", "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245), "ground": (225, 235, 240), "ground_edge": (255, 255, 255), "decor": "snow"},
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


def generate_new_platforms():
    """Generate new platforms HIGH UP above the current lava level.
    Platforms spawn in the upper portion of the screen, well above the lava,
    so the battle naturally migrates upward over time."""
    global PLATFORMS, PLATFORM_SPAWN_ANIMS
    
    # Remove submerged platforms
    submerged = [p for p in PLATFORMS if p.top >= current_lava_y]
    for p in submerged:
        PLATFORMS.remove(p)
    
    needed = TARGET_PLATFORM_COUNT - len(PLATFORMS)
    if needed <= 0 or len(PLATFORMS) >= MAX_PLATFORMS:
        return
    
    # === HIGH SPAWN ZONE ===
    # Platforms spawn in the UPPER third of the screen, well above the lava.
    # The zone shifts upward as lava rises, keeping new ground always high.
    # Top boundary: just below the HUD area
    spawn_top = MIN_PLATFORM_Y
    # Bottom boundary: at least 150px above the lava (high up!)
    spawn_bottom = max(spawn_top + 60, current_lava_y - 150)
    
    # If lava has risen very high, compress the zone near the top
    if spawn_bottom <= spawn_top + 40:
        spawn_bottom = spawn_top + 80
    
    new_platforms = []
    attempts = 0
    max_attempts = 80
    
    while len(new_platforms) < needed and attempts < max_attempts:
        attempts += 1
        
        width = random.randint(80, 150)
        height = random.choice([16, 18, 20])
        
        # Bias toward higher positions: use sqrt to weight toward spawn_top
        raw_t = random.random()
        biased_t = raw_t ** 0.6  # pushes values toward 0 (higher on screen)
        y = int(spawn_top + biased_t * (spawn_bottom - spawn_top))
        
        x = random.randint(60, WIDTH - 60 - width)
        
        new_plat = pygame.Rect(x, y, width, height)
        
        # Check overlap with existing and new platforms (generous gap)
        overlap = False
        for existing in PLATFORMS + new_platforms:
            expanded = existing.inflate(40, 50)
            if new_plat.colliderect(expanded):
                overlap = True
                break
        
        if not overlap:
            new_platforms.append(new_plat)
            PLATFORM_SPAWN_ANIMS.append((new_plat, 25))  # 25 frames of spawn animation
    
    PLATFORMS.extend(new_platforms)
    
    if new_platforms:
        play_platform_spawn_sound()


class Arrow:
    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return
        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class Bomb:
    def __init__(self, x, y, facing, damage, owner):
        self.x = x
        self.y = y
        self.vel_x = facing * 9
        self.vel_y = -11
        self.damage = damage
        self.owner = owner
        self.timer = 70
        self.dead = False
        self.exploded = False

    def update(self, platforms):
        if self.exploded:
            return
        self.vel_y += GRAVITY
        self.x += self.vel_x
        self.y += self.vel_y
        self.timer -= 1
        for plat in platforms:
            if plat.left <= self.x <= plat.right and self.y >= plat.top and self.vel_y >= 0:
                if self.y - self.vel_y <= plat.top + 10:
                    self.y = plat.top
                    self.vel_y = 0
                    self.vel_x *= 0.7
                    self.timer -= 3
        if self.y >= current_lava_y or self.timer <= 0:
            self.exploded = True
            self.dead = True

    def explode(self):
        play_explosion_sound()
        return Explosion(self.x, self.y, self.damage, self.owner)

    def draw(self, surf):
        pygame.draw.circle(surf, (30, 30, 30), (int(self.x), int(self.y)), 8)
        if self.timer % 8 < 4:
            pygame.draw.circle(surf, (255, 200, 50), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 3)
        pygame.draw.line(surf, (100, 100, 100), (int(self.x), int(self.y)), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 2)


class Debris:
    def __init__(self, x, y, color):
        self.x = x
        self.y = y
        self.vel_x = random.uniform(-8, 8)
        self.vel_y = random.uniform(-10, -2)
        self.size = random.randint(4, 10)
        self.color = color
        self.rotation = random.uniform(0, 360)
        self.rot_speed = random.uniform(-15, 15)
        self.life = random.randint(40, 70)

    def update(self):
        self.vel_y += GRAVITY * 0.8
        self.x += self.vel_x
        self.y += self.vel_y
        self.rotation += self.rot_speed
        self.life -= 1
        self.vel_x *= 0.98

    def draw(self, surf):
        if self.life <= 0:
            return
        alpha = min(255, self.life * 6)
        surf_deb = pygame.Surface((self.size * 2, self.size * 2), pygame.SRCALPHA)
        rect = pygame.Rect(0, 0, self.size, self.size)
        pygame.draw.rect(surf_deb, (*self.color, alpha), rect)
        pygame.draw.rect(surf_deb, (0, 0, 0, alpha), rect, 1)
        rotated = pygame.transform.rotate(surf_deb, self.rotation)
        new_rect = rotated.get_rect(center=(int(self.x), int(self.y)))
        surf.blit(rotated, new_rect)


class Explosion:
    def __init__(self, x, y, damage, owner):
        self.x = x
        self.y = y
        self.damage = damage
        self.owner = owner
        self.radius = 10
        self.max_radius = 110
        self.life = 20
        self.has_damaged = False
        self.has_destroyed_platforms = False

    def update(self, p1, p2, debris_list):
        self.life -= 1
        if self.life > 10:
            self.radius += (self.max_radius - self.radius) * 0.4
        else:
            self.radius += (self.max_radius - self.radius) * 0.1
        if not self.has_damaged and self.life == 19:
            for target in (p1, p2):
                target_cx = target.x
                target_cy = target.y - target.height / 2
                dist = math.hypot(target_cx - self.x, target_cy - self.y)
                if dist < self.max_radius:
                    from_left = (self.x < target.x)
                    target.take_hit(self.damage, from_left)
                    distance_factor = 1.0 - (dist / self.max_radius)
                    base_push = 120
                    base_pop = -22
                    push = (base_push * distance_factor) if from_left else -(base_push * distance_factor)
                    target.x = max(target.width, min(WIDTH - target.width, target.x + push))
                    target.vel_y = base_pop * distance_factor
            self.has_damaged = True
        if not self.has_destroyed_platforms and self.life == 17:
            global PLATFORMS
            platforms_to_destroy = []
            for plat in PLATFORMS:
                closest_x = max(plat.left, min(self.x, plat.right))
                closest_y = max(plat.top, min(self.y, plat.bottom))
                dist = math.hypot(closest_x - self.x, closest_y - self.y)
                if dist < self.max_radius:
                    platforms_to_destroy.append(plat)
            if len(PLATFORMS) - len(platforms_to_destroy) >= 1:
                for plat in platforms_to_destroy:
                    plat_color = (110, 90, 70)
                    for _ in range(18):
                        chunk_x = random.uniform(plat.left, plat.right)
                        chunk_y = random.uniform(plat.top, plat.bottom)
                        debris_list.append(Debris(chunk_x, chunk_y, plat_color))
                    PLATFORMS.remove(plat)
                    play_crumble_sound()
            self.has_destroyed_platforms = True

    def draw(self, surf):
        if self.life > 0:
            alpha = int(255 * (self.life / 20))
            surf_exp = pygame.Surface((self.max_radius * 2, self.max_radius * 2), pygame.SRCALPHA)
            pygame.draw.circle(surf_exp, (255, 80, 20, alpha), (self.max_radius, self.max_radius), int(self.radius))
            pygame.draw.circle(surf_exp, (255, 200, 50, int(alpha * 0.8)), (self.max_radius, self.max_radius), int(self.radius * 0.6))
            pygame.draw.circle(surf_exp, (255, 255, 220, int(alpha * 0.9)), (self.max_radius, self.max_radius), int(self.radius * 0.25))
            surf.blit(surf_exp, (int(self.x - self.max_radius), int(self.y - self.max_radius)))


class WeaponPickup:
    def __init__(self, x, y, wtype):
        self.x = x
        self.y = y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [(self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        elif self.wtype == "uzi":
            pygame.draw.rect(surf, color, (self.x - 12, self.y - 26, 24, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (self.x - 2, self.y - 15), (self.x - 6, self.y - 4), 5)
        elif self.wtype == "bomb":
            pygame.draw.circle(surf, (30, 30, 30), (self.x, self.y - 20), 10)
            pygame.draw.line(surf, (100, 100, 100), (self.x, self.y - 20), (self.x + 6, self.y - 30), 2)
            pygame.draw.circle(surf, (255, 200, 50), (self.x + 6, self.y - 30), 3)
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class ArmorPickup:
    def __init__(self, x, y, atype):
        self.x = x
        self.y = y
        self.atype = atype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = ARMOR_TYPES[self.atype]
        color = info["color"]
        glow = pygame.Surface((80, 80), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*color, 80), (40, 40), 34)
        surf.blit(glow, (self.x - 40, self.y - 60))

        chest_points = [
            (self.x - 14, self.y - 38),
            (self.x + 14, self.y - 38),
            (self.x + 16, self.y - 18),
            (self.x + 10, self.y - 8),
            (self.x - 10, self.y - 8),
            (self.x - 16, self.y - 18),
        ]
        pygame.draw.polygon(surf, color, chest_points)
        pygame.draw.polygon(surf, BLACK, chest_points, 2)
        pygame.draw.line(surf, color, (self.x - 12, self.y - 38), (self.x - 14, self.y - 44), 4)
        pygame.draw.line(surf, color, (self.x + 12, self.y - 38), (self.x + 14, self.y - 44), 4)
        pygame.draw.circle(surf, BLACK, (int(self.x), int(self.y - 24)), 3)

        label = font_tiny.render(info["name"] + " Armor", True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 64))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing
        self.controls = controls
        self.name = name
        self.health = MAX_HEALTH
        self.on_ground = True
        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0
        self.attack_type = None
        self.hit_stun = 0
        self.walk_cycle = 0
        self.moving = False
        self.wins = 0
        self.weapon = None
        self.armor = None
        self.armor_flash = 0
        self.reload_timer = 0

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height, self.width, self.height)

    def current_stats(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup_weapon(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {
                    "type": wp.wtype,
                    "durability": info["durability"],
                    "current_mag": info.get("mag_size", 0),
                }
                weapons.remove(wp)
                return

    def try_pickup_armor(self, armors):
        for ap in armors:
            if ap.rect().colliderect(self.rect()):
                info = ARMOR_TYPES[ap.atype]
                self.armor = {"type": ap.atype, "durability": info["durability"]}
                self.armor_flash = 12
                armors.remove(ap)
                play_armor_pickup_sound()
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def is_reloading(self):
        return self.reload_timer > 0

    def start_reload(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            if info.get("mag_size", 0) > 0:
                self.reload_timer = RELOAD_TIME
                play_reload_sound()

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)
        elif self.weapon["type"] == "uzi":
            spawn_y += random.randint(-3, 3)
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))
        if "current_mag" in self.weapon:
            self.weapon["current_mag"] -= 1
            if self.weapon["current_mag"] <= 0:
                self.start_reload()

    def throw_bomb(self, bombs):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 20
        spawn_y = self.y - self.height * 0.7
        bombs.append(Bomb(spawn_x, spawn_y, self.facing, info["damage"], self))
        self.weapon["durability"] -= 1
        if self.weapon["durability"] <= 0:
            self.weapon = None
            play_break_sound()

    def handle_input(self, keys, opponent, weapons, armors, arrows, bombs):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.is_reloading():
                pass
            elif self.weapon and self.weapon["type"] == "bomb":
                self.attack_type = "throw"
                self.attack_anim = 15
                self.throw_bomb(bombs)
            elif self.is_ranged():
                if "current_mag" in self.weapon and self.weapon["current_mag"] <= 0:
                    self.start_reload()
                    play_empty_click_sound()
                else:
                    self.attack_type = "shoot"
                    self.attack_anim = 10
                    self.shoot(arrows)
                    self.weapon["durability"] -= 1
                    if self.weapon["durability"] <= 0:
                        self.weapon = None
                        play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup_weapon(weapons)
        self.try_pickup_armor(armors)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y

        self.on_ground = False
        for plat in PLATFORMS:
            prev_y = self.y - self.vel_y
            if prev_y <= plat.top and self.y >= plat.top and plat.left <= self.x <= plat.right and self.vel_y >= 0:
                self.y = plat.top
                self.vel_y = 0
                self.on_ground = True
                break

        if not self.on_ground:
            if self.y >= current_lava_y:
                self.hazard_hit()

        if self.punch_cd > 0: self.punch_cd -= 1
        if self.kick_cd > 0: self.kick_cd -= 1
        if self.armor_flash > 0: self.armor_flash -= 1

        if self.reload_timer > 0:
            self.reload_timer -= 1
            if self.reload_timer == 0 and self.weapon:
                info = WEAPON_TYPES[self.weapon["type"]]
                if "current_mag" in self.weapon:
                    self.weapon["current_mag"] = info.get("mag_size", 0)

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            absorbed = min(damage, info["absorb"])
            damage = damage - absorbed
            self.armor["durability"] -= 1
            self.armor_flash = 10
            if self.armor["durability"] <= 0:
                self.armor = None
                play_break_sound()
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        if PLATFORMS:
            safe_platforms = [p for p in PLATFORMS if p.top < current_lava_y - 20]
            if safe_platforms:
                # Respawn on the HIGHEST safe platform (closest to top of screen)
                best = min(safe_platforms, key=lambda p: p.top)
                self.x = best.centerx
                self.y = best.top
            else:
                self.x = 500
                self.y = max(MIN_PLATFORM_Y + 20, current_lava_y - 80)
        else:
            self.x = 500
            self.y = max(MIN_PLATFORM_Y + 20, current_lava_y - 80)
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            aura_color = info["color"]
            flash = self.armor_flash > 0
            aura_surf = pygame.Surface((90, 130), pygame.SRCALPHA)
            alpha = 90 if not flash else 180
            pygame.draw.ellipse(aura_surf, (*aura_color, alpha), (0, 0, 90, 130))
            if flash:
                pygame.draw.ellipse(aura_surf, (255, 255, 255, 120), (10, 10, 70, 110))
            surf.blit(aura_surf, (cx - 45, self.y - self.height - 10))

        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        elif self.attack_type == "throw":
            fwd_hand = (cx + self.facing * 20, shoulder_y - 10)
            back_hand = (cx - self.facing * 10, shoulder_y + 10)
        elif self.is_reloading():
            fwd_hand = (cx + self.facing * 8, shoulder_y + 30)
            back_hand = (cx - self.facing * 6, shoulder_y + 28)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            ac = info["color"]
            chest_points = [
                (cx - 12, shoulder_y - 2),
                (cx + 12, shoulder_y - 2),
                (cx + 14, shoulder_y + 18),
                (cx + 8, shoulder_y + 26),
                (cx - 8, shoulder_y + 26),
                (cx - 14, shoulder_y + 18),
            ]
            pygame.draw.polygon(surf, ac, chest_points)
            pygame.draw.polygon(surf, BLACK, chest_points, 2)
            pygame.draw.circle(surf, BLACK, (cx, shoulder_y + 12), 2)

        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "uzi":
            wcolor = WEAPON_TYPES["uzi"]["color"]
            ux, uy = fwd_hand[0], fwd_hand[1] - 3
            barrel_tip = (ux + self.facing * 27, uy)
            grip_bottom = (ux - self.facing * 5, uy + 18)
            pygame.draw.rect(surf, wcolor, pygame.Rect(min(ux, barrel_tip[0]), uy - 7, abs(barrel_tip[0] - ux) + 8, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (ux, uy + 4), grip_bottom, 6)
            pygame.draw.line(surf, (25, 25, 28), barrel_tip, (barrel_tip[0] + self.facing * 10, barrel_tip[1]), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 8, barrel_tip[1])
                pygame.draw.circle(surf, (255, 220, 90), (int(flash[0]), int(flash[1])), 6)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [tipend, (tip[0] + perp[0], tip[1] + perp[1]), (tip[0] - perp[0], tip[1] - perp[1])])
            else:
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK, (guard_center[0] + perp[0], guard_center[1] + perp[1]), (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        label_y = int(head_y) - head_r - 20
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            mag_size = info.get("mag_size", 0)
            if mag_size > 0 and "current_mag" in self.weapon:
                cur = self.weapon["current_mag"]
                mag_color = YELLOW if cur > 0 else RED
                mag_label = font_tiny.render(f"[{cur}/{mag_size}]", True, mag_color)
                surf.blit(mag_label, (cx - mag_label.get_width() // 2, label_y))
                label_y -= 14
            label = font_tiny.render(f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}", True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, label_y))
            label_y -= 16

        if self.is_reloading():
            reload_label = font_tiny.render("RELOADING...", True, YELLOW)
            pygame.draw.rect(surf, BLACK, (cx - reload_label.get_width() // 2 - 2, label_y - 1,
                                            reload_label.get_width() + 4, reload_label.get_height() + 2))
            surf.blit(reload_label, (cx - reload_label.get_width() // 2, label_y))
            label_y -= 16

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            label = font_tiny.render(f"{info['name']} Armor x{self.armor['durability']}", True, info["color"])
            pygame.draw.rect(surf, BLACK, (cx - label.get_width() // 2 - 2, label_y - 1, label.get_width() + 4, label.get_height() + 2))
            surf.blit(label, (cx - label.get_width() // 2, label_y))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]
    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    draw_hazard_pit(surf, current_lava_y)

    for plat in PLATFORMS:
        anim_frames = 0
        for anim_plat, frames in PLATFORM_SPAWN_ANIMS:
            if anim_plat is plat:
                anim_frames = frames
                break
        
        if anim_frames > 0:
            progress = 1.0 - (anim_frames / 25.0)
            scale = 0.2 + 0.8 * progress
            alpha = int(255 * progress)
            
            glow_surf = pygame.Surface((plat.width + 24, plat.height + 24), pygame.SRCALPHA)
            glow_color = (255, 255, 150, int(alpha * 0.7))
            pygame.draw.rect(glow_surf, glow_color, (0, 0, plat.width + 24, plat.height + 24), border_radius=6)
            glow_surf_scaled = pygame.transform.scale(glow_surf, 
                (int((plat.width + 24) * scale), int((plat.height + 24) * scale)))
            surf.blit(glow_surf_scaled, 
                (int(plat.centerx - glow_surf_scaled.get_width() / 2),
                 int(plat.centery - glow_surf_scaled.get_height() / 2)))
        
        pygame.draw.rect(surf, stage["ground"], plat)
        pygame.draw.rect(surf, stage["ground_edge"], (plat.left, plat.top, plat.width, 10))
        pygame.draw.polygon(surf, stage["ground"], [(plat.left, plat.bottom), (plat.left + 12, plat.bottom + 18), (plat.left + 24, plat.bottom)])
        pygame.draw.polygon(surf, stage["ground"], [(plat.right, plat.bottom), (plat.right - 12, plat.bottom + 18), (plat.right - 24, plat.bottom)])


def draw_hazard_pit(surf, lava_y):
    pit_rect = pygame.Rect(0, lava_y, WIDTH, HEIGHT - lava_y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)
    
    for i, (band_y_offset, color) in enumerate([
        (15, (150, 40, 15)),
        (35, (200, 70, 20)),
        (55, (240, 110, 30)),
    ]):
        actual_band_y = lava_y + band_y_offset
        if actual_band_y < HEIGHT:
            for gx in range(0, WIDTH, 26):
                wobble = math.sin((gx + i * 40) * 0.15) * 4
                pygame.draw.circle(surf, color, (gx + 13, int(actual_band_y + wobble)), 9)
                
    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [
            (gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)
        ])
        pygame.draw.polygon(surf, (90, 90, 95), [
            (gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)
        ])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)
    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)
    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def draw_armor_bar(surf, x, y, fighter, align_left=True):
    if not fighter.armor:
        return
    info = ARMOR_TYPES[fighter.armor["type"]]
    max_dur = info["durability"]
    cur_dur = fighter.armor["durability"]
    bar_w, bar_h = 320, 10
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=3)
    fill_w = int((bar_w - 4) * (cur_dur / max_dur))
    if align_left:
        fill_rect = pygame.Rect(x + 2, y + 2, fill_w, bar_h - 4)
    else:
        fill_rect = pygame.Rect(x + bar_w - 2 - fill_w, y + 2, fill_w, bar_h - 4)
    pygame.draw.rect(surf, info["color"], fill_rect, border_radius=2)
    armor_label = font_tiny.render(f"{info['name']} Armor ({cur_dur})", True, info["color"])
    if align_left:
        surf.blit(armor_label, (x, y - 14))
    else:
        surf.blit(armor_label, (x + bar_w - armor_label.get_width(), y - 14))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, y, wtype))


def spawn_armor(armors):
    if len(armors) >= MAX_ARMOR_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    atype = random.choices(list(ARMOR_TYPES.keys()), weights=[5, 3, 1], k=1)[0]
    armors.append(ArmorPickup(x, y, atype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    global PLATFORMS, current_lava_y, lava_rise_timer, lava_rise_warning, PLATFORM_SPAWN_ANIMS
    PLATFORMS = _initial_platforms()
    PLATFORM_SPAWN_ANIMS = []
    current_lava_y = GROUND_Y
    lava_rise_timer = LAVA_RISE_INTERVAL
    lava_rise_warning = 0
    
    p1 = Fighter(210, RED, DARK_RED, 1, (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g), "Player 1")
    p1.y = 450
    p2 = Fighter(790, BLUE, DARK_BLUE, -1, (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l), "Player 2")
    p2.y = 450
    p1.wins = p1_wins
    p2.wins = p2_wins
    if p1_start_weapon:
        info = WEAPON_TYPES[p1_start_weapon]
        p1.weapon = {"type": p1_start_weapon, "durability": info["durability"], "current_mag": info.get("mag_size", 0)}
    if p2_start_weapon:
        info = WEAPON_TYPES[p2_start_weapon]
        p2.weapon = {"type": p2_start_weapon, "durability": info["durability"], "current_mag": info.get("mag_size", 0)}
    weapons, armors, bombs, explosions, arrows, debris = [], [], [], [], [], []
    spawn_weapon(weapons)
    spawn_armor(armors)
    return p1, p2, weapons, armors, arrows, bombs, explosions, debris


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [(cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "uzi":
        pygame.draw.rect(surf, (45, 45, 50), (cx - 22, cy - 8, 44, 16), border_radius=3)
        pygame.draw.line(surf, (25, 25, 28), (cx - 4, cy + 5), (cx - 10, cy + 25), 7)
        pygame.draw.line(surf, (25, 25, 28), (cx + 20, cy), (cx + 34, cy), 5)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)
    elif wtype == "bomb":
        pygame.draw.circle(surf, (40, 40, 40), (cx, cy), 14)
        pygame.draw.line(surf, (100, 100, 100), (cx, cy), (cx + 8, cy - 12), 3)
        pygame.draw.circle(surf, (255, 200, 50), (cx + 8, cy - 12), 4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220
    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))
        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))
    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE
    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))
    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)
    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))
    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))
    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))
    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)
    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render("Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    global current_lava_y, lava_rise_timer, lava_rise_warning, PLATFORM_SPAWN_ANIMS
    state = "select"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0
    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, armors, arrows, bombs, explosions, debris = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL
    armor_spawn_timer = ARMOR_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a: p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d: p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f: p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT: p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT: p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k: p2_ready = True
                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT): stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT): stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k): world_locked = True
                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, armors, arrows, bombs, explosions, debris = reset_fighters(
                        p1_wins, p2_wins, WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    armor_spawn_timer = ARMOR_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, armors, arrows, bombs)
            p2.handle_input(keys, p1, weapons, armors, arrows, bombs)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            for bomb in bombs[:]:
                bomb.update(PLATFORMS)
                if bomb.exploded:
                    explosions.append(bomb.explode())
                    bombs.remove(bomb)
            for exp in explosions[:]:
                exp.update(p1, p2, debris)
                if exp.life <= 0:
                    explosions.remove(exp)
            for d in debris[:]:
                d.update()
                if d.life <= 0:
                    debris.remove(d)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL
            armor_spawn_timer -= 1
            if armor_spawn_timer <= 0:
                spawn_armor(armors)
                armor_spawn_timer = ARMOR_SPAWN_INTERVAL

            for anim in PLATFORM_SPAWN_ANIMS[:]:
                plat, frames = anim
                frames -= 1
                if frames <= 0:
                    PLATFORM_SPAWN_ANIMS.remove(anim)
                else:
                    idx = PLATFORM_SPAWN_ANIMS.index(anim)
                    PLATFORM_SPAWN_ANIMS[idx] = (plat, frames)

            lava_rise_timer -= 1
            if lava_rise_timer <= 120:
                lava_rise_warning = 120
            
            if lava_rise_timer <= 0:
                current_lava_y -= LAVA_RISE_AMOUNT
                if current_lava_y < MIN_PLATFORM_Y - 30:
                    current_lava_y = MIN_PLATFORM_Y - 30
                lava_rise_timer = LAVA_RISE_INTERVAL
                play_hazard_sound()
                generate_new_platforms()

            if lava_rise_warning > 0:
                lava_rise_warning -= 1

            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        for ap in armors:
            ap.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)
        for bomb in bombs:
            bomb.draw(screen)
        for exp in explosions:
            exp.draw(screen)
        for d in debris:
            d.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)
        draw_armor_bar(screen, 30, 62, p1, align_left=True)
        draw_armor_bar(screen, WIDTH - 30 - 320, 62, p2, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        plat_count = font_tiny.render(f"Platforms: {len(PLATFORMS)}", True, WHITE)
        screen.blit(plat_count, (WIDTH // 2 - plat_count.get_width() // 2, 50))

        if lava_rise_warning > 0 and (lava_rise_warning // 5) % 2 == 0:
            warning_text = font_med.render("WARNING: LAVA RISING!", True, RED)
            screen.blit(warning_text, (WIDTH // 2 - warning_text.get_width() // 2, HEIGHT // 2 - 50))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))
            msg = "DRAW!" if winner == "Draw" else f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))
            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   Platforms spawn HIGH as lava rises!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

KeyboardInterrupt: 

In [29]:
"""
Stickman Battle
A simple 2-player stick figure fighting game built with pygame, now with weapons, 
floating platforms, bombs that DESTROY platforms, ARMOR PICKUPS, GUN RELOADING, 
and RISING LAVA with DYNAMIC PLATFORMS THAT SPAWN UPWARD!

Controls:
  Player 1 (Red):  A / D to move, W to jump, F to punch/attack, G to kick
  Player 2 (Blue): Left/Right arrows to move, Up arrow to jump,
                    K to punch/attack, L to kick

Before each match, both players pick a starting weapon (or fists) on the
select screen:
  Player 1: A / D to cycle choices, F to confirm
  Player 2: Left / Right arrows to cycle choices, K to confirm

Then either player picks the battle world (Meadow, Desert, Night City,
Volcano, or Snow Peak) using the same movement/attack keys.

Weapons (uzi, sword, bat, spear, bow, gatling, bomb) AND armor (light/medium/
heavy) spawn randomly on the platforms during the match. Walk over a pickup 
to grab it.

GUN RELOADING: The uzi, gatling, and bow now have 50-round magazines! After 
firing 50 shots, the gun auto-reloads for ~1 second.

RISING LAVA + UPWARD PLATFORMS: Every 10 seconds, the lava pit rises by 25 
pixels. Submerged platforms are DESTROYED, and NEW platforms spawn ABOVE the 
highest existing platform, forcing the battle ever upward! The arena 
constantly climbs toward the sky.

First to reduce the opponent's health to 0 wins.
Press R to restart after a match ends. Press ESC to quit.
"""

import pygame
import sys
import math
import random
import array

pygame.init()

# ---------------------------------------------------------------------------
# Sound
# ---------------------------------------------------------------------------
SAMPLE_RATE = 44100
break_sound = None
hazard_sound = None
explosion_sound = None
crumble_sound = None
armor_pickup_sound = None
reload_sound = None
empty_click_sound = None
platform_spawn_sound = None

try:
    pygame.mixer.quit()
    pygame.mixer.init(frequency=SAMPLE_RATE, size=-16, channels=1)

    def _generate_break_sound():
        duration = 0.28
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 2
            freq = 900 * (1 - progress) + 90
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.55 * noise + 0.45 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_hazard_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) ** 0.6
            freq = 220 + 60 * math.sin(2 * math.pi * 18 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.75 * noise + 0.25 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_explosion_sound():
        duration = 0.45
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 120 * (1 - progress * 0.8)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.7 * noise + 0.3 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_crumble_sound():
        duration = 0.6
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.2
            freq = 180 * (1 - progress * 0.5) + 40 * math.sin(2 * math.pi * 8 * t)
            tone = math.sin(2 * math.pi * freq * t)
            noise = random.uniform(-1, 1)
            mix = 0.85 * noise + 0.15 * tone
            value = int(amplitude * envelope * mix)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_armor_pickup_sound():
        duration = 0.22
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.5
            freq = 1200 - 400 * progress
            tone = math.sin(2 * math.pi * freq * t)
            tone2 = math.sin(2 * math.pi * (freq * 1.5) * t) * 0.5
            value = int(amplitude * envelope * (tone + tone2) * 0.6)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_reload_sound():
        duration = 0.35
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 1.3
            click1 = math.sin(2 * math.pi * 1800 * t) * (1 if progress < 0.1 else 0)
            click2 = math.sin(2 * math.pi * 2200 * t) * (1 if 0.45 < progress < 0.55 else 0)
            metallic = math.sin(2 * math.pi * 900 * t) * 0.3
            mix = click1 + click2 + metallic
            value = int(amplitude * envelope * mix * 0.5)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_empty_click_sound():
        duration = 0.08
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = (1 - progress) ** 3
            freq = 3500 * (1 - progress * 0.5)
            tone = math.sin(2 * math.pi * freq * t)
            value = int(amplitude * envelope * tone * 0.4)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    def _generate_platform_spawn_sound():
        duration = 0.4
        n_samples = int(SAMPLE_RATE * duration)
        amplitude = 32767
        samples = array.array('h')
        for i in range(n_samples):
            t = i / SAMPLE_RATE
            progress = t / duration
            envelope = math.sin(math.pi * progress) * (1 - progress * 0.5)
            freq = 400 + 800 * progress
            tone = math.sin(2 * math.pi * freq * t)
            tone2 = math.sin(2 * math.pi * (freq * 2) * t) * 0.3
            value = int(amplitude * envelope * (tone + tone2) * 0.5)
            value = max(-32768, min(32767, value))
            samples.append(value)
        return pygame.mixer.Sound(buffer=samples.tobytes())

    break_sound = _generate_break_sound()
    hazard_sound = _generate_hazard_sound()
    explosion_sound = _generate_explosion_sound()
    crumble_sound = _generate_crumble_sound()
    armor_pickup_sound = _generate_armor_pickup_sound()
    reload_sound = _generate_reload_sound()
    empty_click_sound = _generate_empty_click_sound()
    platform_spawn_sound = _generate_platform_spawn_sound()
except pygame.error:
    break_sound = None
    hazard_sound = None
    explosion_sound = None
    crumble_sound = None
    armor_pickup_sound = None
    reload_sound = None
    empty_click_sound = None
    platform_spawn_sound = None


def play_break_sound():
    if break_sound: break_sound.play()
def play_hazard_sound():
    if hazard_sound: hazard_sound.play()
def play_explosion_sound():
    if explosion_sound: explosion_sound.play()
def play_crumble_sound():
    if crumble_sound: crumble_sound.play()
def play_armor_pickup_sound():
    if armor_pickup_sound: armor_pickup_sound.play()
def play_reload_sound():
    if reload_sound: reload_sound.play()
def play_empty_click_sound():
    if empty_click_sound: empty_click_sound.play()
def play_platform_spawn_sound():
    if platform_spawn_sound: platform_spawn_sound.play()

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
WIDTH, HEIGHT = 1000, 600
GROUND_Y = HEIGHT - 100
FPS = 60

WHITE = (245, 245, 245)
BLACK = (20, 20, 20)
RED = (220, 60, 60)
DARK_RED = (150, 30, 30)
BLUE = (60, 100, 220)
DARK_BLUE = (30, 60, 150)
GREEN = (60, 200, 90)
YELLOW = (240, 200, 40)
GRAY = (120, 120, 120)
BROWN = (120, 80, 40)
SILVER = (200, 205, 210)
GOLD = (230, 190, 60)

GRAVITY = 0.8
JUMP_STRENGTH = -15
MOVE_SPEED = 5
MAX_HEALTH = 100

LAVA_RISE_INTERVAL = 600
LAVA_RISE_AMOUNT = 25
current_lava_y = GROUND_Y
lava_rise_timer = LAVA_RISE_INTERVAL
lava_rise_warning = 0

# Platform generation
MIN_PLATFORM_Y = 80   # Absolute ceiling for platform spawns (room for HUD)
TARGET_PLATFORM_COUNT = 9
MAX_PLATFORMS = 14

HAZARD_DAMAGE = 40
HAZARD_STUN = 24

RELOAD_TIME = 60

def _initial_platforms():
    return [
        pygame.Rect(150, 450, 120, 18),
        pygame.Rect(730, 450, 120, 18),
        pygame.Rect(280, 400, 140, 20),
        pygame.Rect(580, 400, 140, 20),
        pygame.Rect(430, 360, 140, 20),
        pygame.Rect(200, 320, 130, 18),
        pygame.Rect(670, 320, 130, 18),
        pygame.Rect(440, 280, 120, 18),
        pygame.Rect(320, 220, 110, 16),
        pygame.Rect(570, 220, 110, 16),
        pygame.Rect(450, 180, 100, 16),
    ]

PLATFORMS = _initial_platforms()
PLATFORM_SPAWN_ANIMS = []

PUNCH_RANGE = 70
PUNCH_DAMAGE = 8
PUNCH_COOLDOWN = 25

KICK_RANGE = 90
KICK_DAMAGE = 14
KICK_COOLDOWN = 45

HIT_STUN = 12

WEAPON_TYPES = {
    "uzi":     {"reach": 0,   "damage": 5, "cooldown": 5, "durability": 50, "color": (45, 45, 50),
                "ranged": True, "proj_speed": 28, "proj_kind": "bullet", "mag_size": 50},
    "sword":   {"reach": 110, "damage": 18, "cooldown": 30, "durability": 5,  "color": SILVER, "ranged": False},
    "bat":     {"reach": 85,  "damage": 22, "cooldown": 40, "durability": 8,  "color": BROWN, "ranged": False},
    "spear":   {"reach": 135, "damage": 14, "cooldown": 35, "durability": 6,  "color": (90, 60, 30), "ranged": False},
    "bow":     {"reach": 0,   "damage": 12, "cooldown": 45, "durability": 6,  "color": (101, 67, 33),
                "ranged": True, "proj_speed": 15, "proj_kind": "arrow", "mag_size": 50},
    "gatling": {"reach": 0,   "damage": 4,  "cooldown": 6,  "durability": 40, "color": (70, 70, 80),
                "ranged": True, "proj_speed": 24, "proj_kind": "bullet", "mag_size": 50},
    "bomb":    {"reach": 0,   "damage": 35, "cooldown": 50, "durability": 3,  "color": (40, 40, 40),
                "ranged": True, "proj_speed": 9, "proj_kind": "bomb"},
}

ARMOR_TYPES = {
    "light":  {"absorb": 15, "durability": 3, "color": (120, 220, 120), "name": "Light"},
    "medium": {"absorb": 30, "durability": 4, "color": (120, 170, 240), "name": "Medium"},
    "heavy":  {"absorb": 50, "durability": 5, "color": (240, 200, 80),  "name": "Heavy"},
}

ARROW_SPEED = 15
ARROW_LENGTH = 34

MAX_WEAPONS_ON_FIELD = 2
MAX_ARMOR_ON_FIELD = 1
WEAPON_SPAWN_INTERVAL = 300
ARMOR_SPAWN_INTERVAL = 480

WEAPON_CHOICES = [None, "uzi", "sword", "bat", "spear", "bow", "gatling", "bomb"]

STAGES = [
    {"name": "Meadow", "sky_top": (135, 190, 230), "sky_bottom": (220, 235, 245), "ground": (110, 90, 70), "ground_edge": (80, 150, 80), "decor": "meadow"},
    {"name": "Desert", "sky_top": (250, 200, 120), "sky_bottom": (255, 235, 190), "ground": (200, 165, 100), "ground_edge": (225, 195, 130), "decor": "desert"},
    {"name": "Night City", "sky_top": (20, 20, 45), "sky_bottom": (60, 50, 90), "ground": (40, 40, 50), "ground_edge": (90, 90, 110), "decor": "city"},
    {"name": "Volcano", "sky_top": (60, 20, 15), "sky_bottom": (150, 60, 30), "ground": (50, 35, 30), "ground_edge": (200, 80, 30), "decor": "volcano"},
    {"name": "Snow Peak", "sky_top": (180, 210, 235), "sky_bottom": (235, 240, 245), "ground": (225, 235, 240), "ground_edge": (255, 255, 255), "decor": "snow"},
]

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Stickman Battle")
clock = pygame.time.Clock()
font_big = pygame.font.SysFont("arial", 64, bold=True)
font_med = pygame.font.SysFont("arial", 32, bold=True)
font_small = pygame.font.SysFont("arial", 22)
font_tiny = pygame.font.SysFont("arial", 16, bold=True)


def generate_new_platforms():
    """Generate new platforms ABOVE the highest existing platform.
    As lava rises and swallows low platforms, fresh ones appear higher up,
    pushing the entire battle upward toward the sky."""
    global PLATFORMS, PLATFORM_SPAWN_ANIMS

    # Remove platforms now submerged in lava
    submerged = [p for p in PLATFORMS if p.top >= current_lava_y]
    for p in submerged:
        PLATFORMS.remove(p)

    # How many do we need?
    needed = TARGET_PLATFORM_COUNT - len(PLATFORMS)
    if needed <= 0 or len(PLATFORMS) >= MAX_PLATFORMS:
        return

    # Find the highest existing platform (lowest Y = highest on screen)
    if PLATFORMS:
        highest_top = min(p.top for p in PLATFORMS)
    else:
        highest_top = current_lava_y - 100

    # --- Primary spawn zone: ABOVE the highest platform ---
    # New platforms appear higher than anything currently in the arena
    spawn_zone_bottom = highest_top - 40   # at least 40px above highest
    spawn_zone_top = max(MIN_PLATFORM_Y, spawn_zone_bottom - 180)

    # If the highest platform is already near the ceiling, we have no room above.
    # Fallback: spawn in gaps between lava and existing platforms
    if spawn_zone_bottom <= spawn_zone_top + 20:
        # Try to find gaps between existing platforms
        # Sort platforms by Y (top to bottom)
        sorted_plats = sorted(PLATFORMS, key=lambda p: p.top)
        spawn_zone_top = max(MIN_PLATFORM_Y, current_lava_y - 350)
        spawn_zone_bottom = current_lava_y - 50

    new_platforms = []
    attempts = 0
    max_attempts = 80

    while len(new_platforms) < needed and attempts < max_attempts:
        attempts += 1

        width = random.randint(80, 150)
        height = random.choice([16, 18, 20])
        x = random.randint(60, WIDTH - 60 - width)
        y = random.randint(spawn_zone_top, max(spawn_zone_top, spawn_zone_bottom))

        new_plat = pygame.Rect(x, y, width, height)

        # Check overlap with all existing + newly placed platforms
        overlap = False
        for existing in PLATFORMS + new_platforms:
            expanded = existing.inflate(30, 50)
            if new_plat.colliderect(expanded):
                overlap = True
                break

        if not overlap:
            new_platforms.append(new_plat)
            PLATFORM_SPAWN_ANIMS.append((new_plat, 25))  # 25 frames of spawn anim

    PLATFORMS.extend(new_platforms)

    if new_platforms:
        play_platform_spawn_sound()


class Arrow:
    def __init__(self, x, y, facing, damage, owner, speed=None, kind="arrow"):
        self.x = x
        self.y = y
        self.facing = facing
        self.damage = damage
        self.owner = owner
        self.kind = kind
        self.speed = speed if speed is not None else ARROW_SPEED
        self.length = ARROW_LENGTH if kind == "arrow" else 16
        self.dead = False

    def update(self):
        self.x += self.speed * self.facing
        if self.x < -50 or self.x > WIDTH + 50:
            self.dead = True

    def rect(self):
        front = self.x + self.facing * self.length
        left = min(self.x, front)
        h = 8 if self.kind == "arrow" else 6
        return pygame.Rect(left, self.y - h / 2, self.length, h)

    def draw(self, surf):
        if self.kind == "bullet":
            tail = (self.x - self.facing * 10, self.y)
            tip = (self.x + self.facing * self.length, self.y)
            pygame.draw.line(surf, (255, 210, 90), tail, tip, 3)
            pygame.draw.circle(surf, (255, 235, 160), (int(tip[0]), int(tip[1])), 3)
            return
        tail = (self.x, self.y)
        tip = (self.x + self.facing * self.length, self.y)
        pygame.draw.line(surf, (80, 50, 20), tail, tip, 3)
        back = (tip[0] - self.facing * 8, self.y - 5)
        back2 = (tip[0] - self.facing * 8, self.y + 5)
        pygame.draw.polygon(surf, (60, 60, 60), [tip, back, back2])
        fletch_end = (tail[0] - self.facing * 6, self.y)
        pygame.draw.line(surf, RED, tail, fletch_end, 4)


class Bomb:
    def __init__(self, x, y, facing, damage, owner):
        self.x = x
        self.y = y
        self.vel_x = facing * 9
        self.vel_y = -11
        self.damage = damage
        self.owner = owner
        self.timer = 70
        self.dead = False
        self.exploded = False

    def update(self, platforms):
        if self.exploded:
            return
        self.vel_y += GRAVITY
        self.x += self.vel_x
        self.y += self.vel_y
        self.timer -= 1
        for plat in platforms:
            if plat.left <= self.x <= plat.right and self.y >= plat.top and self.vel_y >= 0:
                if self.y - self.vel_y <= plat.top + 10:
                    self.y = plat.top
                    self.vel_y = 0
                    self.vel_x *= 0.7
                    self.timer -= 3
        if self.y >= current_lava_y or self.timer <= 0:
            self.exploded = True
            self.dead = True

    def explode(self):
        play_explosion_sound()
        return Explosion(self.x, self.y, self.damage, self.owner)

    def draw(self, surf):
        pygame.draw.circle(surf, (30, 30, 30), (int(self.x), int(self.y)), 8)
        if self.timer % 8 < 4:
            pygame.draw.circle(surf, (255, 200, 50), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 3)
        pygame.draw.line(surf, (100, 100, 100), (int(self.x), int(self.y)), (int(self.x + self.vel_x * 0.4), int(self.y - 9)), 2)


class Debris:
    def __init__(self, x, y, color):
        self.x = x
        self.y = y
        self.vel_x = random.uniform(-8, 8)
        self.vel_y = random.uniform(-10, -2)
        self.size = random.randint(4, 10)
        self.color = color
        self.rotation = random.uniform(0, 360)
        self.rot_speed = random.uniform(-15, 15)
        self.life = random.randint(40, 70)

    def update(self):
        self.vel_y += GRAVITY * 0.8
        self.x += self.vel_x
        self.y += self.vel_y
        self.rotation += self.rot_speed
        self.life -= 1
        self.vel_x *= 0.98

    def draw(self, surf):
        if self.life <= 0:
            return
        alpha = min(255, self.life * 6)
        surf_deb = pygame.Surface((self.size * 2, self.size * 2), pygame.SRCALPHA)
        rect = pygame.Rect(0, 0, self.size, self.size)
        pygame.draw.rect(surf_deb, (*self.color, alpha), rect)
        pygame.draw.rect(surf_deb, (0, 0, 0, alpha), rect, 1)
        rotated = pygame.transform.rotate(surf_deb, self.rotation)
        new_rect = rotated.get_rect(center=(int(self.x), int(self.y)))
        surf.blit(rotated, new_rect)


class Explosion:
    def __init__(self, x, y, damage, owner):
        self.x = x
        self.y = y
        self.damage = damage
        self.owner = owner
        self.radius = 10
        self.max_radius = 110
        self.life = 20
        self.has_damaged = False
        self.has_destroyed_platforms = False

    def update(self, p1, p2, debris_list):
        self.life -= 1
        if self.life > 10:
            self.radius += (self.max_radius - self.radius) * 0.4
        else:
            self.radius += (self.max_radius - self.radius) * 0.1
        if not self.has_damaged and self.life == 19:
            for target in (p1, p2):
                target_cx = target.x
                target_cy = target.y - target.height / 2
                dist = math.hypot(target_cx - self.x, target_cy - self.y)
                if dist < self.max_radius:
                    from_left = (self.x < target.x)
                    target.take_hit(self.damage, from_left)
                    distance_factor = 1.0 - (dist / self.max_radius)
                    base_push = 120
                    base_pop = -22
                    push = (base_push * distance_factor) if from_left else -(base_push * distance_factor)
                    target.x = max(target.width, min(WIDTH - target.width, target.x + push))
                    target.vel_y = base_pop * distance_factor
            self.has_damaged = True
        if not self.has_destroyed_platforms and self.life == 17:
            global PLATFORMS
            platforms_to_destroy = []
            for plat in PLATFORMS:
                closest_x = max(plat.left, min(self.x, plat.right))
                closest_y = max(plat.top, min(self.y, plat.bottom))
                dist = math.hypot(closest_x - self.x, closest_y - self.y)
                if dist < self.max_radius:
                    platforms_to_destroy.append(plat)
            if len(PLATFORMS) - len(platforms_to_destroy) >= 1:
                for plat in platforms_to_destroy:
                    plat_color = (110, 90, 70)
                    for _ in range(18):
                        chunk_x = random.uniform(plat.left, plat.right)
                        chunk_y = random.uniform(plat.top, plat.bottom)
                        debris_list.append(Debris(chunk_x, chunk_y, plat_color))
                    PLATFORMS.remove(plat)
                    play_crumble_sound()
            self.has_destroyed_platforms = True

    def draw(self, surf):
        if self.life > 0:
            alpha = int(255 * (self.life / 20))
            surf_exp = pygame.Surface((self.max_radius * 2, self.max_radius * 2), pygame.SRCALPHA)
            pygame.draw.circle(surf_exp, (255, 80, 20, alpha), (self.max_radius, self.max_radius), int(self.radius))
            pygame.draw.circle(surf_exp, (255, 200, 50, int(alpha * 0.8)), (self.max_radius, self.max_radius), int(self.radius * 0.6))
            pygame.draw.circle(surf_exp, (255, 255, 220, int(alpha * 0.9)), (self.max_radius, self.max_radius), int(self.radius * 0.25))
            surf.blit(surf_exp, (int(self.x - self.max_radius), int(self.y - self.max_radius)))


class WeaponPickup:
    def __init__(self, x, y, wtype):
        self.x = x
        self.y = y
        self.wtype = wtype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = WEAPON_TYPES[self.wtype]
        color = info["color"]
        glow = pygame.Surface((70, 70), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*YELLOW, 60), (35, 35), 30)
        surf.blit(glow, (self.x - 35, self.y - 55))

        if self.wtype == "sword":
            pygame.draw.line(surf, color, (self.x, self.y - 5), (self.x, self.y - 42), 5)
            pygame.draw.line(surf, BLACK, (self.x - 10, self.y - 12), (self.x + 10, self.y - 12), 4)
        elif self.wtype == "bat":
            pygame.draw.line(surf, color, (self.x, self.y - 2), (self.x + 6, self.y - 40), 8)
        elif self.wtype == "spear":
            pygame.draw.line(surf, (90, 60, 30), (self.x, self.y), (self.x, self.y - 40), 4)
            pygame.draw.polygon(surf, SILVER, [(self.x, self.y - 40), (self.x - 6, self.y - 28), (self.x + 6, self.y - 28)])
        elif self.wtype == "bow":
            rect = pygame.Rect(self.x - 16, self.y - 44, 32, 44)
            pygame.draw.arc(surf, color, rect, -1.4, 1.4, 4)
            pygame.draw.line(surf, (220, 220, 200), (self.x + 12, self.y - 42), (self.x + 12, self.y - 2), 1)
        elif self.wtype == "gatling":
            pygame.draw.circle(surf, (40, 40, 45), (int(self.x), self.y - 22), 12)
            pygame.draw.rect(surf, color, (self.x - 14, self.y - 26, 32, 8))
        elif self.wtype == "uzi":
            pygame.draw.rect(surf, color, (self.x - 12, self.y - 26, 24, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (self.x - 2, self.y - 15), (self.x - 6, self.y - 4), 5)
        elif self.wtype == "bomb":
            pygame.draw.circle(surf, (30, 30, 30), (self.x, self.y - 20), 10)
            pygame.draw.line(surf, (100, 100, 100), (self.x, self.y - 20), (self.x + 6, self.y - 30), 2)
            pygame.draw.circle(surf, (255, 200, 50), (self.x + 6, self.y - 30), 3)
        label = font_tiny.render(self.wtype.capitalize(), True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 60))


class ArmorPickup:
    def __init__(self, x, y, atype):
        self.x = x
        self.y = y
        self.atype = atype

    def rect(self):
        return pygame.Rect(self.x - 20, self.y - 40, 40, 40)

    def draw(self, surf):
        info = ARMOR_TYPES[self.atype]
        color = info["color"]
        glow = pygame.Surface((80, 80), pygame.SRCALPHA)
        pygame.draw.circle(glow, (*color, 80), (40, 40), 34)
        surf.blit(glow, (self.x - 40, self.y - 60))

        chest_points = [
            (self.x - 14, self.y - 38),
            (self.x + 14, self.y - 38),
            (self.x + 16, self.y - 18),
            (self.x + 10, self.y - 8),
            (self.x - 10, self.y - 8),
            (self.x - 16, self.y - 18),
        ]
        pygame.draw.polygon(surf, color, chest_points)
        pygame.draw.polygon(surf, BLACK, chest_points, 2)
        pygame.draw.line(surf, color, (self.x - 12, self.y - 38), (self.x - 14, self.y - 44), 4)
        pygame.draw.line(surf, color, (self.x + 12, self.y - 38), (self.x + 14, self.y - 44), 4)
        pygame.draw.circle(surf, BLACK, (int(self.x), int(self.y - 24)), 3)

        label = font_tiny.render(info["name"] + " Armor", True, BLACK)
        surf.blit(label, (self.x - label.get_width() // 2, self.y - 64))


class Fighter:
    def __init__(self, x, color, dark_color, facing, controls, name):
        self.x = x
        self.y = GROUND_Y
        self.vel_y = 0
        self.width = 40
        self.height = 110
        self.color = color
        self.dark_color = dark_color
        self.facing = facing
        self.controls = controls
        self.name = name
        self.health = MAX_HEALTH
        self.on_ground = True
        self.punch_cd = 0
        self.kick_cd = 0
        self.attack_anim = 0
        self.attack_type = None
        self.hit_stun = 0
        self.walk_cycle = 0
        self.moving = False
        self.wins = 0
        self.weapon = None
        self.armor = None
        self.armor_flash = 0
        self.reload_timer = 0

    def rect(self):
        return pygame.Rect(self.x - self.width // 2, self.y - self.height, self.width, self.height)

    def current_stats(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            return info["reach"], info["damage"], info["cooldown"]
        return PUNCH_RANGE, PUNCH_DAMAGE, PUNCH_COOLDOWN

    def attack_hitbox(self):
        if self.attack_type == "weapon" and self.weapon:
            reach, _, _ = self.current_stats()
            y_center = self.y - self.height * 0.6
            h = 24
        elif self.attack_type == "punch":
            reach = PUNCH_RANGE
            y_center = self.y - self.height * 0.65
            h = 20
        elif self.attack_type == "kick":
            reach = KICK_RANGE
            y_center = self.y - self.height * 0.35
            h = 26
        else:
            return None
        if self.facing == 1:
            x = self.x
        else:
            x = self.x - reach
        return pygame.Rect(x, y_center - h // 2, reach, h)

    def try_pickup_weapon(self, weapons):
        if self.weapon is not None:
            return
        for wp in weapons:
            if wp.rect().colliderect(self.rect()):
                info = WEAPON_TYPES[wp.wtype]
                self.weapon = {
                    "type": wp.wtype,
                    "durability": info["durability"],
                    "current_mag": info.get("mag_size", 0),
                }
                weapons.remove(wp)
                return

    def try_pickup_armor(self, armors):
        for ap in armors:
            if ap.rect().colliderect(self.rect()):
                info = ARMOR_TYPES[ap.atype]
                self.armor = {"type": ap.atype, "durability": info["durability"]}
                self.armor_flash = 12
                armors.remove(ap)
                play_armor_pickup_sound()
                return

    def is_ranged(self):
        return bool(self.weapon) and WEAPON_TYPES[self.weapon["type"]].get("ranged")

    def is_reloading(self):
        return self.reload_timer > 0

    def start_reload(self):
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            if info.get("mag_size", 0) > 0:
                self.reload_timer = RELOAD_TIME
                play_reload_sound()

    def shoot(self, arrows):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 26
        spawn_y = self.y - self.height * 0.6
        if self.weapon["type"] == "gatling":
            spawn_y += random.randint(-6, 6)
        elif self.weapon["type"] == "uzi":
            spawn_y += random.randint(-3, 3)
        arrows.append(Arrow(spawn_x, spawn_y, self.facing, info["damage"], self,
                             speed=info.get("proj_speed"), kind=info.get("proj_kind", "arrow")))
        if "current_mag" in self.weapon:
            self.weapon["current_mag"] -= 1
            if self.weapon["current_mag"] <= 0:
                self.start_reload()

    def throw_bomb(self, bombs):
        info = WEAPON_TYPES[self.weapon["type"]]
        spawn_x = self.x + self.facing * 20
        spawn_y = self.y - self.height * 0.7
        bombs.append(Bomb(spawn_x, spawn_y, self.facing, info["damage"], self))
        self.weapon["durability"] -= 1
        if self.weapon["durability"] <= 0:
            self.weapon = None
            play_break_sound()

    def handle_input(self, keys, opponent, weapons, armors, arrows, bombs):
        if self.hit_stun > 0:
            self.hit_stun -= 1
            self.moving = False
            return

        left, right, jump, punch, kick = self.controls
        self.moving = False

        if keys[left]:
            self.x -= MOVE_SPEED
            self.facing = -1
            self.moving = True
        if keys[right]:
            self.x += MOVE_SPEED
            self.facing = 1
            self.moving = True

        self.x = max(self.width, min(WIDTH - self.width, self.x))

        if keys[jump] and self.on_ground:
            self.vel_y = JUMP_STRENGTH
            self.on_ground = False

        if self.punch_cd == 0 and keys[punch]:
            if self.is_reloading():
                pass
            elif self.weapon and self.weapon["type"] == "bomb":
                self.attack_type = "throw"
                self.attack_anim = 15
                self.throw_bomb(bombs)
            elif self.is_ranged():
                if "current_mag" in self.weapon and self.weapon["current_mag"] <= 0:
                    self.start_reload()
                    play_empty_click_sound()
                else:
                    self.attack_type = "shoot"
                    self.attack_anim = 10
                    self.shoot(arrows)
                    self.weapon["durability"] -= 1
                    if self.weapon["durability"] <= 0:
                        self.weapon = None
                        play_break_sound()
            elif self.weapon:
                self.attack_type = "weapon"
                self.attack_anim = 12
            else:
                self.attack_type = "punch"
                self.attack_anim = 10
            _, _, cd = self.current_stats()
            self.punch_cd = cd

        if self.kick_cd == 0 and keys[kick]:
            self.attack_type = "kick"
            self.attack_anim = 14
            self.kick_cd = KICK_COOLDOWN

        self.try_pickup_weapon(weapons)
        self.try_pickup_armor(armors)

    def physics(self):
        self.vel_y += GRAVITY
        self.y += self.vel_y

        self.on_ground = False
        for plat in PLATFORMS:
            prev_y = self.y - self.vel_y
            if prev_y <= plat.top and self.y >= plat.top and plat.left <= self.x <= plat.right and self.vel_y >= 0:
                self.y = plat.top
                self.vel_y = 0
                self.on_ground = True
                break

        if not self.on_ground:
            if self.y >= current_lava_y:
                self.hazard_hit()

        if self.punch_cd > 0: self.punch_cd -= 1
        if self.kick_cd > 0: self.kick_cd -= 1
        if self.armor_flash > 0: self.armor_flash -= 1

        if self.reload_timer > 0:
            self.reload_timer -= 1
            if self.reload_timer == 0 and self.weapon:
                info = WEAPON_TYPES[self.weapon["type"]]
                if "current_mag" in self.weapon:
                    self.weapon["current_mag"] = info.get("mag_size", 0)

        if self.attack_anim > 0:
            self.attack_anim -= 1
            if self.attack_anim == 0:
                self.attack_type = None

        if self.moving and self.on_ground:
            self.walk_cycle += 0.25
        else:
            self.walk_cycle = 0

    def register_weapon_hit(self):
        if self.weapon:
            self.weapon["durability"] -= 1
            if self.weapon["durability"] <= 0:
                self.weapon = None
                play_break_sound()

    def take_hit(self, damage, from_left):
        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            absorbed = min(damage, info["absorb"])
            damage = damage - absorbed
            self.armor["durability"] -= 1
            self.armor_flash = 10
            if self.armor["durability"] <= 0:
                self.armor = None
                play_break_sound()
        self.health = max(0, self.health - damage)
        self.hit_stun = HIT_STUN
        push = 8 if from_left else -8
        self.x = max(self.width, min(WIDTH - self.width, self.x + push))

    def hazard_hit(self):
        self.health = max(0, self.health - HAZARD_DAMAGE)
        self.hit_stun = HAZARD_STUN
        if PLATFORMS:
            safe_platforms = [p for p in PLATFORMS if p.top < current_lava_y - 20]
            if safe_platforms:
                # Prefer the highest safe platform to push the player upward
                best = min(safe_platforms, key=lambda p: p.top)
                self.x = best.centerx
                self.y = best.top
            else:
                self.x = 500
                self.y = max(MIN_PLATFORM_Y + 20, current_lava_y - 50)
        else:
            self.x = 500
            self.y = max(MIN_PLATFORM_Y + 20, current_lava_y - 50)
        self.vel_y = JUMP_STRENGTH * 0.5
        self.on_ground = False
        play_hazard_sound()

    def draw(self, surf):
        color = self.dark_color if self.hit_stun > 0 else self.color
        cx = int(self.x)
        head_r = 16
        hip_y = self.y - 55
        shoulder_y = self.y - 85
        head_y = shoulder_y - head_r - 4

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            aura_color = info["color"]
            flash = self.armor_flash > 0
            aura_surf = pygame.Surface((90, 130), pygame.SRCALPHA)
            alpha = 90 if not flash else 180
            pygame.draw.ellipse(aura_surf, (*aura_color, alpha), (0, 0, 90, 130))
            if flash:
                pygame.draw.ellipse(aura_surf, (255, 255, 255, 120), (10, 10, 70, 110))
            surf.blit(aura_surf, (cx - 45, self.y - self.height - 10))

        swing = math.sin(self.walk_cycle) * 18 if self.moving else 0
        leg_spread = 14 if self.on_ground else 8
        if self.attack_type == "kick":
            kick_dir = self.facing
            front_leg = (cx + kick_dir * KICK_RANGE * 0.8, self.y - 40)
        else:
            front_leg = (cx + leg_spread + swing, self.y)
        back_leg = (cx - leg_spread - swing, self.y)

        pygame.draw.line(surf, color, (cx, hip_y), front_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), back_leg, 6)
        pygame.draw.line(surf, color, (cx, hip_y), (cx, shoulder_y), 6)

        if self.attack_type == "weapon" and self.weapon:
            atk_dir = self.facing
            reach, _, _ = self.current_stats()
            fwd_hand = (cx + atk_dir * reach * 0.6, shoulder_y + 10)
            back_hand = (cx - 10, shoulder_y + 20)
        elif self.attack_type == "punch":
            punch_dir = self.facing
            fwd_hand = (cx + punch_dir * PUNCH_RANGE * 0.9, shoulder_y + 10)
            back_hand = (cx - 12, shoulder_y + 20)
        elif self.attack_type == "shoot":
            fwd_hand = (cx + self.facing * 26, shoulder_y + 4)
            back_hand = (cx - self.facing * 14, shoulder_y + 10)
        elif self.attack_type == "throw":
            fwd_hand = (cx + self.facing * 20, shoulder_y - 10)
            back_hand = (cx - self.facing * 10, shoulder_y + 10)
        elif self.is_reloading():
            fwd_hand = (cx + self.facing * 8, shoulder_y + 30)
            back_hand = (cx - self.facing * 6, shoulder_y + 28)
        else:
            arm_swing = math.sin(self.walk_cycle + math.pi) * 14 if self.moving else 0
            fwd_hand = (cx + 14 + arm_swing, shoulder_y + 22)
            back_hand = (cx - 14 - arm_swing, shoulder_y + 22)

        pygame.draw.line(surf, color, (cx, shoulder_y), fwd_hand, 5)
        pygame.draw.line(surf, color, (cx, shoulder_y), back_hand, 5)

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            ac = info["color"]
            chest_points = [
                (cx - 12, shoulder_y - 2),
                (cx + 12, shoulder_y - 2),
                (cx + 14, shoulder_y + 18),
                (cx + 8, shoulder_y + 26),
                (cx - 8, shoulder_y + 26),
                (cx - 14, shoulder_y + 18),
            ]
            pygame.draw.polygon(surf, ac, chest_points)
            pygame.draw.polygon(surf, BLACK, chest_points, 2)
            pygame.draw.circle(surf, BLACK, (cx, shoulder_y + 12), 2)

        if self.weapon and self.weapon["type"] == "bow":
            wcolor = WEAPON_TYPES["bow"]["color"]
            bow_x = fwd_hand[0]
            bow_y = fwd_hand[1] - 20
            bow_rect = pygame.Rect(bow_x - 22, bow_y - 22, 44, 44)
            if self.facing == 1:
                pygame.draw.arc(surf, wcolor, bow_rect, -1.4, 1.4, 4)
                string_x = bow_x + 16
            else:
                pygame.draw.arc(surf, wcolor, bow_rect, math.pi - 1.4, math.pi + 1.4, 4)
                string_x = bow_x - 16
            pull = 10 if self.attack_type == "shoot" else 0
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y - 20), (string_x - self.facing * pull, bow_y), 1)
            pygame.draw.line(surf, (220, 220, 200), (string_x, bow_y + 20), (string_x - self.facing * pull, bow_y), 1)
        elif self.weapon and self.weapon["type"] == "uzi":
            wcolor = WEAPON_TYPES["uzi"]["color"]
            ux, uy = fwd_hand[0], fwd_hand[1] - 3
            barrel_tip = (ux + self.facing * 27, uy)
            grip_bottom = (ux - self.facing * 5, uy + 18)
            pygame.draw.rect(surf, wcolor, pygame.Rect(min(ux, barrel_tip[0]), uy - 7, abs(barrel_tip[0] - ux) + 8, 14), border_radius=3)
            pygame.draw.line(surf, (25, 25, 28), (ux, uy + 4), grip_bottom, 6)
            pygame.draw.line(surf, (25, 25, 28), barrel_tip, (barrel_tip[0] + self.facing * 10, barrel_tip[1]), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 8, barrel_tip[1])
                pygame.draw.circle(surf, (255, 220, 90), (int(flash[0]), int(flash[1])), 6)
        elif self.weapon and self.weapon["type"] == "gatling":
            wcolor = WEAPON_TYPES["gatling"]["color"]
            gx, gy = fwd_hand[0], fwd_hand[1] - 4
            barrel_len = 34
            barrel_tip = (gx + self.facing * barrel_len, gy)
            pygame.draw.circle(surf, (35, 35, 40), (int(gx), int(gy)), 11)
            pygame.draw.line(surf, wcolor, (gx, gy), barrel_tip, 10)
            pygame.draw.circle(surf, (20, 20, 25), (int(barrel_tip[0]), int(barrel_tip[1])), 5)
            if self.attack_type == "shoot":
                flash = (barrel_tip[0] + self.facing * 10, barrel_tip[1])
                pygame.draw.circle(surf, (255, 230, 140), (int(flash[0]), int(flash[1])), 8)
                pygame.draw.circle(surf, (255, 255, 200), (int(flash[0]), int(flash[1])), 4)
        elif self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            wcolor = info["color"]
            reach = info["reach"]
            if self.attack_type == "weapon":
                tip = (cx + self.facing * reach, shoulder_y + 10)
                grip = fwd_hand
            else:
                tip = (cx + self.facing * reach * 0.5, shoulder_y - 20)
                grip = fwd_hand
            if self.weapon["type"] == "bat":
                pygame.draw.line(surf, wcolor, grip, tip, 8)
            elif self.weapon["type"] == "spear":
                pygame.draw.line(surf, (90, 60, 30), grip, tip, 4)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                tipend = (tip[0] + nx * 14, tip[1] + ny * 14)
                perp = (-ny * 5, nx * 5)
                pygame.draw.polygon(surf, SILVER, [tipend, (tip[0] + perp[0], tip[1] + perp[1]), (tip[0] - perp[0], tip[1] - perp[1])])
            else:
                pygame.draw.line(surf, wcolor, grip, tip, 5)
                dx, dy = tip[0] - grip[0], tip[1] - grip[1]
                dist = max(1, math.hypot(dx, dy))
                nx, ny = dx / dist, dy / dist
                perp = (-ny * 8, nx * 8)
                guard_center = (grip[0] + nx * 8, grip[1] + ny * 8)
                pygame.draw.line(surf, BLACK, (guard_center[0] + perp[0], guard_center[1] + perp[1]), (guard_center[0] - perp[0], guard_center[1] - perp[1]), 4)

        pygame.draw.circle(surf, color, (cx, int(head_y)), head_r)
        pygame.draw.circle(surf, BLACK, (cx, int(head_y)), head_r, 2)
        eye_x = cx + self.facing * 6
        pygame.draw.circle(surf, BLACK, (int(eye_x), int(head_y) - 2), 2)

        label_y = int(head_y) - head_r - 20
        if self.weapon:
            info = WEAPON_TYPES[self.weapon["type"]]
            mag_size = info.get("mag_size", 0)
            if mag_size > 0 and "current_mag" in self.weapon:
                cur = self.weapon["current_mag"]
                mag_color = YELLOW if cur > 0 else RED
                mag_label = font_tiny.render(f"[{cur}/{mag_size}]", True, mag_color)
                surf.blit(mag_label, (cx - mag_label.get_width() // 2, label_y))
                label_y -= 14
            label = font_tiny.render(f"{self.weapon['type'].capitalize()} x{self.weapon['durability']}", True, BLACK)
            surf.blit(label, (cx - label.get_width() // 2, label_y))
            label_y -= 16

        if self.is_reloading():
            reload_label = font_tiny.render("RELOADING...", True, YELLOW)
            pygame.draw.rect(surf, BLACK, (cx - reload_label.get_width() // 2 - 2, label_y - 1,
                                            reload_label.get_width() + 4, reload_label.get_height() + 2))
            surf.blit(reload_label, (cx - reload_label.get_width() // 2, label_y))
            label_y -= 16

        if self.armor:
            info = ARMOR_TYPES[self.armor["type"]]
            label = font_tiny.render(f"{info['name']} Armor x{self.armor['durability']}", True, info["color"])
            pygame.draw.rect(surf, BLACK, (cx - label.get_width() // 2 - 2, label_y - 1, label.get_width() + 4, label.get_height() + 2))
            surf.blit(label, (cx - label.get_width() // 2, label_y))


def draw_background(surf, stage=None):
    if stage is None:
        stage = STAGES[0]
    sky_top = stage["sky_top"]
    sky_bottom = stage["sky_bottom"]
    for i in range(HEIGHT):
        t = i / HEIGHT
        r = sky_top[0] + (sky_bottom[0] - sky_top[0]) * t
        g = sky_top[1] + (sky_bottom[1] - sky_top[1]) * t
        b = sky_top[2] + (sky_bottom[2] - sky_top[2]) * t
        pygame.draw.line(surf, (r, g, b), (0, i), (WIDTH, i))

    decor = stage["decor"]
    if decor == "meadow":
        pygame.draw.circle(surf, (255, 245, 200), (860, 90), 45)
        for cx in (120, 300, 700, 900):
            pygame.draw.ellipse(surf, (255, 255, 255), (cx, 60, 90, 30))
            pygame.draw.ellipse(surf, (255, 255, 255), (cx + 30, 45, 70, 30))
    elif decor == "desert":
        pygame.draw.circle(surf, (255, 250, 210), (150, 100), 55)
        for cx, w in ((250, 40), (650, 55), (820, 35)):
            base = GROUND_Y
            pygame.draw.rect(surf, (90, 130, 70), (cx, base - 70, 14, 70), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx - w // 2, base - 45, 12, 30), border_radius=6)
            pygame.draw.rect(surf, (90, 130, 70), (cx + w // 2, base - 55, 12, 40), border_radius=6)
    elif decor == "city":
        random.seed(7)
        for i in range(10):
            bw = random.randint(40, 90)
            bh = random.randint(80, 240)
            bx = int(i * (WIDTH / 10) + random.randint(-10, 10))
            by = GROUND_Y - bh
            pygame.draw.rect(surf, (30, 30, 45), (bx, by, bw, bh))
            for wy in range(by + 10, GROUND_Y - 10, 20):
                for wx in range(bx + 6, bx + bw - 6, 16):
                    if random.random() > 0.4:
                        pygame.draw.rect(surf, (240, 220, 120), (wx, wy, 8, 10))
        random.seed()
        pygame.draw.circle(surf, (230, 230, 210), (500, 80), 30)
    elif decor == "volcano":
        pygame.draw.polygon(surf, (70, 40, 35), [(700, GROUND_Y), (820, 120), (940, GROUND_Y)])
        pygame.draw.polygon(surf, (230, 90, 30), [(790, 200), (820, 130), (850, 200)])
        for gx in range(0, WIDTH, 60):
            pygame.draw.circle(surf, (255, 140, 40), (gx + 20, GROUND_Y - 4), 3)
    elif decor == "snow":
        random.seed(3)
        for _ in range(60):
            sx = random.randint(0, WIDTH)
            sy = random.randint(0, GROUND_Y - 20)
            pygame.draw.circle(surf, WHITE, (sx, sy), 2)
        random.seed()
        pygame.draw.polygon(surf, (210, 220, 230), [(50, GROUND_Y), (150, 90), (250, GROUND_Y)])
        pygame.draw.polygon(surf, (210, 220, 230), [(750, GROUND_Y), (860, 60), (970, GROUND_Y)])

    draw_hazard_pit(surf, current_lava_y)

    for plat in PLATFORMS:
        anim_frames = 0
        for anim_plat, frames in PLATFORM_SPAWN_ANIMS:
            if anim_plat is plat:
                anim_frames = frames
                break

        if anim_frames > 0:
            progress = 1.0 - (anim_frames / 25.0)
            scale = 0.2 + 0.8 * progress
            alpha = int(255 * progress)

            # Rising sparkle effect during spawn
            glow_surf = pygame.Surface((plat.width + 24, plat.height + 24), pygame.SRCALPHA)
            glow_color = (255, 255, 150, int(alpha * 0.7))
            pygame.draw.rect(glow_surf, glow_color, (0, 0, plat.width + 24, plat.height + 24), border_radius=6)
            # Upward arrow indicator
            arrow_cx = (plat.width + 24) // 2
            arrow_cy = (plat.height + 24) // 2
            pygame.draw.polygon(glow_surf, (255, 255, 200, int(alpha * 0.9)), [
                (arrow_cx, arrow_cy - 8), (arrow_cx - 6, arrow_cy + 2), (arrow_cx + 6, arrow_cy + 2)
            ])
            scaled = pygame.transform.scale(glow_surf,
                (int((plat.width + 24) * scale), int((plat.height + 24) * scale)))
            surf.blit(scaled,
                (int(plat.centerx - scaled.get_width() / 2),
                 int(plat.centery - scaled.get_height() / 2)))

        pygame.draw.rect(surf, stage["ground"], plat)
        pygame.draw.rect(surf, stage["ground_edge"], (plat.left, plat.top, plat.width, 10))
        pygame.draw.polygon(surf, stage["ground"], [(plat.left, plat.bottom), (plat.left + 12, plat.bottom + 18), (plat.left + 24, plat.bottom)])
        pygame.draw.polygon(surf, stage["ground"], [(plat.right, plat.bottom), (plat.right - 12, plat.bottom + 18), (plat.right - 24, plat.bottom)])


def draw_hazard_pit(surf, lava_y):
    pit_rect = pygame.Rect(0, lava_y, WIDTH, HEIGHT - lava_y)
    pygame.draw.rect(surf, (40, 10, 8), pit_rect)

    for i, (band_y_offset, color) in enumerate([
        (15, (150, 40, 15)),
        (35, (200, 70, 20)),
        (55, (240, 110, 30)),
    ]):
        actual_band_y = lava_y + band_y_offset
        if actual_band_y < HEIGHT:
            for gx in range(0, WIDTH, 26):
                wobble = math.sin((gx + i * 40) * 0.15) * 4
                pygame.draw.circle(surf, color, (gx + 13, int(actual_band_y + wobble)), 9)

    spike_w = 26
    for gx in range(0, WIDTH, spike_w):
        pygame.draw.polygon(surf, (55, 55, 60), [
            (gx, HEIGHT), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w, HEIGHT)
        ])
        pygame.draw.polygon(surf, (90, 90, 95), [
            (gx + spike_w // 2 - 3, HEIGHT - 30), (gx + spike_w // 2, HEIGHT - 34), (gx + spike_w // 2 + 3, HEIGHT - 30)
        ])


def draw_health_bar(surf, x, y, health, name, align_left=True):
    bar_w, bar_h = 320, 26
    ratio = health / MAX_HEALTH
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=6)
    pygame.draw.rect(surf, (60, 60, 60), bg_rect.inflate(-4, -4), border_radius=5)
    fill_w = int((bar_w - 8) * ratio)
    fill_color = GREEN if ratio > 0.5 else (YELLOW if ratio > 0.25 else RED)
    if align_left:
        fill_rect = pygame.Rect(x + 4, y + 4, fill_w, bar_h - 8)
    else:
        fill_rect = pygame.Rect(x + bar_w - 4 - fill_w, y + 4, fill_w, bar_h - 8)
    pygame.draw.rect(surf, fill_color, fill_rect, border_radius=4)
    label = font_small.render(name, True, WHITE)
    if align_left:
        surf.blit(label, (x, y - 24))
    else:
        surf.blit(label, (x + bar_w - label.get_width(), y - 24))


def draw_armor_bar(surf, x, y, fighter, align_left=True):
    if not fighter.armor:
        return
    info = ARMOR_TYPES[fighter.armor["type"]]
    max_dur = info["durability"]
    cur_dur = fighter.armor["durability"]
    bar_w, bar_h = 320, 10
    bg_rect = pygame.Rect(x, y, bar_w, bar_h)
    pygame.draw.rect(surf, BLACK, bg_rect, border_radius=3)
    fill_w = int((bar_w - 4) * (cur_dur / max_dur))
    if align_left:
        fill_rect = pygame.Rect(x + 2, y + 2, fill_w, bar_h - 4)
    else:
        fill_rect = pygame.Rect(x + bar_w - 2 - fill_w, y + 2, fill_w, bar_h - 4)
    pygame.draw.rect(surf, info["color"], fill_rect, border_radius=2)
    armor_label = font_tiny.render(f"{info['name']} Armor ({cur_dur})", True, info["color"])
    if align_left:
        surf.blit(armor_label, (x, y - 14))
    else:
        surf.blit(armor_label, (x + bar_w - armor_label.get_width(), y - 14))


def check_hits(p1, p2):
    for attacker, defender in ((p1, p2), (p2, p1)):
        if attacker.attack_type not in ("punch", "kick", "weapon"):
            continue
        is_weapon = attacker.attack_type == "weapon"
        trigger_frame = 11 if is_weapon else (9 if attacker.attack_type == "punch" else 12)
        if attacker.attack_anim == trigger_frame:
            box = attacker.attack_hitbox()
            if box and box.colliderect(defender.rect()) and defender.hit_stun == 0:
                _, dmg, _ = attacker.current_stats()
                defender.take_hit(dmg, from_left=(attacker.x < defender.x))
                if is_weapon:
                    attacker.register_weapon_hit()


def update_arrows(arrows, p1, p2):
    for arrow in arrows[:]:
        arrow.update()
        target = p2 if arrow.owner is p1 else p1
        if not arrow.dead and target.hit_stun == 0 and arrow.rect().colliderect(target.rect()):
            target.take_hit(arrow.damage, from_left=(arrow.facing == 1))
            arrow.dead = True
        if arrow.dead:
            arrows.remove(arrow)


def spawn_weapon(weapons):
    if len(weapons) >= MAX_WEAPONS_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    wtype = random.choice(list(WEAPON_TYPES.keys()))
    weapons.append(WeaponPickup(x, y, wtype))


def spawn_armor(armors):
    if len(armors) >= MAX_ARMOR_ON_FIELD:
        return
    if not PLATFORMS:
        return
    plat = random.choice(PLATFORMS)
    margin = 25
    x = random.randint(plat.left + margin, plat.right - margin)
    y = plat.top
    atype = random.choices(list(ARMOR_TYPES.keys()), weights=[5, 3, 1], k=1)[0]
    armors.append(ArmorPickup(x, y, atype))


def reset_fighters(p1_wins, p2_wins, p1_start_weapon=None, p2_start_weapon=None):
    global PLATFORMS, current_lava_y, lava_rise_timer, lava_rise_warning, PLATFORM_SPAWN_ANIMS
    PLATFORMS = _initial_platforms()
    PLATFORM_SPAWN_ANIMS = []
    current_lava_y = GROUND_Y
    lava_rise_timer = LAVA_RISE_INTERVAL
    lava_rise_warning = 0

    p1 = Fighter(210, RED, DARK_RED, 1, (pygame.K_a, pygame.K_d, pygame.K_w, pygame.K_f, pygame.K_g), "Player 1")
    p1.y = 450
    p2 = Fighter(790, BLUE, DARK_BLUE, -1, (pygame.K_LEFT, pygame.K_RIGHT, pygame.K_UP, pygame.K_k, pygame.K_l), "Player 2")
    p2.y = 450
    p1.wins = p1_wins
    p2.wins = p2_wins
    if p1_start_weapon:
        info = WEAPON_TYPES[p1_start_weapon]
        p1.weapon = {"type": p1_start_weapon, "durability": info["durability"], "current_mag": info.get("mag_size", 0)}
    if p2_start_weapon:
        info = WEAPON_TYPES[p2_start_weapon]
        p2.weapon = {"type": p2_start_weapon, "durability": info["durability"], "current_mag": info.get("mag_size", 0)}
    weapons, armors, bombs, explosions, arrows, debris = [], [], [], [], [], []
    spawn_weapon(weapons)
    spawn_armor(armors)
    return p1, p2, weapons, armors, arrows, bombs, explosions, debris


def draw_weapon_icon(surf, cx, cy, wtype, scale=1.0):
    if wtype is None:
        pygame.draw.circle(surf, WHITE, (cx - 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx - 12, cy), int(10 * scale), 2)
        pygame.draw.circle(surf, WHITE, (cx + 12, cy), int(10 * scale))
        pygame.draw.circle(surf, BLACK, (cx + 12, cy), int(10 * scale), 2)
        return
    info = WEAPON_TYPES[wtype]
    color = info["color"]
    if wtype == "sword":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx, cy - 35), 6)
        pygame.draw.line(surf, BLACK, (cx - 14, cy + 10), (cx + 14, cy + 10), 5)
    elif wtype == "bat":
        pygame.draw.line(surf, color, (cx, cy + 35), (cx + 8, cy - 35), 10)
    elif wtype == "spear":
        pygame.draw.line(surf, (90, 60, 30), (cx, cy + 35), (cx, cy - 30), 5)
        pygame.draw.polygon(surf, SILVER, [(cx, cy - 42), (cx - 8, cy - 26), (cx + 8, cy - 26)])
    elif wtype == "bow":
        rect = pygame.Rect(cx - 30, cy - 35, 60, 70)
        pygame.draw.arc(surf, color, rect, -1.3, 1.3, 5)
        pygame.draw.line(surf, (220, 220, 200), (cx + 25, cy - 32), (cx + 25, cy + 32), 1)
    elif wtype == "uzi":
        pygame.draw.rect(surf, (45, 45, 50), (cx - 22, cy - 8, 44, 16), border_radius=3)
        pygame.draw.line(surf, (25, 25, 28), (cx - 4, cy + 5), (cx - 10, cy + 25), 7)
        pygame.draw.line(surf, (25, 25, 28), (cx + 20, cy), (cx + 34, cy), 5)
    elif wtype == "gatling":
        pygame.draw.circle(surf, (40, 40, 45), (cx - 20, cy), 16)
        for a in range(6):
            ang = a * math.pi / 3
            bx = cx - 20 + math.cos(ang) * 12
            by = cy + math.sin(ang) * 12
            pygame.draw.circle(surf, (90, 90, 100), (int(bx), int(by)), 3)
        pygame.draw.rect(surf, color, (cx - 20, cy - 6, 46, 12))
        pygame.draw.rect(surf, (30, 30, 35), (cx - 32, cy - 14, 20, 28), border_radius=4)
    elif wtype == "bomb":
        pygame.draw.circle(surf, (40, 40, 40), (cx, cy), 14)
        pygame.draw.line(surf, (100, 100, 100), (cx, cy), (cx + 8, cy - 12), 3)
        pygame.draw.circle(surf, (255, 200, 50), (cx + 8, cy - 12), 4)


def draw_select_screen(surf, p1_idx, p2_idx, p1_ready, p2_ready):
    draw_background(surf)
    title = font_big.render("CHOOSE YOUR WEAPON", True, BLACK)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    n = len(WEAPON_CHOICES)
    col_w = min(180, (WIDTH - 60) // n)
    box_size = min(140, col_w - 20)
    start_x = WIDTH // 2 - (n * col_w) // 2 + col_w // 2
    icon_y = 220
    for i, choice in enumerate(WEAPON_CHOICES):
        cx = start_x + i * col_w
        box = pygame.Rect(cx - box_size // 2, icon_y - box_size // 2, box_size, box_size)
        pygame.draw.rect(surf, (255, 255, 255), box, border_radius=10)
        pygame.draw.rect(surf, GRAY, box, 3, border_radius=10)
        draw_weapon_icon(surf, cx, icon_y, choice, scale=box_size / 140)
        name = "Fists" if choice is None else choice.capitalize()
        label = font_small.render(name, True, BLACK)
        surf.blit(label, (cx - label.get_width() // 2, icon_y + box_size // 2 + 10))
        if i == p1_idx:
            pygame.draw.rect(surf, RED, box.inflate(16, 16), 5, border_radius=12)
            tag = font_tiny.render("P1", True, RED)
            surf.blit(tag, (box.left, box.top - 22))
        if i == p2_idx:
            pygame.draw.rect(surf, BLUE, box.inflate(28, 28), 5, border_radius=14)
            tag = font_tiny.render("P2", True, BLUE)
            surf.blit(tag, (box.right - 24, box.top - 22))
    p1_status = "READY!" if p1_ready else "A/D choose, F to lock in"
    p2_status = "READY!" if p2_ready else "\u2190/\u2192 choose, K to lock in"
    p1_color = GREEN if p1_ready else RED
    p2_color = GREEN if p2_ready else BLUE
    t1 = font_med.render(f"Player 1: {p1_status}", True, p1_color)
    surf.blit(t1, (WIDTH // 2 - t1.get_width() // 2, 400))
    t2 = font_med.render(f"Player 2: {p2_status}", True, p2_color)
    surf.blit(t2, (WIDTH // 2 - t2.get_width() // 2, 445))
    if p1_ready and p2_ready:
        go = font_med.render("Get ready...", True, YELLOW)
        surf.blit(go, (WIDTH // 2 - go.get_width() // 2, 500))


def draw_world_select_screen(surf, stage_idx, locked):
    stage = STAGES[stage_idx]
    draw_background(surf, stage)
    overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 90))
    surf.blit(overlay, (0, 0))
    title = font_big.render("CHOOSE YOUR WORLD", True, WHITE)
    surf.blit(title, (WIDTH // 2 - title.get_width() // 2, 50))
    name = font_med.render(stage["name"], True, YELLOW)
    surf.blit(name, (WIDTH // 2 - name.get_width() // 2, 160))
    arrow_l = font_big.render("<", True, WHITE)
    arrow_r = font_big.render(">", True, WHITE)
    surf.blit(arrow_l, (WIDTH // 2 - 220, 145))
    surf.blit(arrow_r, (WIDTH // 2 + 200, 145))
    dots_y = 230
    total_w = len(STAGES) * 24
    dot_start = WIDTH // 2 - total_w // 2
    for i in range(len(STAGES)):
        color = YELLOW if i == stage_idx else GRAY
        pygame.draw.circle(surf, color, (dot_start + i * 24, dots_y), 6)
    if locked:
        status = font_med.render("READY! Get ready...", True, GREEN)
    else:
        status = font_med.render("Either player: A/D or Arrows to browse, F or K to lock in", True, WHITE)
    surf.blit(status, (WIDTH // 2 - status.get_width() // 2, 480))


def main():
    global current_lava_y, lava_rise_timer, lava_rise_warning, PLATFORM_SPAWN_ANIMS
    state = "select"
    p1_wins = 0
    p2_wins = 0
    p1_idx, p2_idx = 0, 0
    p1_ready, p2_ready = False, False
    select_confirm_timer = 0
    stage_idx = 0
    world_locked = False
    world_confirm_timer = 0

    p1, p2, weapons, armors, arrows, bombs, explosions, debris = reset_fighters(p1_wins, p2_wins)
    game_over = False
    winner = None
    spawn_timer = WEAPON_SPAWN_INTERVAL
    armor_spawn_timer = ARMOR_SPAWN_INTERVAL

    running = True
    while running:
        clock.tick(FPS)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                if state == "select":
                    if not p1_ready:
                        if event.key == pygame.K_a: p1_idx = (p1_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_d: p1_idx = (p1_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_f: p1_ready = True
                    if not p2_ready:
                        if event.key == pygame.K_LEFT: p2_idx = (p2_idx - 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_RIGHT: p2_idx = (p2_idx + 1) % len(WEAPON_CHOICES)
                        elif event.key == pygame.K_k: p2_ready = True
                elif state == "world_select" and not world_locked:
                    if event.key in (pygame.K_a, pygame.K_LEFT): stage_idx = (stage_idx - 1) % len(STAGES)
                    elif event.key in (pygame.K_d, pygame.K_RIGHT): stage_idx = (stage_idx + 1) % len(STAGES)
                    elif event.key in (pygame.K_f, pygame.K_k): world_locked = True
                if event.key == pygame.K_r and state == "over":
                    state = "select"
                    p1_idx, p2_idx = 0, 0
                    p1_ready, p2_ready = False, False

        if state == "select":
            draw_select_screen(screen, p1_idx, p2_idx, p1_ready, p2_ready)
            pygame.display.flip()
            if p1_ready and p2_ready:
                select_confirm_timer += 1
                if select_confirm_timer > 45:
                    select_confirm_timer = 0
                    world_locked = False
                    world_confirm_timer = 0
                    state = "world_select"
            continue

        if state == "world_select":
            draw_world_select_screen(screen, stage_idx, world_locked)
            pygame.display.flip()
            if world_locked:
                world_confirm_timer += 1
                if world_confirm_timer > 45:
                    p1, p2, weapons, armors, arrows, bombs, explosions, debris = reset_fighters(
                        p1_wins, p2_wins, WEAPON_CHOICES[p1_idx], WEAPON_CHOICES[p2_idx])
                    game_over = False
                    winner = None
                    spawn_timer = WEAPON_SPAWN_INTERVAL
                    armor_spawn_timer = ARMOR_SPAWN_INTERVAL
                    world_confirm_timer = 0
                    state = "playing"
            continue

        keys = pygame.key.get_pressed()
        stage = STAGES[stage_idx]

        if not game_over:
            p1.handle_input(keys, p2, weapons, armors, arrows, bombs)
            p2.handle_input(keys, p1, weapons, armors, arrows, bombs)
            p1.physics()
            p2.physics()
            check_hits(p1, p2)
            update_arrows(arrows, p1, p2)

            for bomb in bombs[:]:
                bomb.update(PLATFORMS)
                if bomb.exploded:
                    explosions.append(bomb.explode())
                    bombs.remove(bomb)
            for exp in explosions[:]:
                exp.update(p1, p2, debris)
                if exp.life <= 0:
                    explosions.remove(exp)
            for d in debris[:]:
                d.update()
                if d.life <= 0:
                    debris.remove(d)

            spawn_timer -= 1
            if spawn_timer <= 0:
                spawn_weapon(weapons)
                spawn_timer = WEAPON_SPAWN_INTERVAL
            armor_spawn_timer -= 1
            if armor_spawn_timer <= 0:
                spawn_armor(armors)
                armor_spawn_timer = ARMOR_SPAWN_INTERVAL

            # Update platform spawn animations
            for anim in PLATFORM_SPAWN_ANIMS[:]:
                plat, frames = anim
                frames -= 1
                if frames <= 0:
                    PLATFORM_SPAWN_ANIMS.remove(anim)
                else:
                    idx = PLATFORM_SPAWN_ANIMS.index(anim)
                    PLATFORM_SPAWN_ANIMS[idx] = (plat, frames)

            # Lava rising logic
            lava_rise_timer -= 1
            if lava_rise_timer <= 120:
                lava_rise_warning = 120

            if lava_rise_timer <= 0:
                current_lava_y -= LAVA_RISE_AMOUNT
                if current_lava_y < MIN_PLATFORM_Y - 50:
                    current_lava_y = MIN_PLATFORM_Y - 50
                lava_rise_timer = LAVA_RISE_INTERVAL
                play_hazard_sound()
                # Generate new platforms ABOVE existing ones!
                generate_new_platforms()

            if lava_rise_warning > 0:
                lava_rise_warning -= 1

            if p1.rect().colliderect(p2.rect()):
                if p1.x < p2.x:
                    p1.x -= 2
                    p2.x += 2
                else:
                    p1.x += 2
                    p2.x -= 2

            if p1.health == 0 or p2.health == 0:
                game_over = True
                state = "over"
                if p1.health == 0 and p2.health == 0:
                    winner = "Draw"
                elif p1.health == 0:
                    winner = p2.name
                    p2.wins += 1
                else:
                    winner = p1.name
                    p1.wins += 1
                p1_wins, p2_wins = p1.wins, p2.wins

        # ---- draw ----
        draw_background(screen, stage)
        for wp in weapons:
            wp.draw(screen)
        for ap in armors:
            ap.draw(screen)
        p1.draw(screen)
        p2.draw(screen)
        for arrow in arrows:
            arrow.draw(screen)
        for bomb in bombs:
            bomb.draw(screen)
        for exp in explosions:
            exp.draw(screen)
        for d in debris:
            d.draw(screen)

        draw_health_bar(screen, 30, 30, p1.health, p1.name, align_left=True)
        draw_health_bar(screen, WIDTH - 30 - 320, 30, p2.health, p2.name, align_left=False)
        draw_armor_bar(screen, 30, 62, p1, align_left=True)
        draw_armor_bar(screen, WIDTH - 30 - 320, 62, p2, align_left=False)

        score_text = font_med.render(f"{p1.wins}  -  {p2.wins}", True, WHITE)
        screen.blit(score_text, (WIDTH // 2 - score_text.get_width() // 2, 20))

        plat_count = font_tiny.render(f"Platforms: {len(PLATFORMS)}", True, WHITE)
        screen.blit(plat_count, (WIDTH // 2 - plat_count.get_width() // 2, 50))

        if lava_rise_warning > 0 and (lava_rise_warning // 5) % 2 == 0:
            warning_text = font_med.render("WARNING: LAVA RISING!", True, RED)
            screen.blit(warning_text, (WIDTH // 2 - warning_text.get_width() // 2, HEIGHT // 2 - 50))

        if game_over:
            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 140))
            screen.blit(overlay, (0, 0))
            msg = "DRAW!" if winner == "Draw" else f"{winner} WINS!"
            text = font_big.render(msg, True, YELLOW)
            screen.blit(text, (WIDTH // 2 - text.get_width() // 2, HEIGHT // 2 - 80))
            sub = font_med.render("Press R to Rematch  |  ESC to Quit", True, WHITE)
            screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, HEIGHT // 2))

        if not game_over:
            hint = font_small.render(
                "P1: A/D move, W jump, F attack, G kick   |   P2: Arrows move/jump, K attack, L kick   |   New platforms spawn UPWARD!",
                True, (40, 40, 40))
            screen.blit(hint, (WIDTH // 2 - hint.get_width() // 2, HEIGHT - 30))

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()

SystemExit: 